# 1) Install dependencies

In [0]:
%pip install seaborn -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 2) Importing packages

In [0]:
# 2) Importing the needed packages
from pyspark.sql.functions import (
    col, count as _count, countDistinct, min as spark_min, max as spark_max,
    when, isnan, isnull, desc, lit, length, trim, lower, upper, current_date,
    sum as _sum, year, month, date_format, explode, split, regexp_extract,
    mean, stddev, abs as spark_abs, skewness, kurtosis, size, array,
    percentile_approx, struct, coalesce
)
from pyspark.sql.types import StringType, DoubleType, LongType, StructType
import re
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 3) Reading raw data

In [0]:
# 3.1) Reading raw data from Unity Catalog
TABLE_NAME = "yelp_poc_uc.bronze_final.yelp_review"

In [0]:

# 3.2) Verify Delta format
table_info = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").collect()[0]
assert table_info["format"].lower() == "delta", f"Expected Delta but found '{table_info['format']}'"
print(f"✅ Confirmed: {TABLE_NAME} is a Delta table")


✅ Confirmed: yelp_poc_uc.bronze_final.yelp_review is a Delta table


In [0]:
# 3.3) Assign the table to a variable
df = spark.table(TABLE_NAME)

print(f"✅ Data loaded successfully from: {TABLE_NAME}")

✅ Data loaded successfully from: yelp_poc_uc.bronze_final.yelp_review


# 4) General Exploration

In [0]:
# 4.1) Head - first 5 rows
print("\n📋 First 5 rows:")
display(df.limit(5))


📋 First 5 rows:


review_id,user_id,business_id,stars,useful,funny,cool,text,date
eYK3GFCCPM_2LTrNBUCsmA,0yrgXkKQUl6aSNAWs25dsQ,Iyopb5uFIBT_HOgp8ipSfw,5.0,3,0,1,"So I'm pretty sure these guys were voted Best Vietnamese Restaurant in Philadelphia for a reason. Considering the many times I have frequented this place, I have never been disappointed. Allow me to elaborate: Excellent decor and environment (certainly not a typical Asian restaurant) Great knowledgeable staff HUGE menu to select from $5 lunch specials I mean seriously, the list could go on. Appetizers: Hn Xúc Bánh Đa -- minced clam meat, beef served in a cute rice cracker break off a piece of the rice cracker and slather some of that minced goodness on Entrees: Ba Vi make your own mini Vietnamese burritos - vermicelli riced noodles, shrimp balls, pork and beef with grapeleaves Caramelized Spare Ribs with quail eggs clay pot goodness with quaint quail eggs and spare rib tips in a sweet brown sauce (bowl of rice, bowl of awesome in-house chicken broth soup and you're good to go) Drinks: Iced Coffee like no other Thai Tea If you haven't been to this affordable Vietnamese destination restaurant, you need to go now.",2013-12-22 20:22:47
ijNVEplkFfEIb8QTFC0ooA,NWmPl8tgT1Q7DWCtUopQHg,lau2jbBCG3XohoAvCRIm1Q,5.0,1,0,1,Quick service and tasty pie! Groupon deal and the whole family ate deliciousness for $26! The garlic knots look and smell fantastic and we can't wait to try it again. The kids loved to chose their own toppings and the atmosphere is good for kiddos too. You can chose a gluten free crust too.,2017-06-17 01:45:38
p5pX5tld7N0DPXaox60W9Q,ZWFevb8rSFP6KCUqeq3FFg,Fz4aGOGRmPo0ki6iAc42Sw,5.0,0,0,0,The business has moved from it's location on Robb Drive to the Scolari's shopping center on Sharlands right off the Robb Drive exit. Love the convenient new location and the new salon is beautiful! All of the stylists are wonderful and I wouldn't go anywhere else!,2011-12-31 21:15:00
PCIUeFCOwJCFJaUCEsNoPQ,7Eq71a1cwHZ-T-k9pusdlw,j-qtdD55OLfSqfsWuQTDJg,4.0,0,0,1,"We sat outside on Rittenhouse Square and did some people watching. We shared the tasty chicken club with bacon and avocado, the lamb sandwich, and fries. We split a bottle of house wine. We didn't spend a ridiculous amount of money. A great lunch experience. Good job, Stephen Starr!",2011-10-23 03:43:48
JpHkJf4_9zMppk6MbJwP2g,BTKCSvPiy7anahF2HhNNNA,IvAa7PvfOYl0xRML5W3ebg,2.0,1,1,0,"Caveat: not a review of the beer, only a review of the brewery itself. So we went with a bunch of out of town friends to try and do the brewery tour, but they weren't offering it that day. OK, that was our fault for not checking before we came. We decided to stay and drink and maybe get something to eat because I was intrigued by the grilled cheese menus, reasonably priced at $3. I was a little wary because the room was huge, dark, hot, and kind of dingy looking, but everyone else was won over by the shuffleboard table. We sat down and a few people went up to the bar to order flights, which (I don't know) seemed priced decently. That's all they drank, and seemed to enjoy most of them, though there were a few varieties that went untouched by anyone until it was evident SOMEONE was going to have to drink it. My main issue was that there was only one guy working. There weren't a lot of people there but he was clearly doing EVERYTHING, and he was clearly overwhelmed. It took about 15 minutes to even catch his attention when I was trying to order food. So frustrating. And when I finally DID order, they were out of EVERYTHING on the menu. Literally EVERYTHING. I asked him what he did have, and he said, ""A few pieces of pumpernickel bread, half a tomato, and some American cheese."" ... okay. I ordered one of whatever he had and we split it. It was all right, but I can make a better grilled cheese at home with ingredients I've actually chosen. We just went to Cafe Estelle afterward for food and that was much, much better.",2010-08-01 13:15:52


In [0]:
# 4.2) Tail - last 6 rows (by business_id desc)
print("\n📋 Last 6 rows:")
display(df.orderBy(col("business_id").desc()).limit(6))



📋 Last 6 rows:


review_id,user_id,business_id,stars,useful,funny,cool,text,date
a_eGYU9cw6O0iw2ob_jXjQ,wg62PofDNK10ZIIV7Q49cg,zzyx5x0Z7xXWWvWnZFuxlQ,2.0,1,0,0,"The pizza is okay. It's not bad. But I wouldn't seek it out unless I were nearby. The service is awful and convoluted. 1. They don't sell slices. Each pizza is ready made and either a half or whole pie. I'm not sure exactly the size difference, but basically a half is about what you'd get in about 2 slices at a normal pizza joint. The made-to-order aspect is nice, but slows things down. While already made slice pies are not as fresh, they are faster to prepare and sell. This is something 99% of pizza shops do and in a food court the current setup makes no sense. 2. The orders are processed as they are rung up. This wouldn't be a big deal if their point-of-sale device worked consistently. I watched today as it crashed 5 times before finally accepting a debit card. If you had one person take the order and then another process payment the whole ordeal would go faster. 3. Have people do one job and do it well. There are about 4-5 working and they all seem to be floaters. One second they are ringing up, the next making pizza. It makes no sense since everything bottlenecks at the register and the oven. Because there's only one of each. Without a dedicated person manning each station, a backlog builds up quickly. 4. The area to order is away from the ""entrance."" The aforementioned point-of-sale device is away from the main entrances (e.g. the stairway next to the car rental and the escalator in the middle of the food court) and to order you have to walk past the entrance to the pizza shop and past a greeter to the register. I hope that the owner(s) consider making some changes to the business. I doubt they will though as the restaurant has been open a month and not much has changed other than they are now selling things like hot dogs (?).",2014-04-30 20:50:32
uOspBIJnqHqgrBTEsdOXvA,mNNDxJQR_LRxblW7m61JYQ,zzyx5x0Z7xXWWvWnZFuxlQ,2.0,1,0,0,"Ok, so I came back. I felt bad because I saw them on GrubHub which links to Yelp and it showed as poorly rated because of my review - their only review. I felt like it was unfair that their listing on GrubHub looked like that. Well, my second visit did not go well. Since it took so bloody long to get my pizza last time, I ordered ahead on Grub Hub at 12:15pm. I arrived at 12:35 and they hadn't even STARTED MAKING IT YET! I stood their and waited forever (like last time) for service. When I was acknowledged, the girl said ""Oh yea!"" and grabbed my order that had printed and handed it to the cook. Awesome. Now, I also ordered something slightly different. I ordered their three cheese pizza on Foccacia (they also offer gluten free.) This was my way of trying to write a well-rounded review AND possibly avoid burnt crust again. Did they remember to make my pizza on Foccacia? Of course not. The pizza wasn't that bad in the end except that they burnt the crust AGAIN. But, the overall experience was bad. I'd like to point out something else. This is located in the very popular Hyatt Bellevue Food Court which was and always is jam-packed. There was only one other customer while I was there. The other food shops had lines 5-7 people deep on average and there were barely any open tables. This says a lot. I definitely won't be back this time.",2014-07-15 17:17:48
lzs0dxvm-3pGFo98ZWDrTw,NGwwlCPEfZRWHxkQ7BD7WA,zzyx5x0Z7xXWWvWnZFuxlQ,4.0,0,0,0,"I was at a conference in Philadelphia and was looking for a quick lunch. I saw the ""Bellevue Gourmet Food Court"" and Walnut Street Pizza looked like my best option. I was really impressed! for $3 and change you get half of a small pizza cut into 4 pieces. I got the Margherita and it was really good - the crust is thin and crispy but still a little chewy. Very good, I'd definitely go back.",2015-02-28 21:36:40
A2PWsT5PbqSFmwyu0woQIA,0v_aUe0nBuPuqFrt3MvtuA,zzyx5x0Z7xXWWvWnZFuxlQ,3.0,2,1,0,"So after waiting a few months, I decided 

In [0]:
# 4.3) Dataset Info
total_rows = df.count()
print(f"\n📊 Dataset Info:")
print(f"   Total rows: {total_rows:,}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Top-level columns: {df.columns}")


📊 Dataset Info:
   Total rows: 6,990,280
   Total columns: 9
   Top-level columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']


In [0]:
# 4.4) Schema info
print("\n📋 Schema Info:")
df.printSchema()


📋 Schema Info:
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)



In [0]:
# 4.5) Identify struct (nested) columns
struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
print(f"   Nested struct columns: {struct_cols}")

   Nested struct columns: []


In [0]:
# 4.6) Missing values summary
print(f"\n📊 Missing Values Summary:")
for c in df.columns:
    null_count = df.filter(col(c).isNull()).count()
    if null_count > 0:
        print(f"   {c}: {null_count:,} nulls ({null_count/total_rows*100:.1f}%)")
    else:
        print(f"   {c}: ✅ No nulls")


📊 Missing Values Summary:
   review_id: ✅ No nulls
   user_id: ✅ No nulls
   business_id: ✅ No nulls
   stars: ✅ No nulls
   useful: ✅ No nulls
   funny: ✅ No nulls
   cool: ✅ No nulls
   text: ✅ No nulls
   date: ✅ No nulls


In [0]:
# 4.7) Duplicate analysis
print(f"\n📊 Duplicate Analysis:")
total_rows_dup = df.count()
distinct_rows = df.distinct().count()
duplicate_rows = total_rows_dup - distinct_rows
print(f"   Total rows: {total_rows_dup:,}")
print(f"   Distinct rows: {distinct_rows:,}")
print(f"   Duplicate rows: {duplicate_rows:,} ({duplicate_rows/total_rows_dup*100:.2f}%)")

# Check duplicates by review_id (primary key candidate)
rid_total = df.select("review_id").count()
rid_distinct = df.select("review_id").distinct().count()
rid_duplicates = rid_total - rid_distinct
print(f"\n   Duplicates by 'review_id' (primary key candidate):")
print(f"   Total review_id: {rid_total:,}")
print(f"   Distinct review_id: {rid_distinct:,}")
print(f"   Duplicate review_id: {rid_duplicates:,} ({rid_duplicates/rid_total*100:.2f}%)")

# Check duplicates by business_id (foreign key — expected non-unique)
bid_total = df.select("business_id").count()
bid_distinct = df.select("business_id").distinct().count()
bid_duplicates = bid_total - bid_distinct
print(f"\n   Duplicates by 'business_id' (FK — expected non-unique):")
print(f"   Total business_id: {bid_total:,}")
print(f"   Distinct business_id: {bid_distinct:,}")
print(f"   Duplicate business_id: {bid_duplicates:,} ({bid_duplicates/bid_total*100:.2f}%)")

# Check duplicates by user_id (foreign key — expected non-unique)
uid_total = df.select("user_id").count()
uid_distinct = df.select("user_id").distinct().count()
uid_duplicates = uid_total - uid_distinct
print(f"\n   Duplicates by 'user_id' (FK — expected non-unique):")
print(f"   Total user_id: {uid_total:,}")
print(f"   Distinct user_id: {uid_distinct:,}")
print(f"   Duplicate user_id: {uid_duplicates:,} ({uid_duplicates/uid_total*100:.2f}%)")

if duplicate_rows == 0:
    print("\n   ✅ No duplicate rows found.")
else:
    print("\n   ⚠️ Duplicate rows detected! Consider deduplication.")


📊 Duplicate Analysis:
   Total rows: 6,990,280
   Distinct rows: 6,990,280
   Duplicate rows: 0 (0.00%)

   Duplicates by 'review_id' (primary key candidate):
   Total review_id: 6,990,280
   Distinct review_id: 6,990,280
   Duplicate review_id: 0 (0.00%)

   Duplicates by 'business_id' (FK — expected non-unique):
   Total business_id: 6,990,280
   Distinct business_id: 150,346
   Duplicate business_id: 6,839,934 (97.85%)

   Duplicates by 'user_id' (FK — expected non-unique):
   Total user_id: 6,990,280
   Distinct user_id: 1,987,929
   Duplicate user_id: 5,002,351 (71.56%)

   ✅ No duplicate rows found.


# 5) Structural & Meta-Data Exploration

In [0]:
# 5.1) Dimensionality/Shape
total_rows = df.count()
total_cols = len(df.columns)
print(f"Shape: {total_rows:,} rows × {total_cols:,} columns")   

Shape: 6,990,280 rows × 9 columns


In [0]:
# 5.2) Data Types and Data Classification

print("="*80)
print("RAW DATA TYPES")
print("="*80)
df.printSchema()

# Collect dtype info and schema-level nullable metadata
dtypes = dict(df.dtypes)
schema_nullable = {field.name: field.nullable for field in df.schema.fields}
total_rows = df.count()

for column in df.columns:
    dtype = dtypes[column]
    n = total_rows
    unique = df.select(countDistinct(col(column))).collect()[0][0]
    null_count = df.filter(col(column).isNull()).count()
    null_pct = null_count / n * 100 if n > 0 else 0
    unique_pct = unique / n * 100 if n > 0 else 0
    col_lower = column.lower()
    
    # Sample values (up to 3)
    samples = df.select(col(column)).filter(col(column).isNotNull()).distinct().limit(3)
    sample_values = [row[0] for row in samples.collect()]
    
    # ---- CLASSIFICATION ----
    if unique == 1:
        classification = "CONSTANT"
    elif col_lower in ['id', 'host_id']:
        classification = "IDENTIFIER"
    elif dtype in ('timestamp', 'date'):
        classification = "DATETIME"
    elif col_lower in ['latitude', 'longitude']:
        classification = "COORDINATE"
    elif dtype in ('int', 'bigint'):
        min_val = df.select(spark_min(col(column))).collect()[0][0]
        if min_val is not None and min_val >= 0:
            if unique == 2:
                classification = "BINARY_FLAG"
            elif unique <= 3:
                classification = "ORDINAL"
            elif unique <= 10:
                classification = "ORDINAL"
            else:
                classification = "COUNT"
        else:
            classification = "NUMERIC_INTEGER"
    elif dtype in ('double', 'float'):
        classification = "CONTINUOUS"
    elif dtype == 'string':
        if unique <= 5:
            classification = "CATEGORICAL"
        elif unique > total_rows * 0.5:
            classification = "FREE_TEXT"
        else:
            classification = "CATEGORICAL"
    else:
        classification = "UNCLASSIFIED"
    
    print(f"\n{'='*60}")
    print(f"Column: {column}")
    print(f"{'='*60}")
    nullable = schema_nullable.get(column, None)
    print(f"  Classification:      {classification}")
    print(f"  Data Type:           {dtype}")
    print(f"  Schema Nullable:     {nullable}")
    print(f"  Unique Values:       {unique:,} ({unique_pct:.1f}%)")
    print(f"  Missing Values:      {null_count} ({null_pct:.1f}%)")
    if nullable and null_count == 0:
        print(f"  ⚠️  Schema allows NULLs but none found — consider adding a NOT NULL constraint")
    elif not nullable and null_count > 0:
        print(f"  🚨  Schema disallows NULLs but {null_count} found — data integrity issue!")
    print(f"  Sample Values:       {', '.join(str(v) for v in sample_values)}")
    
    # ---- NUMERIC PROFILING ----
    if dtype in ('int', 'bigint', 'double', 'float'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        
        mean_val = df.select(mean(col(column))).collect()[0][0]
        median_val = df.select(percentile_approx(col(column), 0.5)).collect()[0][0]
        stddev_val = df.select(stddev(col(column))).collect()[0][0]
        skew_val = df.select(skewness(col(column))).collect()[0][0]
        
        zero_count = df.filter(col(column) == 0).count()
        neg_count = df.filter(col(column) < 0).count()
        
        q1 = df.select(percentile_approx(col(column), 0.25)).collect()[0][0]
        q3 = df.select(percentile_approx(col(column), 0.75)).collect()[0][0]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
        
        if skew_val is not None:
            if abs(skew_val) < 0.5:
                skew_text = "Symmetric"
            elif abs(skew_val) < 1:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (Moderate)"
            else:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (High)"
        else:
            skew_text = "N/A"
        
        print(f"  Min:                 {min_val}")
        print(f"  Max:                 {max_val}")
        print(f"  Mean:                {mean_val:.2f}")
        print(f"  Median:              {median_val}")
        print(f"  Std Dev:             {stddev_val:.2f}")
        print(f"  Zero Count:          {zero_count}")
        print(f"  Negative Count:      {neg_count}")
        print(f"  Skewness:            {skew_val:.2f} ({skew_text})")
        print(f"  IQR Outliers:        {outlier_count} ({'Has outliers' if outlier_count > 0 else 'No outliers'})")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "COORDINATE":
            action = "Use for geospatial analysis"
        elif classification == "ORDINAL":
            action = "Preserve order (ordinal encoding)"
        elif classification == "BINARY_FLAG":
            action = "Use as boolean (0/1)"
        elif zero_count > total_rows * 0.9:
            action = "High zero percentage - consider log transform"
        elif skew_val and abs(skew_val) > 1:
            action = "Scale & check outliers (high skew)"
        else:
            action = "Scale & check outliers"
        print(f"  Action:              {action}")
    
    # ---- STRING PROFILING ----
    elif dtype == 'string':
        empty_count = df.filter((col(column) == "") | (col(column).isNull())).count()
        whitespace_count = df.filter(length(col(column)) != length(trim(col(column)))).count()
        
        total_non_null = n - null_count
        if total_non_null > 0:
            all_upper = df.filter(col(column) == upper(col(column))).count() - null_count
            all_lower = df.filter(col(column) == lower(col(column))).count() - null_count
            upper_pct = all_upper / total_non_null * 100
            lower_pct = all_lower / total_non_null * 100
            
            if upper_pct > 80:
                casing = f"{upper_pct:.0f}% Uppercase"
            elif lower_pct > 80:
                casing = f"{lower_pct:.0f}% Lowercase"
            else:
                casing = "Mixed Casing"
        else:
            casing = "N/A (all null)"
        
        len_stats = df.select(
            spark_min(length(col(column))).alias("min_len"),
            spark_max(length(col(column))).alias("max_len"),
            mean(length(col(column))).alias("avg_len")
        ).collect()[0]
        
        print(f"  Empty Count:         {empty_count}")
        print(f"  Whitespace Issues:   {whitespace_count}")
        print(f"  Casing:              {casing}")
        print(f"  Length:              Min: {len_stats['min_len']}, Max: {len_stats['max_len']}, Avg: {len_stats['avg_len']:.2f}")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "FREE_TEXT":
            action = "Use for text mining / NLP only"
        elif unique <= 5:
            action = "One-hot or frequency encode"
        elif unique <= 50:
            action = "Label encode or one-hot (careful with cardinality)"
        else:
            action = "Frequency encode or target encode"
        print(f"  Action:              {action}")
    
    # ---- DATETIME PROFILING ----
    elif dtype in ('timestamp', 'date'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        future_count = df.filter(col(column) > current_date()).count()
        old_count = df.filter(col(column) < "2008-01-01").count()
        
        print(f"  Date Range:          {min_val} to {max_val}")
        print(f"  Future Dates:        {future_count}")
        print(f"  Before 2008:         {old_count}")
        print(f"  Action:              Convert to datetime, extract features")

RAW DATA TYPES
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)


Column: review_id
  Classification:      FREE_TEXT
  Data Type:           string
  Schema Nullable:     True
  Unique Values:       6,990,280 (100.0%)
  Missing Values:      0 (0.0%)
  ⚠️  Schema allows NULLs but none found — consider adding a NOT NULL constraint
  Sample Values:       Lfphg4pEVNU1TTKTkKAE9Q, 2bpkTF7Au_vIMkVc7MOMQA, -wGmNwzrSfyFtVAQcwqf_w
  Empty Count:         0
  Whitespace Issues:   0
  Casing:              Mixed Casing
  Length:              Min: 22, Max: 22, Avg: 22.00
  Action:              Use for text mining / NLP only

Column: user_id
  Classification:      CATEGORICAL
  Data Type:           string
  Schema 

In [0]:
# 5.3) Column Header Audit
import re

print("="*80)
print("COLUMN HEADER AUDIT — Databricks Naming Convention Check")
print("="*80)

# Databricks / Delta Lake column naming rules:
# - Must start with a letter or underscore
# - Can contain letters, digits, underscores, hyphens, spaces (with column mapping)
# - Should avoid special characters
# - Best practice: lowercase snake_case, no leading/trailing spaces

columns = df.columns
total_cols = len(columns)
issues_found = 0

for c in columns:
    issues = []

    # Check: starts with letter or underscore
    if not re.match(r'^[a-zA-Z_]', c):
        issues.append("❌ Does not start with a letter or underscore")

    # Check: contains only valid characters (letters, digits, underscores)
    if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', c):
        issues.append("⚠️  Contains characters beyond [a-zA-Z0-9_]")

    # Check: leading or trailing whitespace
    if c != c.strip():
        issues.append("❌ Has leading/trailing whitespace")

    # Check: contains spaces
    if ' ' in c:
        issues.append("⚠️  Contains spaces (use underscores instead)")

    # Check: contains hyphens
    if '-' in c:
        issues.append("⚠️  Contains hyphens (use underscores instead)")

    # Check: uppercase letters (best practice is lowercase snake_case)
    if c != c.lower():
        issues.append("⚠️  Contains uppercase letters (prefer lowercase snake_case)")

    # Check: consecutive underscores
    if '__' in c:
        issues.append("⚠️  Contains consecutive underscores")

    # Check: excessively long name
    if len(c) > 128:
        issues.append("⚠️  Column name exceeds 128 characters")

    # Check: reserved SQL keywords (common ones)
    reserved_keywords = {
        'select', 'from', 'where', 'table', 'column', 'index', 'group',
        'order', 'by', 'as', 'join', 'on', 'insert', 'update', 'delete',
        'create', 'drop', 'alter', 'into', 'values', 'set', 'null',
        'not', 'and', 'or', 'in', 'is', 'between', 'like', 'exists',
        'having', 'limit', 'offset', 'union', 'all', 'distinct', 'case',
        'when', 'then', 'else', 'end', 'cast', 'primary', 'key',
        'foreign', 'references', 'default', 'constraint', 'check',
        'unique', 'database', 'schema', 'view', 'replace', 'if',
        'true', 'false', 'date', 'time', 'timestamp', 'interval',
        'row', 'rows', 'window', 'partition', 'range', 'current',
        'user', 'role', 'grant', 'revoke'
    }
    if c.lower() in reserved_keywords:
        issues.append("⚠️  Column name is a SQL reserved keyword")

    # Report
    if issues:
        issues_found += 1
        print(f"\n  Column: '{c}'")
        for issue in issues:
            print(f"    {issue}")
        # Suggest corrected name
        suggested = re.sub(r'[^a-zA-Z0-9_]', '_', c)  # replace invalid chars
        suggested = re.sub(r'_+', '_', suggested)       # collapse multiple underscores
        suggested = suggested.strip('_').lower()         # lowercase, strip edge underscores
        if suggested != c:
            print(f"    💡 Suggested rename: '{c}' → '{suggested}'")
    else:
        print(f"\n  Column: '{c}' — ✅ Compliant")

print(f"\n{'='*80}")
print(f"SUMMARY: {total_cols} columns checked, {issues_found} with issues, {total_cols - issues_found} compliant")
if issues_found == 0:
    print("✅ All column names follow Databricks naming conventions!")
else:
    print(f"⚠️  {issues_found} column(s) have naming convention issues — review suggestions above.")
print("="*80)

COLUMN HEADER AUDIT — Databricks Naming Convention Check

  Column: 'review_id' — ✅ Compliant

  Column: 'user_id' — ✅ Compliant

  Column: 'business_id' — ✅ Compliant

  Column: 'stars' — ✅ Compliant

  Column: 'useful' — ✅ Compliant

  Column: 'funny' — ✅ Compliant

  Column: 'cool' — ✅ Compliant

  Column: 'text' — ✅ Compliant

  Column: 'date'
    ⚠️  Column name is a SQL reserved keyword

SUMMARY: 9 columns checked, 1 with issues, 8 compliant
⚠️  1 column(s) have naming convention issues — review suggestions above.


In [0]:
# 5.4) Memory Footprint Estimation
print("="*80)
print("MEMORY FOOTPRINT ESTIMATION")
print("="*80)

total_rows = df.count()

# Estimate bytes per column based on data type
type_sizes = {
    'string': 50,      # average estimate
    'int': 4,
    'bigint': 8,
    'double': 8,
    'float': 4,
    'boolean': 1,
    'timestamp': 8,
    'date': 4,
}

total_estimated_bytes = 0
results = []

for field in df.schema.fields:
    col_name = field.name
    dtype = field.dataType.simpleString()
    
    if dtype == 'string':
        # Use actual average length for strings
        avg_len_row = df.select(mean(length(col(col_name)))).collect()[0][0]
        bytes_per_row = int(avg_len_row) if avg_len_row else 0
    else:
        bytes_per_row = type_sizes.get(dtype, 8)
    
    col_total = bytes_per_row * total_rows
    total_estimated_bytes += col_total
    results.append({
        "Column": col_name,
        "Type": dtype,
        "Est. Bytes/Row": bytes_per_row,
        "Est. Total (MB)": round(col_total / (1024**2), 2)
    })

# Add totals row
results.append({
    "Column": "TOTAL",
    "Type": "",
    "Est. Bytes/Row": "",
    "Est. Total (MB)": round(total_estimated_bytes / (1024**2), 2)
})

memory_df = pd.DataFrame(results)
display(memory_df)

print(f"\n  Total Estimated Size: {total_estimated_bytes / (1024**2):.2f} MB ({total_estimated_bytes / (1024**3):.2f} GB)")

# Warn if large
if total_estimated_bytes > 1 * 1024**3:
    print("  ⚠️  Estimated size > 1 GB — consider partitioning or filtering for local operations.")
else:
    print("  ✅ Dataset fits comfortably in memory.")
print("="*80)

MEMORY FOOTPRINT ESTIMATION


Column,Type,Est. Bytes/Row,Est. Total (MB)
review_id,string,22,146.66
user_id,string,22,146.66
business_id,string,22,146.66
stars,double,8,53.33
useful,bigint,8,53.33
funny,bigint,8,53.33
cool,bigint,8,53.33
text,string,567,3779.88
date,string,19,126.66
TOTAL,,,4559.85



  Total Estimated Size: 4559.85 MB (4.45 GB)
  ⚠️  Estimated size > 1 GB — consider partitioning or filtering for local operations.


In [0]:
# 5.5) Column Constraints & Key Candidates
print("="*80)
print("COLUMN CONSTRAINTS & KEY CANDIDATES")
print("="*80)

# 1) Table properties (Delta table properties, owner, etc.)
print("\n📋 Table Properties:")
try:
    props_df = spark.sql(f"SHOW TBLPROPERTIES {TABLE_NAME}")
    display(props_df)
except Exception as e:
    print(f"   Could not retrieve table properties: {e}")

# 2) NOT NULL constraints per column
print("\n📋 NOT NULL Constraints per Column:")
print(f"{'Column':<30} {'Data Type':<20} {'Nullable':<10}")
print("-" * 60)
for field in df.schema.fields:
    nullable_str = "YES" if field.nullable else "NO (🔒 enforced)"
    print(f"{field.name:<30} {field.dataType.simpleString():<20} {nullable_str:<10}")

# 3) Inferred uniqueness — identify candidate keys
print("\n📋 Candidate Key Analysis:")
total_rows = df.count()
for c in df.columns:
    distinct_count = df.select(countDistinct(col(c))).collect()[0][0]
    null_count = df.filter(col(c).isNull()).count()
    uniqueness_ratio = distinct_count / total_rows * 100 if total_rows > 0 else 0

    if distinct_count == total_rows and null_count == 0:
        status = "🔑 Candidate Primary Key (unique, no nulls)"
    elif distinct_count == total_rows:
        status = "🔑 Unique but has NULLs"
    elif uniqueness_ratio > 95:
        status = f"🔶 Near-unique ({uniqueness_ratio:.1f}%)"
    elif distinct_count <= 2:
        status = f"🔘 Low cardinality ({distinct_count} values)"
    else:
        status = f"📊 {distinct_count:,} distinct ({uniqueness_ratio:.1f}%)"

    print(f"   {c:<30} {status}")

print("\n" + "="*80)
print("META-DATA EXPLORATION COMPLETE")
print("="*80)

COLUMN CONSTRAINTS & KEY CANDIDATES

📋 Table Properties:


key,value
delta.enableDeletionVectors,true
delta.feature.appendOnly,supported
delta.feature.deletionVectors,supported
delta.feature.invariants,supported
delta.minReaderVersion,3
delta.minWriterVersion,7
delta.parquet.compression.codec,zstd
delta.parquet.format.version,2.12.0
delta.parquet.format.version.afe.internal,2.12.0
delta.workloadBasedColumns.deltaFileStatistics,"`text`,`review_id`,`date`,`business_id`,`stars`"



📋 NOT NULL Constraints per Column:
Column                         Data Type            Nullable  
------------------------------------------------------------
review_id                      string               YES       
user_id                        string               YES       
business_id                    string               YES       
stars                          double               YES       
useful                         bigint               YES       
funny                          bigint               YES       
cool                           bigint               YES       
text                           string               YES       
date                           string               YES       

📋 Candidate Key Analysis:
   review_id                      🔑 Candidate Primary Key (unique, no nulls)
   user_id                        📊 1,987,929 distinct (28.4%)
   business_id                    📊 150,346 distinct (2.2%)
   stars                          📊 5 distinct

# 6) Data Quality Checks/Assessment


### 6.1) STRUCTURAL / SCHEMA CHECKS
*Verify the dataset's shape, schema, and declared types match expectations — and that declared types actually reflect the semantic content of the data.*

- **Column Presence & Order** — Expected columns exist; no missing or unexpected columns; order matches schema contract(if it exists).
  - *Example:* Expected `[id, fname, lname, hire_date]`, found `[id, name, hire_date]` → missing `fname`, `lname`; unexpected `name`.
  - *Columns present but reordered → flag if downstream consumers depend on positional access.*

- **Data Type Conformance** — Checks the column's declared type matches the type specified in the schema contract(if it exists).
  - *Examples:*
    - Contract says age INT, dataset declares age VARCHAR → schema mismatch.
    - Contract says hire_date DATE, dataset declares hire_date TIMESTAMP → widening mismatch (may be acceptable; configurable).
    - Contract says salary DECIMAL(10,2), dataset declares salary FLOAT → precision mismatch.

- **Semantic Data Type Violations** - The column's actual data values match the type the schema claims they are. The semantic type is inferred from the values themselves — not read from the schema — so the check is non-circular.
    - *Example:* A column can declare INT in the schema but contain "twenty-five", "unknown", "$1,200", or empty strings. Schema-vs-schema checks never catch this. Semantic inference does.

- **Schema Drift Detection** — Compare current schema against baseline/columns in the schema contract(if it exists); flag added/removed/retyped columns.
  - *Example:* `salary` changed from `INT` to `VARCHAR` between loads.

---

### 6.2) DOMAIN & RANGE CHECKS
*Verify values belong to allowed sets and fall within permitted bounds. All scope variants live here.*

- **Allowed Value Conformance (single column)** — Values belong to the permitted domain/category list.
  - *Example:* `sex` allows only `M`, `F`, `Other`; found `Male`, `1`, `Unknown`.
  - *Sub:* Geographic codes — `country_code` must be ISO-3166 alpha-2.
  - *Sub:* Department names — `department` must match organizational list.
  - *Sub:* Status codes — `employee_status` must be in `{Active, Terminated, Leave}`.

- **Range Conformance (single column)** — Values within min/max bounds.
  - *Example:* `age` must be 0–120; found `-3` or `250`.
  - *Sub:* Salary bands — `salary` within role-specific band.
  - *Sub:* Date bounds — `hire_date` after 1980-01-01.

- **Pattern Conformance** — Values match required format (email, phone, postal code, ID patterns).
  - *Example:* `email` must match RFC-5322; `phone` must match E.164.

- **Impossible Value Detection** — Values that cannot exist regardless of bounds.
  - *Example:* `birth_date` in the future; `age` negative; `death_date` before `birth_date` (single-column view).

---

### 6.3) FORMAT & ENCODING STANDARDIZATION
*Detect and normalize inconsistent representations. Detection only — normalization is a separate remediation step.*

- **Categorical Representation Consistency** — Same concept, different encodings.
  - *Example:* `Male`, `male`, `M`, `1` all representing the same category.
  - *Sub:* Category mapping consistency across related columns — `country_code = 'US'` but `country_name = 'Canada'`.

- **Date Format Consistency** — Dates stored in non-uniform formats.
  - *Example:* `2024-01-15`, `01/15/2024`, `15-Jan-2024` co-existing in one column.

- **Numeric Format Consistency** — Numbers stored with embedded symbols, separators, or units.
  - *Example:* `"$1,200.50"`, `"1.200,50 EUR"`, `"5 kg"` in a numeric column.

- **Boolean Representation Consistency** — Mixed boolean encodings co-existing.
  - *Example:* `Yes/No`, `True/False`, `1/0`, `Y/N` all present in one column.

- **Text Normalization Issues**
  - *Sub:* Leading/trailing whitespace — `" John "` vs `"John"`.
  - *Sub:* Multiple consecutive whitespaces — `"John    Doe"`.
  - *Sub:* Inconsistent capitalization — `"New York"`, `"new york"`, `"NEW YORK"`.
  - *Sub:* Escape/control characters — embedded `\n`, `\t`, `\r`, `\0`, non-printable characters that corrupt parsing.

- **Null Representation Consistency** — Non-standard nulls masquerading as data.
  - *Example:* `"N/A"`, `"NULL"`, `"None"`, `""`, `" "`, `"—"`, `"unknown"`, `"TBD"`, `"."`, `"NIL"`.
  - *Note:* This check only **detects and normalizes**. Counting missingness happens in 6.4.

---

### 6.4) COMPLETENESS CHECKS
*Measure missingness after null normalization. Understand patterns and impact.*

- **Missingness Measurement**
  - *Sub:* Per column — null percentage per column.
  - *Sub:* Per row — rows with excessive missing values (e.g., >50% null).
  - *Sub:* Per cell — total null rate after normalizing variants from 6.3.

- **Missingness Pattern Analysis**
  - *Sub:* Randomness classification — MCAR / MAR / MNAR.
  - *Sub:* Missing value correlation — missingness in `income` correlated with missingness in `employment_status`.
  - *Sub:* Temporal gaps — missing time periods in time-series data (e.g., no records for Q3 2023).

- **Critical Field Coverage** — Required fields (per business rules) must be non-null.
  - *Example:* `customer_id` must never be null.

---

### 6.5) UNIQUENESS CHECKS
*Ensure no duplicate data and each record is uniquely identifiable.*

- **Row-Level Uniqueness** — Entire rows duplicated.
  - *Example:* Two identical rows for `customer_id = 101` with all columns matching.

- **Key-Level Uniqueness** — Primary/unique key columns contain no duplicates.
  - *Example:* `order_id` appears twice.

- **Composite Key Uniqueness** — Multi-column keys are unique together.
  - *Example:* `(store_id, product_id, date)` should be unique; found duplicate.

- **Distribution Uniqueness** — Distinct value counts in categorical columns (too few/too many).
  - *Example:* `state` has 200 distinct values in a US-only dataset (expected ≤ 50) → likely encoding inconsistency (cross-reference 6.3).
  - *Example:* `country` has 1 distinct value in a global dataset → suspiciously low.

---

### 6.6) CROSS-COLUMN CONSISTENCY CHECKS
*Validate logical relationships and contradictions across columns within a row.*

- **Logical Relationship Validation** — Derived columns match their source columns.
  - *Example:* `full_name = fname + ' ' + lname` must hold.
  - *Example:* `total = subtotal + tax`.

- **Logical Contradiction Detection** — Values that conflict logically.
  - *Example:* `age < 18` AND `job_title = 'CEO'`.
  - *Example:* `marital_status = 'Single'` AND `spouse_name` is populated.

- **Cross-Column Range Logic** — Ranges that don't make sense together.
  - *Example:* `birth_date > hire_date`.
  - *Example:* `end_date < start_date`.
  - *Example:* `death_date < birth_date`.

- **Dependency Rule Validation** — Conditional required/forbidden fields.
  - *Example:* `employee_status = 'Terminated'` but `end_date` is null.
  - *Example:* `payment_method = 'Credit Card'` but `card_number` is null.

- **Category Mapping Consistency** — Codes and descriptions align across related columns.
  - *Example:* `country_code = 'US'` but `country_name = 'Canada'`.
  - *Example:* `dept_code = 'ENG'` but `dept_name = 'Sales'`.

---

### 6.7) REFERENTIAL INTEGRITY CHECKS
*Ensure relationships between tables are valid. One check, three views.*

- **Foreign Key Validity** — Every FK value exists in the parent table's PK.
  - *View 1 — Orphan records:* Child row's FK has no matching parent.
    - *Example:* `orders.customer_id = 999` but no `customers.id = 999`.
  - *View 2 — Missing parent records:* Parent row absent for an existing child reference.
    - *Example:* All orders for `customer_id = 500` exist, but customer 500 was deleted.
  - *View 3 — Invalid FK format:* FK value itself is malformed.
    - *Example:* `customer_id = "ABC"` in a numeric FK column.

- **Lookup Table Conformance** — Referenced values exist in lookup/reference tables.
  - *Example:* `orders.status_code = 'X9'` not present in `status_lookup`.

- **Cross-Table Consistency** — Aggregates reconcile across tables.
  - *Example:* `SUM(orders.amount)` per customer ≠ `customers.total_spend`.

---

### 6.8) BUSINESS RULE & ACCURACY CHECKS
*Domain-specific rules and statistical anomalies. All business rules live here — nowhere else.*

- **Business Rule Violations** — Values that break defined business constraints.
  - *Example:* `salary > 1,000,000` for a non-executive role.
  - *Example:* `discount_percent > 100`.
  - *Example:* `order_quantity > max_order_limit`.

- **Statistical Outlier Detection** — Values deviating significantly from mean/median.
  - *Example:* `age = 200` in a customer dataset (Z-score > 5).
  - *Example:* `transaction_amount` in the 99.99th percentile.

- **Distribution Anomalies** — Imbalances, sudden spikes/drops.
  - *Example:* Daily order count drops 90% on a single day with no holiday.
  - *Example:* `gender` distribution 99% `M` in a general population dataset.

---

### 6.9) TIMELINESS CHECKS
*Monitor data freshness, latency, and temporal logic. Only time-related checks live here.*

- **Data Freshness** — Age of data and last update timestamp.
  - *Example:* `last_updated` is 45 days old for a daily-refresh table.

- **Load Timestamp Analysis** — Consistency of ETL/load timestamps.
  - *Example:* `load_timestamp` < `source_timestamp` (loaded before generated).
  - *Example:* Load timestamps out of order across partitions.

- **Source-to-Target Latency** — Delay between source generation and target availability.
  - *Example:* Source event at 10:00, loaded at 14:00 → 4-hour latency exceeds 1-hour SLA.

- **Temporal Consistency** — Timestamps are logical.
  - *Example:* Future-dated `created_at`.
  - *Example:* `updated_at` before `created_at`.
  - *Example:* Data older than retention policy.

- **Temporal Gaps** — Missing time periods in time-series data.
  - *Note:* If treated as a **completeness** issue, keep in 6.4 only. Recommended home: **6.4** (missingness) with a cross-reference here for monitoring freshness cadence. Choose one and stick to it.
  - Currently I placed this check at **6.4** (missingness), so no need to implement it again here.


### 6.1) STRUCTURAL / SCHEMA CHECKS
*Verify the dataset's shape, schema, and declared types match expectations — and that declared types actually reflect the semantic content of the data.*

- **Column Presence & Order** — Expected columns exist; no missing or unexpected columns; order matches schema contract(if it exists).
  - *Example:* Expected `[id, fname, lname, hire_date]`, found `[id, name, hire_date]` → missing `fname`, `lname`; unexpected `name`.
  - *Columns present but reordered → flag if downstream consumers depend on positional access.*

- **Data Type Conformance** — Checks the column's declared type matches the type specified in the schema contract(if it exists).
  - *Examples:*
    - Contract says age INT, dataset declares age VARCHAR → schema mismatch.
    - Contract says hire_date DATE, dataset declares hire_date TIMESTAMP → widening mismatch (may be acceptable; configurable).
    - Contract says salary DECIMAL(10,2), dataset declares salary FLOAT → precision mismatch.

- **Semantic Data Type Violations** - The column's actual data values match the type the schema claims they are. The semantic type is inferred from the values themselves — not read from the schema — so the check is non-circular.
    - *Example:* A column can declare INT in the schema but contain "twenty-five", "unknown", "$1,200", or empty strings. Schema-vs-schema checks never catch this. Semantic inference does.

- **Schema Drift Detection** — Compare current schema against baseline/columns in the schema contract(if it exists); flag added/removed/retyped columns.
  - *Example:* `salary` changed from `INT` to `VARCHAR` between loads.


In [0]:
# 6.1.1) Column Presence & Order
# ─────────────────────────────────────────────────────────────────────
# Expected columns exist; no missing or unexpected columns;
# order matches schema contract.

print("=" * 80)
print("6.1.1) COLUMN PRESENCE & ORDER")
print("    Verifying expected columns exist, no extras, and correct order")
print("=" * 80)

# HYPOTHETICAL DATA CONTRACT
# ══ Expected schema contract for raw yelp_review ══
expected_columns = ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool", "text", "date"]
actual_columns   = df.columns

print(f"\n   Expected columns ({len(expected_columns)}): {expected_columns}")
print(f"   Actual columns   ({len(actual_columns)}): {actual_columns}")

findings = []

# ── 1) Missing columns ──
missing = [c for c in expected_columns if c not in actual_columns]
if missing:
    print(f"\n   ⚠️  MISSING columns (in contract but not in data): {missing}")
else:
    print(f"\n   ✅ No missing columns — all expected columns are present")

# ── 2) Unexpected / extra columns ──
unexpected = [c for c in actual_columns if c not in expected_columns]
if unexpected:
    print(f"   ⚠️  UNEXPECTED columns (in data but not in contract): {unexpected}")
else:
    print(f"   ✅ No unexpected columns — no extras beyond the contract")

# ── 3) Column order ──
if list(actual_columns) == expected_columns:
    print(f"   ✅ Column order matches the expected schema contract")
else:
    print(f"   ⚠️  Column order differs from expected schema contract")
    max_len = max(len(expected_columns), len(actual_columns))
    for i in range(max_len):
        exp = expected_columns[i] if i < len(expected_columns) else "<N/A>"
        act = actual_columns[i]   if i < len(actual_columns)   else "<N/A>"
        marker = "✅" if exp == act else "⚠️ "
        print(f"      Position {i}: expected '{exp}', got '{act}' {marker}")

findings.append({"Check": "Missing Columns",    "Detail": str(missing) if missing else "None",       "Status": "PASS" if not missing else "FINDING"})
findings.append({"Check": "Unexpected Columns",  "Detail": str(unexpected) if unexpected else "None", "Status": "PASS" if not unexpected else "FINDING"})
findings.append({"Check": "Column Order",        "Detail": "Matches contract" if list(actual_columns) == expected_columns else "Order differs", "Status": "PASS" if list(actual_columns) == expected_columns else "FINDING"})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
if violation_count == 0:
    print("\n✅ Column presence and order match the schema contract perfectly.")
else:
    print(f"\n⚠️  {violation_count} issue(s) detected — review above.")
print("COLUMN PRESENCE & ORDER CHECK COMPLETE")
print("=" * 80)

6.1.1) COLUMN PRESENCE & ORDER
    Verifying expected columns exist, no extras, and correct order

   Expected columns (9): ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']
   Actual columns   (9): ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']

   ✅ No missing columns — all expected columns are present
   ✅ No unexpected columns — no extras beyond the contract
   ✅ Column order matches the expected schema contract

SUMMARY


Check,Detail,Status
Missing Columns,None,PASS
Unexpected Columns,None,PASS
Column Order,Matches contract,PASS



✅ Column presence and order match the schema contract perfectly.
COLUMN PRESENCE & ORDER CHECK COMPLETE


In [0]:
# 6.1.2) Data Type Conformance
# ─────────────────────────────────────────────────────────────────────
# Check the column's declared (schema) type matches the type specified
# in the schema contract. Detects widening / narrowing / precision mismatches.

print("=" * 80)
print("6.1.2) DATA TYPE CONFORMANCE")
print("    Checking declared types match the schema contract")
print("=" * 80)

# HYPOTHETICAL DATA CONTRACT
# ══ Expected type contract (raw yelp_review) ══
expected_types = {
    "review_id":   "string",
    "user_id":     "string",
    "business_id": "string",
    "stars":       "int",      # ordinal 1-5 — should be integer, raw table stores as double
    "useful":      "bigint",   # vote count
    "funny":       "bigint",   # vote count
    "cool":        "bigint",   # vote count
    "text":        "string",   # free-text review body
    "date":        "timestamp",  # single review timestamp — raw table stores as string
}

actual_types = dict(df.dtypes)
findings = []

# Type relationship helpers
widening_pairs  = {("int", "bigint"), ("float", "double"), ("date", "timestamp"), ("int", "double"), ("int", "float")}
narrowing_pairs = {(v, k) for k, v in widening_pairs}

for col_name, expected_type in expected_types.items():
    actual_type = actual_types.get(col_name, "COLUMN MISSING")
    match = (actual_type == expected_type)
    icon   = "✅" if match else "⚠️ "
    status = "PASS" if match else "FINDING"

    print(f"\n   {icon} {col_name}:")
    print(f"      Contract type: {expected_type}")
    print(f"      Actual type:   {actual_type}")
    print(f"      Status:        {status}")

    if not match and actual_type != "COLUMN MISSING":
        pair = (expected_type, actual_type)
        if pair in widening_pairs:
            print(f"      ℹ️  Widening mismatch — actual is a broader type (may be acceptable)")
        elif pair in narrowing_pairs:
            print(f"      ⚠️  Narrowing mismatch — actual is narrower (potential data loss)")
        else:
            print(f"      ⚠️  Type mismatch — incompatible types")

    findings.append({
        "Column": col_name,
        "Expected Type": expected_type,
        "Actual Type": actual_type,
        "Match": "Yes" if match else "No",
        "Status": status
    })

# ── Extra columns not in contract ──
extra_cols = [c for c in actual_types if c not in expected_types]
if extra_cols:
    print(f"\n   ℹ️  Extra columns not in schema contract:")
    for c in extra_cols:
        print(f"      + {c}: {actual_types[c]}")
        findings.append({"Column": c, "Expected Type": "N/A (not in contract)", "Actual Type": actual_types[c], "Match": "N/A", "Status": "INFO"})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
if violation_count == 0:
    print("\n✅ All column types match the schema contract.")
else:
    print(f"\n⚠️  {violation_count} type mismatch(es) detected — review above.")
print("DATA TYPE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

6.1.2) DATA TYPE CONFORMANCE
    Checking declared types match the schema contract

   ✅ review_id:
      Contract type: string
      Actual type:   string
      Status:        PASS

   ✅ user_id:
      Contract type: string
      Actual type:   string
      Status:        PASS

   ✅ business_id:
      Contract type: string
      Actual type:   string
      Status:        PASS

   ⚠️  stars:
      Contract type: int
      Actual type:   double
      Status:        FINDING
      ℹ️  Widening mismatch — actual is a broader type (may be acceptable)

   ✅ useful:
      Contract type: bigint
      Actual type:   bigint
      Status:        PASS

   ✅ funny:
      Contract type: bigint
      Actual type:   bigint
      Status:        PASS

   ✅ cool:
      Contract type: bigint
      Actual type:   bigint
      Status:        PASS

   ✅ text:
      Contract type: string
      Actual type:   string
      Status:        PASS

   ⚠️  date:
      Contract type: timestamp
      Actual type:   str

Column,Expected Type,Actual Type,Match,Status
review_id,string,string,Yes,PASS
user_id,string,string,Yes,PASS
business_id,string,string,Yes,PASS
stars,int,double,No,FINDING
useful,bigint,bigint,Yes,PASS
funny,bigint,bigint,Yes,PASS
cool,bigint,bigint,Yes,PASS
text,string,string,Yes,PASS
date,timestamp,string,No,FINDING



⚠️  2 type mismatch(es) detected — review above.
DATA TYPE CONFORMANCE CHECK COMPLETE


---
#### 🟠 DQ-REVIEW-001 — Data Type Mismatch: `stars` Column

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-001` |
| **Description** | The `stars` column’s declared schema type (`double`) does not match the data contract type (`int`). Star ratings are ordinal values (1–5) with no fractional component, yet the source stores them as 64-bit floating-point. |
| **Detection Phase** | Phase 6.1 — Structural / Schema Checks → **6.1.2) Data Type Conformance** |
| **Detected In** | Cell 27 of this notebook |
| **Confirmed By** | Cell 30 — **6.1.3) Semantic Data Type Violations**: all 6,990,280 values are whole numbers (samples: `1.0`, `4.0`, `5.0`). Also reflected in Cell 35 — **6.1.4) Schema Drift Detection** as a RETYPED finding (`int` → `double`). |
| **Severity** | 🟠 Medium |
| **Affected Rows** | 6,990,280 (100% of the dataset) |
| **Impact** | Wastes 4 bytes/row (double=8B vs int=4B — \~27 MB across the dataset), misleads downstream models into treating a discrete ordinal variable as continuous, and causes floating-point display artefacts (e.g., `4.0` instead of `4`). |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Cast `stars` to `IntegerType`** during the Silver-layer cleaning step | Most direct fix: preserves the 1–5 ordinal semantics, halves storage, prevents continuous-variable misinterpretation, and is universally compatible with Spark, ML pipelines, and BI tools. |
| 2 | Cast to `ShortType` (2 bytes) | Even smaller footprint (2B per row), valid since range 1–5 fits comfortably. However, `ShortType` is less commonly supported across ML frameworks and BI connectors, which may cause downstream compatibility issues. |
| 3 | Leave as `double` and add a `CHECK` constraint (`stars BETWEEN 1 AND 5`) | Avoids a schema change but does **not** fix the type mismatch, wasted storage, or the risk of continuous misinterpretation. Only adds a validation guard. |

##### ✅ Best Logical Solution
**Solution 1 — Cast to `IntegerType`** during the Silver-layer cleaning step. It correctly represents the ordinal domain (1–5), is universally supported, reduces storage by 50%, and is already validated in this notebook’s temporary fix cell (TE.1, Cell 33) with **0 failed casts** across all 6,990,280 rows.

---

---
#### 🔴 DQ-REVIEW-002 — Data Type Mismatch: `date` Column

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-002` |
| **Description** | The `date` column’s declared schema type (`string`) does not match the data contract type (`timestamp`). Each row contains a single review timestamp in the format `"YYYY-MM-DD HH:MM:SS"` (e.g., `"2015-08-06 04:21:46"`), but it is stored as a flat string rather than a proper `TimestampType`. |
| **Detection Phase** | Phase 6.1 — Structural / Schema Checks → **6.1.2) Data Type Conformance** |
| **Detected In** | Cell 27 of this notebook |
| **Confirmed By** | Cell 30 — **6.1.3) Semantic Data Type Violations**: 100% of 6,990,280 values are parseable as timestamps (samples: `2015-08-06 04:21:46`, `2014-01-12 02:30:08`, `2019-07-01 03:21:00`). Also reflected in Cell 35 — **6.1.4) Schema Drift Detection** as ADDED (`date:string`) + REMOVED (`review_timestamp:timestamp`). |
| **Severity** | 🔴 High |
| **Affected Rows** | 6,990,280 (100% of the dataset) |
| **Impact** | Prevents native temporal operations (date-range filtering, `year()`/`month()`/`hour()` extraction, window functions), blocks partition pruning and temporal indexing, forces repeated string parsing in every downstream query, and makes time-zone-aware processing impossible. This is the highest-impact finding because temporal analysis is central to review-trend, seasonality, and recency studies. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Cast to `TimestampType`** using `to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")` during the Silver-layer cleaning step | Direct and lossless — the format is uniform across all 6.99M rows (0 null casts validated). Unlocks all temporal operations, enables partition pruning on date ranges, and preserves full datetime precision (hour, minute, second) needed for time-of-day and temporal trend analysis. |
| 2 | Cast to `DateType` (date-only, discard time component) | Simpler type, but **loses the HH:MM:SS component** which is valuable for time-of-day analysis (review hour, AM/PM patterns, peak posting times). Not recommended for a review dataset where time-of-day is analytically significant. |
| 3 | Keep as `string` and parse on-the-fly in downstream queries | Avoids a schema change but forces every downstream consumer to re-parse the same string, creates significant performance overhead on 7M rows, and prevents predicate pushdown on temporal filters. |

##### ✅ Best Logical Solution
**Solution 1 — Cast to `TimestampType`** during the Silver-layer cleaning step. The format is perfectly consistent across all rows (0 failed casts validated in Cell 30), preserves the full datetime precision needed for review-hour and temporal-trend analysis, and is already applied in this notebook’s temporary fix cell (TE.1, Cell 33) as `review_timestamp`.

---

In [0]:
# 6.1.3) Semantic Data Type Violations
# Instead of comparing schema types against themselves (which is circular and always passes),
# this infers the SEMANTIC type from actual data values and flags mismatches with the stored type.

from pyspark.sql.types import (
    IntegerType, LongType, StringType, DoubleType, FloatType,
    DateType, TimestampType, BooleanType, ArrayType, MapType, StructType
)
from pyspark.sql.functions import expr as _expr
from datetime import datetime as _dt

print("=" * 80)
print("6.1.3) SEMANTIC DATA TYPE VIOLATIONS")
print("    Infers the likely intended type from actual values vs. stored schema type")
print("=" * 80)

total_rows = df.count()
findings = []

for field in df.schema.fields:
    col_name = field.name
    actual_type = field.dataType
    actual_type_str = actual_type.simpleString()

    non_null_df = df.select(col(col_name)).filter(col(col_name).isNotNull())
    non_null_count = non_null_df.count()
    sample_values = [row[0] for row in non_null_df.distinct().limit(10).collect()]

    violations = []
    inferred_type = actual_type_str  # default: type is correct

    # ── Detect nested / complex types ──
    if isinstance(actual_type, ArrayType):
        print(f"\n{'─'*60}")
        print(f"📌 Column: {col_name}")
        print(f"   🔎 Nested: ArrayType — element: {actual_type.elementType}")
    elif isinstance(actual_type, MapType):
        print(f"\n{'─'*60}")
        print(f"📌 Column: {col_name}")
        print(f"   🔎 Nested: MapType — key: {actual_type.keyType}, value: {actual_type.valueType}")
    elif isinstance(actual_type, StructType):
        sub = [f"{sf.name}: {sf.dataType.simpleString()}" for sf in actual_type.fields]
        print(f"\n{'─'*60}")
        print(f"📌 Column: {col_name}")
        print(f"   🔎 Nested: StructType — fields: {', '.join(sub)}")

    # ════════════════════════════════════════════════════════════
    # STRING columns: probe what the values actually represent
    # ════════════════════════════════════════════════════════════
    if isinstance(actual_type, StringType) and non_null_count > 0:

        # 1) Comma-separated / delimited multi-value check (highest priority)
        comma_list_count = non_null_df.filter(col(col_name).contains(", ")).count()
        comma_ratio = comma_list_count / non_null_count

        # Guard: skip comma inference for free-text columns (natural language has commas)
        _avg_len = non_null_df.select(mean(length(col(col_name)))).collect()[0][0] if comma_ratio > 0.10 else 0
        if comma_ratio > 0.10 and (_avg_len is not None and _avg_len <= 100):
            # Probe whether individual elements look like timestamps
            sample_str = str(sample_values[0]) if sample_values else ""
            elements = [e.strip() for e in sample_str.split(",")][:5]
            ts_hits = 0
            for elem in elements:
                try:
                    _dt.strptime(elem, "%Y-%m-%d %H:%M:%S")
                    ts_hits += 1
                except Exception:
                    pass

            if ts_hits == len(elements) and elements:
                inferred_type = "array<timestamp>"
                violations.append(
                    f"🔶 {comma_ratio*100:.1f}% of rows contain comma-separated TIMESTAMPS packed in a single string"
                )
                violations.append("   → Recommended: split(',') then cast elements to TimestampType, or explode into rows")
            else:
                inferred_type = "array<string>"
                violations.append(
                    f"🔶 {comma_ratio*100:.1f}% of rows contain comma-separated values packed in a single string"
                )
                violations.append("   → Recommended: split into ArrayType or explode into rows")
        else:
            # 2) JSON-embedded check
            json_count = non_null_df.filter(
                (trim(col(col_name)).startswith("{") & trim(col(col_name)).endswith("}")) |
                (trim(col(col_name)).startswith("[") & trim(col(col_name)).endswith("]"))
            ).count()
            json_ratio = json_count / non_null_count

            if json_ratio > 0.50:
                inferred_type = "struct / map (JSON)"
                violations.append(f"🔶 {json_ratio*100:.1f}% of values appear to be JSON")
                violations.append("   → Recommended: parse with from_json() / schema_of_json()")
            else:
                # 3) Numeric check  (try_cast avoids ANSI-mode errors / optimizer false positives)
                castable_numeric = non_null_df.filter(_expr(f"try_cast(`{col_name}` AS DOUBLE) IS NOT NULL")).count()
                numeric_ratio = castable_numeric / non_null_count

                # 4) Timestamp check
                castable_ts = non_null_df.filter(_expr(f"try_cast(`{col_name}` AS TIMESTAMP) IS NOT NULL")).count()
                ts_ratio = castable_ts / non_null_count

                # 5) Boolean-pattern check
                bool_patterns = ['true','false','yes','no','0','1','y','n','t','f']
                bool_count = non_null_df.filter(lower(trim(col(col_name))).isin(bool_patterns)).count()
                bool_ratio = bool_count / non_null_count

                # Determine best semantic match (highest confidence first)
                # Timestamp checked first — timestamps contain digits that can false-positive as numeric
                if ts_ratio > 0.95:
                    inferred_type = "timestamp"
                    violations.append(f"🔶 {ts_ratio*100:.1f}% parseable as timestamp")
                    violations.append("   → Recommended: cast to TimestampType")
                elif numeric_ratio > 0.95:
                    # Guard: check if values are truly numeric or just hex/alphanumeric IDs
                    has_alpha = non_null_df.filter(col(col_name).rlike("[a-zA-Z]")).count()
                    alpha_ratio = has_alpha / non_null_count
                    if alpha_ratio > 0.05:
                        pass  # Not truly numeric — likely IDs/hashes; skip numeric inference
                    else:
                        castable_long = non_null_df.filter(_expr(f"try_cast(`{col_name}` AS BIGINT) IS NOT NULL")).count()
                        if castable_long / non_null_count > 0.95:
                            inferred_type = "long / integer"
                            violations.append(f"🔶 {numeric_ratio*100:.1f}% castable to numeric (integer-like)")
                            violations.append("   → Recommended: cast to LongType or IntegerType")
                        else:
                            inferred_type = "double / float"
                            violations.append(f"🔶 {numeric_ratio*100:.1f}% castable to numeric (decimal)")
                            violations.append("   → Recommended: cast to DoubleType")
                elif bool_ratio > 0.95:
                    inferred_type = "boolean"
                    violations.append(f"🔶 {bool_ratio*100:.1f}% match boolean patterns")
                    violations.append("   → Recommended: cast to BooleanType")
                elif 0.05 < numeric_ratio <= 0.95:
                    violations.append(f"⚠️  Mixed content: {numeric_ratio*100:.1f}% castable to numeric")
                    bad_examples = (
                        non_null_df.filter(col(col_name).cast("double").isNull())
                        .distinct().limit(5)
                    )
                    violations.append(f"      Non-numeric examples: {[r[0] for r in bad_examples.collect()]}")

    # ════════════════════════════════════════════════════════════
    # INTEGER / LONG columns: detect boolean-like or constant flags
    # ════════════════════════════════════════════════════════════
    elif isinstance(actual_type, (IntegerType, LongType)) and non_null_count > 0:
        min_val = df.agg(spark_min(col(col_name))).collect()[0][0]
        max_val = df.agg(spark_max(col(col_name))).collect()[0][0]
        distinct_count = df.select(countDistinct(col(col_name))).collect()[0][0]

        if distinct_count == 2 and min_val == 0 and max_val == 1:
            inferred_type = "boolean"
            violations.append("ℹ️  Binary (0/1) — could be stored as BooleanType")
        elif distinct_count == 1:
            inferred_type = f"{actual_type_str} (constant)"
            violations.append(f"ℹ️  Column is constant (single value: {min_val})")

    # ════════════════════════════════════════════════════════════
    # DOUBLE / FLOAT columns: detect integer-only values
    # ════════════════════════════════════════════════════════════
    elif isinstance(actual_type, (DoubleType, FloatType)) and non_null_count > 0:
        frac_check = non_null_df.filter(
            (col(col_name) != col(col_name).cast("long").cast(actual_type_str))
        ).count()
        if frac_check == 0:
            inferred_type = "long / integer (no fractional part)"
            violations.append("ℹ️  All values are whole numbers — consider LongType for efficiency")

    # ── Report per column ──
    status = "✅ PASS" if not violations else "⚠️  FINDING"
    print(f"\n{'─'*60}")
    print(f"📌 Column: {col_name}")
    print(f"   Schema Type:    {actual_type_str}")
    print(f"   Inferred Type:  {inferred_type}")
    print(f"   Non-Null Count: {non_null_count:,} / {total_rows:,}")
    print(f"   Status:         {status}")

    if sample_values:
        display_samples = [str(s)[:80] + ("..." if len(str(s)) > 80 else "") for s in sample_values[:3]]
        print(f"   Samples:        {display_samples}")
    for v in violations:
        print(f"   {v}")

    findings.append({
        "Column": col_name,
        "Schema Type": actual_type_str,
        "Inferred Semantic Type": inferred_type,
        "Status": "PASS" if not violations else "FINDING",
        "Detail": "; ".join(v.strip() for v in violations) if violations else "Types aligned"
    })

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("="*80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} columns checked, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All column schema types match their inferred semantic types.")
else:
    print(f"⚠️  {violation_count} column(s) have schema-vs-semantic type mismatches — review above.")
print("SEMANTIC DATA TYPE VIOLATIONS CHECK COMPLETE")
print("=" * 80)

6.1.3) SEMANTIC DATA TYPE VIOLATIONS
    Infers the likely intended type from actual values vs. stored schema type

────────────────────────────────────────────────────────────
📌 Column: review_id
   Schema Type:    string
   Inferred Type:  string
   Non-Null Count: 6,990,280 / 6,990,280
   Status:         ✅ PASS
   Samples:        ['Lfphg4pEVNU1TTKTkKAE9Q', '2bpkTF7Au_vIMkVc7MOMQA', '-wGmNwzrSfyFtVAQcwqf_w']

────────────────────────────────────────────────────────────
📌 Column: user_id
   Schema Type:    string
   Inferred Type:  string
   Non-Null Count: 6,990,280 / 6,990,280
   Status:         ✅ PASS
   Samples:        ['byI0j1O-MqV4TpnL7WWTbQ', 'xGv9o9nIq-762Vxrw1n0CQ', 'I7Fmd8WQA0FozWOEZQNdkw']

────────────────────────────────────────────────────────────
📌 Column: business_id
   Schema Type:    string
   Inferred Type:  string
   Non-Null Count: 6,990,280 / 6,990,280
   Status:         ✅ PASS
   Samples:        ['8Rj8N102MbhuwW-afz0Yuw', 'aqAiTlfcI0RfbE4AqLXZRg', 'Bj2_M9hl2ZcDp

Column,Schema Type,Inferred Semantic Type,Status,Detail
review_id,string,string,PASS,Types aligned
user_id,string,string,PASS,Types aligned
business_id,string,string,PASS,Types aligned
stars,double,long / integer (no fractional part),FINDING,ℹ️ All values are whole numbers — consider LongType for efficiency
useful,bigint,bigint,PASS,Types aligned
funny,bigint,bigint,PASS,Types aligned
cool,bigint,bigint,PASS,Types aligned
text,string,string,PASS,Types aligned
date,string,timestamp,FINDING,🔶 100.0% parseable as timestamp; → Recommended: cast to TimestampType



9 columns checked, 2 finding(s)
⚠️  2 column(s) have schema-vs-semantic type mismatches — review above.
SEMANTIC DATA TYPE VIOLATIONS CHECK COMPLETE


---
#### ✅ 6.1.3 Semantic Analysis — Independent Confirmation of DQ-REVIEW-001 & DQ-REVIEW-002

The semantic data-type inference in **6.1.3)** scans the actual **values** of every column (rather than comparing declared types against a contract) and independently confirms both findings from 6.1.2:

| Error ID | Column | Schema Type | Inferred Semantic Type | Evidence |
| --- | --- | --- | --- | --- |
| `DQ-REVIEW-001` | `stars` | `double` | `long / integer` | All 6,990,280 values are whole numbers with no fractional part (e.g., `1.0`, `4.0`, `5.0`) |
| `DQ-REVIEW-002` | `date` | `string` | `timestamp` | 100% of 6,990,280 values parse as valid timestamps (e.g., `2015-08-06 04:21:46`, `2019-07-01 03:21:00`) |

**Why this matters:** Phase 6.1.2 compares **declared types against a contract** (top-down), while 6.1.3 infers types **from the actual data values** (bottom-up). The fact that both approaches independently flag the same two columns — and no others — provides strong, cross-validated evidence that these are genuine type mismatches, not contract errors.

**Remaining 7 columns** (`review_id`, `user_id`, `business_id`, `useful`, `funny`, `cool`, `text`) passed both checks with types aligned.

---

#### For this Comprehensive Assessment to be logical we will need to
1) Fix the Data Types — cast the `date` string to `TimestampType`, cast `stars` from double to integer (ordinal 1–5)
2) Remodel the data(IF NEEDED) — extract temporal features from the review timestamp + add text analytics features (length, word count, sentence count) and a total_votes aggregate

#### Important Note: These fixes won't change anything yet in the original tables, they are just made temporarily here till the cleaning step of the pipeline

In [0]:
# Fix Data Types based on 6.1.3) Semantic Data Type Violations findings
# ─────────────────────────────────────────────────────────────────────
# review_id   : string  ✅  (alphanumeric hash — correctly typed)
# user_id     : string  ✅  (alphanumeric hash — correctly typed)
# business_id : string  ✅  (alphanumeric hash — correctly typed)
# stars       : double  ⚠️  → cast to integer (ordinal 1–5)
# useful      : bigint  ✅  (vote count — correctly typed)
# funny       : bigint  ✅  (vote count — correctly typed)
# cool        : bigint  ✅  (vote count — correctly typed)
# text        : string  ✅  (free-text review body — correctly typed)
# date        : string  ⚠️  → cast to timestamp
#
# Important: These fixes are TEMPORARY for this EDA notebook.
# Original tables in bronze_final are NOT modified.

from pyspark.sql.functions import to_timestamp

print("=" * 80)
print("FIX DATA TYPES")
print("=" * 80)

print("\n📋 BEFORE fix:")
df.printSchema()

df_typed = (
    df
    # Cast date string → timestamp (format: "2018-01-14 20:54:14")
    .withColumn("review_timestamp", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .drop("date")
    # Cast stars double → integer (ordinal 1–5)
    .withColumn("stars", col("stars").cast("int"))
)

print("\n📋 AFTER fix:")
df_typed.printSchema()

# Validation: check for null timestamps (failed casts)
null_ts = df_typed.filter(col("review_timestamp").isNull()).count()
# Validation: check for null stars (failed casts)
null_stars = df_typed.filter(col("stars").isNull()).count()
total = df_typed.count()
print(f"\n✅ Rows: {total:,}")
print(f"   Null timestamps (failed casts): {null_ts:,}")
print(f"   Null stars (failed casts):      {null_stars:,}")

# Spot-check: show a few rows with their typed columns
print("\n📋 Sample (typed):")
display(df_typed.limit(5))

FIX DATA TYPES

📋 BEFORE fix:
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)


📋 AFTER fix:
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: integer (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)


✅ Rows: 6,990,280
   Null timestamps (failed casts): 0
   Null stars (failed casts):      0

📋 Sample (typed):


review_id,user_id,business_id,stars,useful,funny,cool,text,review_timestamp
eYK3GFCCPM_2LTrNBUCsmA,0yrgXkKQUl6aSNAWs25dsQ,Iyopb5uFIBT_HOgp8ipSfw,5,3,0,1,"So I'm pretty sure these guys were voted Best Vietnamese Restaurant in Philadelphia for a reason. Considering the many times I have frequented this place, I have never been disappointed. Allow me to elaborate: Excellent decor and environment (certainly not a typical Asian restaurant) Great knowledgeable staff HUGE menu to select from $5 lunch specials I mean seriously, the list could go on. Appetizers: Hn Xúc Bánh Đa -- minced clam meat, beef served in a cute rice cracker break off a piece of the rice cracker and slather some of that minced goodness on Entrees: Ba Vi make your own mini Vietnamese burritos - vermicelli riced noodles, shrimp balls, pork and beef with grapeleaves Caramelized Spare Ribs with quail eggs clay pot goodness with quaint quail eggs and spare rib tips in a sweet brown sauce (bowl of rice, bowl of awesome in-house chicken broth soup and you're good to go) Drinks: Iced Coffee like no other Thai Tea If you haven't been to this affordable Vietnamese destination restaurant, you need to go now.",2013-12-22T20:22:47.000Z
ijNVEplkFfEIb8QTFC0ooA,NWmPl8tgT1Q7DWCtUopQHg,lau2jbBCG3XohoAvCRIm1Q,5,1,0,1,Quick service and tasty pie! Groupon deal and the whole family ate deliciousness for $26! The garlic knots look and smell fantastic and we can't wait to try it again. The kids loved to chose their own toppings and the atmosphere is good for kiddos too. You can chose a gluten free crust too.,2017-06-17T01:45:38.000Z
p5pX5tld7N0DPXaox60W9Q,ZWFevb8rSFP6KCUqeq3FFg,Fz4aGOGRmPo0ki6iAc42Sw,5,0,0,0,The business has moved from it's location on Robb Drive to the Scolari's shopping center on Sharlands right off the Robb Drive exit. Love the convenient new location and the new salon is beautiful! All of the stylists are wonderful and I wouldn't go anywhere else!,2011-12-31T21:15:00.000Z
PCIUeFCOwJCFJaUCEsNoPQ,7Eq71a1cwHZ-T-k9pusdlw,j-qtdD55OLfSqfsWuQTDJg,4,0,0,1,"We sat outside on Rittenhouse Square and did some people watching. We shared the tasty chicken club with bacon and avocado, the lamb sandwich, and fries. We split a bottle of house wine. We didn't spend a ridiculous amount of money. A great lunch experience. Good job, Stephen Starr!",2011-10-23T03:43:48.000Z
JpHkJf4_9zMppk6MbJwP2g,BTKCSvPiy7anahF2HhNNNA,IvAa7PvfOYl0xRML5W3ebg,2,1,1,0,"Caveat: not a review of the beer, only a review of the brewery itself. So we went with a bunch of out of town friends to try and do the brewery tour, but they weren't offering it that day. OK, that was our fault for not checking before we came. We decided to stay and drink and maybe get something to eat because I was intrigued by the grilled cheese menus, reasonably priced at $3. I was a little wary because the room was huge, dark, hot, and kind of dingy looking, but everyone else was won over by the shuffleboard table. We sat down and a few people went up to the bar to order flights, which (I don't know) seemed priced decently. That's all they drank, and seemed to enjoy most of them, though there were a few varieties that went untouched by anyone until it was evident SOMEONE was going to have to drink it. My main issue was that there was only one guy working. There weren't a lot of people there but he was clearly doing EVERYTHING, and he was clearly overwhelmed. It took about 15 minutes to even catch his attention when I was trying to order food. So frustrating. And when I finally DID order, they were out of EVERYTHING on the menu. Literally EVERYTHING. I asked him what he did have, and he said, ""A few pieces of pumpernickel bread, half a tomato, and some American cheese."" ... okay. I ordered one of whatever he had and we split it. It was all right, but I can make a better grilled cheese at home with ingredients I've actually chosen. We just went to Cafe Estelle afterward for food and that was much, much better.",2010-08-01T13:15:52.000Z

In [0]:
# # Remodel: Extract temporal features from the review timestamp
# # + add text analytics features for downstream NLP / analysis
# # ─────────────────────────────────────────────────────────────────────
# from pyspark.sql.functions import (
#     hour, minute, second, dayofweek, dayofmonth, dayofyear,
#     weekofyear, month, year, quarter, date_format, last_day,
#     next_day, trunc, months_between, lit, when, floor, ceil, concat,
#     length as spark_length, size as spark_size, regexp_replace
# )

# print("=" * 80)
# print("REMODEL DATA  —  Temporal + Text Feature Extraction")
# print("=" * 80)

# df_remodeled = (
#     df_typed

#     # ════════════════════════════════════════════════════════════════════
#     # DATE-LEVEL ATTRIBUTES  (date dimension style)
#     # ════════════════════════════════════════════════════════════════════
#     .withColumn("review_date",        col("review_timestamp").cast("date"))
#     .withColumn("review_year",        year(col("review_timestamp")))
#     .withColumn("review_month",       month(col("review_timestamp")))
#     .withColumn("review_month_name",  date_format(col("review_timestamp"), "MMMM"))        # January…
#     .withColumn("review_month_abbr",  date_format(col("review_timestamp"), "MMM"))          # Jan…
#     .withColumn("review_day",         dayofmonth(col("review_timestamp")))
#     .withColumn("review_day_of_year", dayofyear(col("review_timestamp")))
#     .withColumn("review_dow_number",  dayofweek(col("review_timestamp")))                   # 1=Sun…7=Sat
#     .withColumn("review_dow_name",    date_format(col("review_timestamp"), "EEEE"))         # Monday…
#     .withColumn("review_dow_abbr",    date_format(col("review_timestamp"), "EEE"))          # Mon…
#     .withColumn("review_quarter",     quarter(col("review_timestamp")))
#     .withColumn("review_quarter_name",concat(lit("Q"), quarter(col("review_timestamp"))))
#     .withColumn("review_week_of_year",weekofyear(col("review_timestamp")))
#     .withColumn("review_year_month",  date_format(col("review_timestamp"), "yyyy-MM"))      # 2019-07
#     .withColumn("review_year_quarter",
#         concat(year(col("review_timestamp")).cast("string"), lit("-Q"), quarter(col("review_timestamp")).cast("string"))
#     )

#     # ── Half-year / Semester ──
#     .withColumn("review_half_year",
#         when(month(col("review_timestamp")) <= 6, lit(1)).otherwise(lit(2))
#     )
#     .withColumn("review_half_year_name",
#         when(month(col("review_timestamp")) <= 6, lit("H1")).otherwise(lit("H2"))
#     )

#     # ── Month position flags ──
#     .withColumn("review_month_start_date", trunc(col("review_timestamp"), "month"))
#     .withColumn("review_month_end_date",   last_day(col("review_timestamp")))
#     .withColumn("is_month_start",
#         (dayofmonth(col("review_timestamp")) == 1).cast("boolean")
#     )
#     .withColumn("is_month_end",
#         (col("review_date") == last_day(col("review_timestamp"))).cast("boolean")
#     )

#     # ── Year position flags ──
#     .withColumn("review_year_start_date", trunc(col("review_timestamp"), "year"))
#     .withColumn("is_year_start",
#         ((month(col("review_timestamp")) == 1) & (dayofmonth(col("review_timestamp")) == 1)).cast("boolean")
#     )
#     .withColumn("is_year_end",
#         ((month(col("review_timestamp")) == 12) & (dayofmonth(col("review_timestamp")) == 31)).cast("boolean")
#     )

#     # ── Weekend / Weekday flag ──
#     .withColumn("is_weekend",
#         dayofweek(col("review_timestamp")).isin(1, 7).cast("boolean")  # 1=Sun, 7=Sat
#     )
#     .withColumn("is_weekday",
#         (~dayofweek(col("review_timestamp")).isin(1, 7)).cast("boolean")
#     )

#     # ════════════════════════════════════════════════════════════════════
#     # TIME-LEVEL ATTRIBUTES
#     # ════════════════════════════════════════════════════════════════════
#     .withColumn("review_hour",   hour(col("review_timestamp")))
#     .withColumn("review_minute", minute(col("review_timestamp")))
#     .withColumn("review_second", second(col("review_timestamp")))
#     .withColumn("review_time_of_day",
#         when(hour(col("review_timestamp")).between(5, 11),  lit("Morning"))
#         .when(hour(col("review_timestamp")).between(12, 16), lit("Afternoon"))
#         .when(hour(col("review_timestamp")).between(17, 20), lit("Evening"))
#         .otherwise(lit("Night"))
#     )
#     .withColumn("review_am_pm",
#         when(hour(col("review_timestamp")) < 12, lit("AM")).otherwise(lit("PM"))
#     )
#     .withColumn("review_hour_12",
#         when(hour(col("review_timestamp")) == 0, lit(12))
#         .when(hour(col("review_timestamp")) > 12, hour(col("review_timestamp")) - 12)
#         .otherwise(hour(col("review_timestamp")))
#     )
#     .withColumn("review_hour_bin_4h",
#         (floor(hour(col("review_timestamp")) / 4) * 4).cast("int")
#     )

#     # ════════════════════════════════════════════════════════════════════
#     # TEXT ANALYTICS FEATURES  (NLP-readiness)
#     # ════════════════════════════════════════════════════════════════════
#     .withColumn("text_length",     spark_length(col("text")))
#     .withColumn("text_word_count", spark_size(split(col("text"), "\\s+")))
#     .withColumn("text_sentence_count",
#         spark_size(split(regexp_replace(col("text"), "([.!?])\\s+", "$1|||SEP|||"), "\\|\\|\\|SEP\\|\\|\\|"))
#     )

#     # ════════════════════════════════════════════════════════════════════
#     # ENGAGEMENT AGGREGATE
#     # ════════════════════════════════════════════════════════════════════
#     .withColumn("total_votes", col("useful") + col("funny") + col("cool"))
# )

# # Reassign df and total_rows so ALL downstream cells use the remodeled data
# df = df_remodeled
# total_rows = df.count()

# print(f"\n📊 Remodeled shape: {total_rows:,} rows × {len(df.columns)} columns")
# print(f"   (Original grain preserved: 1 row per review — no row count change)")
# print("\n📋 Schema:")
# df.printSchema()
# print("📋 Sample (remodeled):")
# display(df.limit(10))

REMODEL DATA  —  Temporal + Text Feature Extraction

📊 Remodeled shape: 6,990,280 rows × 46 columns
   (Original grain preserved: 1 row per review — no row count change)

📋 Schema:
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: integer (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)
 |-- review_month_name: string (nullable = true)
 |-- review_month_abbr: string (nullable = true)
 |-- review_day: integer (nullable = true)
 |-- review_day_of_year: integer (nullable = true)
 |-- review_dow_number: integer (nullable = true)
 |-- review_dow_name: string (nullable = true)
 |-- review_dow_abbr: string (nullable = t

review_id,user_id,business_id,stars,useful,funny,cool,text,review_timestamp,review_date,review_year,review_month,review_month_name,review_month_abbr,review_day,review_day_of_year,review_dow_number,review_dow_name,review_dow_abbr,review_quarter,review_quarter_name,review_week_of_year,review_year_month,review_year_quarter,review_half_year,review_half_year_name,review_month_start_date,review_month_end_date,is_month_start,is_month_end,review_year_start_date,is_year_start,is_year_end,is_weekend,is_weekday,review_hour,review_minute,review_second,review_time_of_day,review_am_pm,review_hour_12,review_hour_bin_4h,text_length,text_word_count,text_sentence_count,total_votes
XRuN-EMRlEsWxofLF2FA-Q,HjdI42gPOxhHLvGO-iAVTw,rW_9aLoDvGnkEaqJTdhiBA,5,1,0,1,Absolutely the best meat loaf sandwich on the planet. Everything is wonderful really.,2014-10-24T21:47:23.000Z,2014-10-24,2014,10,October,Oct,24,297,6,Friday,Fri,4,Q4,43,2014-10,2014-Q4,2,H2,2014-10-01,2014-10-31,false,false,2014-01-01,false,false,false,true,21,47,23,Night,PM,9,20,86,13,2,2
QAmZ5WI-wi-_0CsnqljnzQ,Nm3pTyfLdW9lauuA0lwTmQ,AsuCvLDKNati-J0J-3q5ZA,5,2,0,2,"I seriously had a moment with these donuts this morning! I like donuts, but never thought I was a ""donut person"".... I can always say no to the box in the kitchen at work... Until Donut refinery! The donuts are unique enjoy to be interesting but it's obvious there's a lot of attention to quality and freshness! I think the donut was so fresh it was slightly warm still after making the journey from Carmel to my castleton office and from the parking lot to the car! I personally enjoyed the two cookies and cream varieties - filled and traditional. I tried a little of a flavor that tasted like Maple and one that had powdered cinnamon sugar on top. All were melt in your mouth delicious. This place is definitely on my donut hot spot list!",2019-12-05T00:55:08.000Z,2019-12-05,2019,12,December,Dec,5,339,5,Thursday,Thu,4,Q4,49,2019-12,2019-Q4,2,H2,2019-12-01,2019-12-31,false,false,2019-01-01,false,false,false,true,0,55,8,Night,AM,12,0,748,137,10,4
HO1KKTSn_oRmBTlEU88iwg,52cDG6Szzdt3wJQ3sKigPA,hI7m7riS39CrgGOhHOLH4w,5,2,0,1,"I'll start by saying this. . . probably the best Indian lamb dish I've had in the past decade. The Rogan Josh was perfect. Tender chunks of lean lamb with a sauce that was completely nuanced with every herb and spice detectable on the palate. Teresa's Chicken Jalfrezie was a solid offering with fresh chicken, vegetables, and a nice heat and viscosity. The portion sizes were generous and came with the offer of additional rice and naan once we finished our original order. Service was a tad on the slow side to start, but was attentive once they got going. While you wouldn't necessarily say the ambiance is fine dining, it's certainly appropriate for a date night. We will definitely be trying their lunch buffet to sample a greater array of their offerings. India Cafe. . .welcome to the rotation.",2017-03-31T00:54:38.000Z,2017-03-31,2017,3,March,Mar,31,90,6,Friday,Fri,1,Q1,13,2017-03,2017-Q1,1,H1,2017-03-01,2017-03-31,false,true,2017-01-01,false,false,false,true,0,54,38,Night,AM,12,0,809,140,14,3
8Jo6E7fAEQiTxen0qMVZ3Q,EvSDm_nNVxGcyy8xf4qmjw,8gV3YUl8uqHnA-wZ3UZ5Dg,5,0,0,1,"I'm a bit surprised to see so many negative reviews, this place is great. It looks like the main issues people have had are slow service and value-for-the-money, so let me give you my perspective on those. In the dozens of times I've been here, I can only think of two times the service was slow. Both times it was due to a large online order coming in shortly before I arrived, so I'd agree that they probably need a new approach to handling those. Even when it's slow though, I would still describe the service as good: the staff are always very friendly and unfailingly helpful. And when they haven't been slammed by an online order, the food comes out extremely quickly. They've gotten my order right every time except one, and I've never asked for something an

In [0]:
# 6.1.4) Schema Drift Detection
# ─────────────────────────────────────────────────────────────────────
# Compare current remodeled schema against expected baseline;
# flag added / removed / retyped columns.

print("=" * 80)
print("6.1.4) SCHEMA DRIFT DETECTION")
print("    Comparing current schema against expected remodeled baseline")
print("=" * 80)

# ══ Expected pre-remodel baseline (original bronze table) ══
baseline_schema = {
    "review_id":   "string",
    "user_id":     "string",
    "business_id": "string",
    "stars":       "int",
    "useful":      "bigint",
    "funny":       "bigint",
    "cool":        "bigint",
    "text":        "string",
    "date": "timestamp",
}

current_schema = dict(df.dtypes)
findings = []

# 1) Added columns
added = {c: current_schema[c] for c in current_schema if c not in baseline_schema}
if added:
    print(f"\n   ⚠️  ADDED columns (not in baseline):")
    for c, t in added.items():
        print(f"      + {c}: {t}")
else:
    print(f"\n   ✅ No new columns added beyond baseline")

# 2) Removed columns
removed = {c: baseline_schema[c] for c in baseline_schema if c not in current_schema}
if removed:
    print(f"\n   ⚠️  REMOVED columns (missing from current):")
    for c, t in removed.items():
        print(f"      - {c}: {t}")
else:
    print(f"   ✅ No columns removed from baseline")

# 3) Retyped columns
retyped = {}
for c in baseline_schema:
    if c in current_schema and baseline_schema[c] != current_schema[c]:
        retyped[c] = (baseline_schema[c], current_schema[c])
if retyped:
    print(f"\n   ⚠️  RETYPED columns:")
    for c, (old_t, new_t) in retyped.items():
        print(f"      ~ {c}: {old_t} → {new_t}")
else:
    print(f"   ✅ No columns changed type from baseline")

for c in added:
    findings.append({"Column": c, "Drift": "ADDED",    "Baseline": "N/A",            "Current": current_schema[c], "Status": "FINDING"})
for c in removed:
    findings.append({"Column": c, "Drift": "REMOVED",  "Baseline": baseline_schema[c], "Current": "N/A",            "Status": "FINDING"})
for c, (old_t, new_t) in retyped.items():
    findings.append({"Column": c, "Drift": "RETYPED",  "Baseline": old_t,             "Current": new_t,            "Status": "FINDING"})
for c in baseline_schema:
    if c in current_schema and baseline_schema[c] == current_schema[c]:
        findings.append({"Column": c, "Drift": "UNCHANGED", "Baseline": baseline_schema[c], "Current": current_schema[c], "Status": "PASS"})

print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

drift_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} columns compared, {drift_count} drift(s) detected")
if drift_count == 0:
    print("✅ No schema drift — current schema matches the expected baseline exactly.")
else:
    print(f"⚠️  {drift_count} schema drift(s) detected — review above.")
print("SCHEMA DRIFT DETECTION COMPLETE")
print("=" * 80)

6.1.4) SCHEMA DRIFT DETECTION
    Comparing current schema against expected remodeled baseline

   ✅ No new columns added beyond baseline
   ✅ No columns removed from baseline

   ⚠️  RETYPED columns:
      ~ stars: int → double
      ~ date: timestamp → string

SUMMARY


Column,Drift,Baseline,Current,Status
stars,RETYPED,int,double,FINDING
date,RETYPED,timestamp,string,FINDING
review_id,UNCHANGED,string,string,PASS
user_id,UNCHANGED,string,string,PASS
business_id,UNCHANGED,string,string,PASS
useful,UNCHANGED,bigint,bigint,PASS
funny,UNCHANGED,bigint,bigint,PASS
cool,UNCHANGED,bigint,bigint,PASS
text,UNCHANGED,string,string,PASS



9 columns compared, 2 drift(s) detected
⚠️  2 schema drift(s) detected — review above.
SCHEMA DRIFT DETECTION COMPLETE


---
#### 📡 6.1.4 Schema Drift Findings — Trace-back to DQ-REVIEW-001 & DQ-REVIEW-002

The **2 drift findings** detected in 6.1.4 are **not new errors** — they are a direct consequence of the same two root-cause issues already documented above. The baseline represents the expected **post-fix** schema (with `stars` as `int` and `date` as `timestamp`), while the current `df` is still the **raw, unfixed** table.

| Drift Type | Column | Baseline (Expected) | Current (Raw) | Root Cause |
| --- | --- | --- | --- | --- |
| **RETYPED** | `stars` | `int` | `double` | **DQ-REVIEW-001** — ordinal 1–5 stored as 64-bit float instead of integer |
| **RETYPED** | `date` | `timestamp` | `string` | **DQ-REVIEW-002** — review timestamps stored as flat string instead of `TimestampType` |

**No columns ADDED or REMOVED** — the raw table has the same 9 column names as the baseline.

**7 columns UNCHANGED**: `review_id`, `user_id`, `business_id`, `useful`, `funny`, `cool`, `text` — all match the baseline exactly.

> **Note:** Once the type fixes from TE.1 (Cell 33) are applied to `df`, both RETYPED findings will resolve to PASS. If the remodel step (TE.2, Cell 34) is then uncommented and re-run, the drift check will show the 37 additional temporal + text feature columns as ADDED.

---


### 6.2) DOMAIN & RANGE CHECKS
*Verify values belong to allowed sets and fall within permitted bounds. All scope variants live here.*

- **Allowed Value Conformance (single column)** — Values belong to the permitted domain/category list.
  - *Example:* `sex` allows only `M`, `F`, `Other`; found `Male`, `1`, `Unknown`.
  - *Sub:* Geographic codes — `country_code` must be ISO-3166 alpha-2.
  - *Sub:* Department names — `department` must match organizational list.
  - *Sub:* Status codes — `employee_status` must be in `{Active, Terminated, Leave}`.

- **Range Conformance (single column)** — Values within min/max bounds.
  - *Example:* `age` must be 0–120; found `-3` or `250`.
  - *Sub:* Salary bands — `salary` within role-specific band.
  - *Sub:* Date bounds — `hire_date` after 1980-01-01.

- **Pattern Conformance** — Values match required format (email, phone, postal code, ID patterns).
  - *Example:* `email` must match RFC-5322; `phone` must match E.164.

- **Impossible Value Detection** — Values that cannot exist regardless of bounds.
  - *Example:* `birth_date` in the future; `age` negative; `death_date` before `birth_date` (single-column view).


In [0]:
# 6.2.1) Allowed Value Conformance (single column)
# ─────────────────────────────────────────────────────────────────────
# Verify that every column with a CLOSED value domain contains only
# permitted values.  For the raw review table (pre-remodel) the only
# true categorical is `stars` (ordinal 1–5).  Vote counts have an
# implicit domain constraint (non-negative integers).
# ─────────────────────────────────────────────────────────────────────

print("=" * 80)
print("6.2.1) ALLOWED VALUE CONFORMANCE  —  Review Table (Pre-Remodel)")
print("    Checking values against closed/allowed sets")
print("=" * 80)

total_rows = df.count()
findings_621 = []

# ══════════════════════════════════════════════════════════════════════
# 1) stars — Yelp star ratings are strictly {1, 2, 3, 4, 5}
#    Column is stored as DOUBLE, so allowed doubles are 1.0–5.0 whole.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 1: stars — Allowed values {1, 2, 3, 4, 5}")

allowed_stars = [1.0, 2.0, 3.0, 4.0, 5.0]
distinct_stars = sorted([r[0] for r in df.select("stars").distinct().collect()])
print(f"   Distinct values found: {distinct_stars}")

bad_stars = df.filter(~col("stars").isin(allowed_stars)).count()
null_stars = df.filter(col("stars").isNull()).count()
status = "PASS" if (bad_stars == 0 and null_stars == 0) else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Non-allowed values: {bad_stars:,}  |  NULLs: {null_stars:,}  — {status}")
if bad_stars > 0:
    bad_vals = [r[0] for r in df.filter(~col("stars").isin(allowed_stars) & col("stars").isNotNull()).select("stars").distinct().limit(10).collect()]
    print(f"      ⚠️  Unexpected values: {bad_vals}")

findings_621.append({
    "Column": "stars",
    "Allowed Values": "{1, 2, 3, 4, 5}",
    "Description": "Yelp ordinal star rating (closed set)",
    "Violations": bad_stars + null_stars,
    "Status": status
})

# Star distribution for context
print("\n   📊 Star rating distribution:")
star_dist = (
    df.groupBy("stars").count()
    .orderBy("stars")
    .collect()
)
for row in star_dist:
    pct = row["count"] / total_rows * 100
    print(f"      ⭐ {int(row['stars'])}: {row['count']:>10,}  ({pct:5.2f}%)")

# ══════════════════════════════════════════════════════════════════════
# 2) Vote counts (useful, funny, cool) — domain: non-negative integers
#    Yelp votes are whole counts; negative or fractional = invalid.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 2: Vote counts — Domain: non-negative integers (≥ 0)")

vote_cols = [("useful", "Useful votes"), ("funny", "Funny votes"), ("cool", "Cool votes")]
for col_name, desc in vote_cols:
    neg_count = df.filter(col(col_name) < 0).count()
    null_count = df.filter(col(col_name).isNull()).count()
    status = "PASS" if (neg_count == 0 and null_count == 0) else "FINDING"
    icon = "✅" if status == "PASS" else "⚠️"
    print(f"   {icon} {col_name}: negatives={neg_count:,}, NULLs={null_count:,} — {status}")
    if neg_count > 0:
        bad_vals = [r[0] for r in df.filter(col(col_name) < 0).select(col_name).distinct().limit(10).collect()]
        print(f"      ⚠️  Negative values: {bad_vals}")

    findings_621.append({
        "Column": col_name,
        "Allowed Values": "≥ 0 (non-negative integer)",
        "Description": desc,
        "Violations": neg_count + null_count,
        "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# 3) Fractional star check — stars SHOULD be whole numbers
#    If stored as double, check for fractional values like 2.5, 3.7
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 3: stars — Whole number integrity (no fractional ratings)")

fractional_stars = df.filter(
    (col("stars") != col("stars").cast("int").cast("double")) &
    col("stars").isNotNull()
).count()
status = "PASS" if fractional_stars == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Fractional star values: {fractional_stars:,} — {status}")
if fractional_stars > 0:
    frac_vals = [r[0] for r in df.filter(col("stars") != col("stars").cast("int").cast("double")).select("stars").distinct().limit(10).collect()]
    print(f"      ⚠️  Fractional values: {frac_vals}")

findings_621.append({
    "Column": "stars (fractional check)",
    "Allowed Values": "Whole numbers only",
    "Description": "Stars must be integer-valued (no 2.5, 3.7 etc.)",
    "Violations": fractional_stars,
    "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 4) text — Must be non-empty (a review must have content)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 4: text — Non-null and non-empty (review must have body)")

null_text = df.filter(col("text").isNull()).count()
empty_text = df.filter(
    (col("text").isNotNull()) & (trim(col("text")) == "")
).count()
status = "PASS" if (null_text == 0 and empty_text == 0) else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} NULL text: {null_text:,}  |  Empty/whitespace-only: {empty_text:,} — {status}")

findings_621.append({
    "Column": "text",
    "Allowed Values": "Non-null, non-empty string",
    "Description": "Review body must contain content",
    "Violations": null_text + empty_text,
    "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 5) date — Must be non-null (every review must have a timestamp)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 5: date — Non-null (every review must be timestamped)")

null_date = df.filter(col("date").isNull()).count()
empty_date = df.filter(
    (col("date").isNotNull()) & (trim(col("date")) == "")
).count()
status = "PASS" if (null_date == 0 and empty_date == 0) else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} NULL dates: {null_date:,}  |  Empty strings: {empty_date:,} — {status}")

findings_621.append({
    "Column": "date",
    "Allowed Values": "Non-null, non-empty string",
    "Description": "Review timestamp string must exist",
    "Violations": null_date + empty_date,
    "Status": status
})

# ══ SUMMARY ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_621_df = pd.DataFrame(findings_621)
display(findings_621_df)

violation_count = sum(1 for f in findings_621 if f["Status"] == "FINDING")
print(f"\n{len(findings_621)} allowed-value checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All columns contain only valid/allowed values.")
else:
    print(f"⚠️  {violation_count} column(s) have value domain violations — review above.")
print("ALLOWED VALUE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

6.2.1) ALLOWED VALUE CONFORMANCE  —  Review Table (Pre-Remodel)
    Checking values against closed/allowed sets

────────────────────────────────────────────────────────────
📌 Check 1: stars — Allowed values {1, 2, 3, 4, 5}
   Distinct values found: [1.0, 2.0, 3.0, 4.0, 5.0]
   ✅ Non-allowed values: 0  |  NULLs: 0  — PASS

   📊 Star rating distribution:
      ⭐ 1:  1,069,561  (15.30%)
      ⭐ 2:    544,240  ( 7.79%)
      ⭐ 3:    691,934  ( 9.90%)
      ⭐ 4:  1,452,918  (20.78%)
      ⭐ 5:  3,231,627  (46.23%)

────────────────────────────────────────────────────────────
📌 Check 2: Vote counts — Domain: non-negative integers (≥ 0)
   ⚠️ useful: negatives=4, NULLs=0 — FINDING
      ⚠️  Negative values: [-1]
   ⚠️ funny: negatives=2, NULLs=0 — FINDING
      ⚠️  Negative values: [-1]
   ⚠️ cool: negatives=1, NULLs=0 — FINDING
      ⚠️  Negative values: [-1]

────────────────────────────────────────────────────────────
📌 Check 3: stars — Whole number integrity (no fractional ratings)
   ✅ 

Column,Allowed Values,Description,Violations,Status
stars,"{1, 2, 3, 4, 5}",Yelp ordinal star rating (closed set),0,PASS
useful,≥ 0 (non-negative integer),Useful votes,4,FINDING
funny,≥ 0 (non-negative integer),Funny votes,2,FINDING
cool,≥ 0 (non-negative integer),Cool votes,1,FINDING
stars (fractional check),Whole numbers only,"Stars must be integer-valued (no 2.5, 3.7 etc.)",0,PASS
text,"Non-null, non-empty string",Review body must contain content,0,PASS
date,"Non-null, non-empty string",Review timestamp string must exist,0,PASS



7 allowed-value checks performed, 3 finding(s)
⚠️  3 column(s) have value domain violations — review above.
ALLOWED VALUE CONFORMANCE CHECK COMPLETE


---
#### 🚨 DQ-REVIEW-003 — Negative Vote Counts

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-003` |
| **Severity** | FINDING (data integrity violation) |
| **Detected In** | Phase 6.2.1 — Allowed Value Conformance (Cell 38) |
| **Also Confirmed In** | Phase 6.2.2 — Range Conformance (Cell 40); Phase 6.2.4 — Impossible Value Detection (Cell 44) |
| **Affected Columns** | `useful` (4 rows), `funny` (2 rows), `cool` (1 row) |
| **Total Affected Rows** | **7** out of 6,990,280 (0.0001%) |
| **Invalid Value** | `-1` (all three columns share the same offending value) |
| **Domain Rule** | Vote counts are community-generated non-negative integers; users can add votes but cannot subtract them |

**Description:** Seven review records contain a vote count of `-1` across the `useful`, `funny`, and `cool` columns. On the Yelp platform, vote counts represent cumulative community endorsements and are monotonically non-negative — a user can mark a review as "useful" but cannot "un-useful" it into a negative count. The value `-1` is therefore physically impossible under normal platform behaviour.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Clamp to zero** (`SET col = 0 WHERE col < 0`) | Preserves the row and all other fields; treats `-1` as a floor violation and resets to the minimum valid value. Simple, reversible, and safe for downstream aggregations (sums, averages). |
| 2 | **Set to NULL** (`SET col = NULL WHERE col < 0`) | Signals "unknown" rather than asserting a value. Downstream code that already handles NULLs (e.g., `AVG` ignores NULLs) will naturally skip these. Useful if the root cause is a data-pipeline bug and the true value is genuinely unknown. |
| 3 | **Delete the 7 rows entirely** | Removes all trace of the corrupted records. Justified only if the rows are confirmed as test/bot artefacts. Risky because it discards the review text and star rating, which are otherwise valid. |
| 4 | **Flag and quarantine** (add `is_vote_anomaly = True`) | Non-destructive; keeps the data for investigation while letting analysts filter it out. Best for audit-trail workflows. |

**Best Logical Solution:** **Option 1 — Clamp to zero.** The 7 affected rows represent 0.0001% of the dataset, and the only anomalous value is `-1` (exactly one unit below the valid floor). Clamping to `0` is the least disruptive fix: it corrects the domain violation without losing the review's text, star rating, or other valid metadata. The magnitude of the error (`-1` vs `0`) has negligible impact on any aggregate statistic, making this the most proportionate and reversible remediation.

---

In [0]:
# 6.2.2) Range Conformance (single column)
# ─────────────────────────────────────────────────────────────────────
# For the raw review table: verify that numeric columns fall within
# physically / business-valid bounds.  Also derives the year from
# the date string for temporal plausibility without requiring the
# remodel step.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import (
    substring, to_timestamp, length as spark_length,
    percentile_approx, min as spark_min, max as spark_max,
    mean as spark_mean, stddev as spark_stddev
)

print("=" * 80)
print("6.2.2) RANGE CONFORMANCE  —  Review Table (Pre-Remodel)")
print("    Checking numeric/temporal values fall within physically valid bounds")
print("=" * 80)

total_rows = df.count()
findings_622 = []

# ══════════════════════════════════════════════════════════════════════
# 1) Numeric column range checks
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Numeric column range checks")

# (column, min, max_or_None, description)
range_checks = [
    # Star rating: Yelp ordinal 1–5
    ("stars",   1, 5,    "Star rating (ordinal 1–5)"),
    # Vote counts: non-negative, no hard ceiling but extreme outliers noted
    ("useful",  0, None, "Useful vote count (non-negative)"),
    ("funny",   0, None, "Funny vote count (non-negative)"),
    ("cool",    0, None, "Cool vote count (non-negative)"),
]

for col_name, min_val, max_val, desc in range_checks:
    conditions = col(col_name) < min_val
    if max_val is not None:
        conditions = conditions | (col(col_name) > max_val)
    out_of_range = df.filter(conditions).count()
    status = "PASS" if out_of_range == 0 else "FINDING"
    icon = "✅" if status == "PASS" else "⚠️"
    range_label = f"[{min_val}–{max_val}]" if max_val is not None else f"[≥ {min_val}]"
    print(f"   {icon} {col_name} {range_label}: {out_of_range:,} violations — {status}")

    if out_of_range > 0:
        bad_vals = [r[0] for r in df.filter(conditions).select(col_name).distinct().limit(10).collect()]
        print(f"      Out-of-range values: {bad_vals}")

    findings_622.append({
        "Column": col_name,
        "Valid Range": range_label,
        "Description": f"{desc} within valid range",
        "Violations": out_of_range,
        "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# 2) Vote count statistical profiling (outlier awareness)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Vote count statistical profiling (detect extreme outliers)")
print("   Vote counts have no hard ceiling, but extreme values may indicate")
print("   bot activity, coordinated voting, or data errors.")

vote_stats = df.select(
    spark_mean("useful").alias("useful_mean"),
    spark_stddev("useful").alias("useful_std"),
    spark_max("useful").alias("useful_max"),
    spark_mean("funny").alias("funny_mean"),
    spark_stddev("funny").alias("funny_std"),
    spark_max("funny").alias("funny_max"),
    spark_mean("cool").alias("cool_mean"),
    spark_stddev("cool").alias("cool_std"),
    spark_max("cool").alias("cool_max"),
).collect()[0]

for vc in ["useful", "funny", "cool"]:
    avg_v = vote_stats[f"{vc}_mean"]
    std_v = vote_stats[f"{vc}_std"]
    max_v = vote_stats[f"{vc}_max"]
    threshold = avg_v + 5 * std_v  # 5-sigma outlier
    extreme_count = df.filter(col(vc) > threshold).count()
    print(f"\n   {vc}:  mean={avg_v:.2f}  std={std_v:.2f}  max={max_v:,}")
    print(f"      5σ threshold: {threshold:,.0f}  |  Rows above: {extreme_count:,}")

    findings_622.append({
        "Column": f"{vc} (outlier)",
        "Valid Range": f"≤ {threshold:,.0f} (5σ)",
        "Description": f"{vc} extreme outlier detection",
        "Violations": extreme_count,
        "Status": "INFO" if extreme_count > 0 else "PASS"
    })

# ══════════════════════════════════════════════════════════════════════
# 3) Temporal range: date string year ∈ [2004, 2026]
#    Yelp was founded Oct 2004; dataset cuts off ~2022.
#    Extract year from the date string directly (no remodel needed).
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Temporal range: date year plausibility [2004 – 2026]")

# Extract 4-digit year from the left of the date string
df_with_year = df.withColumn("_date_year", substring(col("date"), 1, 4).cast("int"))

ts_stats = df_with_year.agg(
    spark_min("_date_year").alias("min_year"),
    spark_max("_date_year").alias("max_year"),
    spark_min("date").alias("min_date"),
    spark_max("date").alias("max_date"),
).collect()[0]
print(f"   Earliest date string: {ts_stats['min_date']}")
print(f"   Latest date string:   {ts_stats['max_date']}")
print(f"   Year range: {ts_stats['min_year']} – {ts_stats['max_year']}")

implausible = df_with_year.filter(
    (col("_date_year") < 2004) | (col("_date_year") > 2026)
).count()
status = "PASS" if implausible == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Implausible years (< 2004 or > 2026): {implausible:,} — {status}")
if implausible > 0:
    bad_years = [r[0] for r in df_with_year.filter((col("_date_year") < 2004) | (col("_date_year") > 2026)).select("_date_year").distinct().limit(10).collect()]
    print(f"      Out-of-range years: {bad_years}")

findings_622.append({
    "Column": "date (year)",
    "Valid Range": "[2004–2026]",
    "Description": "Review year within Yelp's operational period",
    "Violations": implausible,
    "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 4) Text length range
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Text length range (review body must have content)")

text_stats = df.select(
    spark_min(spark_length(col("text"))).alias("min_len"),
    spark_max(spark_length(col("text"))).alias("max_len"),
    spark_mean(spark_length(col("text"))).alias("avg_len"),
).collect()[0]
print(f"   Min length: {text_stats['min_len']:,} chars")
print(f"   Max length: {text_stats['max_len']:,} chars")
print(f"   Avg length: {text_stats['avg_len']:,.0f} chars")

zero_len = df.filter(spark_length(col("text")) < 1).count()
status = "PASS" if zero_len == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Reviews with < 1 char: {zero_len:,} — {status}")

# Also flag suspiciously short reviews (< 10 chars)
very_short = df.filter(
    (spark_length(col("text")) >= 1) & (spark_length(col("text")) < 10)
).count()
print(f"   ℹ️  Very short reviews (1–9 chars): {very_short:,} (informational)")
if very_short > 0:
    short_samples = [r[0] for r in df.filter(spark_length(col("text")) < 10).select("text").limit(5).collect()]
    print(f"      Samples: {short_samples}")

findings_622.append({
    "Column": "text (length)",
    "Valid Range": "[≥ 1 char]",
    "Description": "Review text length must be positive",
    "Violations": zero_len,
    "Status": status
})
findings_622.append({
    "Column": "text (suspiciously short)",
    "Valid Range": "[≥ 10 chars recommended]",
    "Description": "Very short reviews (< 10 chars)",
    "Violations": very_short,
    "Status": "INFO" if very_short > 0 else "PASS"
})

print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_622_df = pd.DataFrame(findings_622)
display(findings_622_df)

finding_count = sum(1 for f in findings_622 if f["Status"] == "FINDING")
info_count = sum(1 for f in findings_622 if f["Status"] == "INFO")
print(f"\n{len(findings_622)} range checks performed, {finding_count} finding(s), {info_count} informational")
if finding_count == 0:
    print("✅ All numeric/temporal values fall within valid bounds.")
else:
    print(f"⚠️  {finding_count} column(s) have range violations — review above.")
if info_count > 0:
    print(f"ℹ️  {info_count} informational flag(s) — not failures, but worth reviewing.")
print("RANGE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

6.2.2) RANGE CONFORMANCE  —  Review Table (Pre-Remodel)
    Checking numeric/temporal values fall within physically valid bounds

────────────────────────────────────────────────────────────
📌 Numeric column range checks
   ✅ stars [1–5]: 0 violations — PASS
   ⚠️ useful [≥ 0]: 4 violations — FINDING
      Out-of-range values: [-1]
   ⚠️ funny [≥ 0]: 2 violations — FINDING
      Out-of-range values: [-1]
   ⚠️ cool [≥ 0]: 1 violations — FINDING
      Out-of-range values: [-1]

────────────────────────────────────────────────────────────
📌 Vote count statistical profiling (detect extreme outliers)
   Vote counts have no hard ceiling, but extreme values may indicate
   bot activity, coordinated voting, or data errors.

   useful:  mean=1.18  std=3.25  max=1,182
      5σ threshold: 17  |  Rows above: 27,845

   funny:  mean=0.33  std=1.69  max=792
      5σ threshold: 9  |  Rows above: 26,919

   cool:  mean=0.50  std=2.17  max=404
      5σ threshold: 11  |  Rows above: 27,873

───────────

Column,Valid Range,Description,Violations,Status
stars,[1–5],Star rating (ordinal 1–5) within valid range,0,PASS
useful,[≥ 0],Useful vote count (non-negative) within valid range,4,FINDING
funny,[≥ 0],Funny vote count (non-negative) within valid range,2,FINDING
cool,[≥ 0],Cool vote count (non-negative) within valid range,1,FINDING
useful (outlier),≤ 17 (5σ),useful extreme outlier detection,27845,INFO
funny (outlier),≤ 9 (5σ),funny extreme outlier detection,26919,INFO
cool (outlier),≤ 11 (5σ),cool extreme outlier detection,27873,INFO
date (year),[2004–2026],Review year within Yelp's operational period,0,PASS
text (length),[≥ 1 char],Review text length must be positive,0,PASS
text (suspiciously short),[≥ 10 chars recommended],Very short reviews (< 10 chars),156,INFO



10 range checks performed, 3 finding(s), 4 informational
⚠️  3 column(s) have range violations — review above.
ℹ️  4 informational flag(s) — not failures, but worth reviewing.
RANGE CONFORMANCE CHECK COMPLETE




---

#### ⚠️ DQ-REVIEW-004 — Extreme Vote Count Outliers

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-005` |
| **Severity** | INFO (advisory — not a data error, but a quality signal) |
| **Detected In** | Phase 6.2.2 — Range Conformance, Statistical Profiling (Cell 40) |
| **Affected Columns** | `useful` (27,845 rows), `funny` (26,919 rows), `cool` (27,873 rows) |
| **Threshold** | 5σ above the mean for each column |
| **Observed Maximums** | `useful` = 1,182 &nbsp;\|&nbsp; `funny` = 792 &nbsp;\|&nbsp; `cool` = 404 |

**Description:** Approximately 27K reviews per vote column exceed the 5-sigma statistical threshold (`useful` > 17, `funny` > 9, `cool` > 11). While these are not *invalid* values — Yelp does not cap vote counts — such extremes may indicate viral reviews, bot-driven voting, coordinated campaigns, or long-tenured highly-visible reviews that organically accumulated votes over many years.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Retain as-is, document for analysts** | Extreme votes are a legitimate feature of the platform. Analysts can apply their own thresholds during modelling. No data is lost. |
| 2 | **Winsorize at the 99th percentile** during feature engineering | Caps extreme values to reduce skew in ML features (e.g., `useful_capped`). Preserves the original column alongside the capped version. |
| 3 | **Log-transform** vote columns during modelling | Compresses the heavy tail (`log1p(useful)`) without altering the stored data. Standard practice for count features. |
| 4 | **Add a boolean flag** `is_vote_outlier` | Non-destructive labelling for downstream filtering or stratified analysis. |

**Best Logical Solution:** **Option 1 — Retain as-is and document.** At the pre-cleaning discovery stage, these are informational observations, not errors. The correct place to handle vote skew is during feature engineering (Silver/Gold layer), where either winsorization or log-transform can be applied based on the specific modelling task. Altering the Bronze data would lose fidelity.

---

#### ℹ️ DQ-REVIEW-005 — Very Short Reviews (1–9 characters)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-006` |
| **Severity** | INFO (advisory — quality signal for NLP pipelines) |
| **Detected In** | Phase 6.2.2 — Range Conformance, Text Length (Cell 40) |
| **Affected Rows** | **156** out of 6,990,280 (0.002%) |
| **Sample Values** | `*`, `S`, `Au jus`, `-----`, `Thank you` |
| **Text Length Stats** | min = 1 char, max = 5,000 chars, avg = 568 chars |

**Description:** 156 reviews have a text body shorter than 10 characters. While technically non-empty (passing the ≥ 1 char rule), these ultra-short texts carry minimal semantic content for NLP tasks such as sentiment analysis, topic modelling, or review summarisation. Some appear to be single punctuation marks (`*`, `-----`) or fragmentary phrases (`S`, `Au jus`).

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag with `is_short_review = True`** (threshold < 10 chars) | Non-destructive; lets NLP pipelines exclude them without losing the data from aggregation queries. |
| 2 | **Exclude during NLP preprocessing** (Silver layer filter) | Standard NLP practice: minimum token count filters (e.g., ≥ 3 words) are applied during text cleaning. No Bronze modification needed. |
| 3 | **Delete the 156 rows** | Overly aggressive for 0.002% of data. The star rating and vote metadata are still valid even if the text is too short for NLP. |

**Best Logical Solution:** **Option 2 — Exclude during NLP preprocessing at the Silver layer.** These 156 rows are not data errors; they are legitimate but low-information reviews. Minimum token/character filters are a standard NLP preprocessing step and should be applied where text analysis occurs, not at the Bronze/discovery stage.

---

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# ROOT CAUSE ANALYSIS — Short Text Reviews (< 10 chars)
# ═══════════════════════════════════════════════════════════════════════════════
# Objective: Determine whether the 156 very short reviews are placeholder /
# test artefacts (removable for ML) or genuine, meaningful micro-reviews.
# ═══════════════════════════════════════════════════════════════════════════════

from pyspark.sql.functions import (
    col, length as spark_length, trim, lower, regexp_replace,
    when, lit, count as _count, mean as spark_mean,
    min as spark_min, max as spark_max, year as spark_year,
    to_timestamp
)
import pandas as pd

print("=" * 80)
print("ROOT CAUSE ANALYSIS — Short Text Reviews (< 10 chars)")
print("    Classifying short reviews as placeholder/noise vs genuine content")
print("=" * 80)

# ── 1) Isolate the short-text cohort ─────────────────────────────────
short_df = df.filter(spark_length(col("text")) < 10)
short_count = short_df.count()
total_rows = df.count()
print(f"\n📌 Cohort: {short_count:,} reviews with text length < 10 chars")
print(f"   ({short_count / total_rows * 100:.4f}% of {total_rows:,} total reviews)")

# ── 2) Show ALL short reviews with metadata ──────────────────────────
print(f"\n{'─'*60}")
print("📌 Full listing of short reviews with metadata")
display(
    short_df.select(
        "review_id", "user_id", "business_id", "stars", "date",
        "useful", "funny", "cool", "text"
    ).orderBy(spark_length(col("text")))
)

# ── 3) Content classification ────────────────────────────────────────
#    Classify each short review into categories to understand nature
print(f"\n{'─'*60}")
print("📌 Content classification")

# Strip whitespace and compute a cleaned version for classification
short_classified = short_df.withColumn(
    "text_trimmed", trim(col("text"))
).withColumn(
    "text_len", spark_length(trim(col("text")))
).withColumn(
    # Remove all non-alphanumeric chars; if empty → punctuation-only
    "alpha_only", regexp_replace(trim(col("text")), r"[^a-zA-Z0-9]", "")
).withColumn(
    "category",
    when(spark_length(col("alpha_only")) == 0, lit("PUNCTUATION_ONLY"))
    .when(spark_length(trim(col("text"))) == 1, lit("SINGLE_CHAR"))
    .when(
        lower(trim(col("text"))).isin(
            "test", "testing", "asdf", "aaa", "bbb", "xxx",
            "n/a", "na", "none", "null", ".", "..", "..."
        ),
        lit("PLACEHOLDER")
    )
    .otherwise(lit("MICRO_REVIEW"))
)

category_summary = (
    short_classified
    .groupBy("category")
    .agg(
        _count("*").alias("count"),
        spark_mean("stars").alias("avg_stars"),
        spark_mean("useful").alias("avg_useful"),
        spark_mean("funny").alias("avg_funny"),
        spark_mean("cool").alias("avg_cool"),
    )
    .orderBy(col("count").desc())
)
print("\n   Category breakdown with average engagement:")
display(category_summary)

# ── 4) Sample texts per category ─────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Sample texts per category")
for row in category_summary.collect():
    cat = row["category"]
    samples = (
        short_classified
        .filter(col("category") == cat)
        .select("text_trimmed")
        .distinct()
        .limit(15)
        .collect()
    )
    sample_list = [r["text_trimmed"] for r in samples]
    print(f"\n   [{cat}] ({row['count']} reviews):")
    for s in sample_list:
        print(f"      • \"{s}\"")

# ── 5) Engagement signal analysis ────────────────────────────────────
#    Do short reviews receive any community votes?  Non-zero votes
#    suggest the community found them meaningful.
print(f"\n{'─'*60}")
print("📌 Engagement signal analysis")

has_votes = short_df.filter(
    (col("useful") > 0) | (col("funny") > 0) | (col("cool") > 0)
).count()
no_votes = short_count - has_votes
print(f"   With ≥ 1 community vote:  {has_votes:,}  ({has_votes / short_count * 100:.1f}%)")
print(f"   Zero community votes:     {no_votes:,}  ({no_votes / short_count * 100:.1f}%)")

# ── 6) Star rating distribution ──────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Star rating distribution (short reviews vs overall)")

short_star_dist = (
    short_df.groupBy("stars")
    .agg(_count("*").alias("short_count"))
    .orderBy("stars")
)
overall_star_dist = (
    df.groupBy("stars")
    .agg(_count("*").alias("overall_count"))
    .orderBy("stars")
)
star_comparison = (
    short_star_dist
    .join(overall_star_dist, "stars", "outer")
    .withColumn("short_pct", col("short_count") / lit(short_count) * 100)
    .withColumn("overall_pct", col("overall_count") / lit(total_rows) * 100)
    .orderBy("stars")
)
display(star_comparison)

# ── 7) Temporal distribution ─────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Temporal distribution of short reviews")

from pyspark.sql.functions import substring
short_by_year = (
    short_df
    .withColumn("review_year", substring(col("date"), 1, 4).cast("int"))
    .groupBy("review_year")
    .agg(_count("*").alias("count"))
    .orderBy("review_year")
)
display(short_by_year)

# ── 8) User repeat analysis ──────────────────────────────────────────
#    Do any users repeatedly post very short reviews?  That signals
#    low-effort or bot/placeholder behaviour.
print(f"\n{'─'*60}")
print("📌 Repeat short-review users")

repeat_users = (
    short_df.groupBy("user_id")
    .agg(_count("*").alias("short_review_count"))
    .filter(col("short_review_count") > 1)
    .orderBy(col("short_review_count").desc())
)
repeat_user_count = repeat_users.count()
print(f"   Users with > 1 short review: {repeat_user_count}")
if repeat_user_count > 0:
    display(repeat_users.limit(10))
else:
    print("   ✅ No repeat offenders — short reviews come from distinct users")

# ══════════════════════════════════════════════════════════════════════
# VERDICT
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'='*80}")
print("ROOT CAUSE ANALYSIS VERDICT")
print("=" * 80)

# Collect category counts for the verdict
cat_counts = {r["category"]: r["count"] for r in category_summary.collect()}
noise_count = cat_counts.get("PUNCTUATION_ONLY", 0) + cat_counts.get("SINGLE_CHAR", 0) + cat_counts.get("PLACEHOLDER", 0)
micro_count = cat_counts.get("MICRO_REVIEW", 0)

print(f"\n   NOISE (punctuation-only + single-char + placeholder):  {noise_count:,} reviews")
print(f"   MICRO-REVIEWS (genuine short text):                    {micro_count:,} reviews")
print(f"   Total short reviews:                                  {short_count:,} reviews")

print(f"\n   📋 ML Recommendation:")
if noise_count > 0:
    print(f"      ❌ REMOVE {noise_count:,} noise records — these are punctuation, single")
    print(f"         characters, or placeholders with no semantic value for NLP/ML.")
if micro_count > 0:
    print(f"      ⚠️  REVIEW {micro_count:,} micro-reviews — these contain real words")
    print(f"         (e.g. 'Thank you', 'Au jus') but are too short for meaningful")
    print(f"         NLP feature extraction. Consider:")
    print(f"         • Keeping for sentiment analysis (short but opinionated)")
    print(f"         • Excluding from topic modelling / text embeddings")
    print(f"         • Flagging with a 'short_text' indicator column")
print(f"\n   Overall: {short_count:,} / {total_rows:,} = {short_count / total_rows * 100:.4f}%")
print(f"   Impact is minimal but cleaning improves NLP pipeline quality.")
print("=" * 80)

ROOT CAUSE ANALYSIS — Short Text Reviews (< 10 chars)
    Classifying short reviews as placeholder/noise vs genuine content

📌 Cohort: 156 reviews with text length < 10 chars
   (0.0022% of 6,990,280 total reviews)

────────────────────────────────────────────────────────────
📌 Full listing of short reviews with metadata


review_id,user_id,business_id,stars,date,useful,funny,cool,text
Sywlx9_vZ-AUDdGQ_S4LiA,1HAGg_5G7oQV9hwO3EQBYA,_GnEXTiDmy5lT3yxVwbZGw,1.0,2017-09-03 18:37:44,0,0,0,.
rJ5RGQLwzIfpnU3XBGT6DQ,jr48Tk3dgMDw5hZuc3aT3Q,tuDmwyx080IEpHsV78tC5A,5.0,2017-01-27 01:38:29,1,0,1,N
Uwa1RPOnWYjDshVqcNEzdQ,5iJ4Pgpp__xzTCKsM3c5Mw,89COhCW-cAD5-fTXPfXYTw,1.0,2014-10-22 03:06:29,0,0,0,.
WkCvXyIDbCK3fY5SSR9uPg,ZvOkC-fWP8zbZi76i6IB6A,TpfKgu0UN4BBzyL92nihTQ,3.0,2017-01-28 19:13:27,0,0,0,Q
gieC2jHQDii3-BuYkfOu7g,W7JGDSHc07E7sFy6-Hr_CQ,zlNtAPz4GvSUiljQ-D3cvQ,5.0,2014-09-15 10:21:15,0,0,0,K
kgoGvfik0cS5knATzRYLAg,XWbRA7dIA8Ib8qm0NUNa6g,V6mqd8fC6-y_KKsfJTQyIA,3.0,2013-04-09 05:03:19,1,0,0,.
omrXRoV1pXIWPXmtIITcyA,R2YyA4uaVGNyNk10S6ZnTw,dfAotOzMI8i4K2wFPPB3Hg,1.0,2014-08-26 02:27:07,4,2,0,.
GqX_2GSqmLIeEb0bsW2C9w,24ayJLOQI1r3yls5skJK-A,BJOGo_upuBElDT_xOaurIA,2.0,2011-01-26 04:24:59,0,0,0,*
AUwrGR8B7Ul83j4XGIyG7Q,WZchojYjIa2-Tedq9zd3cg,hRskO0RDRW3Cq3k7_Kggcg,1.0,2015-08-26 03:45:52,0,0,0,.
j1Q_NKfZlguBBjEqU9Fh2A,_L3N3ur_5OC8Zlrsuf7wtQ,FoYjHEoY1TJU6g4xnq_x_w,4.0,2017-07-13 22:34:18,0,0,0,L



────────────────────────────────────────────────────────────
📌 Content classification

   Category breakdown with average engagement:


category,count,avg_stars,avg_useful,avg_funny,avg_cool
MICRO_REVIEW,110,3.309090909090909,0.6272727272727273,0.4090909090909091,0.3
PUNCTUATION_ONLY,30,2.3666666666666667,0.9333333333333333,0.5333333333333333,0.43333333333333335
SINGLE_CHAR,16,2.5625,0.625,0.6875,0.3125



────────────────────────────────────────────────────────────
📌 Sample texts per category

   [MICRO_REVIEW] (110 reviews):
      • "For ortho"
      • "Au jus"
      • "Thank you"
      • "Nope."
      • "Dont go"
      • "Bla"
      • "Ok"
      • "the best."
      • "Nice"
      • "Great"
      • "Blah."
      • "not bad."
      • "stop by!"
      • "NOM :)"
      • "Tbd"

   [PUNCTUATION_ONLY] (30 reviews):
      • "-----"
      • "*"
      • ".."
      • "......."
      • "."
      • "........"
      • ":)"
      • "..."
      • "饺子还不错"
      • ":("
      • "!!"
      • "沸腾水煮鱼"
      • "......"

   [SINGLE_CHAR] (16 reviews):
      • "Y"
      • "S"
      • "X"
      • "N"
      • "M"
      • "Q"
      • "K"
      • "L"
      • "H"
      • "O"
      • "D"

────────────────────────────────────────────────────────────
📌 Engagement signal analysis
   With ≥ 1 community vote:  60  (38.5%)
   Zero community votes:     96  (61.5%)

───────────────────────────────────────────────────────

...


────────────────────────────────────────────────────────────
📌 Temporal distribution of short reviews


review_year,count
2008,1
2009,3
2010,2
2011,16
2012,14
2013,27
2014,23
2015,29
2016,16
2017,15



────────────────────────────────────────────────────────────
📌 Repeat short-review users
   Users with > 1 short review: 8


user_id,short_review_count
ouODopBKF3AqfCkuQEnrDg,5
SDXRbMcnXeiPUAbHf3CQGw,3
CS6mIboxyuYyVMAHWbyptw,2
0YyF7OrD2VEUVUiYmKZ8MA,2
l_5iaa0MREofT7KIZIH4ug,2
Ase_kJIYuT6yOsqqVPuWUA,2
f_CPBJKFAML1XnHxUnf0hg,2
oOsjFifsg7-owhmL-GLssA,2



ROOT CAUSE ANALYSIS VERDICT

   NOISE (punctuation-only + single-char + placeholder):  46 reviews
   MICRO-REVIEWS (genuine short text):                    110 reviews
   Total short reviews:                                  156 reviews

   📋 ML Recommendation:
      ❌ REMOVE 46 noise records — these are punctuation, single
         characters, or placeholders with no semantic value for NLP/ML.
      ⚠️  REVIEW 110 micro-reviews — these contain real words
         (e.g. 'Thank you', 'Au jus') but are too short for meaningful
         NLP feature extraction. Consider:
         • Keeping for sentiment analysis (short but opinionated)
         • Excluding from topic modelling / text embeddings
         • Flagging with a 'short_text' indicator column

   Overall: 156 / 6,990,280 = 0.0022%
   Impact is minimal but cleaning improves NLP pipeline quality.



---

#### 🔍 RCA Documentation — Short Text Reviews (< 10 chars): Useless vs Useful Classification

**Source:** Root Cause Analysis — Cell 42 &nbsp;|&nbsp; **Finding:** DQ-REVIEW-006 &nbsp;|&nbsp; **Cohort:** 156 / 6,990,280 (0.0022%)

##### Classification Summary

| Category | Count | % of Short | Avg Stars | Avg Useful | Avg Funny | Avg Cool | ML Verdict |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| **MICRO_REVIEW** (useful) | 110 | 70.5% | 3.31 | 0.63 | 0.41 | 0.30 | ⚠️ Review per task |
| **PUNCTUATION_ONLY** (useless) | 30 | 19.2% | 2.37 | 0.93 | 0.53 | 0.43 | ❌ Remove |
| **SINGLE_CHAR** (useless) | 16 | 10.3% | 2.56 | 0.63 | 0.69 | 0.31 | ❌ Remove |

> ⚠️ **Correction:** 2 of the 30 "PUNCTUATION_ONLY" reviews are genuine **Chinese-language reviews** misclassified by the ASCII-only regex (`[a-zA-Z0-9]` strips CJK characters). **True noise = 44, true useful = 112.**

---

##### ❌ USELESS — Noise Text (44 true noise reviews)

These carry zero semantic value for NLP/ML and should be removed or nullified during Silver-layer cleaning.

**Punctuation-Only (28 true noise reviews):**

| Text | Count | Stars Range | Note |
| --- | ---: | --- | --- |
| `.` | 9 | 1–4 | Single period — most common noise pattern |
| `..` | 4 | 1–4 | Double period |
| `...` | 5 | 1–2 | Ellipsis |
| `......` | 1 | 4 | Extended dots |
| `.......` | 1 | 5 | Extended dots |
| `........` | 1 | 4 | Extended dots |
| `*` | 1 | 2 | Asterisk placeholder |
| `-----` | 1 | 1 | Dash separator |
| `:)` | 2 | 3, 5 | Emoticon only |
| `:(` | 1 | 1 | Emoticon only |
| `!!` | 2 | 2, 4 | Exclamation only |

**Single Character (16 reviews):**

| Text | Count | &nbsp; | Text | Count |
| --- | ---: | --- | --- | ---: |
| `Y` | 2 | | `S` | 1 |
| `N` | 2 | | `X` | 1 |
| `O` | 2 | | `M` | 1 |
| `H` | 2 | | `L` | 1 |
| `K` | 2 | | `Q` | 1 |
| | | | `D` | 1 |

---

##### ⚠️ MISCLASSIFIED — Chinese-Language Reviews (2 reviews → reclassify as useful)

| Text | Translation | Stars | Useful | Funny | Cool |
| --- | --- | ---: | ---: | ---: | ---: |
| `沸腾水煮鱼` | "Boiling water fish" (Sichuan dish name) | 4 | 0 | 0 | 0 |
| `饺子还不错` | "Dumplings are not bad" | 4 | 0 | 0 | 0 |

> Both are genuine food reviews in Chinese, correctly rated with plausible 4-star ratings. Misclassified because the ASCII-only regex `[a-zA-Z0-9]` strips CJK characters entirely.

---

##### ✅ USEFUL — Genuine Micro-Reviews (110 classified; 112 corrected)

Real words carrying sentiment, business status, or contextual signals.

**🏢 Business Status Indicators (13 reviews):**
`Closed` ×4 &nbsp;·&nbsp; `closed` ×4 &nbsp;·&nbsp; `Closed.` ×2 &nbsp;·&nbsp; `Closed?` ×1 &nbsp;·&nbsp; `Closed :(` ×1 &nbsp;·&nbsp; `closed.` ×1

> Valuable for business operational intelligence — flags businesses that have permanently closed.

**👍 Positive Sentiment (\~55 reviews):**
`Great` ×3 · `Great!` ×2 · `Awesome` ×3 · `Good food` ×3 · `Yum` ×3 · `Awesome!` · `awesome!` · `Wow` · `Wow!` · `Amazing!` · `Excellent` · `Love it!` · `Loved it!` · `the best.` · `Heaven.` · `So good!` · `Good` · `Dank food` · `Dank` · `Rules` · `Always!` · `Yeah` · `Nice` · `Solid.` · `Fresh!` · `Thank you` · `So money!` · `Huge!` · `great job` · `Muy bueno` · `BLT!!!` · `stop by!` · `NOM :)` · `:) good` · `HOME (-:` · `im home!` · `Noms` · `Yum.` · `Mmm` · `pas mal` · `So` · `#1` · `Okay` · `Ok` · `ok` · `not bad.`

**👎 Negative Sentiment (\~13 reviews):**
`Nope.` · `No` · `No!` · `Boo` · `Booo` · `Boooo` · `Awful` · `Horrible!` · `Not happy` · `Sorry bad` · `Dont go` · `pure scum` · `no Bueno`

**😐 Neutral / Ambiguous (\~15 reviews):**
`Meh...` · `meh...` · `So-so.` · `Blah.` · `Bla` · `Eh.` · `typical` · `South` · `Done` · `Salut` · `For ortho` · `Au jus` · `Wendy's` · `Nancy L` · `fun af`

**📝 Meta / Placeholder-like (\~14 reviews):**
`Edited` ×2 · `Removed` · `Update` · `See below` · `Tbd` · `Hhjjgfhh` · `Ii` · `:-D` · `Lll` ×3 · `"Yo` · `Thanks`

> ⚠️ Several items in this group (`Tbd`, `Hhjjgfhh`, `Edited`, `Removed`, `Lll` ×3, `Ii`) are arguably **gibberish or editorial artefacts** rather than genuine reviews, raising the effective noise count by \~9 additional records.

---

##### 📊 Supporting Signals

| Signal | Detail |
| --- | --- |
| **Engagement** | 60 (38.5%) received ≥ 1 community vote; 96 (61.5%) had zero votes |
| **Star skew** | 1★ over-represented: 31.4% vs 15.3% overall; 5★ under-represented: 26.9% vs 46.2% |
| **Temporal peak** | 2013–2015 concentrated (50.6% of all short reviews); near-zero after 2018 |
| **Repeat users** | 8 users posted > 1 short review; top offender posted 5 |

##### 🏁 Corrected Final Verdict

| Category | Count | Recommendation |
| --- | ---: | --- |
| **True Noise** (punctuation + single chars) | 44 | ❌ Remove — zero semantic value for any ML task |
| **Chinese Reviews** (misclassified) | 2 | ✅ Keep — genuine non-English micro-reviews |
| **Genuine Micro-Reviews** | 110 | ⚠️ Task-dependent: keep for sentiment; flag/exclude for topic models & embeddings |
| → _of which arguably placeholder_ | _\~9_ | _Consider reclassifying as noise (Tbd, Hhjjgfhh, Edited, Removed, Lll×3, Ii, :-D)_ |
| **Total** | **156** | |

---

In [0]:
# 6.2.3) Pattern Conformance
# ─────────────────────────────────────────────────────────────────────
# Verify that string-valued columns match their expected structural
# format.  For the raw review table: three 22-char Yelp hashes,
# a date string in "yyyy-MM-dd HH:mm:ss" format, and a non-trivial
# review body.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import to_timestamp, length as spark_length

print("=" * 80)
print("6.2.3) PATTERN CONFORMANCE  —  Review Table (Pre-Remodel)")
print("    Checking structural format of string-valued columns")
print("=" * 80)

total_rows = df.count()
findings_623 = []
# ══════════════════════════════════════════════════════════════════════
# 1) ID columns — 22-char alphanumeric Yelp hash
#    Yelp IDs are Base62-like 22-character hashes with [a-zA-Z0-9_-]
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Checks 1–3: ID columns — 22-char alphanumeric hash")
print("   Expected pattern: ^[a-zA-Z0-9_-]{22}$")

id_pattern = "^[a-zA-Z0-9_-]{22}$"
id_cols = [
    ("review_id",   "Review ID (primary key)"),
    ("user_id",     "User ID (foreign key to yelp_user)"),
    ("business_id", "Business ID (foreign key to yelp_business)"),
]

for col_name, desc in id_cols:
    bad_count = df.filter(
        col(col_name).isNull() | ~col(col_name).rlike(id_pattern)
    ).count()
    status = "PASS" if bad_count == 0 else "FINDING"
    icon = "✅" if status == "PASS" else "⚠️"
    print(f"   {icon} {col_name}: {bad_count:,} / {total_rows:,} non-conforming — {status}")
    if bad_count > 0:
        bad_samples = [r[0] for r in df.filter(col(col_name).isNull() | ~col(col_name).rlike(id_pattern)).select(col_name).distinct().limit(10).collect()]
        print(f"      Examples: {bad_samples}")
    findings_623.append({
        "Column": col_name,
        "Pattern": id_pattern,
        "Description": desc,
        "Violations": bad_count,
        "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# 4) date — "yyyy-MM-dd HH:mm:ss" string format
#    Must parse cleanly to a timestamp (no partial dates, no garbage).
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 4: date — 'yyyy-MM-dd HH:mm:ss' string format")

date_pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
bad_date_format = df.filter(
    col("date").isNull() | ~col("date").rlike(date_pattern)
).count()
status = "PASS" if bad_date_format == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Regex format violations: {bad_date_format:,} — {status}")
if bad_date_format > 0:
    bad_dates = [r[0] for r in df.filter(col("date").isNull() | ~col("date").rlike(date_pattern)).select("date").distinct().limit(10).collect()]
    print(f"      Examples: {bad_dates}")

findings_623.append({
    "Column": "date (format)",
    "Pattern": date_pattern,
    "Description": "yyyy-MM-dd HH:mm:ss timestamp string",
    "Violations": bad_date_format,
    "Status": status
})

# Also verify the string actually parses to a valid timestamp
print(f"\n   🔍  Semantic parse check (cast to timestamp, count NULLs):")
null_after_cast = df.withColumn(
    "_ts_check", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")
).filter(col("_ts_check").isNull() & col("date").isNotNull()).count()

status2 = "PASS" if null_after_cast == 0 else "FINDING"
icon2 = "✅" if status2 == "PASS" else "⚠️"
print(f"   {icon2} Failed timestamp casts (format ok but invalid date): {null_after_cast:,} — {status2}")
if null_after_cast > 0:
    parse_fails = [r[0] for r in df.withColumn("_ts_check", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")).filter(col("_ts_check").isNull() & col("date").isNotNull()).select("date").limit(10).collect()]
    print(f"      Examples: {parse_fails}")

findings_623.append({
    "Column": "date (parseable)",
    "Pattern": "Valid calendar date + time",
    "Description": "String parses to a valid timestamp (no Feb 30 etc.)",
    "Violations": null_after_cast,
    "Status": status2
})

# ══════════════════════════════════════════════════════════════════════
# 5) text — Non-trivial review content
#    Check for whitespace-only, control chars, or suspiciously short.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 5: text — Review body content quality")

# a) Whitespace-only reviews
ws_only = df.filter(trim(col("text")) == "").count()
icon = "✅" if ws_only == 0 else "⚠️"
print(f"   {icon} Whitespace-only reviews: {ws_only:,}")

# b) Reviews with embedded NUL / control characters (\x00-\x1F except \n\r\t)
control_char_count = df.filter(
    col("text").rlike(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
).count()
icon = "✅" if control_char_count == 0 else "⚠️"
print(f"   {icon} Reviews with control characters: {control_char_count:,}")

# c) Reviews that look like placeholder / test data
placeholder_count = df.filter(
    lower(trim(col("text"))).isin(
        "test", "n/a", "na", "none", ".", "..", "...", "xxx", "asdf", "lorem ipsum"
    )
).count()
icon = "✅" if placeholder_count == 0 else "⚠️"
print(f"   {icon} Placeholder/test-data reviews: {placeholder_count:,}")

total_text_issues = ws_only + control_char_count + placeholder_count
status = "PASS" if total_text_issues == 0 else "FINDING"
findings_623.append({
    "Column": "text (content quality)",
    "Pattern": "Non-trivial prose",
    "Description": "No whitespace-only, control chars, or placeholder text",
    "Violations": total_text_issues,
    "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 6) ID length distribution (verify uniform 22-char length)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 6: ID length uniformity (all IDs should be exactly 22 chars)")

for col_name, desc in id_cols:
    len_dist = (
        df.withColumn("_len", spark_length(col(col_name)))
        .groupBy("_len").count()
        .orderBy("_len")
        .collect()
    )
    lengths = {r["_len"]: r["count"] for r in len_dist}
    non_22 = sum(v for k, v in lengths.items() if k != 22)
    icon = "✅" if non_22 == 0 else "⚠️"
    print(f"   {icon} {col_name}: lengths found = {lengths}  |  non-22-char: {non_22:,}")
    findings_623.append({
        "Column": f"{col_name} (length)",
        "Pattern": "Exactly 22 characters",
        "Description": f"{desc} — uniform length",
        "Violations": non_22,
        "Status": "PASS" if non_22 == 0 else "FINDING"
    })

# ══ SUMMARY ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_623_df = pd.DataFrame(findings_623)
display(findings_623_df)

violation_count = sum(1 for f in findings_623 if f["Status"] == "FINDING")
print(f"\n{len(findings_623)} pattern checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All values conform to their expected patterns and formats.")
else:
    print(f"⚠️  {violation_count} check(s) detected pattern mismatches — review above.")
print("PATTERN CONFORMANCE CHECK COMPLETE")
print("=" * 80)

6.2.3) PATTERN CONFORMANCE  —  Review Table (Pre-Remodel)
    Checking structural format of string-valued columns

────────────────────────────────────────────────────────────
📌 Checks 1–3: ID columns — 22-char alphanumeric hash
   Expected pattern: ^[a-zA-Z0-9_-]{22}$
   ✅ review_id: 0 / 6,990,280 non-conforming — PASS
   ✅ user_id: 0 / 6,990,280 non-conforming — PASS
   ✅ business_id: 0 / 6,990,280 non-conforming — PASS

────────────────────────────────────────────────────────────
📌 Check 4: date — 'yyyy-MM-dd HH:mm:ss' string format
   ✅ Regex format violations: 0 — PASS

   🔍  Semantic parse check (cast to timestamp, count NULLs):
   ✅ Failed timestamp casts (format ok but invalid date): 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 5: text — Review body content quality
   ✅ Whitespace-only reviews: 0
   ✅ Reviews with control characters: 0
   ⚠️ Placeholder/test-data reviews: 18

────────────────────────────────────────────────────────────
📌 Check 

Column,Pattern,Description,Violations,Status
review_id,^[a-zA-Z0-9_-]{22}$,Review ID (primary key),0,PASS
user_id,^[a-zA-Z0-9_-]{22}$,User ID (foreign key to yelp_user),0,PASS
business_id,^[a-zA-Z0-9_-]{22}$,Business ID (foreign key to yelp_business),0,PASS
date (format),^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$,yyyy-MM-dd HH:mm:ss timestamp string,0,PASS
date (parseable),Valid calendar date + time,String parses to a valid timestamp (no Feb 30 etc.),0,PASS
text (content quality),Non-trivial prose,"No whitespace-only, control chars, or placeholder text",18,FINDING
review_id (length),Exactly 22 characters,Review ID (primary key) — uniform length,0,PASS
user_id (length),Exactly 22 characters,User ID (foreign key to yelp_user) — uniform length,0,PASS
business_id (length),Exactly 22 characters,Business ID (foreign key to yelp_business) — uniform length,0,PASS



9 pattern checks performed, 1 finding(s)
⚠️  1 check(s) detected pattern mismatches — review above.
PATTERN CONFORMANCE CHECK COMPLETE


---
#### ⚠️ DQ-REVIEW-006 — Placeholder/Test-Data Reviews

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-004` |
| **Severity** | FINDING (data quality issue — semi-artificial records) |
| **Detected In** | Phase 6.2.3 — Pattern Conformance (Cell 42) |
| **Affected Rows** | **18** out of 6,990,280 (0.00026%) |
| **Pattern** | Placeholder content or "test" markers in review text |

**Description:** Detected 18 reviews containing placeholder or test data in the `text` column. Examples include generic strings, test artefacts, or boilerplate indicating synthetic or QA-originated reviews. While these pass basic domain and range checks (non-empty, parseable dates, valid IDs), their content is not genuine user-generated feedback and may distort downstream NLP or engagement metrics.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag with `is_placeholder_review = True`** | Non-destructive; lets downstream quality pipelines ignore or stratify without removal. |
| 2 | **Exclude from NLP and engagement modelling** | Treat flagged values as uninformative for analytics. Preserves all linked metadata. |
| 3 | **Delete the 18 rows** | Removes all test-artefact data, but irreversibly discards all review and metadata (votes, stars, business/user linkage).
Risk: overkill at 0.00026%. |

**Best Logical Solution:** **Option 2 — Exclude from downstream analytics.** These placeholder/test records are extremely rare (0.00026%) and may be valuable for audit-trail consistency. Removing them from sentiment/NLP and engagement pipelines prevents quality contamination while preserving a complete record of ingest history. Adding an explicit flag further supports analytical transparency and reporting.

---

In [0]:
# 6.2.4) Impossible Value Detection
# ─────────────────────────────────────────────────────────────────────
# Values that CANNOT physically exist regardless of configurable bounds.
# Works directly on the raw review table (9 columns, pre-remodel).
# Uses to_timestamp cast to derive temporal checks from the string `date`.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import (
    to_timestamp, substring, current_timestamp,
    length as spark_length
)

print("=" * 80)
print("6.2.4) IMPOSSIBLE VALUE DETECTION  —  Review Table (Pre-Remodel)")
print("    Checking for values that cannot physically exist")
print("=" * 80)

total_rows = df.count()
findings_624 = []

# Helper: derive timestamp from string once for multiple checks
df_ts = df.withColumn("_review_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))

# ══ 1) Future dates (review cannot happen in the future) ══
print(f"\n{'─'*60}")
print("📌 Check 1: Future dates (review cannot happen after today)")
future_count = df_ts.filter(col("_review_ts") > current_timestamp()).count()
status = "PASS" if future_count == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Future timestamps: {future_count:,} — {status}")
if future_count > 0:
    samples = [r[0] for r in df_ts.filter(col("_review_ts") > current_timestamp()).select("date").limit(5).collect()]
    print(f"      Examples: {samples}")
findings_624.append({"Check": "Future timestamps", "Description": "date > now()", "Violations": future_count, "Status": status})

# ══ 2) Pre-Yelp dates (Yelp founded October 2004) ══
print(f"\n{'─'*60}")
print("📌 Check 2: Pre-Yelp dates (Yelp founded October 2004)")
pre_yelp = df_ts.filter(col("_review_ts") < lit("2004-10-01")).count()
status = "PASS" if pre_yelp == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Pre-Yelp timestamps (before 2004-10-01): {pre_yelp:,} — {status}")
if pre_yelp > 0:
    samples = [r[0] for r in df_ts.filter(col("_review_ts") < lit("2004-10-01")).select("date").limit(5).collect()]
    print(f"      Examples: {samples}")
findings_624.append({"Check": "Pre-Yelp timestamps", "Description": "date < 2004-10-01", "Violations": pre_yelp, "Status": status})

# ══ 3) Impossible calendar dates (Feb 30, Apr 31, etc.) ══
print(f"\n{'─'*60}")
print("📌 Check 3: Impossible calendar dates (e.g., Feb 30, Apr 31)")
print("   Spark rejects invalid dates during to_timestamp cast → NULL")
null_ts = df_ts.filter(col("_review_ts").isNull() & col("date").isNotNull()).count()
status = "PASS" if null_ts == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Failed casts (valid string but impossible date): {null_ts:,} — {status}")
if null_ts > 0:
    samples = [r[0] for r in df_ts.filter(col("_review_ts").isNull() & col("date").isNotNull()).select("date").limit(10).collect()]
    print(f"      Examples: {samples}")
findings_624.append({"Check": "Impossible calendar dates", "Description": "NULL after to_timestamp cast", "Violations": null_ts, "Status": status})

# ══ 4) Fractional stars (stars stored as double, must be whole 1–5) ══
print(f"\n{'─'*60}")
print("📌 Check 4: Fractional star ratings (impossible on Yelp)")
print("   Yelp only allows integer ratings 1–5; half-stars don't exist.")
fractional = df.filter(
    (col("stars") != col("stars").cast("int").cast("double")) &
    col("stars").isNotNull()
).count()
status = "PASS" if fractional == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Fractional star values: {fractional:,} — {status}")
if fractional > 0:
    frac_vals = [r[0] for r in df.filter(col("stars") != col("stars").cast("int").cast("double")).select("stars").distinct().limit(10).collect()]
    print(f"      Values: {frac_vals}")
findings_624.append({"Check": "Fractional stars", "Description": "stars ≠ int(stars) (half-stars impossible)", "Violations": fractional, "Status": status})

# ══ 5) Negative vote counts (physically impossible) ══
print(f"\n{'─'*60}")
print("📌 Check 5: Negative vote counts (impossible — can't un-vote)")
total_neg = 0
for vc in ["useful", "funny", "cool"]:
    neg = df.filter(col(vc) < 0).count()
    total_neg += neg
    icon = "✅" if neg == 0 else "⚠️"
    print(f"   {icon} {vc} < 0: {neg:,}")
    if neg > 0:
        neg_vals = [r[0] for r in df.filter(col(vc) < 0).select(vc).distinct().limit(10).collect()]
        print(f"      Values: {neg_vals}")
status = "PASS" if total_neg == 0 else "FINDING"
findings_624.append({"Check": "Negative vote counts", "Description": "useful/funny/cool < 0", "Violations": total_neg, "Status": status})

# ══ 6) Zero-length / NULL review text (review without content is impossible) ══
print(f"\n{'─'*60}")
print("📌 Check 6: Empty review text (a review must have a body)")
null_text = df.filter(col("text").isNull()).count()
empty_text = df.filter(trim(col("text")) == "").count()
total_empty = null_text + empty_text
status = "PASS" if total_empty == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} NULL text: {null_text:,}  |  Empty/whitespace-only: {empty_text:,}  |  Total: {total_empty:,} — {status}")
findings_624.append({"Check": "Empty review text", "Description": "text IS NULL or TRIM(text) = ''", "Violations": total_empty, "Status": status})

# ══ 7) Temporal component range from date string ══
print(f"\n{'─'*60}")
print("📌 Check 7: Impossible time components in date string")
print("   Extracted month [01-12], day [01-31], hour [00-23], minute/second [00-59]")

df_parts = df.select(
    substring(col("date"), 6, 2).cast("int").alias("_month"),
    substring(col("date"), 9, 2).cast("int").alias("_day"),
    substring(col("date"), 12, 2).cast("int").alias("_hour"),
    substring(col("date"), 15, 2).cast("int").alias("_minute"),
    substring(col("date"), 18, 2).cast("int").alias("_second"),
)

time_checks = [
    ("_month",  1, 12, "Month [1-12]"),
    ("_day",    1, 31, "Day [1-31]"),
    ("_hour",   0, 23, "Hour [0-23]"),
    ("_minute", 0, 59, "Minute [0-59]"),
    ("_second", 0, 59, "Second [0-59]"),
]
total_time_violations = 0
for part_col, lo, hi, desc in time_checks:
    bad = df_parts.filter((col(part_col) < lo) | (col(part_col) > hi) | col(part_col).isNull()).count()
    total_time_violations += bad
    icon = "✅" if bad == 0 else "⚠️"
    print(f"   {icon} {desc}: {bad:,} violations")

status = "PASS" if total_time_violations == 0 else "FINDING"
findings_624.append({"Check": "Impossible time components", "Description": "Month/Day/Hour/Min/Sec out of range", "Violations": total_time_violations, "Status": status})

# ══ 8) Cross-column: votes on a NULL-text review (suspicious) ══
print(f"\n{'─'*60}")
print("📌 Check 8: Cross-column — votes on empty/NULL review text")
print("   A review with no text shouldn't accumulate community votes.")
votes_no_text = df.filter(
    ((col("text").isNull()) | (trim(col("text")) == "")) &
    ((col("useful") > 0) | (col("funny") > 0) | (col("cool") > 0))
).count()
status = "PASS" if votes_no_text == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Empty-text reviews with votes > 0: {votes_no_text:,} — {status}")
findings_624.append({"Check": "Votes on empty review", "Description": "text empty but useful/funny/cool > 0", "Violations": votes_no_text, "Status": status})

# ══ 9) Cross-column: review_id uniqueness (PK constraint) ══
print(f"\n{'─'*60}")
print("📌 Check 9: review_id uniqueness (primary key integrity)")
print("   Each review_id must appear exactly once.")
distinct_reviews = df.select("review_id").distinct().count()
duplicates = total_rows - distinct_reviews
status = "PASS" if duplicates == 0 else "FINDING"
icon = "✅" if status == "PASS" else "⚠️"
print(f"   {icon} Total rows: {total_rows:,}  |  Distinct review_ids: {distinct_reviews:,}  |  Duplicates: {duplicates:,} — {status}")
if duplicates > 0:
    dup_samples = (
        df.groupBy("review_id").count()
        .filter(col("count") > 1)
        .orderBy(col("count").desc())
        .limit(5).collect()
    )
    print(f"      Top duplicate review_ids: {[(r['review_id'], r['count']) for r in dup_samples]}")
findings_624.append({"Check": "review_id uniqueness", "Description": "PK: each review_id appears once", "Violations": duplicates, "Status": status})

# ══ SUMMARY ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_624_df = pd.DataFrame(findings_624)
display(findings_624_df)

violation_count = sum(1 for f in findings_624 if f["Status"] == "FINDING")
print(f"\n{len(findings_624)} impossible-value checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ No physically impossible values detected.")
else:
    print(f"⚠️  {violation_count} check(s) found impossible values — review above.")
print("IMPOSSIBLE VALUE DETECTION COMPLETE")
print("=" * 80)

6.2.4) IMPOSSIBLE VALUE DETECTION  —  Review Table (Pre-Remodel)
    Checking for values that cannot physically exist

────────────────────────────────────────────────────────────
📌 Check 1: Future dates (review cannot happen after today)
   ✅ Future timestamps: 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 2: Pre-Yelp dates (Yelp founded October 2004)
   ✅ Pre-Yelp timestamps (before 2004-10-01): 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 3: Impossible calendar dates (e.g., Feb 30, Apr 31)
   Spark rejects invalid dates during to_timestamp cast → NULL
   ✅ Failed casts (valid string but impossible date): 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 4: Fractional star ratings (impossible on Yelp)
   Yelp only allows integer ratings 1–5; half-stars don't exist.
   ✅ Fractional star values: 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 5: Negative vote c

Check,Description,Violations,Status
Future timestamps,date > now(),0,PASS
Pre-Yelp timestamps,date < 2004-10-01,0,PASS
Impossible calendar dates,NULL after to_timestamp cast,0,PASS
Fractional stars,stars ≠ int(stars) (half-stars impossible),0,PASS
Negative vote counts,useful/funny/cool < 0,7,FINDING
Empty review text,text IS NULL or TRIM(text) = '',0,PASS
Impossible time components,Month/Day/Hour/Min/Sec out of range,0,PASS
Votes on empty review,text empty but useful/funny/cool > 0,0,PASS
review_id uniqueness,PK: each review_id appears once,0,PASS



9 impossible-value checks performed, 1 finding(s)
⚠️  1 check(s) found impossible values — review above.
IMPOSSIBLE VALUE DETECTION COMPLETE



### 6.3) FORMAT & ENCODING STANDARDIZATION
*Detect and normalize inconsistent representations. Detection only — normalization is a separate remediation step.*

- **Categorical Representation Consistency** — Same concept, different encodings.
  - *Example:* `Male`, `male`, `M`, `1` all representing the same category.
  - *Sub:* Category mapping consistency across related columns — `country_code = 'US'` but `country_name = 'Canada'`.

- **Date Format Consistency** — Dates stored in non-uniform formats.
  - *Example:* `2024-01-15`, `01/15/2024`, `15-Jan-2024` co-existing in one column.

- **Numeric Format Consistency** — Numbers stored with embedded symbols, separators, or units.
  - *Example:* `"$1,200.50"`, `"1.200,50 EUR"`, `"5 kg"` in a numeric column.

- **Boolean Representation Consistency** — Mixed boolean encodings co-existing.
  - *Example:* `Yes/No`, `True/False`, `1/0`, `Y/N` all present in one column.

- **Text Normalization Issues**
  - *Sub:* Leading/trailing whitespace — `" John "` vs `"John"`.
  - *Sub:* Multiple consecutive whitespaces — `"John    Doe"`.
  - *Sub:* Inconsistent capitalization — `"New York"`, `"new york"`, `"NEW YORK"`.
  - *Sub:* Escape/control characters — embedded `\n`, `\t`, `\r`, `\0`, non-printable characters that corrupt parsing.

- **Null Representation Consistency** — Non-standard nulls masquerading as data.
  - *Example:* `"N/A"`, `"NULL"`, `"None"`, `""`, `" "`, `"—"`, `"unknown"`, `"TBD"`, `"."`, `"NIL"`.
  - *Note:* This check only **detects and normalizes**. Counting missingness happens in 6.4.

In [0]:
# 6.3.1) Categorical Representation Consistency
# ─────────────────────────────────────────────────────────────────────
# For the RAW review table (pre-remodel) there are NO string categorical
# columns (month names, DOW names, etc. are created by the remodel step).
# Instead we verify:
#   1) stars ordinal representation — stored as DOUBLE, all values must be
#      exactly {1.0, 2.0, 3.0, 4.0, 5.0} with no fractional encoding variants
#   2) ID character-set uniformity — all 3 ID columns (review_id, user_id,
#      business_id) should use the same Base62-like 22-char character repertoire
#   3) Cross-column ID format agreement — length and charset match across IDs
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import (
    col, lower, countDistinct, collect_set, length as spark_length,
    sum as spark_sum, when, lit, regexp_extract
)

print("=" * 80)
print("6.3.1) CATEGORICAL REPRESENTATION CONSISTENCY  —  Review Table (Pre-Remodel)")
print("    Detecting inconsistent encodings of the same concept")
print("=" * 80)

total_rows = df.count()
findings_631 = []

# ══════════════════════════════════════════════════════════════════════
# 1) Stars — Ordinal representation consistency
#    Stored as DOUBLE: valid are exactly 1.0, 2.0, 3.0, 4.0, 5.0.
#    Detect fractional variants (1.5), string residue ("5.0"), or
#    encoding variants (5.00 vs 5.0 — at the Spark numeric level these
#    are identical, but we verify the distinct set is exactly 5 values).
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 1: stars — Ordinal representation consistency")
print("   Expected: exactly 5 distinct DOUBLE values {1.0, 2.0, 3.0, 4.0, 5.0}")

distinct_stars = sorted([r[0] for r in df.select("stars").distinct().collect() if r[0] is not None])
expected_stars = [1.0, 2.0, 3.0, 4.0, 5.0]
match = distinct_stars == expected_stars
status = "PASS" if match else "FINDING"
icon = "✅" if match else "⚠️"
print(f"   Distinct star values: {distinct_stars}")
print(f"   Expected values:      {expected_stars}")
print(f"   {icon} Exact match: {match} — {status}")
if not match:
    unexpected = [v for v in distinct_stars if v not in expected_stars]
    if unexpected:
        print(f"      ⚠️  Unexpected encodings: {unexpected}")

findings_631.append({
    "Check": "stars ordinal representation",
    "Column(s)": "stars",
    "Expected": "{1.0, 2.0, 3.0, 4.0, 5.0}",
    "Actual Distinct": len(distinct_stars),
    "Issue": "None" if match else f"Unexpected values: {[v for v in distinct_stars if v not in expected_stars]}",
    "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 2) ID character-set uniformity
#    Yelp IDs are 22-char Base62-like hashes [a-zA-Z0-9_-].
#    Check that every ID value uses the SAME character repertoire.
#    Also detect: mixed case, unexpected characters, length outliers.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 2: ID character-set uniformity")
print("   Expected: 22-char strings from [a-zA-Z0-9_-] for all 3 ID columns")

id_cols = ["review_id", "user_id", "business_id"]
id_pattern = r"^[a-zA-Z0-9_-]{22}$"

# Single-pass aggregation for all ID checks
agg_exprs = []
for c in id_cols:
    # Non-conforming pattern
    agg_exprs.append(
        spark_sum(when(~col(c).rlike(id_pattern) & col(c).isNotNull(), 1).otherwise(0)).alias(f"{c}__bad_pattern")
    )
    # Length != 22
    agg_exprs.append(
        spark_sum(when(spark_length(col(c)) != 22, 1).otherwise(0)).alias(f"{c}__bad_len")
    )
    # Contains uppercase
    agg_exprs.append(
        spark_sum(when(col(c).rlike("[A-Z]") & col(c).isNotNull(), 1).otherwise(0)).alias(f"{c}__has_upper")
    )
    # Contains lowercase
    agg_exprs.append(
        spark_sum(when(col(c).rlike("[a-z]") & col(c).isNotNull(), 1).otherwise(0)).alias(f"{c}__has_lower")
    )
    # Contains digits
    agg_exprs.append(
        spark_sum(when(col(c).rlike("[0-9]") & col(c).isNotNull(), 1).otherwise(0)).alias(f"{c}__has_digit")
    )
    # Contains underscore or hyphen
    agg_exprs.append(
        spark_sum(when(col(c).rlike("[_-]") & col(c).isNotNull(), 1).otherwise(0)).alias(f"{c}__has_special")
    )

result = df.agg(*agg_exprs).collect()[0]

for c in id_cols:
    bad_pat  = int(result[f"{c}__bad_pattern"] or 0)
    bad_len  = int(result[f"{c}__bad_len"]     or 0)
    has_up   = int(result[f"{c}__has_upper"]   or 0)
    has_lo   = int(result[f"{c}__has_lower"]   or 0)
    has_dig  = int(result[f"{c}__has_digit"]   or 0)
    has_spec = int(result[f"{c}__has_special"] or 0)

    status = "PASS" if bad_pat == 0 else "FINDING"
    icon = "✅" if bad_pat == 0 else "⚠️"
    print(f"\n   {icon} {c}: {bad_pat:,} pattern violations, {bad_len:,} length violations — {status}")
    print(f"      Character composition ({total_rows:,} rows):")
    print(f"        Contains uppercase: {has_up:,}  |  lowercase: {has_lo:,}")
    print(f"        Contains digits: {has_dig:,}  |  underscore/hyphen: {has_spec:,}")
    # Determine if mixed-case is the norm (expected for Base62)
    if has_up > 0 and has_lo > 0:
        print(f"        → Mixed-case is consistent (Base62 norm) ✅")

    findings_631.append({
        "Check": f"ID charset: {c}",
        "Column(s)": c,
        "Expected": "[a-zA-Z0-9_-]{22}",
        "Actual Distinct": f"{bad_pat} violations",
        "Issue": f"{bad_pat} non-conforming IDs" if bad_pat > 0 else "None",
        "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# 3) Cross-column ID format agreement
#    Verify all 3 ID columns share the same structural properties:
#    same length distribution, same character-set footprint.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 3: Cross-column ID format agreement")
print("   All 3 ID columns should share the same structural format")

# Check that all 3 columns have identical length distributions
len_agg = []
for c in id_cols:
    len_agg.append(countDistinct(spark_length(col(c))).alias(f"{c}__n_lens"))
    len_agg.append(collect_set(spark_length(col(c))).alias(f"{c}__lens"))

len_result = df.agg(*len_agg).collect()[0]

all_lens = {}
for c in id_cols:
    lens = sorted(len_result[f"{c}__lens"])
    all_lens[c] = lens
    print(f"   {c}: length values = {lens}")

# All should be {22}
uniform = all(lens == [22] for lens in all_lens.values())
status = "PASS" if uniform else "FINDING"
icon = "✅" if uniform else "⚠️"
print(f"   {icon} All 3 ID columns have uniform length = 22: {uniform} — {status}")

findings_631.append({
    "Check": "Cross-column ID format agreement",
    "Column(s)": "review_id, user_id, business_id",
    "Expected": "All length = 22",
    "Actual Distinct": str(all_lens),
    "Issue": "None" if uniform else "Inconsistent lengths across ID columns",
    "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 4) Note: No string categorical columns in raw review table
#    Columns like review_month_name, review_dow_abbr, review_time_of_day
#    are created during the remodel step and do not exist here.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Note: No string categorical columns in raw review table")
print("   Columns like review_month_name, review_dow_abbr, review_am_pm")
print("   are created by the remodel step. Categorical consistency for")
print("   those derived columns should be checked post-remodel.")

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_631_df = pd.DataFrame(findings_631)
display(findings_631_df)
v = sum(1 for f in findings_631 if f["Status"] == "FINDING")
print(f"\n{len(findings_631)} checks performed, {v} finding(s)")
if v == 0:
    print("✅ All categorical/representation values are consistently encoded.")
else:
    print(f"⚠️  {v} inconsistency(ies) detected — review above.")
print("CATEGORICAL REPRESENTATION CONSISTENCY CHECK COMPLETE")
print("=" * 80)

6.3.1) CATEGORICAL REPRESENTATION CONSISTENCY  —  Review Table (Pre-Remodel)
    Detecting inconsistent encodings of the same concept

────────────────────────────────────────────────────────────
📌 Check 1: stars — Ordinal representation consistency
   Expected: exactly 5 distinct DOUBLE values {1.0, 2.0, 3.0, 4.0, 5.0}
   Distinct star values: [1.0, 2.0, 3.0, 4.0, 5.0]
   Expected values:      [1.0, 2.0, 3.0, 4.0, 5.0]
   ✅ Exact match: True — PASS

────────────────────────────────────────────────────────────
📌 Check 2: ID character-set uniformity
   Expected: 22-char strings from [a-zA-Z0-9_-] for all 3 ID columns

   ✅ review_id: 0 pattern violations, 0 length violations — PASS
      Character composition (6,990,280 rows):
        Contains uppercase: 6,990,199  |  lowercase: 6,990,209
        Contains digits: 6,792,996  |  underscore/hyphen: 3,401,015
        → Mixed-case is consistent (Base62 norm) ✅

   ✅ user_id: 0 pattern violations, 0 length violations — PASS
      Character co

Check,Column(s),Expected,Actual Distinct,Issue,Status
stars ordinal representation,stars,"{1.0, 2.0, 3.0, 4.0, 5.0}",5,None,PASS
ID charset: review_id,review_id,[a-zA-Z0-9_-]{22},0 violations,None,PASS
ID charset: user_id,user_id,[a-zA-Z0-9_-]{22},0 violations,None,PASS
ID charset: business_id,business_id,[a-zA-Z0-9_-]{22},0 violations,None,PASS
Cross-column ID format agreement,"review_id, user_id, business_id",All length = 22,"{'review_id': [22], 'user_id': [22], 'business_id': [22]}",None,PASS



5 checks performed, 0 finding(s)
✅ All categorical/representation values are consistently encoded.
CATEGORICAL REPRESENTATION CONSISTENCY CHECK COMPLETE


In [0]:
# 6.3.2) Date Format Consistency
# ─────────────────────────────────────────────────────────────────────
# For the RAW review table (pre-remodel) there is ONE date column:
#   `date` — stored as STRING in expected format "yyyy-MM-dd HH:mm:ss".
# We check:
#   1) Uniform regex format (no mixed separators or alternative patterns)
#   2) No timezone suffixes, fractional seconds, or AM/PM markers
#   3) Consistent separator characters (dash, colon, space)
#   4) Successful Spark timestamp parse (no invalid calendar dates)
#   5) Time component value ranges (24-hour format)
# Optimized: single aggregation pass.

from pyspark.sql.functions import (
    col, sum as spark_sum, when, to_timestamp, substring,
    length as spark_length, trim
)

print("=" * 80)
print("6.3.2) DATE FORMAT CONSISTENCY  —  Review Table (Pre-Remodel)")
print("    Detecting non-uniform or broken date representations")
print("=" * 80)

total_rows = df.count()
findings_632 = []

# The review table has ONE date column: `date` (STRING).
# Expected canonical format: "yyyy-MM-dd HH:mm:ss" (e.g., "2019-03-15 14:22:07")
canonical_pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"

# ══════════════════════════════════════════════════════════════════════
# Single-pass aggregation for all format checks
# ══════════════════════════════════════════════════════════════════════
agg_exprs = [
    # 1) Canonical format match
    spark_sum(when(
        ~col("date").rlike(canonical_pattern) & col("date").isNotNull(), 1
    ).otherwise(0)).alias("non_canonical"),

    # 2) Alternative separators: slashes (MM/dd/yyyy)
    spark_sum(when(col("date").rlike(r"/"), 1).otherwise(0)).alias("has_slash"),

    # 3) ISO 8601 T separator (2019-03-15T14:22:07)
    spark_sum(when(col("date").rlike(r"T"), 1).otherwise(0)).alias("has_T_sep"),

    # 4) Timezone suffixes (Z, +00:00, UTC, etc.)
    spark_sum(when(
        col("date").rlike(r"(Z|[+-]\d{2}:?\d{2}|UTC|GMT)\s*$"), 1
    ).otherwise(0)).alias("has_tz"),

    # 5) Fractional seconds (.123, .123456)
    spark_sum(when(col("date").rlike(r"\.\d+"), 1).otherwise(0)).alias("has_fractional"),

    # 6) AM/PM markers
    spark_sum(when(
        col("date").rlike(r"(?i)(AM|PM)\s*$"), 1
    ).otherwise(0)).alias("has_ampm"),

    # 7) Leading/trailing whitespace
    spark_sum(when(
        col("date") != trim(col("date")), 1
    ).otherwise(0)).alias("has_ws"),

    # 8) Null or empty
    spark_sum(when(col("date").isNull(), 1).otherwise(0)).alias("nulls"),
    spark_sum(when(trim(col("date")) == "", 1).otherwise(0)).alias("empty"),

    # 9) Length uniformity (canonical = 19 chars)
    spark_sum(when(
        (spark_length(col("date")) != 19) & col("date").isNotNull(), 1
    ).otherwise(0)).alias("non_19_len"),

    # 10) Timestamp parse failure
    spark_sum(when(
        to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss").isNull() & col("date").isNotNull(), 1
    ).otherwise(0)).alias("parse_fail"),
]

result = df.agg(*agg_exprs).collect()[0]

# ─── Check 1: Canonical format ───
print(f"\n{'─'*60}")
print("📌 Check 1: Canonical format — 'yyyy-MM-dd HH:mm:ss'")
non_can = int(result["non_canonical"] or 0)
status = "PASS" if non_can == 0 else "FINDING"
icon = "✅" if non_can == 0 else "⚠️"
print(f"   {icon} Non-canonical format: {non_can:,} / {total_rows:,} — {status}")
if non_can > 0:
    samples = df.filter(~col("date").rlike(canonical_pattern) & col("date").isNotNull()).select("date").distinct().limit(5).collect()
    print(f"      Samples: {[r[0] for r in samples]}")
findings_632.append({"Check": "Canonical format", "Expected": "yyyy-MM-dd HH:mm:ss", "Violations": non_can, "Status": status})

# ─── Check 2: Mixed separator detection ───
print(f"\n{'─'*60}")
print("📌 Check 2: Mixed separator detection")
slash_ct = int(result["has_slash"] or 0)
t_sep_ct = int(result["has_T_sep"] or 0)
print(f"   Slash separators (MM/dd/yyyy): {slash_ct:,}")
print(f"   ISO T separators (yyyy-MM-ddTHH): {t_sep_ct:,}")
mixed = slash_ct + t_sep_ct
status = "PASS" if mixed == 0 else "FINDING"
icon = "✅" if mixed == 0 else "⚠️"
print(f"   {icon} Alternative separators: {mixed:,} — {status}")
findings_632.append({"Check": "Mixed separators", "Expected": "Dashes and spaces only", "Violations": mixed, "Status": status})

# ─── Check 3: Timezone / fractional seconds / AM-PM ───
print(f"\n{'─'*60}")
print("📌 Check 3: Unexpected date suffixes")
tz_ct   = int(result["has_tz"]         or 0)
frac_ct = int(result["has_fractional"] or 0)
ampm_ct = int(result["has_ampm"]       or 0)
print(f"   Timezone suffixes (Z, +00:00, UTC): {tz_ct:,}")
print(f"   Fractional seconds (.123):          {frac_ct:,}")
print(f"   AM/PM markers:                      {ampm_ct:,}")
suffix_total = tz_ct + frac_ct + ampm_ct
status = "PASS" if suffix_total == 0 else "FINDING"
icon = "✅" if suffix_total == 0 else "⚠️"
print(f"   {icon} Total unexpected suffixes: {suffix_total:,} — {status}")
findings_632.append({"Check": "Unexpected suffixes", "Expected": "No TZ/fractional/AMPM", "Violations": suffix_total, "Status": status})

# ─── Check 4: Length uniformity ───
print(f"\n{'─'*60}")
print("📌 Check 4: Date string length uniformity (expected: 19 chars)")
non_19 = int(result["non_19_len"] or 0)
ws_ct  = int(result["has_ws"]     or 0)
print(f"   Non-19-char date strings: {non_19:,}")
print(f"   Leading/trailing whitespace: {ws_ct:,}")
len_issues = non_19 + ws_ct
status = "PASS" if len_issues == 0 else "FINDING"
icon = "✅" if len_issues == 0 else "⚠️"
print(f"   {icon} Length/whitespace issues: {len_issues:,} — {status}")
findings_632.append({"Check": "Length uniformity", "Expected": "19 chars, no padding", "Violations": len_issues, "Status": status})

# ─── Check 5: Timestamp parse success ───
print(f"\n{'─'*60}")
print("📌 Check 5: Spark timestamp parse success")
parse_fail = int(result["parse_fail"] or 0)
null_ct    = int(result["nulls"]      or 0)
empty_ct   = int(result["empty"]      or 0)
print(f"   NULL dates: {null_ct:,}  |  Empty strings: {empty_ct:,}")
print(f"   Format-valid but calendar-invalid (failed cast): {parse_fail:,}")
status = "PASS" if (parse_fail + null_ct + empty_ct) == 0 else "FINDING"
icon = "✅" if (parse_fail + null_ct + empty_ct) == 0 else "⚠️"
print(f"   {icon} Total parse issues: {parse_fail + null_ct + empty_ct:,} — {status}")
findings_632.append({"Check": "Timestamp parse", "Expected": "All parse to valid timestamp", "Violations": parse_fail + null_ct + empty_ct, "Status": status})

# ─── Check 6: Date format distribution (sample first 10 distinct patterns) ───
print(f"\n{'─'*60}")
print("📌 Check 6: Date format pattern distribution")
print("   Extracting structural pattern (digits → D, alpha → A)")
# Replace digits with D, letters with A to see structural patterns
from pyspark.sql.functions import regexp_replace
df_pat = df.withColumn(
    "_date_pattern",
    regexp_replace(regexp_replace(col("date"), r"\d", "D"), r"[a-zA-Z]", "A")
)
patterns = df_pat.groupBy("_date_pattern").count().orderBy(col("count").desc()).limit(10).collect()
for p in patterns:
    pct = p["count"] / total_rows * 100
    print(f"   Pattern: '{p['_date_pattern']}'  →  {p['count']:,} rows ({pct:.4f}%)")
n_patterns = df_pat.select("_date_pattern").distinct().count()
status = "PASS" if n_patterns == 1 else "FINDING"
icon = "✅" if n_patterns == 1 else "⚠️"
print(f"   {icon} Distinct structural patterns: {n_patterns} — {status}")
findings_632.append({"Check": "Pattern uniformity", "Expected": "1 structural pattern", "Violations": n_patterns - 1 if n_patterns > 1 else 0, "Status": status})

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_632_df = pd.DataFrame(findings_632)
display(findings_632_df)
v = sum(1 for f in findings_632 if f["Status"] == "FINDING")
print(f"\n{len(findings_632)} checks performed, {v} finding(s)")
if v == 0:
    print("✅ The `date` column is consistently formatted — no anomalies.")
else:
    print(f"⚠️  {v} date format inconsistency(ies) detected — review above.")
print("DATE FORMAT CONSISTENCY CHECK COMPLETE")
print("=" * 80)

6.3.2) DATE FORMAT CONSISTENCY  —  Review Table (Pre-Remodel)
    Detecting non-uniform or broken date representations

────────────────────────────────────────────────────────────
📌 Check 1: Canonical format — 'yyyy-MM-dd HH:mm:ss'
   ✅ Non-canonical format: 0 / 6,990,280 — PASS

────────────────────────────────────────────────────────────
📌 Check 2: Mixed separator detection
   Slash separators (MM/dd/yyyy): 0
   ISO T separators (yyyy-MM-ddTHH): 0
   ✅ Alternative separators: 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 3: Unexpected date suffixes
   Timezone suffixes (Z, +00:00, UTC): 0
   Fractional seconds (.123):          0
   AM/PM markers:                      0
   ✅ Total unexpected suffixes: 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 4: Date string length uniformity (expected: 19 chars)
   Non-19-char date strings: 0
   Leading/trailing whitespace: 0
   ✅ Length/whitespace issues: 0 — PASS

───────────────

Check,Expected,Violations,Status
Canonical format,yyyy-MM-dd HH:mm:ss,0,PASS
Mixed separators,Dashes and spaces only,0,PASS
Unexpected suffixes,No TZ/fractional/AMPM,0,PASS
Length uniformity,"19 chars, no padding",0,PASS
Timestamp parse,All parse to valid timestamp,0,PASS
Pattern uniformity,1 structural pattern,0,PASS



6 checks performed, 0 finding(s)
✅ The `date` column is consistently formatted — no anomalies.
DATE FORMAT CONSISTENCY CHECK COMPLETE


In [0]:
# 6.3.3) Numeric Format Consistency
# ─────────────────────────────────────────────────────────────────────
# For the RAW review table (pre-remodel) the numeric columns are:
#   stars  (DOUBLE) — should contain only whole doubles {1.0..5.0}
#   useful, funny, cool (LONG) — integer vote counts
# String columns: review_id, user_id, business_id, text, date.
#
# We check:
#   1) stars storage type anomaly — DOUBLE when should be INT (cross-ref DQ-001)
#   2) stars fractional residue — any non-integer doubles?
#   3) Vote columns (LONG) — inherently integer, verify type purity
#   4) String columns — disguised numeric values with formatting artifacts
#   5) text column — reviews that are entirely numeric (possible data entry error)
# Optimized: single aggregation pass where possible.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import col, sum as spark_sum, when, trim
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType

print("=" * 80)
print("6.3.3) NUMERIC FORMAT CONSISTENCY  —  Review Table (Pre-Remodel)")
print("    Detecting numbers with embedded symbols, type mismatches, or artifacts")
print("=" * 80)

total_rows = df.count()
findings_633 = []

# ══════════════════════════════════════════════════════════════════════
# 1) stars storage type verification
#    stars is an ordinal 1–5 integer stored as DOUBLE (cf. DQ-REVIEW-001).
#    Verify the actual Spark type and flag the type mismatch.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 1: stars storage type — DOUBLE vs expected INT")
stars_type = str(df.schema["stars"].dataType)
print(f"   Current Spark type: {stars_type}")
print(f"   Expected type:      IntegerType (ordinal 1–5)")
is_double = isinstance(df.schema["stars"].dataType, DoubleType)
status = "INFO" if is_double else "PASS"
icon = "ℹ️" if is_double else "✅"
print(f"   {icon} stars stored as DOUBLE (wider than needed): {is_double} — {status}")
print(f"      → Cross-reference: DQ-REVIEW-001 (type fix prescribed in remodel)")
findings_633.append({
    "Column": "stars", "Current Type": stars_type,
    "Expected Type": "int", "Issue": "DOUBLE instead of INT (DQ-REVIEW-001)" if is_double else "None",
    "Violations": total_rows if is_double else 0, "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 2) stars fractional residue — any non-integer doubles?
#    e.g., 3.5, 4.7, 2.1 — which shouldn't exist on Yelp.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 2: stars fractional residue (non-integer doubles)")
fractional = df.filter(
    (col("stars") != col("stars").cast("int").cast("double")) &
    col("stars").isNotNull()
).count()
status = "PASS" if fractional == 0 else "FINDING"
icon = "✅" if fractional == 0 else "⚠️"
print(f"   {icon} Fractional star values: {fractional:,} — {status}")
if fractional > 0:
    frac_vals = [r[0] for r in df.filter(col("stars") != col("stars").cast("int").cast("double")).select("stars").distinct().limit(10).collect()]
    print(f"      Values: {frac_vals}")
findings_633.append({
    "Column": "stars", "Current Type": stars_type,
    "Expected Type": "int (whole numbers)", "Issue": f"{fractional} fractional values" if fractional > 0 else "None",
    "Violations": fractional, "Status": status
})

# ══════════════════════════════════════════════════════════════════════
# 3) Vote columns (LONG) — type purity
#    Integer types cannot hold fractional residue by definition,
#    but verify they are truly LONG (not silently cast from STRING).
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 3: Vote columns — integer type purity")
vote_cols = ["useful", "funny", "cool"]
for vc in vote_cols:
    vc_type = str(df.schema[vc].dataType)
    is_long = isinstance(df.schema[vc].dataType, LongType)
    is_int  = isinstance(df.schema[vc].dataType, (IntegerType, LongType))
    status = "PASS" if is_int else "FINDING"
    icon = "✅" if is_int else "⚠️"
    print(f"   {icon} {vc}: type = {vc_type} — {status}")
    findings_633.append({
        "Column": vc, "Current Type": vc_type,
        "Expected Type": "long/int", "Issue": f"Unexpected type: {vc_type}" if not is_int else "None",
        "Violations": total_rows if not is_int else 0, "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# 4) String columns — disguised numeric values with formatting artifacts
#    Check review_id, user_id, business_id, date for values that look
#    like formatted numbers ($1,200, 5kg, etc.).
#    Exclude `text` from this check (review prose naturally contains numbers).
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 4: String columns — disguised numeric formatting artifacts")
print("   (Excluding `text` — review prose naturally contains numbers)")

structural_str_cols = ["review_id", "user_id", "business_id", "date"]
numeric_artifact_pattern = r"^[\$\u20ac\u00a3\u00a5]?\s*[\d,]+\.?\d*\s*(%|kg|lbs?|mi|km|hrs?|mins?|secs?)?$"

agg_exprs = [
    spark_sum(
        when(col(c).rlike(numeric_artifact_pattern) & col(c).isNotNull(), 1).otherwise(0)
    ).alias(f"{c}__num")
    for c in structural_str_cols
]
result = df.agg(*agg_exprs).collect()[0]
for c in structural_str_cols:
    count = int(result[f"{c}__num"] or 0)
    status = "FINDING" if count > 0 else "PASS"
    icon = "⚠️" if count > 0 else "✅"
    print(f"   {icon} {c}: {count:,} formatted-number artifacts — {status}")
    if count > 0:
        samples = df.filter(col(c).rlike(numeric_artifact_pattern) & col(c).isNotNull()).select(c).distinct().limit(5).collect()
        print(f"      Examples: {[r[0] for r in samples]}")
    findings_633.append({
        "Column": c, "Current Type": "string",
        "Expected Type": "string (non-numeric)", "Issue": f"{count} numeric artifacts" if count > 0 else "None",
        "Violations": count, "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# 5) text column — reviews that are entirely numeric
#    A review body that is ONLY digits (no prose) is suspect.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 5: text column — reviews that are entirely numeric")
print("   A review body consisting only of digits is suspicious data")

all_numeric_text = df.filter(
    col("text").rlike(r"^\s*\d+\s*$") & col("text").isNotNull()
).count()
status = "INFO" if all_numeric_text > 0 else "PASS"
icon = "ℹ️" if all_numeric_text > 0 else "✅"
print(f"   {icon} Reviews with entirely numeric text: {all_numeric_text:,} — {status}")
if all_numeric_text > 0:
    samples = df.filter(col("text").rlike(r"^\s*\d+\s*$")).select("text").limit(5).collect()
    print(f"      Samples: {[r[0] for r in samples]}")
findings_633.append({
    "Column": "text", "Current Type": "string",
    "Expected Type": "string (prose)", "Issue": f"{all_numeric_text} all-numeric reviews" if all_numeric_text > 0 else "None",
    "Violations": all_numeric_text, "Status": status
})

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_633_df = pd.DataFrame(findings_633)
display(findings_633_df)
v = sum(1 for f in findings_633 if f["Status"] == "FINDING")
i = sum(1 for f in findings_633 if f["Status"] == "INFO")
print(f"\n{len(findings_633)} checks performed, {v} finding(s), {i} informational")
if v == 0 and i == 0:
    print("✅ All numeric representations are clean — no format artifacts.")
elif v == 0:
    print("✅ No format errors. Informational flags noted — review above.")
else:
    print(f"⚠️  {v} numeric format issue(s) detected — review above.")
print("NUMERIC FORMAT CONSISTENCY CHECK COMPLETE")
print("=" * 80)

6.3.3) NUMERIC FORMAT CONSISTENCY  —  Review Table (Pre-Remodel)
    Detecting numbers with embedded symbols, type mismatches, or artifacts

────────────────────────────────────────────────────────────
📌 Check 1: stars storage type — DOUBLE vs expected INT
   Current Spark type: DoubleType()
   Expected type:      IntegerType (ordinal 1–5)
   ℹ️ stars stored as DOUBLE (wider than needed): True — INFO
      → Cross-reference: DQ-REVIEW-001 (type fix prescribed in remodel)

────────────────────────────────────────────────────────────
📌 Check 2: stars fractional residue (non-integer doubles)
   ✅ Fractional star values: 0 — PASS

────────────────────────────────────────────────────────────
📌 Check 3: Vote columns — integer type purity
   ✅ useful: type = LongType() — PASS
   ✅ funny: type = LongType() — PASS
   ✅ cool: type = LongType() — PASS

────────────────────────────────────────────────────────────
📌 Check 4: String columns — disguised numeric formatting artifacts
   (Excluding `tex

Column,Current Type,Expected Type,Issue,Violations,Status
stars,DoubleType(),int,DOUBLE instead of INT (DQ-REVIEW-001),6990280,INFO
stars,DoubleType(),int (whole numbers),None,0,PASS
useful,LongType(),long/int,None,0,PASS
funny,LongType(),long/int,None,0,PASS
cool,LongType(),long/int,None,0,PASS
review_id,string,string (non-numeric),None,0,PASS
user_id,string,string (non-numeric),None,0,PASS
business_id,string,string (non-numeric),None,0,PASS
date,string,string (non-numeric),None,0,PASS
text,string,string (prose),None,0,PASS



10 checks performed, 0 finding(s), 1 informational
✅ No format errors. Informational flags noted — review above.
NUMERIC FORMAT CONSISTENCY CHECK COMPLETE


In [0]:
# 6.3.4) Boolean Representation Consistency
# ─────────────────────────────────────────────────────────────────────
# For the RAW review table (pre-remodel):
#   • NO boolean-typed columns exist in the schema.
#   • String columns: review_id, user_id, business_id, text, date.
# We check:
#   1) Confirm no boolean-typed columns (schema verification)
#   2) Structural string columns (IDs, date) for boolean-like values
#   3) text column for boolean-only reviews (entire text is just
#      "yes"/"no"/"true"/"false" — suspicious as review content)
# Optimized: single aggregation pass.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import col, sum as spark_sum, when, lower, trim
from pyspark.sql.types import BooleanType, StringType

print("=" * 80)
print("6.3.4) BOOLEAN REPRESENTATION CONSISTENCY  —  Review Table (Pre-Remodel)")
print("    Detecting mixed boolean encodings across all columns")
print("=" * 80)

total_rows = df.count()
findings_634 = []

# ══════════════════════════════════════════════════════════════════════
# 1) Schema verification: no boolean-typed columns
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 1: Boolean-typed columns in schema")

bool_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, BooleanType)]
if bool_cols:
    print(f"   Found {len(bool_cols)} boolean column(s): {bool_cols}")
    agg_exprs = [
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"{c}__nulls")
        for c in bool_cols
    ]
    result = df.agg(*agg_exprs).collect()[0]
    for c in bool_cols:
        n = int(result[f"{c}__nulls"] or 0)
        status = "PASS" if n == 0 else "FINDING"
        icon = "✅" if n == 0 else "⚠️"
        print(f"   {icon} {c}: {n:,} nulls — {status}")
        findings_634.append({"Column": c, "Type": "boolean", "Issue": f"{n:,} nulls" if n > 0 else "None", "Status": status})
else:
    print("   ✅ No boolean-typed columns in the raw review schema.")
    print("   This is expected — the raw table has: 3 string IDs, 1 double,")
    print("   3 longs, 1 text string, and 1 date string.")

# ══════════════════════════════════════════════════════════════════════
# 2) Structural string columns: boolean-like values masquerading as IDs/dates
#    Check review_id, user_id, business_id, date for values that are
#    entirely boolean tokens ("yes", "no", "true", "false", etc.).
#    These would indicate data corruption in structural columns.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 2: Structural string columns — boolean-like corruption")
print("   Checking IDs and date for values that are entirely boolean tokens")

bool_like_tokens = [
    "yes", "no", "y", "n", "true", "false", "t", "f",
    "1", "0", "on", "off", "active", "inactive", "enabled", "disabled"
]
structural_cols = ["review_id", "user_id", "business_id", "date"]

agg_exprs = [
    spark_sum(
        when(lower(trim(col(c))).isin(bool_like_tokens) & col(c).isNotNull(), 1).otherwise(0)
    ).alias(f"{c}__bl")
    for c in structural_cols
]
result = df.agg(*agg_exprs).collect()[0]
for c in structural_cols:
    count = int(result[f"{c}__bl"] or 0)
    status = "FINDING" if count > 0 else "PASS"
    icon = "⚠️" if count > 0 else "✅"
    print(f"   {icon} {c}: {count:,} boolean-like values — {status}")
    if count > 0:
        samples = df.filter(lower(trim(col(c))).isin(bool_like_tokens)).select(c).distinct().limit(10).collect()
        print(f"      Values: {[r[0] for r in samples]}")
    findings_634.append({"Column": c, "Type": "string (structural)", "Issue": f"{count:,} boolean-like values" if count > 0 else "None", "Status": status})

# ══════════════════════════════════════════════════════════════════════
# 3) text column: reviews whose ENTIRE body is a boolean token
#    A review that says ONLY "yes" or "no" (and nothing else) is
#    suspicious — it may be placeholder, bot, or survey data.
#    (This is distinct from reviews that *contain* "yes" in prose.)
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 3: text column — reviews that are entirely boolean tokens")
print("   A review body of just 'yes'/'no'/'true'/'false' is suspicious")

bool_only_text = df.filter(
    lower(trim(col("text"))).isin(bool_like_tokens) & col("text").isNotNull()
).count()
status = "INFO" if bool_only_text > 0 else "PASS"
icon = "ℹ️" if bool_only_text > 0 else "✅"
print(f"   {icon} Boolean-only review text: {bool_only_text:,} — {status}")
if bool_only_text > 0:
    samples = df.filter(lower(trim(col("text"))).isin(bool_like_tokens)).select("text", "stars").limit(10).collect()
    print(f"      Samples (text, stars): {[(r['text'], r['stars']) for r in samples]}")
findings_634.append({"Column": "text", "Type": "string (content)", "Issue": f"{bool_only_text:,} boolean-only reviews" if bool_only_text > 0 else "None", "Status": status})

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_634_df = pd.DataFrame(findings_634)
display(findings_634_df)
v = sum(1 for f in findings_634 if f["Status"] == "FINDING")
i = sum(1 for f in findings_634 if f["Status"] == "INFO")
print(f"\n{len(findings_634)} checks performed, {v} finding(s), {i} informational")
if v == 0 and i == 0:
    print("✅ No boolean representation issues detected.")
elif v == 0:
    print("✅ No boolean encoding errors. Informational flags noted.")
else:
    print(f"⚠️  {v} boolean encoding inconsistency(ies) detected — review above.")
print("BOOLEAN REPRESENTATION CONSISTENCY CHECK COMPLETE")
print("=" * 80)

6.3.4) BOOLEAN REPRESENTATION CONSISTENCY  —  Review Table (Pre-Remodel)
    Detecting mixed boolean encodings across all columns

────────────────────────────────────────────────────────────
📌 Check 1: Boolean-typed columns in schema
   ✅ No boolean-typed columns in the raw review schema.
   This is expected — the raw table has: 3 string IDs, 1 double,
   3 longs, 1 text string, and 1 date string.

────────────────────────────────────────────────────────────
📌 Check 2: Structural string columns — boolean-like corruption
   Checking IDs and date for values that are entirely boolean tokens
   ✅ review_id: 0 boolean-like values — PASS
   ✅ user_id: 0 boolean-like values — PASS
   ✅ business_id: 0 boolean-like values — PASS
   ✅ date: 0 boolean-like values — PASS

────────────────────────────────────────────────────────────
📌 Check 3: text column — reviews that are entirely boolean tokens
   A review body of just 'yes'/'no'/'true'/'false' is suspicious
   ℹ️ Boolean-only review text: 5 — 

Column,Type,Issue,Status
review_id,string (structural),None,PASS
user_id,string (structural),None,PASS
business_id,string (structural),None,PASS
date,string (structural),None,PASS
text,string (content),5 boolean-only reviews,INFO



5 checks performed, 0 finding(s), 1 informational
✅ No boolean encoding errors. Informational flags noted.
BOOLEAN REPRESENTATION CONSISTENCY CHECK COMPLETE


In [0]:
# 6.3.5) Text Normalization Issues
# ─────────────────────────────────────────────────────────────────────
# For the RAW review table (pre-remodel) the string columns are:
#   Structural: review_id, user_id, business_id, date
#   Content:    text (the review body — primary NLP input)
#
# This cell is CRITICAL for NLP pipeline readiness.  We check:
#   1) Leading/trailing whitespace (all string columns)
#   2) Multiple consecutive spaces (all string columns)
#   3) Control characters: \t, \r, \n, \x00, etc. (all columns)
#   4) HTML entities (&amp;, &lt;, &gt;, &nbsp;, &#NNN;) in text
#   5) Mojibake / encoding artifacts (Ã©, Ã±, Ã, etc.) in text
#   6) Unicode smart punctuation (‘’“”–—…) in text (informational)
#   7) Excessive newlines / paragraph breaks in text
#   8) Empty strings after trimming
# Optimized: single aggregation pass per column group.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import (
    col, sum as spark_sum, when, trim, lower, upper,
    length as spark_length
)
from pyspark.sql.types import StringType

print("=" * 80)
print("6.3.5) TEXT NORMALIZATION ISSUES  —  Review Table (Pre-Remodel)")
print("    Detecting whitespace, control chars, encoding, and NLP-readiness issues")
print("=" * 80)

total_rows = df.count()
findings_635 = []
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

# ══════════════════════════════════════════════════════════════════════
# Part A: ALL string columns — structural normalization
# Single-pass aggregation for basic text hygiene across all 5 columns
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Part A: Basic text hygiene — all string columns")
print("   Checking: whitespace, multi-space, control chars, empty strings")

agg_exprs = []
for c in string_cols:
    agg_exprs.extend([
        # Leading/trailing whitespace
        spark_sum(when(
            (col(c) != trim(col(c))) & col(c).isNotNull(), 1
        ).otherwise(0)).alias(f"{c}__ws"),
        # Multiple consecutive spaces
        spark_sum(when(
            col(c).rlike("  +") & col(c).isNotNull(), 1
        ).otherwise(0)).alias(f"{c}__msp"),
        # Control characters (\x00-\x1f except \n\r\t, plus \x7f)
        # For `text`: \n is expected (paragraphs), so check separately
        spark_sum(when(
            col(c).rlike("[\\x00-\\x08\\x0b\\x0c\\x0e-\\x1f\\x7f]") & col(c).isNotNull(), 1
        ).otherwise(0)).alias(f"{c}__ctrl"),
        # Empty after trim
        spark_sum(when(
            (trim(col(c)) == "") & col(c).isNotNull(), 1
        ).otherwise(0)).alias(f"{c}__empty"),
    ])

result_a = df.agg(*agg_exprs).collect()[0]

for c in string_cols:
    ws    = int(result_a[f"{c}__ws"]    or 0)
    msp   = int(result_a[f"{c}__msp"]   or 0)
    ctrl  = int(result_a[f"{c}__ctrl"]  or 0)
    empty = int(result_a[f"{c}__empty"] or 0)

    issues = []
    if ws    > 0: issues.append(f"leading/trailing whitespace: {ws:,}")
    if msp   > 0: issues.append(f"multiple consecutive spaces: {msp:,}")
    if ctrl  > 0: issues.append(f"control characters: {ctrl:,}")
    if empty > 0: issues.append(f"empty strings: {empty:,}")

    if issues:
        print(f"   ⚠️  {c}: {'; '.join(issues)}")
        status = "FINDING"
    else:
        print(f"   ✅ {c}: clean")
        status = "PASS"

    findings_635.append({
        "Column": c, "Check": "Basic hygiene",
        "Whitespace": ws, "Multi-Space": msp,
        "Control Chars": ctrl, "Empty Strings": empty,
        "Status": status
    })

# ══════════════════════════════════════════════════════════════════════
# Part B: `text` column — NLP-specific normalization checks
# These are checks that matter for downstream NLP/sentiment pipelines.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Part B: NLP-specific checks on `text` column")

nlp_agg = [
    # B1) HTML entities: &amp; &lt; &gt; &nbsp; &#NNN; &#xHH;
    spark_sum(when(
        col("text").rlike(r"&(amp|lt|gt|nbsp|quot|apos|#\d{1,5}|#x[0-9a-fA-F]{1,4});") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("html_entities"),

    # B2) Mojibake / encoding artifacts (common UTF-8 double-encoding)
    #     Ã© (é), Ã± (ñ), Ã¼ (ü), Ã  (à), Â  (NBSP), etc.
    spark_sum(when(
        col("text").rlike("[\u00c3][\u0080-\u00bf]") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("mojibake"),

    # B3) Unicode smart punctuation (‘ ’ “ ” – — … •)
    #     These are valid Unicode but may need normalizing for some NLP models
    spark_sum(when(
        col("text").rlike("[\u2018\u2019\u201c\u201d\u2013\u2014\u2026\u2022]") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("smart_punct"),

    # B4) Excessive newlines (more than 3 consecutive \n)
    spark_sum(when(
        col("text").rlike("\n{4,}") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("excess_newlines"),

    # B5) Tab characters in review text
    spark_sum(when(
        col("text").rlike("\t") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("has_tabs"),

    # B6) Carriage return (\r) — Windows-style line endings
    spark_sum(when(
        col("text").rlike("\r") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("has_cr"),

    # B7) Reviews containing HTML tags (<br>, <p>, <b>, etc.)
    spark_sum(when(
        col("text").rlike(r"<\s*/?(br|p|b|i|a|div|span|img|em|strong|ul|li|ol|h[1-6])[\s/>]") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("html_tags"),

    # B8) URL-like content in reviews (http://, https://, www.)
    spark_sum(when(
        col("text").rlike(r"(https?://|www\.)[\S]+") & col("text").isNotNull(), 1
    ).otherwise(0)).alias("has_urls"),
]

result_b = df.agg(*nlp_agg).collect()[0]

nlp_checks = [
    ("html_entities",   "HTML entities (&amp; &lt; etc.)",         "FINDING"),
    ("mojibake",        "Mojibake/encoding artifacts (Ã© Ã± etc.)", "FINDING"),
    ("smart_punct",     "Unicode smart punctuation (‘’“”–—…)",     "INFO"),
    ("excess_newlines", "Excessive newlines (4+ consecutive)",      "INFO"),
    ("has_tabs",        "Tab characters in review text",            "INFO"),
    ("has_cr",          "Carriage returns (\\r) in text",           "INFO"),
    ("html_tags",       "HTML tags (<br>, <p>, etc.)",              "FINDING"),
    ("has_urls",        "URLs in review text",                      "INFO"),
]

for key, desc, severity_if_found in nlp_checks:
    count = int(result_b[key] or 0)
    if count > 0:
        status = severity_if_found
        icon = "⚠️" if severity_if_found == "FINDING" else "ℹ️"
    else:
        status = "PASS"
        icon = "✅"
    print(f"   {icon} {desc}: {count:,} — {status}")
    findings_635.append({
        "Column": "text", "Check": desc,
        "Whitespace": "-", "Multi-Space": "-",
        "Control Chars": "-", "Empty Strings": "-",
        "Status": status
    })

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_635_df = pd.DataFrame(findings_635)
display(findings_635_df)
v = sum(1 for f in findings_635 if f["Status"] == "FINDING")
i = sum(1 for f in findings_635 if f["Status"] == "INFO")
print(f"\n{len(findings_635)} checks performed, {v} finding(s), {i} informational")
if v == 0 and i == 0:
    print("✅ All string columns are clean — no normalization issues detected.")
elif v == 0:
    print("✅ No critical normalization issues. Informational NLP flags noted.")
else:
    print(f"⚠️  {v} column(s)/check(s) have normalization issues — review above.")
print("TEXT NORMALIZATION CHECK COMPLETE")
print("=" * 80)

6.3.5) TEXT NORMALIZATION ISSUES  —  Review Table (Pre-Remodel)
    Detecting whitespace, control chars, encoding, and NLP-readiness issues

────────────────────────────────────────────────────────────
📌 Part A: Basic text hygiene — all string columns
   Checking: whitespace, multi-space, control chars, empty strings
   ✅ review_id: clean
   ✅ user_id: clean
   ✅ business_id: clean
   ⚠️  text: multiple consecutive spaces: 2,381,849; control characters: 18
   ✅ date: clean

────────────────────────────────────────────────────────────
📌 Part B: NLP-specific checks on `text` column
   ⚠️ HTML entities (&amp; &lt; etc.): 97 — FINDING
   ✅ Mojibake/encoding artifacts (Ã© Ã± etc.): 0 — PASS
   ✅ Unicode smart punctuation (‘’“”–—…): 0 — PASS
   ℹ️ Excessive newlines (4+ consecutive): 4,566 — INFO
   ✅ Tab characters in review text: 0 — PASS
   ℹ️ Carriage returns (\r) in text: 389 — INFO
   ✅ HTML tags (<br>, <p>, etc.): 0 — PASS
   ℹ️ URLs in review text: 12,551 — INFO

SUMMARY


Column,Check,Whitespace,Multi-Space,Control Chars,Empty Strings,Status
review_id,Basic hygiene,0,0,0,0,PASS
user_id,Basic hygiene,0,0,0,0,PASS
business_id,Basic hygiene,0,0,0,0,PASS
text,Basic hygiene,0,2381849,18,0,FINDING
date,Basic hygiene,0,0,0,0,PASS
text,HTML entities (& < etc.),-,-,-,-,FINDING
text,Mojibake/encoding artifacts (Ã© Ã± etc.),-,-,-,-,PASS
text,Unicode smart punctuation (‘’“”–—…),-,-,-,-,PASS
text,Excessive newlines (4+ consecutive),-,-,-,-,INFO
text,Tab characters in review text,-,-,-,-,PASS



13 checks performed, 2 finding(s), 3 informational
⚠️  2 column(s)/check(s) have normalization issues — review above.
TEXT NORMALIZATION CHECK COMPLETE


---
#### 🚨 DQ-REVIEW-007 — Multiple Consecutive Spaces in Review Text

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-007` |
| **Severity** | FINDING (NLP data quality issue) |
| **Detected In** | Phase 6.3.5 — Text Normalization, Part A: Basic Hygiene (Cell 51) |
| **Affected Column** | `text` |
| **Affected Rows** | **2,381,849** out of 6,990,280 (34.07%) |
| **Pattern** | Two or more consecutive space characters within the review body |

**Description:** Over one-third of all reviews contain multiple consecutive space characters (e.g., "Great  place" or "The food   was amazing"). While not a data integrity error, this is the most pervasive text normalization issue in the dataset and directly impacts downstream NLP tasks: tokenizers may produce empty tokens, TF-IDF vectors may be distorted by space-inflated token counts, and text similarity measures may be skewed.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Regex collapse during Silver-layer text preprocessing** (`regexp_replace(text, '  +', ' ')`) | Standard NLP preprocessing step. Preserves all semantic content while normalising whitespace. Non-destructive to the Bronze record. |
| 2 | **Apply in-place on the Bronze table** (`UPDATE ... SET text = regexp_replace(text, '  +', ' ')`) | Faster downstream queries since every consumer gets clean text. Risk: loses the original representation if the extra spaces were intentional formatting. |
| 3 | **Retain as-is, handle in each consumer** | Zero data modification, but every NLP pipeline must independently handle it — error-prone and duplicates effort. |

**Best Logical Solution:** **Option 1 — Regex collapse at the Silver layer.** At 34% prevalence this is a systemic formatting artefact, not an error. Collapsing consecutive spaces is a universal NLP preprocessing step that should happen once during the Bronze→Silver transformation, preserving the raw Bronze record for audit while giving all Silver consumers clean input.

---

#### 🚨 DQ-REVIEW-008 — Control Characters in Review Text

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-008` |
| **Severity** | FINDING (data integrity / corruption risk) |
| **Detected In** | Phase 6.3.5 — Text Normalization, Part A: Basic Hygiene (Cell 51) |
| **Affected Column** | `text` |
| **Affected Rows** | **18** out of 6,990,280 (0.00026%) |
| **Pattern** | Non-printable control characters (`\x00`–`\x08`, `\x0B`, `\x0C`, `\x0E`–`\x1F`, `\x7F`) excluding normal whitespace (`\n`, `\r`, `\t`) |

**Description:** 18 reviews contain non-printable control characters that are invisible but can corrupt downstream processing: JSON/CSV serialisation may break, tokenizers may produce garbage tokens, and full-text search indexing can fail silently.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Strip control characters during Silver-layer preprocessing** (`regexp_replace(text, '[\\x00-\\x08\\x0B\\x0C\\x0E-\\x1F\\x7F]', '')`) | Removes invisible corruption without altering readable content. Standard text-cleaning practice. |
| 2 | **Replace with space** instead of removing | Avoids accidentally merging words (e.g., "word1\x00word2" → "word1 word2" instead of "word1word2"). Marginally safer for NLP tokenisation. |
| 3 | **Flag and quarantine the 18 rows** | Preserves original data for investigation, but at 0.00026% the investigation value is minimal. |

**Best Logical Solution:** **Option 1 — Strip control characters at the Silver layer.** The 18 affected rows are an infinitesimal fraction and the characters carry zero semantic meaning. Stripping them is universally safe and prevents silent downstream failures.

---

#### ⚠️ DQ-REVIEW-009 — HTML Entities in Review Text

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-009` |
| **Severity** | FINDING (encoding issue impacting NLP) |
| **Detected In** | Phase 6.3.5 — Text Normalization, Part B: NLP Checks (Cell 51) |
| **Affected Column** | `text` |
| **Affected Rows** | **97** out of 6,990,280 (0.0014%) |
| **Pattern** | HTML entity sequences: `&amp;`, `&lt;`, `&gt;`, `&nbsp;`, `&quot;`, `&#NNN;`, `&#xHH;` |

**Description:** 97 reviews contain HTML-encoded entities that were not decoded during data ingestion. For example, `&amp;` should be `&`, `&lt;` should be `<`. These artefacts indicate the text was scraped from a web page and the HTML decoding step was incomplete. They corrupt NLP features: a tokenizer sees `&amp;` as a single token instead of the punctuation it represents.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **HTML-unescape during Silver-layer text preprocessing** (Python `html.unescape()` via UDF or `regexp_replace` for common entities) | Restores the text to its intended readable form. Handles both named (`&amp;`) and numeric (`&#38;`) entities. Standard web-data cleaning step. |
| 2 | **Replace only known common entities** (`&amp;`→`&`, `&lt;`→`<`, `&gt;`→`>`, `&nbsp;`→` `) | Simpler regex, covers 95%+ of real-world cases, avoids importing Python’s html module as a UDF. |
| 3 | **Leave as-is, let tokenizers handle it** | Some modern tokenizers (SentencePiece, BPE) can learn entity patterns. But standard Spark NLP and NLTK tokenizers will not. |

**Best Logical Solution:** **Option 1 — Full HTML unescape at the Silver layer.** A comprehensive unescape is trivially implemented and handles all entity forms. At only 97 rows the performance cost is nil, and it ensures all downstream NLP consumers receive semantically correct text.

---

#### ℹ️ DQ-REVIEW-010 — NLP Readiness Advisory Signals

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-010` |
| **Severity** | INFO (advisory — not data errors, quality signals for NLP pipelines) |
| **Detected In** | Phase 6.3.5 — Text Normalization, Part B: NLP Checks (Cell 51) |
| **Affected Column** | `text` |

| Signal | Count | % of Dataset | Description |
| --- | --- | --- | --- |
| Excessive newlines (4+ consecutive) | 4,566 | 0.065% | Reviews with large paragraph gaps; may indicate copy-paste from other sources |
| Carriage returns (`\r`) | 389 | 0.006% | Windows-style line endings (`\r\n`); inconsistent with Unix-style (`\n`) in the rest of the dataset |
| URLs in review text | 12,551 | 0.180% | Reviews containing `http://`, `https://`, or `www.` links |

**Description:** These three signals are **not data errors** but important quality indicators for NLP preprocessing design. Excessive newlines may need collapsing, carriage returns should be normalised to `\n`, and URLs may need removal or replacement with a `<URL>` token depending on the NLP task.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Normalise all three during Silver-layer NLP preprocessing**: collapse `\n{3,}` → `\n\n`, replace `\r\n` → `\n` and `\r` → `\n`, optionally replace URLs with `<URL>` token | One-pass regex chain handles all three. Standard NLP text-cleaning practice. |
| 2 | **Retain as-is at Silver, handle per-task at Gold** | Different NLP tasks may want URLs preserved (e.g., link analysis) vs. removed (sentiment). Flexible but duplicates effort. |

**Best Logical Solution:** **Option 1 — Normalise at the Silver layer** for the newline/CR issues (universally beneficial), but **Option 2 for URLs** (task-dependent, better deferred to Gold-layer feature engineering).

---

In [0]:
# 6.3.6) Null Representation Consistency  —  REVIEW TABLE ADAPTED
# ─────────────────────────────────────────────────────────────────────
# For the RAW review table (pre-remodel) detect non-standard null
# representations masquerading as real data in string columns:
#   Structural: review_id, user_id, business_id, date
#   Content:    text (the review body)
# Optimized: single aggregation pass across all string columns.

from pyspark.sql.functions import col, sum as spark_sum, when, trim, lower
from pyspark.sql.types import StringType

print("=" * 80)
print("6.3.6) NULL REPRESENTATION CONSISTENCY  —  Review Table (Pre-Remodel)")
print("    Detecting non-standard nulls masquerading as data")
print("=" * 80)

total_rows = df.count()
findings_636 = []
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

# Case-insensitive matching via lower(trim())
null_patterns_lower = [
    # Classic null tokens
    "n/a", "na", "null", "none", "nan", "nil",
    # Semantic indicators
    "unknown", "tbd", "missing", "undefined",
    "not available", "not applicable",
    # Punctuation-based
    ".", "-", "--", "---", "\u2014", "\u2013",
    # Abbreviation variants
    "n.a.", "n/a.",
    # Question marks / asterisks
    "?", "??", "???", "*", "**", "***",
    # Spreadsheet error codes
    "#n/a", "#value!", "#ref!", "#name?", "#div/0!", "#null!",
    # Empty-like labels
    "empty", "blank",
]

print(f"\n{'─'*60}")
print("📌 Check 1: Masked null tokens across all string columns")
print(f"   Checking {len(null_patterns_lower)} null-like patterns (case-insensitive)")

# Single-pass aggregation for all string columns
agg_exprs = []
for c in string_cols:
    agg_exprs.append(
        spark_sum(when(
            lower(trim(col(c))).isin(null_patterns_lower) & col(c).isNotNull(), 1
        ).otherwise(0)).alias(f"{c}__masked")
    )
    agg_exprs.append(
        spark_sum(when(
            (trim(col(c)) == "") & col(c).isNotNull() & (col(c) != ""), 1
        ).otherwise(0)).alias(f"{c}__ws_only")
    )

result = df.agg(*agg_exprs).collect()[0]

for c in string_cols:
    masked  = int(result[f"{c}__masked"]  or 0)
    ws_only = int(result[f"{c}__ws_only"] or 0)
    total   = masked + ws_only

    if total > 0:
        parts = []
        if masked > 0:
            bad_vals = df.filter(
                lower(trim(col(c))).isin(null_patterns_lower) & col(c).isNotNull()
            ).select(c).distinct().limit(10).collect()
            parts.append(f"{masked:,} masked nulls — values: {[r[0] for r in bad_vals]}")
        if ws_only > 0:
            parts.append(f"{ws_only:,} whitespace-only strings")
        print(f"   ⚠️  {c}: {'; '.join(parts)}")
        status = "FINDING"
    else:
        print(f"   ✅ {c}: no masked nulls")
        status = "PASS"

    findings_636.append({"Column": c, "Masked Nulls": masked, "Whitespace-Only": ws_only, "Total": total, "Status": status})

# ══════════════════════════════════════════════════════════════════════
# Check 2: Review-specific "no comment" placeholders in `text`
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 2: Review-specific 'no-comment' placeholders in text")

no_comment_patterns = [
    "no comment", "no comments", "no review",
    "nothing to say", "nothing to add", "no opinion",
    "declined to comment", "no feedback",
    "...", "..", "---", "-----",
    "test", "testing", "test review", "asdf", "aaa", "xxx",
    "lorem ipsum",
]

no_comment_count = df.filter(
    lower(trim(col("text"))).isin(no_comment_patterns) & col("text").isNotNull()
).count()
status = "FINDING" if no_comment_count > 0 else "PASS"
icon = "⚠️" if no_comment_count > 0 else "✅"
print(f"   {icon} Reviews with no-comment placeholders: {no_comment_count:,} — {status}")
if no_comment_count > 0:
    samples = df.filter(
        lower(trim(col("text"))).isin(no_comment_patterns)
    ).select("text", "stars").limit(10).collect()
    print(f"      Samples (text, stars): {[(r['text'], r['stars']) for r in samples]}")
    print(f"      → Overlaps with DQ-REVIEW-004 (placeholder reviews)")

findings_636.append({
    "Column": "text (no-comment)", "Masked Nulls": no_comment_count,
    "Whitespace-Only": 0, "Total": no_comment_count, "Status": status
})

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_636_df = pd.DataFrame(findings_636)
display(findings_636_df)
v = sum(1 for f in findings_636 if f["Status"] == "FINDING")
print(f"\n{len(findings_636)} checks performed, {v} finding(s)")
if v == 0:
    print("✅ No masked null values detected — all nulls are properly represented.")
else:
    print(f"⚠️  {v} check(s) found masked nulls or placeholders — review above.")
print("NULL REPRESENTATION CONSISTENCY CHECK COMPLETE")
print("=" * 80)

6.3.6) NULL REPRESENTATION CONSISTENCY  —  Review Table (Pre-Remodel)
    Detecting non-standard nulls masquerading as data

────────────────────────────────────────────────────────────
📌 Check 1: Masked null tokens across all string columns
   Checking 34 null-like patterns (case-insensitive)
   ✅ review_id: no masked nulls
   ✅ user_id: no masked nulls
   ✅ business_id: no masked nulls
   ⚠️  text: 11 masked nulls — values: ['*', 'Tbd', '.']
   ✅ date: no masked nulls

────────────────────────────────────────────────────────────
📌 Check 2: Review-specific 'no-comment' placeholders in text
   ⚠️ Reviews with no-comment placeholders: 10 — FINDING
      Samples (text, stars): [('-----', 1.0), ('..', 4.0), ('..', 3.0), ('..', 1.0), ('...', 1.0), ('...', 1.0), ('...', 1.0), ('...', 2.0), ('..', 3.0), ('...', 1.0)]
      → Overlaps with DQ-REVIEW-004 (placeholder reviews)

SUMMARY


Column,Masked Nulls,Whitespace-Only,Total,Status
review_id,0,0,0,PASS
user_id,0,0,0,PASS
business_id,0,0,0,PASS
text,11,0,11,FINDING
date,0,0,0,PASS
text (no-comment),10,0,10,FINDING



6 checks performed, 2 finding(s)
⚠️  2 check(s) found masked nulls or placeholders — review above.
NULL REPRESENTATION CONSISTENCY CHECK COMPLETE


---
#### ⚠️ DQ-REVIEW-011 — Masked Null Tokens in Review Text

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-011` |
| **Severity** | FINDING (data integrity — non-standard null representations) |
| **Detected In** | Phase 6.3.6 — Null Representation Consistency, Check 1 (Cell 53) |
| **Affected Column** | `text` |
| **Affected Rows** | **11** out of 6,990,280 (0.00016%) |
| **Values Found** | `*`, `Tbd`, `.` |

**Description:** 11 reviews contain text that matches well-known null-masking tokens: a single asterisk (`*`), the abbreviation `Tbd` ("to be determined"), and a lone period (`.`). These are not genuine review content — they represent cases where the review body was left effectively blank but a placeholder character was entered instead of a proper NULL. They will pollute NLP pipelines with noise tokens and inflate non-null row counts, masking the true completeness picture.

**Suggested Solutions:**

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Convert to proper NULL** (`SET text = NULL WHERE LOWER(TRIM(text)) IN ('*', 'tbd', '.')`) | Makes missingness explicit and correctable in section 6.4 Completeness Checks. Standard null-normalisation practice. |
| 2 | **Flag with `is_masked_null = True`** | Non-destructive; preserves the original value for investigation while letting downstream pipelines exclude flagged rows. |
| 3 | **Delete the 11 rows** | Overkill — the star rating and vote metadata on these rows are still valid. |

**Best Logical Solution:** **Option 1 — Convert to proper NULL at the Silver layer.** These are unambiguously non-content values. Converting them to NULL makes the dataset’s completeness semantics honest and allows NLP pipelines to exclude them via a simple `WHERE text IS NOT NULL` filter instead of maintaining a list of masked-null patterns.

---

#### 🔁 6.3.6 Confirmation: DQ-REVIEW-004 (Placeholder Reviews)

Phase 6.3.6 Null Representation Consistency, Check 2 (Cell 53) independently detected **10 no-comment placeholder reviews** (`-----`, `..`, `...`). These are a **subset** of the 18 placeholder/test-data reviews already fully documented under **DQ-REVIEW-004** after 6.2.3 (Cell 42).

- The 10 rows detected here used the no-comment pattern list (`-----`, `..`, `...`, etc.).
- The original DQ-REVIEW-004 detection used a broader placeholder pattern list and captured all 18.
- No new remediation is needed beyond what DQ-REVIEW-004 already prescribes.

---


### 6.4) COMPLETENESS CHECKS
*Measure missingness after null normalization. Understand patterns and impact.*

- **Missingness Measurement**
  - *Sub:* Per column — null percentage per column.
  - *Sub:* Per row — rows with excessive missing values (e.g., >50% null).
  - *Sub:* Per cell — total null rate after normalizing variants from 6.3.

- **Missingness Pattern Analysis**
  - *Sub:* Randomness classification — MCAR / MAR / MNAR.
  - *Sub:* Missing value correlation — missingness in `text` correlated with missingness in `stars` (e.g., are reviews missing text when they also lack a rating?).
  - *Sub:* Temporal gaps — missing time periods in time-series data (e.g., no reviews in certain months during 2005–2022).

- **Critical Field Coverage** — Required fields (per business rules) must be non-null.
  - *Example:* `review_id`, `user_id`, `business_id`, `stars`, `text`, and `date` must never be null for a valid review record.

In [0]:
# 6.4.1) Per-Column Missingness
# ─────────────────────────────────────────────────────────────────────
# Null percentage per column — single-pass aggregation (optimized).
# Note: Section 4.6 performed initial null analysis on the raw 9-column review df;
# this re-evaluates on the review dataset after null-normalization checks from section 6.3.

from pyspark.sql.functions import (
    sum as _sum, when, col, lit, count as _count, mean, corr,
    min as spark_min, max as spark_max
)
import pandas as pd

print("=" * 80)
print("6.4.1) PER-COLUMN MISSINGNESS")
print("    Null percentage per column (single-pass optimized)")
print("    ℹ️  Section 4.6 performed basic null counts on the raw 9-col review df;")
print("       this re-evaluates on the review dataset after null-normalization.")
print("=" * 80)

total_rows = df.count()
total_cols = len(df.columns)

# ── Optimized: single aggregation pass for ALL null counts ──
null_agg_exprs = [
    _sum(when(col(c).isNull(), lit(1)).otherwise(lit(0))).alias(c)
    for c in df.columns
]
null_counts_row = df.agg(*null_agg_exprs).collect()[0]

col_null_data = []
for c in df.columns:
    null_cnt = null_counts_row[c] or 0
    pct = (null_cnt / total_rows * 100) if total_rows > 0 else 0
    severity = "NONE" if null_cnt == 0 else ("LOW" if pct < 1 else ("MEDIUM" if pct < 5 else "HIGH"))
    icon = "✅" if null_cnt == 0 else ("⚠️ " if pct < 5 else "🚨")
    print(f"   {icon} {c}: {null_cnt:,} nulls ({pct:.2f}%) — Severity: {severity}")
    col_null_data.append({
        "Column": c, "Null Count": null_cnt, "Null %": round(pct, 2),
        "Severity": severity, "Status": "PASS" if null_cnt == 0 else "FINDING"
    })

col_null_df = pd.DataFrame(col_null_data)
print("\n")
display(col_null_df)

cols_with_nulls_list = [d["Column"] for d in col_null_data if d["Null Count"] > 0]
print(f"\n   Columns with nulls: {len(cols_with_nulls_list)} / {total_cols}")
if not cols_with_nulls_list:
    print("   ✅ All columns are fully populated — no per-column missingness issues.")
else:
    print(f"   ⚠️  {len(cols_with_nulls_list)} column(s) have nulls: {cols_with_nulls_list}")
print("=" * 80)

6.4.1) PER-COLUMN MISSINGNESS
    Null percentage per column (single-pass optimized)
    ℹ️  Section 4.6 performed basic null counts on the raw 9-col review df;
       this re-evaluates on the review dataset after null-normalization.
   ✅ review_id: 0 nulls (0.00%) — Severity: NONE
   ✅ user_id: 0 nulls (0.00%) — Severity: NONE
   ✅ business_id: 0 nulls (0.00%) — Severity: NONE
   ✅ stars: 0 nulls (0.00%) — Severity: NONE
   ✅ useful: 0 nulls (0.00%) — Severity: NONE
   ✅ funny: 0 nulls (0.00%) — Severity: NONE
   ✅ cool: 0 nulls (0.00%) — Severity: NONE
   ✅ text: 0 nulls (0.00%) — Severity: NONE
   ✅ date: 0 nulls (0.00%) — Severity: NONE




Column,Null Count,Null %,Severity,Status
review_id,0,0.0,NONE,PASS
user_id,0,0.0,NONE,PASS
business_id,0,0.0,NONE,PASS
stars,0,0.0,NONE,PASS
useful,0,0.0,NONE,PASS
funny,0,0.0,NONE,PASS
cool,0,0.0,NONE,PASS
text,0,0.0,NONE,PASS
date,0,0.0,NONE,PASS



   Columns with nulls: 0 / 9
   ✅ All columns are fully populated — no per-column missingness issues.


In [0]:
# 6.4.2) Per-Row Missingness
# ─────────────────────────────────────────────────────────────────────
# Rows with excessive missing values flagged at 25% and 50% thresholds.

print("=" * 80)
print("6.4.2) PER-ROW MISSINGNESS")
print("    Identifying rows with excessive null values")
print("=" * 80)

# Count nulls per row (vectorized expression)
null_per_row_expr = sum([when(col(c).isNull(), lit(1)).otherwise(lit(0)) for c in df.columns])
df_with_null_count = df.withColumn("_null_count", null_per_row_expr)

# Multiple thresholds
threshold_50 = int(total_cols * 0.50)
threshold_25 = int(total_cols * 0.25)

# Single-pass aggregation for all row-level metrics
row_metrics = df_with_null_count.agg(
    _sum(when(col("_null_count") > 0, lit(1)).otherwise(lit(0))).alias("rows_any_null"),
    _sum(when(col("_null_count") > threshold_25, lit(1)).otherwise(lit(0))).alias("rows_25"),
    _sum(when(col("_null_count") > threshold_50, lit(1)).otherwise(lit(0))).alias("rows_50"),
    spark_max("_null_count").alias("max_nulls_in_row"),
    mean("_null_count").alias("avg_nulls_per_row")
).collect()[0]

rows_any_null = row_metrics["rows_any_null"] or 0
rows_25 = row_metrics["rows_25"] or 0
rows_50 = row_metrics["rows_50"] or 0
max_nulls = row_metrics["max_nulls_in_row"] or 0
avg_nulls = row_metrics["avg_nulls_per_row"] or 0

print(f"   Total columns: {total_cols}  |  Total rows: {total_rows:,}")
print(f"\n   Rows with any null:           {rows_any_null:,}  ({rows_any_null/total_rows*100:.2f}%)")
print(f"   Rows with >25% null (>{threshold_25} cols): {rows_25:,}  ({rows_25/total_rows*100:.2f}%)")
print(f"   Rows with >50% null (>{threshold_50} cols): {rows_50:,}  ({rows_50/total_rows*100:.2f}%)")
print(f"   Max nulls in a single row:    {max_nulls}")
print(f"   Avg nulls per row:            {avg_nulls:.2f}")

# Distribution of null counts per row
print("\n   ── Null-count distribution per row ──")
null_dist = df_with_null_count.groupBy("_null_count").count().orderBy("_null_count")
display(null_dist)

if rows_any_null == 0:
    print("\n   ✅ No rows have any null values — dataset is row-complete.")
elif rows_50 == 0:
    print(f"\n   ⚠️  No excessively incomplete rows (>50%), but {rows_any_null:,} rows have some nulls.")
else:
    print(f"\n   🚨 {rows_50:,} rows exceed 50% null threshold — consider dropping or imputing.")
print("=" * 80)

6.4.2) PER-ROW MISSINGNESS
    Identifying rows with excessive null values
   Total columns: 9  |  Total rows: 6,990,280

   Rows with any null:           0  (0.00%)
   Rows with >25% null (>2 cols): 0  (0.00%)
   Rows with >50% null (>4 cols): 0  (0.00%)
   Max nulls in a single row:    0
   Avg nulls per row:            0.00

   ── Null-count distribution per row ──


_null_count,count
0,6990280



   ✅ No rows have any null values — dataset is row-complete.


In [0]:
# 6.4.3) Per-Cell Null Rate
# ─────────────────────────────────────────────────────────────────────
# Overall null rate across the entire dataset matrix (rows × columns)
# after null normalization from section 6.3.

print("=" * 80)
print("6.4.3) PER-CELL NULL RATE")
print("    Total null rate across the entire dataset matrix")
print("=" * 80)

total_null_cells = sum(d["Null Count"] for d in col_null_data)
total_cells = total_rows * total_cols
null_rate = (total_null_cells / total_cells * 100) if total_cells > 0 else 0
completeness = 100.0 - null_rate

print(f"   Total cells (rows × columns): {total_rows:,} × {total_cols} = {total_cells:,}")
print(f"   Null cells:        {total_null_cells:,}")
print(f"   Null rate:         {null_rate:.4f}%")
print(f"   Completeness rate: {completeness:.4f}%")

# Severity classification
if null_rate == 0:
    print("\n   ✅ 100.00% cell completeness — zero null values in the dataset.")
elif null_rate < 1:
    print(f"\n   ✅ Very low null rate ({null_rate:.4f}%) — dataset is near-complete.")
elif null_rate < 5:
    print(f"\n   ⚠️  Moderate null rate ({null_rate:.2f}%) — investigate column-level patterns.")
else:
    print(f"\n   🚨 High null rate ({null_rate:.2f}%) — significant data loss risk.")
print("=" * 80)

6.4.3) PER-CELL NULL RATE
    Total null rate across the entire dataset matrix
   Total cells (rows × columns): 6,990,280 × 9 = 62,912,520
   Null cells:        0
   Null rate:         0.0000%
   Completeness rate: 100.0000%

   ✅ 100.00% cell completeness — zero null values in the dataset.


In [0]:
# 6.4.4) Missingness Pattern Analysis (MCAR / MAR / MNAR)
# ─────────────────────────────────────────────────────────────────────
# Classify missingness randomness:
#   MCAR — Missing Completely At Random (null independent of any data)
#   MAR  — Missing At Random (null depends on other observed columns)
#   MNAR — Missing Not At Random (null depends on the missing value itself)

print("=" * 80)
print("6.4.4) MISSINGNESS PATTERN ANALYSIS (MCAR / MAR / MNAR)")
print("    Classifying the randomness of null patterns")
print("=" * 80)

if not cols_with_nulls_list:
    print("\n   ✅ No columns have nulls — missingness pattern analysis is NOT APPLICABLE.")
    print("   Classification: N/A (dataset is fully complete)")
else:
    # Build null-indicator columns (1 = null, 0 = present)
    df_null_ind = df
    null_ind_names = []
    for c in cols_with_nulls_list:
        ind_name = f"_null_{c}"
        df_null_ind = df_null_ind.withColumn(ind_name, when(col(c).isNull(), lit(1)).otherwise(lit(0)))
        null_ind_names.append(ind_name)

    # ── MCAR heuristic: split-consistency ──
    # If null rate is consistent across random partitions → supports MCAR
    print(f"\n   Columns with nulls: {cols_with_nulls_list}")
    print(f"\n   ── MCAR Heuristic: Random-split consistency ──")
    print(f"   (Null rate should be similar in both halves if MCAR)")

    splits = df_null_ind.randomSplit([0.5, 0.5], seed=42)
    # Single-pass for both splits
    agg_a = [mean(col(f"_null_{c}")).alias(c) for c in cols_with_nulls_list]
    agg_b = [mean(col(f"_null_{c}")).alias(c) for c in cols_with_nulls_list]
    rates_a = splits[0].agg(*agg_a).collect()[0]
    rates_b = splits[1].agg(*agg_b).collect()[0]

    mcar_results = []
    for c in cols_with_nulls_list:
        rate_a = rates_a[c] or 0
        rate_b = rates_b[c] or 0
        diff = abs(rate_a - rate_b)
        verdict = "Likely MCAR" if diff < 0.02 else "Possibly MAR/MNAR"
        icon = "✅" if diff < 0.02 else "⚠️ "
        print(f"   {icon} {c}: Split A = {rate_a:.4f}, Split B = {rate_b:.4f}, Δ = {diff:.4f} → {verdict}")
        mcar_results.append({"Column": c, "Split A Rate": round(rate_a, 4),
                             "Split B Rate": round(rate_b, 4), "Delta": round(diff, 4), "Verdict": verdict})

    display(pd.DataFrame(mcar_results))

    # ── MAR heuristic: correlate null indicators with observed numeric columns ──
    numeric_cols = [
        f.name for f in df.schema.fields
        if f.dataType.simpleString() in ('int', 'bigint', 'double', 'float')
        and f.name not in cols_with_nulls_list
    ]

    if numeric_cols:
        print(f"\n   ── MAR Heuristic: Null-indicator × numeric-column correlation ──")
        print(f"   (|r| > 0.3 suggests missingness depends on an observed column → MAR)")
        # Batch correlation computation
        corr_exprs = []
        corr_labels = []
        for ind in null_ind_names[:5]:
            for nc in numeric_cols[:10]:
                corr_exprs.append(corr(col(ind), col(nc)).alias(f"{ind}__{nc}"))
                corr_labels.append((ind, nc))

        if corr_exprs:
            corr_row = df_null_ind.agg(*corr_exprs).collect()[0]
            mar_found = False
            for (ind, nc), label in zip(corr_labels, [f"{i}__{n}" for i, n in corr_labels]):
                r = corr_row[label]
                if r is not None and abs(r) > 0.3:
                    src = ind.replace("_null_", "")
                    print(f"   ⚠️  Null in '{src}' correlated with '{nc}': r = {r:.3f} → Suggests MAR")
                    mar_found = True
            if not mar_found:
                print(f"   ✅ No strong correlations found — no MAR evidence from numeric columns.")
    else:
        print(f"\n   No numeric columns available for MAR correlation test.")

    print(f"\n   Note: MNAR cannot be confirmed from observed data alone (by definition).")

print("=" * 80)

6.4.4) MISSINGNESS PATTERN ANALYSIS (MCAR / MAR / MNAR)
    Classifying the randomness of null patterns

   ✅ No columns have nulls — missingness pattern analysis is NOT APPLICABLE.
   Classification: N/A (dataset is fully complete)


In [0]:
# 6.4.5) Missing Value Correlation
# ─────────────────────────────────────────────────────────────────────
# Detect if nulls in one column predict nulls in another column.
# e.g., missingness in 'income' correlated with missingness in 'employment_status'.

print("=" * 80)
print("6.4.5) MISSING VALUE CORRELATION")
print("    Checking if nulls in one column are correlated with nulls in another")
print("=" * 80)

if not cols_with_nulls_list:
    print("\n   ✅ No columns have nulls — missing value correlation is NOT APPLICABLE.")
elif len(cols_with_nulls_list) < 2:
    print(f"\n   Only 1 column with nulls ({cols_with_nulls_list[0]}) — pairwise correlation requires ≥ 2.")
    print("   Skipping.")
else:
    # Reuse df_null_ind from 6.4.4
    # Batch all pairwise correlations in a single aggregation
    pair_corr_exprs = []
    pair_labels = []
    for i, c1 in enumerate(cols_with_nulls_list):
        for c2 in cols_with_nulls_list[i+1:]:
            pair_corr_exprs.append(corr(f"_null_{c1}", f"_null_{c2}").alias(f"{c1}__{c2}"))
            pair_labels.append((c1, c2))

    corr_results = df_null_ind.agg(*pair_corr_exprs).collect()[0]
    correlated_pairs = []
    for c1, c2 in pair_labels:
        r = corr_results[f"{c1}__{c2}"]
        if r is not None and abs(r) > 0.3:
            correlated_pairs.append({
                "Column A": c1, "Column B": c2,
                "Correlation": round(r, 3),
                "Strength": "Strong" if abs(r) > 0.7 else "Moderate"
            })

    print(f"\n   Pairs checked: {len(pair_labels)}")
    if correlated_pairs:
        print(f"   ⚠️  {len(correlated_pairs)} correlated null-pair(s) found (|r| > 0.3):")
        display(pd.DataFrame(correlated_pairs))
        print("   → Correlated missingness suggests a shared root cause or structural dependency.")
    else:
        print(f"   ✅ No strong null correlations found — missingness appears independent across columns.")

print("=" * 80)

6.4.5) MISSING VALUE CORRELATION
    Checking if nulls in one column are correlated with nulls in another

   ✅ No columns have nulls — missing value correlation is NOT APPLICABLE.


In [0]:
# 6.4.6) Temporal Gap Analysis  —  REVIEW TABLE ADAPTED
# ─────────────────────────────────────────────────────────────────────
# Detect missing time periods (months/years with no review data).
# The raw review table has a `date` string column ("YYYY-MM-DD HH:MM:SS").
# We parse year/month inline — no pre-existing temporal columns needed.
# For review time-series data, gaps may indicate data collection issues
# or partial dataset boundaries.
#
# Checks performed:
#   1) Year-month gap grid — expected vs. present months
#   2) Yearly review volume — detect sparse/boundary years
#   3) Monthly volume — detect seasonal patterns & collection anomalies
# ─────────────────────────────────────────────────────────────────────

from collections import defaultdict
from pyspark.sql.functions import (
    year as _year, month as _month, to_timestamp
)

print("=" * 80)
print("6.4.6) TEMPORAL GAP ANALYSIS  —  Review Table (Pre-Remodel)")
print("    Detecting missing time periods in the review date timeline")
print("=" * 80)

if "date" not in df.columns:
    print("   ⚠️  `date` column not available — skipping temporal analysis.")
else:
    # ── Parse year and month from the date string column ──
    df_temporal = df.withColumn(
        "_review_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")
    ).withColumn(
        "_review_year", _year("_review_ts")
    ).withColumn(
        "_review_month", _month("_review_ts")
    )

    # ── Get data range in single pass ──
    range_row = df_temporal.agg(
        spark_min("_review_year").alias("min_y"),
        spark_max("_review_year").alias("max_y")
    ).collect()[0]
    min_year, max_year = range_row["min_y"], range_row["max_y"]
    print(f"   Data year range: {min_year} – {max_year}")

    # ── Collect all present (year, month) pairs ──
    present_ym = set(
        (r["_review_year"], r["_review_month"])
        for r in df_temporal.select("_review_year", "_review_month").distinct().collect()
    )

    # ── Generate full expected grid ──
    expected_ym = {(y, m) for y in range(min_year, max_year + 1) for m in range(1, 13)}
    missing_ym = sorted(expected_ym - present_ym)

    print(f"   Expected year-months: {len(expected_ym)}")
    print(f"   Present year-months:  {len(present_ym)}")
    print(f"   Missing year-months:  {len(missing_ym)}")

    if missing_ym:
        print(f"\n   ⚠️  Temporal gaps detected:")
        gaps_by_year = defaultdict(list)
        for y, m in missing_ym:
            gaps_by_year[y].append(m)
        for y in sorted(gaps_by_year):
            months_str = ", ".join(str(m) for m in gaps_by_year[y])
            gap_pct = len(gaps_by_year[y]) / 12 * 100
            print(f"      {y}: month(s) {months_str}  ({gap_pct:.0f}% of year missing)")

        gap_df = pd.DataFrame(missing_ym, columns=["Year", "Month"])
        display(gap_df)
    else:
        print("\n   ✅ No temporal gaps — all months within the year range have review data.")

    # ── Yearly review volume — detect sparse/boundary years ──
    print(f"\n{'─'*60}")
    print("📌 Yearly review volume (detect sparse years)")
    yearly_vol = df_temporal.groupBy("_review_year").count().orderBy("_review_year")
    yearly_vol_display = yearly_vol.withColumnRenamed("_review_year", "review_year")
    display(yearly_vol_display)

    # Compute volume stats for anomaly detection
    yearly_pd = yearly_vol.toPandas()
    yearly_pd.columns = ["review_year", "count"]
    if len(yearly_pd) > 2:
        median_vol = yearly_pd["count"].median()
        sparse_years = yearly_pd[yearly_pd["count"] < median_vol * 0.1]
        if not sparse_years.empty:
            print(f"\n   ⚠️  Sparse years (<10% of median volume {median_vol:,.0f}):")
            for _, row in sparse_years.iterrows():
                print(f"      {int(row['review_year'])}: {int(row['count']):,} reviews")
        else:
            print(f"   ✅ No anomalously sparse years detected (median volume: {median_vol:,.0f}).")

    # ── Monthly review volume — detect seasonal patterns & collection gaps ──
    print(f"\n{'─'*60}")
    print("📌 Monthly review volume (detect seasonal patterns)")
    monthly_vol = (
        df_temporal.groupBy("_review_year", "_review_month")
        .count()
        .orderBy("_review_year", "_review_month")
    )
    monthly_pd = monthly_vol.toPandas()
    monthly_pd.columns = ["Year", "Month", "Review Count"]
    display(monthly_pd)

    # Flag months with anomalously low volume
    if len(monthly_pd) > 6:
        month_median = monthly_pd["Review Count"].median()
        anomalous = monthly_pd[monthly_pd["Review Count"] < month_median * 0.05]
        if not anomalous.empty:
            print(f"\n   ⚠️  Anomalously low months (<5% of median {month_median:,.0f}):")
            for _, row in anomalous.iterrows():
                print(f"      {int(row['Year'])}-{int(row['Month']):02d}: {int(row['Review Count']):,} reviews")
        else:
            print(f"   ✅ No anomalously low months detected (median monthly volume: {month_median:,.0f}).")

print("=" * 80)

6.4.6) TEMPORAL GAP ANALYSIS  —  Review Table (Pre-Remodel)
    Detecting missing time periods in the review date timeline
   Data year range: 2005 – 2022
   Expected year-months: 216
   Present year-months:  204
   Missing year-months:  12

   ⚠️  Temporal gaps detected:
      2005: month(s) 1  (8% of year missing)
      2022: month(s) 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12  (92% of year missing)


Year,Month
2005,1
2022,2
2022,3
2022,4
2022,5
2022,6
2022,7
2022,8
2022,9
2022,10



────────────────────────────────────────────────────────────
📌 Yearly review volume (detect sparse years)


review_year,count
2005,854
2006,3853
2007,15363
2008,48226
2009,74387
2010,138587
2011,230813
2012,286570
2013,383950
2014,522275



   ⚠️  Sparse years (<10% of median volume 335,260):
      2005: 854 reviews
      2006: 3,853 reviews
      2007: 15,363 reviews
      2022: 31,665 reviews

────────────────────────────────────────────────────────────
📌 Monthly review volume (detect seasonal patterns)


Year,Month,Review Count
2005,2,3
2005,3,74
2005,4,26
2005,5,108
2005,6,38
2005,7,259
2005,8,66
2005,9,69
2005,10,39
2005,11,70



   ⚠️  Anomalously low months (<5% of median 31,461):
      2005-02: 3 reviews
      2005-03: 74 reviews
      2005-04: 26 reviews
      2005-05: 108 reviews
      2005-06: 38 reviews
      2005-07: 259 reviews
      2005-08: 66 reviews
      2005-09: 69 reviews
      2005-10: 39 reviews
      2005-11: 70 reviews
      2005-12: 102 reviews
      2006-01: 221 reviews
      2006-02: 185 reviews
      2006-03: 191 reviews
      2006-04: 196 reviews
      2006-05: 224 reviews
      2006-06: 180 reviews
      2006-07: 389 reviews
      2006-08: 492 reviews
      2006-09: 461 reviews
      2006-10: 433 reviews
      2006-11: 426 reviews
      2006-12: 455 reviews
      2007-01: 802 reviews
      2007-02: 1,420 reviews
      2007-03: 1,219 reviews
      2007-04: 1,111 reviews
      2007-05: 908 reviews
      2007-06: 1,141 reviews
      2007-09: 1,442 reviews
      2007-10: 1,429 reviews
      2007-11: 1,377 reviews
      2007-12: 1,189 reviews


---
#### 🟠 DQ-REVIEW-012 — Temporal Gaps: Incomplete Boundary Years (2005 & 2022)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-012` |
| **Severity** | 🟠 **Medium** — the core dataset (2006–2021) has all 192 expected year-months fully covered; however, the two boundary years are incomplete and will distort year-over-year and seasonal trend analyses if included without adjustment. |
| **Detected In** | Phase 6.4 — Completeness Checks → **6.4.6) Temporal Gap Analysis** (Cell 61) |
| **Affected Column** | `date` (parsed to year/month) |
| **Affected Rows** | **2005:** 854 rows (0.012%); **2022:** 31,665 rows (0.453%) — combined 32,519 of 6,990,280 total rows (0.47%) |

**Description:** The review dataset spans **2005–2022** (18 years) but has **12 missing year-months** concentrated at the temporal boundaries:

* **2005:** January is missing — the earliest review is February 2005, with only **3 reviews** that month. The entire year has just **854 reviews**, confirming it represents the Yelp platform’s launch ramp rather than a data collection gap.
* **2022:** February through December are missing — only January 2022 is present, with **31,665 reviews**. This represents a dataset extraction cutoff, not a data quality issue.

Additionally, **4 years** fall below 10% of the median annual volume (335,260):

| Year | Reviews | % of Median | Nature |
| --- | --- | --- | --- |
| 2005 | 854 | 0.25% | Platform launch — negligible volume |
| 2006 | 3,853 | 1.15% | Early growth — statistically fragile |
| 2007 | 15,363 | 4.58% | Growth phase — approaching viability |
| 2022 | 31,665 | 9.45% | Extraction cutoff — January only |

> **📝 Contextual Note — COVID-19 Impact:** A visible volume dip appears in **2020** (554,557 reviews, \~39% drop from the 2019 peak of 907,284) and **2021** (618,189). This is consistent with the pandemic’s impact on restaurant foot traffic and is a **real-world phenomenon**, not a data quality issue. It should be acknowledged in temporal trend analyses but does not require remediation.

---

##### 💡 Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Filter to complete years (2006–2021) for temporal analyses** | Removes both boundary years, ensuring all year-over-year and seasonal comparisons use complete 12-month periods. Simplest approach — eliminates distortion with no schema changes. Discards 32,519 valid review records from temporal queries. |
| 2 | **Retain all years but add an `is_complete_year` boolean flag** | Preserves the full dataset while giving downstream consumers a queryable indicator to filter at will. Temporal trend queries filter on `is_complete_year = True`; exploratory or business-level queries retain full access to boundary data. Non-destructive and metadata-driven. |
| 3 | **Exclude 2005 entirely; annotate 2022 with a cutoff marker** | 2005 has only 854 records — statistically meaningless for trend analysis. 2022’s 31,665 January records carry analytical value for month-level or business-level questions. This hybrid approach removes only the truly negligible boundary. |
| 4 | **Document as a known limitation — no transformation** | If the cleaning pipeline is scope-limited to structural/value fixes, document the boundary-year incompleteness as a known limitation for downstream analysts. |

##### ✅ Recommended Solution

**Solution 2 — Add an `is_complete_year` flag column.** This preserves every record while providing an explicit, queryable indicator of year completeness. Temporal trend analyses filter to `is_complete_year = True` (yielding the clean 2006–2021 window) for reliable year-over-year and seasonal comparisons, while exploratory analyses retain full access. The decision stays in the downstream analyst’s hands — critical for a reusable Bronze → Silver transformation.

---

#### ℹ️ DQ-REVIEW-013 — Sparse Early Years: Platform Growth Phase (2005–2008)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-013` |
| **Severity** | ℹ️ **Info** (advisory — not a data error, platform growth artefact) |
| **Detected In** | Phase 6.4 — Completeness Checks → **6.4.6) Temporal Gap Analysis** (Cell 61) |
| **Affected Rows** | 2005–2008 combined: **68,296 rows** (0.98% of dataset) |

**Description:** The first four years of the dataset (2005–2008) represent Yelp’s early platform growth phase. Monthly volumes are extremely low — 2005 averages **78 reviews/month**, while the mature period (2015–2019) averages **\~63,000 reviews/month**. These years are all present (11/12 months in 2005, all 12 in 2006–2008) but are statistically fragile for any per-month or per-business aggregation.

| Signal | 2005 | 2006 | 2007 | 2008 | 2009+ |
| --- | --- | --- | --- | --- | --- |
| Annual volume | 854 | 3,853 | 15,363 | 48,226 | 74,387+ |
| Avg monthly volume | 78 | 321 | 1,280 | 4,019 | 6,199+ |
| % of median year | 0.25% | 1.15% | 4.58% | 14.39% | 22.19%+ |

**Suggested Handling:** No remediation needed. These years reflect genuine platform adoption and should be retained. However, downstream analyses involving per-month, per-business, or per-user aggregations should consider setting a **minimum-volume threshold** (e.g., 2010+ for reliable trend analysis) or weighting early observations accordingly.

---

In [0]:
# 6.4.7) Critical Field Coverage
# ─────────────────────────────────────────────────────────────────────
# Required fields (per business rules) must be 100% non-null.

print("=" * 80)
print("6.4.7) CRITICAL FIELD COVERAGE")
print("    Required fields must be non-null per business rules")
print("=" * 80)

# Review table critical fields — these define a valid review record:
#   review_id    — primary key, unique identifier
#   user_id      — foreign key to user (attribution)
#   business_id  — foreign key to business (target of the review)
#   stars        — the rating, core analytic metric
#   text         — the review body, primary NLP input
#   date         — temporal anchor, required for time-series analysis
# Note: useful/funny/cool are vote counts — legitimately 0 but structurally non-critical.
critical_fields = ["review_id", "user_id", "business_id", "stars", "text", "date"]
available_critical = [c for c in critical_fields if c in df.columns]
missing_critical_cols = [c for c in critical_fields if c not in df.columns]

if missing_critical_cols:
    print(f"   🚨 Critical fields NOT in schema: {missing_critical_cols}")

# Single-pass null count for critical fields
crit_null_exprs = [
    _sum(when(col(c).isNull(), lit(1)).otherwise(lit(0))).alias(c)
    for c in available_critical
]
crit_row = df.agg(*crit_null_exprs).collect()[0]

critical_findings = []
for c in available_critical:
    null_cnt = crit_row[c] or 0
    pct = (null_cnt / total_rows * 100) if total_rows > 0 else 0
    status = "PASS" if null_cnt == 0 else "CRITICAL FINDING"
    icon = "✅" if null_cnt == 0 else "🚨"
    print(f"   {icon} {c}: {null_cnt:,} nulls ({pct:.2f}%) — {status}")
    critical_findings.append({
        "Critical Field": c, "Null Count": null_cnt,
        "Null %": round(pct, 2), "Status": status
    })

print("\n")
display(pd.DataFrame(critical_findings))

violations = sum(1 for f in critical_findings if f["Status"] != "PASS")
if violations == 0 and not missing_critical_cols:
    print("   ✅ All critical fields are 100% populated — no coverage violations.")
else:
    print(f"   🚨 {violations} critical field violation(s) detected — data integrity at risk.")
print("=" * 80)

6.4.7) CRITICAL FIELD COVERAGE
    Required fields must be non-null per business rules
   ✅ review_id: 0 nulls (0.00%) — PASS
   ✅ user_id: 0 nulls (0.00%) — PASS
   ✅ business_id: 0 nulls (0.00%) — PASS
   ✅ stars: 0 nulls (0.00%) — PASS
   ✅ text: 0 nulls (0.00%) — PASS
   ✅ date: 0 nulls (0.00%) — PASS




Critical Field,Null Count,Null %,Status
review_id,0,0.0,PASS
user_id,0,0.0,PASS
business_id,0,0.0,PASS
stars,0,0.0,PASS
text,0,0.0,PASS
date,0,0.0,PASS


   ✅ All critical fields are 100% populated — no coverage violations.


In [0]:
# 6.4.8) Completeness Summary
# ─────────────────────────────────────────────────────────────────────
# Consolidated summary of all completeness checks (6.4.1 – 6.4.7).

print("=" * 80)
print("6.4) COMPLETENESS CHECKS — CONSOLIDATED SUMMARY")
print("=" * 80)

# Gather metrics from previous cells
n_cols_with_nulls = len(cols_with_nulls_list)
n_critical_violations = sum(1 for f in critical_findings if f["Status"] != "PASS")

summary_data = [
    {"Check": "6.4.1 Per-Column Missingness",    "Result": f"{n_cols_with_nulls}/{total_cols} columns with nulls",              "Status": "PASS" if n_cols_with_nulls == 0 else "FINDING"},
    {"Check": "6.4.2 Per-Row Missingness",        "Result": f"{rows_any_null:,} rows with nulls, {rows_50:,} >50%",            "Status": "PASS" if rows_50 == 0 else "FINDING"},
    {"Check": "6.4.3 Per-Cell Null Rate",          "Result": f"{null_rate:.4f}% null ({completeness:.4f}% complete)",            "Status": "PASS" if null_rate == 0 else ("FINDING" if null_rate < 5 else "CRITICAL")},
    {"Check": "6.4.4 MCAR/MAR/MNAR Analysis",     "Result": "N/A (no nulls)" if not cols_with_nulls_list else "See 6.4.4",    "Status": "PASS" if not cols_with_nulls_list else "REVIEW"},
    {"Check": "6.4.5 Missing Value Correlation",   "Result": "N/A (no nulls)" if not cols_with_nulls_list else "See 6.4.5",    "Status": "PASS" if not cols_with_nulls_list else "REVIEW"},
    {"Check": "6.4.6 Temporal Gap Analysis",       "Result": "See 6.4.6 details",                                               "Status": "REVIEW"},
    {"Check": "6.4.7 Critical Field Coverage",     "Result": f"{n_critical_violations} violation(s)",                            "Status": "PASS" if n_critical_violations == 0 else "CRITICAL"},
]

summary_df_result = pd.DataFrame(summary_data)
display(summary_df_result)

print(f"\n   Overall null cell rate:   {total_null_cells:,} / {total_cells:,} ({null_rate:.4f}%)")
print(f"   Rows with >50% null:      {rows_50:,}")
print(f"   Critical field violations: {n_critical_violations}")

all_pass = (n_cols_with_nulls == 0 and rows_50 == 0 and n_critical_violations == 0)
if all_pass:
    print("\n   ✅ Dataset is FULLY COMPLETE — zero nulls across all columns, rows, and critical fields.")
else:
    print("\n   ⚠️  Completeness issues detected — review individual check results above.")

print("\nCOMPLETENESS CHECKS COMPLETE")
print("=" * 80)

6.4) COMPLETENESS CHECKS — CONSOLIDATED SUMMARY


Check,Result,Status
6.4.1 Per-Column Missingness,0/9 columns with nulls,PASS
6.4.2 Per-Row Missingness,"0 rows with nulls, 0 >50%",PASS
6.4.3 Per-Cell Null Rate,0.0000% null (100.0000% complete),PASS
6.4.4 MCAR/MAR/MNAR Analysis,N/A (no nulls),PASS
6.4.5 Missing Value Correlation,N/A (no nulls),PASS
6.4.6 Temporal Gap Analysis,See 6.4.6 details,REVIEW
6.4.7 Critical Field Coverage,0 violation(s),PASS



   Overall null cell rate:   0 / 62,912,520 (0.0000%)
   Rows with >50% null:      0
   Critical field violations: 0

   ✅ Dataset is FULLY COMPLETE — zero nulls across all columns, rows, and critical fields.

COMPLETENESS CHECKS COMPLETE


### 6.5) UNIQUENESS CHECKS
*Ensure no duplicate data and each record is uniquely identifiable.*

- **Row-Level Uniqueness** — Entire rows duplicated.
  - *Example:* Two identical rows for the same `review_id` with all 9 columns matching.

- **Key-Level Uniqueness** — Primary/unique key columns contain no duplicates.
  - *Example:* `review_id` (the designated PK) must appear exactly once per row.
  - *Sub:* FK columns (`user_id`, `business_id`) — distribution of reviews per user/business.

- **Composite Key Uniqueness** — Multi-column keys are unique together.
  - *Example:* `(user_id, business_id)` — on Yelp, a user should have at most one active review per business; duplicates suggest retained edits.
  - *Example:* `(user_id, business_id, date)` — exact timestamp collisions are highly suspicious.
  - *Sub:* Near-duplicate detection — same user reviewing the same business within 24 hours.

- **Distribution Uniqueness** — Distinct value counts in each column match domain expectations.
  - *Example:* `stars` should have exactly 5 distinct values (1–5); finding 6+ suggests data corruption.
  - *Example:* `text` should be near-unique; very low cardinality suggests mass-duplicated boilerplate.


In [0]:
# 6.5.1) Row-Level Uniqueness
# ─────────────────────────────────────────────────────────────────────
# Detect exact duplicate rows (all columns identical).
# ℹ️  Basic duplicate analysis was done in §4.7 on the raw 9-column
#     bronze review data. Re-verifying here as part of systematic
#     uniqueness checks on the review table.
# Review table: review_id (PK), user_id, business_id, stars, useful,
#               funny, cool, text, date  — 9 columns, ~6.99M rows.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import (
    count as _count, col, countDistinct, sum as _sum,
    min as spark_min, max as spark_max, mean, stddev,
    percentile_approx, when, lit
)
import pandas as pd

print("=" * 80)
print("6.5.1) ROW-LEVEL UNIQUENESS")
print("    Detecting exact duplicate rows in the review dataframe")
print("=" * 80)

print("\n   ℹ️  Previously checked in §4.7 on the raw 9-column bronze")
print("      review data. Re-verifying here systematically.\n")

total_rows = df.count()

# review_id is the designated primary key — check both PK and full-row uniqueness
review_id_distinct = df.select("review_id").distinct().count()
full_distinct = df.distinct().count()
duplicate_rows = total_rows - full_distinct
dup_pct = (duplicate_rows / total_rows * 100) if total_rows > 0 else 0

icon = "✅" if duplicate_rows == 0 else "⚠️ "
status = "PASS" if duplicate_rows == 0 else "FINDING"
print(f"   {icon} Total rows:                {total_rows:,}")
print(f"   {icon} Distinct rows (all 9 cols): {full_distinct:,}")
print(f"   {icon} Distinct review_ids:        {review_id_distinct:,}")
print(f"   {icon} Exact duplicate rows:       {duplicate_rows:,} ({dup_pct:.4f}%)")
print(f"   {icon} Status:                     {status}")

if review_id_distinct == total_rows:
    print(f"\n   ✅ review_id is 100% unique — confirms PK integrity.")
    if full_distinct == total_rows:
        print(f"   ✅ Full-row distinct matches total — zero exact duplicates.")
else:
    pk_dups = total_rows - review_id_distinct
    print(f"\n   ⚠️  review_id duplicates: {pk_dups:,}")
    print(f"      → PK violation: review_id is NOT fully unique!")

if review_id_distinct == full_distinct:
    print(f"\n   ✅ review_id distinct ({review_id_distinct:,}) == full-row distinct ({full_distinct:,})")
    print(f"      → Every duplicate row shares the same review_id (no partial dups).")
else:
    print(f"\n   ⚠️  review_id distinct ({review_id_distinct:,}) ≠ full-row distinct ({full_distinct:,})")
    print(f"      → Some rows share a review_id but differ in other columns!")

# Duplicate frequency distribution
if duplicate_rows > 0:
    print(f"\n{'─'*60}")
    print("   📊 Duplicate Frequency Distribution:")
    print("      (how many times each duplicated review_id appears)")
    dup_freq = (
        df.groupBy("review_id")
        .agg(_count("*").alias("occurrence_count"))
        .filter(col("occurrence_count") > 1)
        .groupBy("occurrence_count")
        .agg(
            _count("*").alias("num_review_ids"),
            _sum("occurrence_count").alias("total_rows_in_group")
        )
        .orderBy("occurrence_count")
    )
    display(dup_freq)

    print(f"\n   📋 Top 10 Most-Duplicated review_ids:")
    top_dup_ids = (
        df.groupBy("review_id")
        .agg(_count("*").alias("cnt"))
        .filter(col("cnt") > 1)
        .orderBy(col("cnt").desc())
        .limit(10)
    )
    display(top_dup_ids)

# Store for summary cell
_uq_results = {
    "total_rows": total_rows,
    "row_level": {
        "distinct": full_distinct, "duplicates": duplicate_rows,
        "pct": dup_pct, "status": status,
        "pk_distinct": review_id_distinct
    }
}

print(f"\n{'='*80}")
print(f"   Row-Level Uniqueness: {status}")
print("=" * 80)

6.5.1) ROW-LEVEL UNIQUENESS
    Detecting exact duplicate rows in the review dataframe

   ℹ️  Previously checked in §4.7 on the raw 9-column bronze
      review data. Re-verifying here systematically.

   ✅ Total rows:                6,990,280
   ✅ Distinct rows (all 9 cols): 6,990,280
   ✅ Distinct review_ids:        6,990,280
   ✅ Exact duplicate rows:       0 (0.0000%)
   ✅ Status:                     PASS

   ✅ review_id is 100% unique — confirms PK integrity.
   ✅ Full-row distinct matches total — zero exact duplicates.

   ✅ review_id distinct (6,990,280) == full-row distinct (6,990,280)
      → Every duplicate row shares the same review_id (no partial dups).

   Row-Level Uniqueness: PASS


In [0]:
# 6.5.2) Key-Level Uniqueness
# ─────────────────────────────────────────────────────────────────────
# Check uniqueness of individual key/identifier columns.
# Review table keys:
#   review_id   — Primary key (expect 100% unique)
#   user_id     — FK to yelp_user (non-unique: users write many reviews)
#   business_id — FK to yelp_business (non-unique: businesses get many reviews)
# ─────────────────────────────────────────────────────────────────────

print("=" * 80)
print("6.5.2) KEY-LEVEL UNIQUENESS")
print("    Checking individual key/identifier columns")
print("=" * 80)

print("\n   ℹ️  review_id is the designated PK; user_id and business_id")
print("      are foreign keys expected to repeat.\n")

total_rows = _uq_results["total_rows"]

# ── Batch distinct + null counts in a single aggregation ──
key_cols = ["review_id", "user_id", "business_id"]
agg_exprs = (
    [countDistinct(col(c)).alias(f"{c}__dist") for c in key_cols] +
    [_sum(col(c).isNull().cast("int")).alias(f"{c}__nulls") for c in key_cols]
)
key_stats = df.agg(*agg_exprs).collect()[0]

key_meta = {
    "review_id":    ("Primary key — review identifier", True),
    "user_id":      ("FK to yelp_user — reviewer",      False),
    "business_id":  ("FK to yelp_business — reviewed",   False),
}

findings = []
for c in key_cols:
    desc, expect_unique = key_meta[c]
    dist = key_stats[f"{c}__dist"]
    nulls = key_stats[f"{c}__nulls"]
    dups = total_rows - dist
    ratio = dist / total_rows if total_rows > 0 else 0

    if expect_unique:
        status = "PASS" if dist == total_rows and nulls == 0 else "FINDING"
    else:
        status = "INFO (non-unique expected)"
    icon = "✅" if "FINDING" not in status else "⚠️ "

    print(f"\n   {icon} {c} — {desc}:")
    print(f"      Distinct:    {dist:,}")
    print(f"      Nulls:       {nulls:,}")
    print(f"      Duplicates:  {dups:,}")
    print(f"      Uniqueness:  {ratio*100:.4f}%")
    print(f"      Status:      {status}")
    findings.append({"Column": c, "Distinct": dist, "Nulls": nulls,
                      "Duplicates": dups, "Uniqueness %": round(ratio*100, 4),
                      "Status": status})

display(pd.DataFrame(findings))

# ── Reviews-per-User distribution ──
print(f"\n{'─'*60}")
print("   📊 Reviews-per-User Distribution:")
user_freq = df.groupBy("user_id").agg(_count("*").alias("review_count"))
user_stats = user_freq.agg(
    spark_min("review_count").alias("min"),
    spark_max("review_count").alias("max"),
    mean("review_count").alias("mean"),
    percentile_approx("review_count", 0.5).alias("median"),
    stddev("review_count").alias("stddev"),
    percentile_approx("review_count", [0.25, 0.75, 0.90, 0.95, 0.99]).alias("pctiles")
).collect()[0]

pcts = user_stats["pctiles"]
print(f"      Min={user_stats['min']:,}  Max={user_stats['max']:,}  Mean={user_stats['mean']:.1f}  Median={user_stats['median']:,}  StdDev={user_stats['stddev']:.1f}")
print(f"      P25={pcts[0]:,}  P75={pcts[1]:,}  P90={pcts[2]:,}  P95={pcts[3]:,}  P99={pcts[4]:,}")

print(f"\n   📊 Top 10 Most Prolific Reviewers:")
display(user_freq.orderBy(col("review_count").desc()).limit(10))

# ── Reviews-per-Business distribution ──
print(f"\n{'─'*60}")
print("   📊 Reviews-per-Business Distribution:")
biz_freq = df.groupBy("business_id").agg(_count("*").alias("review_count"))
biz_stats = biz_freq.agg(
    spark_min("review_count").alias("min"),
    spark_max("review_count").alias("max"),
    mean("review_count").alias("mean"),
    percentile_approx("review_count", 0.5).alias("median"),
    stddev("review_count").alias("stddev"),
    percentile_approx("review_count", [0.25, 0.75, 0.90, 0.95, 0.99]).alias("pctiles")
).collect()[0]

pcts = biz_stats["pctiles"]
print(f"      Min={biz_stats['min']:,}  Max={biz_stats['max']:,}  Mean={biz_stats['mean']:.1f}  Median={biz_stats['median']:,}  StdDev={biz_stats['stddev']:.1f}")
print(f"      P25={pcts[0]:,}  P75={pcts[1]:,}  P90={pcts[2]:,}  P95={pcts[3]:,}  P99={pcts[4]:,}")

print(f"\n   📊 Top 10 Most Reviewed Businesses:")
display(biz_freq.orderBy(col("review_count").desc()).limit(10))

_uq_results["key_level"] = findings

print(f"\n{'='*80}")
print("   Key-Level Uniqueness: Complete")
print("=" * 80)

6.5.2) KEY-LEVEL UNIQUENESS
    Checking individual key/identifier columns

   ℹ️  review_id is the designated PK; user_id and business_id
      are foreign keys expected to repeat.


   ✅ review_id — Primary key — review identifier:
      Distinct:    6,990,280
      Nulls:       0
      Duplicates:  0
      Uniqueness:  100.0000%
      Status:      PASS

   ✅ user_id — FK to yelp_user — reviewer:
      Distinct:    1,987,929
      Nulls:       0
      Duplicates:  5,002,351
      Uniqueness:  28.4385%
      Status:      INFO (non-unique expected)

   ✅ business_id — FK to yelp_business — reviewed:
      Distinct:    150,346
      Nulls:       0
      Duplicates:  6,839,934
      Uniqueness:  2.1508%
      Status:      INFO (non-unique expected)


Column,Distinct,Nulls,Duplicates,Uniqueness %,Status
review_id,6990280,0,0,100.0,PASS
user_id,1987929,0,5002351,28.4385,INFO (non-unique expected)
business_id,150346,0,6839934,2.1508,INFO (non-unique expected)



────────────────────────────────────────────────────────────
   📊 Reviews-per-User Distribution:
      Min=1  Max=3,048  Mean=3.5  Median=1  StdDev=12.8
      P25=1  P75=3  P90=6  P95=11  P99=36

   📊 Top 10 Most Prolific Reviewers:


user_id,review_count
_BcWyKQL16ndpBdggh2kNA,3048
Xw7ZjaGfr0WNVt6s_5KZfA,1840
0Igx-a1wAstiBDerGxXk2A,1747
-G7Zkl1wIWBBmD0KRy_sCw,1682
ET8n-r7glWYqZhuR6GcdNw,1653
bYENop4BuQepBjM1-BI3fA,1578
1HM81n6n4iPIFU5d2Lokhw,1554
fr1Hz2acAb3OaL3l6DyKNg,1447
wXdbkFZsfDR7utJvbWElyA,1396
Um5bfs5DH6eizgjH3xZsvg,1391



────────────────────────────────────────────────────────────
   📊 Reviews-per-Business Distribution:
      Min=5  Max=7,673  Mean=46.5  Median=15  StdDev=124.5
      P25=8  P75=38  P90=101  P95=180  P99=490

   📊 Top 10 Most Reviewed Businesses:


business_id,review_count
_ab50qdWOk0DdB6XOrBitw,7673
ac1AeYqs8Z4_e2X5M3if2A,7516
GXFMD0Z4jEVZBCsbPf4CTQ,6160
ytynqOUb3hjKeJfRj5Tshw,5778
oBNrLz4EDhiscSlbOl8uAw,5264
iSRTaT9WngzB8JJ2YKJUig,5254
VQcCL9PiNL_wkGf-uF3fjg,5146
_C7QiQQc47AOEv4PE3Kong,4969
GBTPC53ZrG1ZBY3DT8Mbcw,4661
6a4gLLFSgr-Q6CZXDLzBGQ,4480



   Key-Level Uniqueness: Complete


In [0]:
# 6.5.3) Composite Key Uniqueness
# ─────────────────────────────────────────────────────────────────────
# Review domain composite keys:
#   (user_id, business_id)       — Yelp allows one active review per
#                                   user-business pair; duplicates may
#                                   indicate retained edits/updates.
#   (user_id, business_id, date) — exact timestamp collisions are
#                                   highly suspicious (organic duplicates
#                                   at the same second are near-impossible).
# Also: near-duplicate detection for same user reviewing same business
#       within a short time window.
# ─────────────────────────────────────────────────────────────────────

from pyspark.sql.window import Window
from pyspark.sql.functions import lag, unix_timestamp, to_timestamp

print("=" * 80)
print("6.5.3) COMPOSITE KEY UNIQUENESS")
print("    Checking review-domain composite keys")
print("=" * 80)

total_rows = _uq_results["total_rows"]

# ═══════════════════════════════════════════════════════════════════
# Check 1: (user_id, business_id) — one review per user-business pair
# ═══════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("   CHECK 1: (user_id, business_id) — one review per user-business pair")
print("   On Yelp, a user should have at most one active review per business.")
print("   Duplicates suggest retained historical edits/updates.\n")

ub_distinct = df.select("user_id", "business_id").distinct().count()
ub_dups = total_rows - ub_distinct
ub_pct = (ub_dups / total_rows * 100) if total_rows > 0 else 0

icon = "✅" if ub_dups == 0 else "⚠️ "
status_ub = "PASS" if ub_dups == 0 else "FINDING"
print(f"   {icon} Distinct (user_id, business_id) pairs: {ub_distinct:,} / {total_rows:,}")
print(f"   {icon} Multi-review user-business pairs:      {ub_dups:,} ({ub_pct:.4f}%)")
print(f"   {icon} Status:                                {status_ub}")

if ub_dups > 0:
    # Distribution: how many reviews per (user, business) pair
    ub_freq = (
        df.groupBy("user_id", "business_id")
        .agg(_count("*").alias("review_count"))
        .filter(col("review_count") > 1)
    )
    multi_review_pairs = ub_freq.count()

    ub_freq_dist = (
        ub_freq.groupBy("review_count")
        .agg(
            _count("*").alias("num_pairs"),
            _sum("review_count").alias("total_rows")
        )
        .orderBy("review_count")
    )
    print(f"\n   📊 Multi-Review Pair Frequency (reviews per user-business):")
    print(f"      {multi_review_pairs:,} user-business pairs have >1 review")
    display(ub_freq_dist)

    print(f"\n   📋 Top 10 User-Business Pairs by Review Count:")
    display(ub_freq.orderBy(col("review_count").desc()).limit(10))

# ═══════════════════════════════════════════════════════════════════
# Check 2: (user_id, business_id, date) — exact timestamp collisions
# ═══════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("   CHECK 2: (user_id, business_id, date) — exact timestamp collisions")
print("   Same user, same business, same exact timestamp = very suspicious.\n")

ubd_distinct = df.select("user_id", "business_id", "date").distinct().count()
ubd_dups = total_rows - ubd_distinct
ubd_pct = (ubd_dups / total_rows * 100) if total_rows > 0 else 0

icon = "✅" if ubd_dups == 0 else "🚨"
status_ubd = "PASS" if ubd_dups == 0 else "FINDING"
print(f"   {icon} Distinct (user, business, date) tuples: {ubd_distinct:,} / {total_rows:,}")
print(f"   {icon} Exact timestamp collisions:             {ubd_dups:,} ({ubd_pct:.4f}%)")
print(f"   {icon} Status:                                 {status_ubd}")

if ubd_dups > 0:
    collision_sample = (
        df.groupBy("user_id", "business_id", "date")
        .agg(_count("*").alias("cnt"))
        .filter(col("cnt") > 1)
        .orderBy(col("cnt").desc())
        .limit(10)
    )
    print(f"\n   📋 Sample Timestamp Collisions:")
    display(collision_sample)

# ═══════════════════════════════════════════════════════════════════
# Check 3: Near-duplicate detection
# Same user + same business, reviews within 24 hours of each other
# ═══════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("   CHECK 3: Near-Duplicate Detection")
print("   Same user reviewing the same business within 24 hours")
print("   (potential review edits, double-submissions, or spam)\n")

df_ts = df.withColumn("_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))

w = Window.partitionBy("user_id", "business_id").orderBy("_ts")
df_near = (
    df_ts.select("user_id", "business_id", "date", "_ts", "review_id", "stars")
    .withColumn("prev_ts", lag("_ts").over(w))
    .withColumn("prev_review_id", lag("review_id").over(w))
    .filter(col("prev_ts").isNotNull())
    .withColumn("gap_hours",
        (unix_timestamp("_ts") - unix_timestamp("prev_ts")) / 3600.0
    )
    .filter((col("gap_hours") >= 0) & (col("gap_hours") <= 24))
)
near_dup_count = df_near.count()

if near_dup_count > 0:
    print(f"   ⚠️  Near-duplicate pairs found (≤24h gap): {near_dup_count:,}")

    # Sub-breakdown by time window
    near_breakdown = df_near.agg(
        _sum(when(col("gap_hours") == 0, lit(1)).otherwise(lit(0))).alias("same_timestamp"),
        _sum(when((col("gap_hours") > 0) & (col("gap_hours") <= 1), lit(1)).otherwise(lit(0))).alias("within_1h"),
        _sum(when((col("gap_hours") > 1) & (col("gap_hours") <= 6), lit(1)).otherwise(lit(0))).alias("within_6h"),
        _sum(when((col("gap_hours") > 6) & (col("gap_hours") <= 24), lit(1)).otherwise(lit(0))).alias("within_24h"),
    ).collect()[0]

    print(f"      Same timestamp (0h):  {near_breakdown['same_timestamp']:,}")
    print(f"      Within 1 hour:        {near_breakdown['within_1h']:,}")
    print(f"      Within 1–6 hours:     {near_breakdown['within_6h']:,}")
    print(f"      Within 6–24 hours:    {near_breakdown['within_24h']:,}")

    print(f"\n   📋 Sample Near-Duplicates (sorted by gap):")
    display(
        df_near.select("user_id", "business_id", "review_id", "prev_review_id",
                       "date", "stars", "gap_hours")
        .orderBy("gap_hours")
        .limit(15)
    )
else:
    print(f"   ✅ No near-duplicates found — no user reviewed the same")
    print(f"      business twice within 24 hours.")

# Store results
_uq_results["composite_key"] = {
    "ub_distinct": ub_distinct,
    "ub_dups": ub_dups,
    "ub_pct": ub_pct,
    "ub_status": status_ub,
    "ubd_distinct": ubd_distinct,
    "ubd_dups": ubd_dups,
    "ubd_pct": ubd_pct,
    "ubd_status": status_ubd,
    "near_dups": near_dup_count
}

overall_status = "FINDING" if (status_ub == "FINDING" or status_ubd == "FINDING" or near_dup_count > 0) else "PASS"
print(f"\n{'='*80}")
print(f"   Composite Key Uniqueness: {overall_status}")
print("=" * 80)

6.5.3) COMPOSITE KEY UNIQUENESS
    Checking review-domain composite keys

────────────────────────────────────────────────────────────
   CHECK 1: (user_id, business_id) — one review per user-business pair
   On Yelp, a user should have at most one active review per business.
   Duplicates suggest retained historical edits/updates.

   ⚠️  Distinct (user_id, business_id) pairs: 6,745,760 / 6,990,280
   ⚠️  Multi-review user-business pairs:      244,520 (3.4980%)
   ⚠️  Status:                                FINDING

   📊 Multi-Review Pair Frequency (reviews per user-business):
      212,788 user-business pairs have >1 review


review_count,num_pairs,total_rows
2,190088,380176
3,17601,52803
4,3278,13112
5,987,4935
6,412,2472
7,168,1176
8,101,808
9,47,423
10,32,320
11,23,253



   📋 Top 10 User-Business Pairs by Review Count:


user_id,business_id,review_count
a0DnfD31lNdiBTY2-YBBFA,C6YaSrMAzy3jJqinlFVudw,40
a0DnfD31lNdiBTY2-YBBFA,FCCnOgjjvwQJBAy3u2fcLA,38
I6x-ZBHeCNlMnmtfUVf5lg,PUfhLM-MkjQYZw4ldnJqzQ,32
xWmYN57XXZbg0LOK8WbbFQ,gfLsBY-xsNE9-ktiTvTvGA,31
EryvaUTsMnk26eiauFWapQ,3_icitB518vI8Hr0wEzl8A,29
IiUGGCANINjDS4egim7eQw,Yv9KUZ-N3otoKUtToPPxAg,24
I6x-ZBHeCNlMnmtfUVf5lg,cuXUuoOSZ4VbV6jvh3BC0A,22
mVtmlFPxA8cBzhaumweIIA,aYHjJJ64whQsxZjx5rUe_w,20
EryvaUTsMnk26eiauFWapQ,YOdGWNE5nl73AUoWpFvyIg,20
EYC10zgoEv-d--NSE05V3A,SCXkyyKWzS7qWl2jBiWQiQ,18



────────────────────────────────────────────────────────────
   CHECK 2: (user_id, business_id, date) — exact timestamp collisions
   Same user, same business, same exact timestamp = very suspicious.

   🚨 Distinct (user, business, date) tuples: 6,990,255 / 6,990,280
   🚨 Exact timestamp collisions:             25 (0.0004%)
   🚨 Status:                                 FINDING

   📋 Sample Timestamp Collisions:


user_id,business_id,date,cnt
6d4TXsdOZAsw8_h1r730uA,t1H2xPVsbTRu48bZ5TgItw,2010-12-27 23:22:50,3
FHIZVv4yXhSAULPkxqsHxQ,zwXT4m9svdg-xh2cKjEp-Q,2015-04-10 16:05:49,3
twOonjoH_7giux8Hg692Rw,uSNue_cXl17-UzEjyopxKg,2012-05-24 20:05:43,2
EOWu2buXing_iDkvJEE86A,GnNVTRNyUCtQ-Lebes-4bA,2012-09-25 17:41:23,2
awo9Pa1I-D-tVBYoLCfBZA,3xHkEBrXjrTrAY3vpxy23A,2014-06-01 19:45:52,2
zKVF3EeQBpAXkU_KjFHPEw,34WZxssK6o9uVzJh36MrOg,2012-11-14 15:00:11,2
3B5wkOgsrjBjXRpGdrmTSA,n3Q7DXPyBzGjPK1l09SomQ,2021-02-01 17:04:06,2
xju5TzascSMHqt0kJFHQ5A,QFksheewp5cTlegJ9mA-qw,2014-12-01 20:03:02,2
139uCVv_RVdGG8prx9HCKQ,cv-SmPhbpwQCtlI2Q7KJIQ,2014-08-02 13:34:27,2
DOQG3_F5sBdjPpTyUPZ93g,SGp9Bg7Y2UCErFMGnbMGWA,2015-02-09 17:23:31,2



────────────────────────────────────────────────────────────
   CHECK 3: Near-Duplicate Detection
   Same user reviewing the same business within 24 hours
   (potential review edits, double-submissions, or spam)

   ⚠️  Near-duplicate pairs found (≤24h gap): 1,409
      Same timestamp (0h):  25
      Within 1 hour:        1,144
      Within 1–6 hours:     108
      Within 6–24 hours:    132

   📋 Sample Near-Duplicates (sorted by gap):


user_id,business_id,review_id,prev_review_id,date,stars,gap_hours
zKVF3EeQBpAXkU_KjFHPEw,34WZxssK6o9uVzJh36MrOg,zob_5ZIMoHjEGwrmAyx8zA,fQukeICgSVNLz0qLLLkB6A,2012-11-14 15:00:11,1.0,0.0
FHIZVv4yXhSAULPkxqsHxQ,zwXT4m9svdg-xh2cKjEp-Q,d-2yAAWXkT9iMUcIly1zVQ,FmEWJou0hHUiEGCMmThz5Q,2015-04-10 16:05:49,2.0,0.0
6d4TXsdOZAsw8_h1r730uA,t1H2xPVsbTRu48bZ5TgItw,6FJ-pUVQsMFHhW4RJltm6w,gZU6syBqLk210QhnwWbSyw,2010-12-27 23:22:50,1.0,0.0
xZ0ioQJ0Jyo_Uc0FzmyFQQ,O_kAd89gPls-dXda40NWzA,8JcvaBpFPAe7lIVSazi0hQ,XjlaQ5ImGg1uL2XelQA2xw,2019-11-22 07:18:46,5.0,0.0
awo9Pa1I-D-tVBYoLCfBZA,3xHkEBrXjrTrAY3vpxy23A,YMEGrI5NQ93dAbhx1VuTiA,CWym6dXXj68b0_Q7Z2plsw,2014-06-01 19:45:52,2.0,0.0
T870ZkavUI1rqaO_FrJFiw,mfTyBrvx9uCTSPjmrl8PjA,qkbLeGVNwxY9-5iMDcwZGw,1lyNZzBeJXrAMxvZU_-7Aw,2008-08-27 09:49:33,5.0,0.0
3B5wkOgsrjBjXRpGdrmTSA,n3Q7DXPyBzGjPK1l09SomQ,QacdXke_oZAD662oeFyUxg,lzBHkMZayoGSpIPDOIIEug,2021-02-01 17:04:06,4.0,0.0
wtwIlzIwl42kNeW_7o1i1w,PEkt4TW5RU4BZoI_-hBTVA,bpnrw4-gGh4gHMdaHQlIIw,XLJSd-c9fTFEH6-u5yRzCg,2014-04-20 01:53:53,1.0,0.0
6d4TXsdOZAsw8_h1r730uA,t1H2xPVsbTRu48bZ5TgItw,uN7sRdsxIJgmD5REegSpBA,6FJ-pUVQsMFHhW4RJltm6w,2010-12-27 23:22:50,1.0,0.0
139uCVv_RVdGG8prx9HCKQ,cv-SmPhbpwQCtlI2Q7KJIQ,n0Y8M08lrRgB0dbJrAE8Kg,57vXYLGAc-vapQv_vipH9A,2014-08-02 13:34:27,5.0,0.0



   Composite Key Uniqueness: FINDING


---
#### 🟠 DQ-REVIEW-014 — Multi-Review User-Business Pairs

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-014` |
| **Severity** | 🟠 **Medium** — 3.50% of total rows are affected; the core review data is sound (`review_id` is 100% unique), but duplicate user-business pairs inflate review counts per business and per user, distorting popularity rankings, rating averages, and recommendation models. |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.3) Composite Key Uniqueness**, Check 1 |
| **Affected Records** | **244,520 excess reviews** across **212,788 user-business pairs** with >1 review (457,308 total rows involved, 6.54% of dataset). |
| **Pattern** | Long-tail distribution: 190,088 pairs have 2 reviews, 17,601 have 3, tapering to 1 pair with 40 reviews. Top offender: user `a0DnfD31lNdiBTY2-YBBFA` — 40 reviews for business `C6YaSrMAzy3jJqinlFVudw`, 38 for `FCCnOgjjvwQJBAy3u2fcLA`. |
| **Root Cause** | On the live Yelp platform, a user may have only one active review per business. When users update their review, the platform replaces the old version. However, the Yelp Academic Dataset appears to retain **historical versions** of each review as separate rows (each with a distinct `review_id`). This is not a data quality defect per se — it is a design characteristic of the export. |

---

##### 💡 Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep only the latest review per (user_id, business_id)** — `ROW_NUMBER() OVER (PARTITION BY user_id, business_id ORDER BY date DESC) = 1` | Mirrors the live Yelp experience where only the most recent review is visible. Suitable for recommendation engines and rating aggregation. |
| 2 | **Add an `is_latest_review` boolean flag** (window-based, non-destructive) | Preserves all historical versions for temporal analysis (e.g., sentiment drift, rating evolution) while giving downstream consumers a simple filter for current-state views. |
| 3 | **Retain all versions as-is; document the multi-review pattern** | If the analytical goal includes studying how users revise their opinions over time (review update behavior, rating trajectory), all versions are valuable. Downstream aggregations must account for this by deduplicating per (user_id, business_id) when computing per-business metrics. |

##### ✅ Best Logical Solution: **Option 2 — Add `is_latest_review` flag at the Silver layer.**

##### 🔧 Chosen Cleaning Approach: **Option 2 — `is_latest_review` boolean flag** (Slowly Changing Dimension pattern). RCA confirmed 89.3% of pairs have exactly 2 versions, median 160-day gap, 41.7% changed stars. Implemented as `ROW_NUMBER() OVER (PARTITION BY user_id, business_id ORDER BY review_timestamp DESC, review_id DESC) = 1`.

This non-destructive approach preserves the full review history (valuable for NLP sentiment-over-time analysis and review-update behavior studies) while providing a clean filter (`WHERE is_latest_review = TRUE`) for all standard analytics: rating averages, review counts, business popularity, and recommendation pipelines. It satisfies both the current-state and historical-analysis use cases without data loss.

---

#### 🟠 DQ-REVIEW-015 — Exact Timestamp Collisions & Near-Duplicate Reviews

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-015` |
| **Severity** | 🟠 **Medium** — the 25 exact collisions are a clear anomaly (a user cannot organically submit two distinct reviews at the same second); the 1,409 near-duplicates (≤24h) represent a broader pattern of rapid re-reviews that may or may not be legitimate. |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.3) Composite Key Uniqueness**, Checks 2 & 3 |
| **Affected Records** | **25 exact timestamp collisions** (0.0004%) + **1,409 near-duplicate pairs** within 24h window (same user, same business). |
| **Near-Dup Breakdown** | Same timestamp: 25 \| Within 1h: 1,144 \| 1–6h: 108 \| 6–24h: 132 |
| **Key Observation** | Each collision/near-dup pair has **different `review_id` values** — these are genuinely distinct records in the source data, not duplicate ingestion artifacts. Combined with DQ-REVIEW-014 (multi-review pairs), these are the most temporally proximate subset of the retained historical review versions. |
| **Root Cause** | The 25 exact-second collisions likely represent rapid review edits captured as separate records with identical timestamps (system-generated timestamp precision). The 1,144 within-1-hour near-duplicates are consistent with review update behavior where a user posts, then quickly revises or replaces their review. |
| **Related Finding** | `DQ-REVIEW-014` — all 1,409 near-duplicate pairs are a subset of the 244,520 multi-review user-business pairs. |

---

##### 💡 Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Subsume under DQ-REVIEW-014 resolution** — the `is_latest_review` flag naturally handles these cases | If the latest-review dedup from DQ-REVIEW-014 is applied, near-duplicates are implicitly resolved because only the most recent review survives per (user_id, business_id). No additional transformation needed. |
| 2 | **Add `is_near_duplicate` boolean flag** (window-based, ≤24h gap detection) | Non-destructive flag that marks temporally suspicious pairs. Useful if the analysis retains all historical versions but wants to identify likely rapid-edit pairs. |
| 3 | **Drop exact-second collisions only** (25 rows) | Conservative: removes only the clearly anomalous records while preserving the broader near-duplicate set for investigation. |

##### ✅ Best Logical Solution: **Option 1 — Subsume under DQ-REVIEW-014 resolution.**

The `is_latest_review` flag from DQ-REVIEW-014 already resolves this issue: keeping only the latest review per (user_id, business_id) pair eliminates both exact collisions and near-duplicates by construction. Adding a separate `is_near_duplicate` flag (Option 2) provides additional granularity only if the analysis specifically studies rapid review-edit behavior. For standard analytics pipelines, the DQ-REVIEW-014 solution is sufficient.

---

#### ℹ️ 6.5.4 Cross-Reference — Negative Vote Values

The distribution summary in 6.5.4 revealed that `useful`, `funny`, and `cool` all have **min = -1**. This anomaly is formally documented as **`DQ-REVIEW-016`** in the findings cell immediately after Cell 70 (6.5.4 Distribution Uniqueness).

---

In [0]:
# 6.5.4) Distribution Uniqueness
# ─────────────────────────────────────────────────────────────────────
# Per-column cardinality analysis — flag columns with suspiciously
# low or high distinct-value counts vs. domain expectations.
# Review table: 9 columns with well-known domain ranges.
# ─────────────────────────────────────────────────────────────────────

print("=" * 80)
print("6.5.4) DISTRIBUTION UNIQUENESS")
print("    Per-column cardinality analysis (review table — 9 columns)")
print("=" * 80)

total_rows = _uq_results["total_rows"]
all_cols = df.columns

# ── Optimized: single aggregation pass for ALL columns ──
dist_row = df.agg(
    *[countDistinct(col(c)).alias(c) for c in all_cols]
).collect()[0]

# Expected cardinality based on review domain knowledge
# None = open-ended (no fixed expected count)
expected = {
    "review_id":    None,   # PK — should be fully unique
    "user_id":      None,   # FK — ~1.99M distinct users
    "business_id":  None,   # FK — ~150K distinct businesses
    "stars":        5,      # Rating: 1, 2, 3, 4, 5
    "useful":       None,   # Vote count — unbounded integer
    "funny":        None,   # Vote count — unbounded integer
    "cool":         None,   # Vote count — unbounded integer
    "text":         None,   # Free text — near-unique
    "date":         None,   # Timestamp string — high cardinality
}

dist_data = []
for c in all_cols:
    distinct_cnt = dist_row[c]
    ratio = distinct_cnt / total_rows if total_rows > 0 else 0
    exp = expected.get(c)

    if distinct_cnt == 1:
        flag = "🔴 CONSTANT"
    elif distinct_cnt == total_rows:
        flag = "FULLY UNIQUE"
    elif exp and distinct_cnt > exp:
        flag = f"⚠️  OVER ({distinct_cnt} > expected {exp})"
    elif exp and distinct_cnt < exp:
        flag = f"⚠️  UNDER ({distinct_cnt} < expected {exp})"
    elif ratio > 0.95:
        flag = "NEAR-UNIQUE (>95%)"
    else:
        flag = "✅ OK"

    dist_data.append({
        "Column": c, "Distinct": distinct_cnt,
        "Expected": exp if exp else "—",
        "Cardinality %": round(ratio * 100, 4), "Flag": flag
    })

dist_df = pd.DataFrame(dist_data)
display(dist_df)

ok_count = sum(
    1 for d in dist_data
    if "OK" in d["Flag"] or d["Flag"] in ("FULLY UNIQUE", "NEAR-UNIQUE (>95%)")
)
issue_count = len(dist_data) - ok_count

_uq_results["distribution"] = {
    "total_cols": len(all_cols), "ok": ok_count, "flagged": issue_count
}

print(f"\n   {len(all_cols)} columns: {ok_count} OK, {issue_count} flagged")

# ── Stars distribution (rating column) — quick sanity check ──
print(f"\n{'─'*60}")
print("   📊 Stars Rating Distribution (expected: 1–5 only):")
stars_dist = df.groupBy("stars").agg(_count("*").alias("count")).orderBy("stars")
display(stars_dist)

# ── Vote columns distribution summary ──
print(f"\n{'─'*60}")
print("   📊 Vote Columns (useful, funny, cool) — Summary Stats:")
vote_cols = ["useful", "funny", "cool"]
vote_agg = []
for vc in vote_cols:
    vote_agg.extend([
        spark_min(col(vc)).alias(f"{vc}_min"),
        spark_max(col(vc)).alias(f"{vc}_max"),
        mean(col(vc)).alias(f"{vc}_mean"),
        percentile_approx(col(vc), 0.5).alias(f"{vc}_median"),
        _sum((col(vc) == 0).cast("int")).alias(f"{vc}_zeros"),
    ])
vote_stats = df.agg(*vote_agg).collect()[0]

vote_summary = []
for vc in vote_cols:
    zeros = vote_stats[f"{vc}_zeros"]
    zero_pct = (zeros / total_rows * 100) if total_rows > 0 else 0
    vote_summary.append({
        "Column": vc,
        "Min": vote_stats[f"{vc}_min"],
        "Max": vote_stats[f"{vc}_max"],
        "Mean": round(vote_stats[f"{vc}_mean"], 2),
        "Median": vote_stats[f"{vc}_median"],
        "Zeros": f"{zeros:,} ({zero_pct:.1f}%)"
    })
display(pd.DataFrame(vote_summary))

print(f"\n{'='*80}")
print("   Distribution Uniqueness: Complete")
print("=" * 80)

6.5.4) DISTRIBUTION UNIQUENESS
    Per-column cardinality analysis (review table — 9 columns)


Column,Distinct,Expected,Cardinality %,Flag
review_id,6990280,—,100.0,FULLY UNIQUE
user_id,1987929,—,28.4385,✅ OK
business_id,150346,—,2.1508,✅ OK
stars,5,5,1.0E-4,✅ OK
useful,259,—,0.0037,✅ OK
funny,201,—,0.0029,✅ OK
cool,210,—,0.003,✅ OK
text,6974127,—,99.7689,NEAR-UNIQUE (>95%)
date,6885665,—,98.5034,NEAR-UNIQUE (>95%)



   9 columns: 9 OK, 0 flagged

────────────────────────────────────────────────────────────
   📊 Stars Rating Distribution (expected: 1–5 only):


stars,count
1.0,1069561
2.0,544240
3.0,691934
4.0,1452918
5.0,3231627



────────────────────────────────────────────────────────────
   📊 Vote Columns (useful, funny, cool) — Summary Stats:


Column,Min,Max,Mean,Median,Zeros
useful,-1,1182,1.18,0,"3,840,492 (54.9%)"
funny,-1,792,0.33,0,"5,894,117 (84.3%)"
cool,-1,404,0.5,0,"5,377,964 (76.9%)"



   Distribution Uniqueness: Complete


---
#### 🟡 DQ-REVIEW-016 — Negative Vote Counts (useful, funny, cool)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-016` |
| **Severity** | 🟡 **Low** — only **5 rows** (0.000072% of 6,990,280) are affected. The anomaly is cosmetically minor and has near-zero impact on aggregate metrics, but it violates the domain constraint that vote counts must be ≥ 0 and should be corrected for data integrity. |
| **Detected In** | Phase 6.5 — Uniqueness Checks → **6.5.4) Distribution Uniqueness** (Vote Columns Summary Stats sub-section) |
| **Detected In Cell** | Cell 70: `6.5.4) Distribution Uniqueness` |
| **Description** | All three vote columns (`useful`, `funny`, `cool`) contain a minimum value of **-1**. Vote counts represent how many users found a review useful, funny, or cool — they are by definition non-negative integers. A value of -1 is semantically invalid and likely represents a system-level placeholder or encoding artifact rather than an actual negative vote. |
| **Affected Records** | **5 rows total**: `useful < 0`: 4 rows \| `funny < 0`: 2 rows \| `cool < 0`: 1 row \| all three negative: 1 row. All negative values are exactly **-1** (no -2, -3, etc.). |
| **Pattern** | 3 of the 5 affected rows involve the **same business** (`SJQePJL1bkeEk-6gp-6eeA`) — all with `useful = -1`. The remaining 2 rows target distinct businesses. Dates span 2016–2018. The concentration on one business suggests a business-level flag or system artifact rather than random corruption. |
| **Affected `review_id`s** | `JcWLycg0Hdsz6LBNAqS37A` (all three = -1), `gk_9N_c-7oEgzrXVDaR8-g`, `iYS4pfUCUPT_TEmW6BOnrA`, `plhadSWfXZFKRAhEztb6Dw` (useful = -1), `7tlF2NEDiFqMRjKBgTPJ3g` (funny = -1, but useful = 2, cool = 2) |
| **Root Cause** | The Yelp platform likely uses -1 as a sentinel value to indicate a special state — e.g., a review whose votes were invalidated, a review flagged by moderators, or a vote-count rollback. The Yelp Academic Dataset exports this raw internal representation without mapping it back to a neutral value (0 or NULL). |

---

##### 💡 Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Clamp to zero: `GREATEST(col, 0)`** at the Silver layer | Simple and deterministic — replaces -1 with 0 for all three vote columns. Since -1 is semantically "no votes" or "invalid," mapping to 0 preserves the intended meaning without losing non-negative vote data. Applies universally and requires no conditional logic. |
| 2 | **Replace with NULL** to distinguish from genuine zero-vote reviews | Marks the 5 rows as "vote data unavailable" rather than asserting zero votes. Downstream aggregations (`AVG`, `SUM`) automatically exclude NULLs, preventing even marginal distortion. However, it introduces NULLs into otherwise 100%-complete columns, which may complicate downstream NULL-handling logic. |
| 3 | **Flag and retain: add an `has_invalid_votes` boolean column** | Non-destructive approach that preserves the raw -1 values for investigation while flagging them for downstream consumers. Useful if the root cause needs further upstream investigation before deciding on the correction. |

##### ✅ Best Logical Solution: **Option 1 — Clamp to zero with `GREATEST(col, 0)`.**

With only 5 rows affected and the sentinel meaning of -1 being functionally equivalent to "no valid votes," clamping to 0 is the simplest, most maintainable solution. It eliminates the domain violation without introducing NULLs or extra columns. The transformation `GREATEST(useful, 0)` is also idempotent and safe to apply across all rows — it has no effect on the 6,990,275 rows that already have values ≥ 0.

---

In [0]:
# 6.5) Uniqueness Checks — Consolidated Summary
# ─────────────────────────────────────────────────────────────────────

print("=" * 80)
print("6.5) UNIQUENESS CHECKS — CONSOLIDATED SUMMARY")
print("=" * 80)

r = _uq_results
rl = r["row_level"]
ck = r["composite_key"]
dk = r.get("distribution", {})

summary_data = [
    {"Check": "6.5.1 Row-Level Uniqueness",
     "Result": f"{rl['duplicates']:,} exact duplicates ({rl['pct']:.4f}%); review_id distinct: {rl['pk_distinct']:,}",
     "Status": rl["status"]},
    {"Check": "6.5.2 Key-Level Uniqueness",
     "Result": "review_id (PK), user_id (FK), business_id (FK) — see per-column breakdown above",
     "Status": "INFO"},
    {"Check": "6.5.3a (user_id, business_id)",
     "Result": f"{ck['ub_dups']:,} multi-review pairs ({ck['ub_pct']:.4f}%)",
     "Status": ck["ub_status"]},
    {"Check": "6.5.3b (user_id, business_id, date)",
     "Result": f"{ck['ubd_dups']:,} timestamp collisions ({ck['ubd_pct']:.4f}%)",
     "Status": ck["ubd_status"]},
    {"Check": "6.5.3c Near-Duplicates (≤24h)",
     "Result": f"{ck['near_dups']:,} near-duplicate pairs",
     "Status": "FINDING" if ck["near_dups"] > 0 else "PASS"},
    {"Check": "6.5.4 Distribution Uniqueness",
     "Result": f"{dk.get('total_cols', '?')} cols: {dk.get('ok', '?')} OK, {dk.get('flagged', '?')} flagged",
     "Status": "FINDING" if dk.get('flagged', 0) > 0 else "PASS"},
]

display(pd.DataFrame(summary_data))

finding_count = sum(1 for s in summary_data if s["Status"] == "FINDING")
print(f"\n   {len(summary_data)} uniqueness checks performed, {finding_count} finding(s)")
if finding_count == 0:
    print("   ✅ No unexpected uniqueness issues detected.")
else:
    print(f"   ⚠️  {finding_count} uniqueness issue(s) — review above for details.")
print("\n   UNIQUENESS CHECKS COMPLETE")
print("=" * 80)

6.5) UNIQUENESS CHECKS — CONSOLIDATED SUMMARY


Check,Result,Status
6.5.1 Row-Level Uniqueness,"0 exact duplicates (0.0000%); review_id distinct: 6,990,280",PASS
6.5.2 Key-Level Uniqueness,"review_id (PK), user_id (FK), business_id (FK) — see per-column breakdown above",INFO
"6.5.3a (user_id, business_id)","244,520 multi-review pairs (3.4980%)",FINDING
"6.5.3b (user_id, business_id, date)",25 timestamp collisions (0.0004%),FINDING
6.5.3c Near-Duplicates (≤24h),"1,409 near-duplicate pairs",FINDING
6.5.4 Distribution Uniqueness,"9 cols: 9 OK, 0 flagged",PASS



   6 uniqueness checks performed, 3 finding(s)
   ⚠️  3 uniqueness issue(s) — review above for details.

   UNIQUENESS CHECKS COMPLETE



### 6.6) CROSS-COLUMN CONSISTENCY CHECKS
*Validate logical relationships and contradictions across columns within a row.*

- **Logical Relationship Validation** — Derived columns match their deterministic source function.
  - *Example:* `total_votes == useful + funny + cool`.
  - *Example:* `text_length == length(text)`.
  - *Example:* `review_date == review_timestamp.cast('date')`.

- **Derived-Column Logic Checks** — Business-logic transformations are correct.
  - *Example:* ID columns follow Yelp's 22-char base62 hash format.
  - *Example:* `stars` is a whole number in the valid Yelp rating set {1, 2, 3, 4, 5}.
  - *Example:* `review_half_year` correctly maps months 1–6 → H1, 7–12 → H2.

- **Logical Contradiction Detection** — Mutually exclusive states or impossible value combinations.
  - *Example:* `stars = 0` (impossible: Yelp's scale is 1–5).
  - *Example:* `useful < 0` (negative vote count is impossible).
  - *Example:* `review_id == user_id` (different ID namespaces should never collide).

- **Cross-Column Range Logic** — Range relationships between related columns.
  - *Example:* `review_month_start_date ≤ review_date ≤ review_month_end_date`.
  - *Example:* Review date between Yelp founding (2004) and today.
  - *Example:* Vote counts within plausible bounds.

- **Dependency Rule Validation** — If condition A holds, then condition B must also hold.
  - *Example:* `is_year_start == true` → `month == 1 AND day == 1`.
  - *Example:* Vote count > 0 → text should be non-empty (votes require content).
  - *Example:* `review_time_of_day == 'Morning'` → `review_am_pm == 'AM'`.

- **Category Mapping Consistency** — Codes and descriptions align across related column pairs.
  - *Example:* `review_month == 1` → `review_month_name == 'January'`.
  - *Example:* `review_quarter == 1` → `review_quarter_name == 'Q1'`.


In [0]:
# 6.6) Cross-Column Consistency Checks — Setup & 6.6.1
# ─────────────────────────────────────────────────────────────────────
# Comprehensive REVIEW-TABLE cross-column consistency validation.
#
# The review table has two states depending on which TE cells ran:
#   • Raw/type-fixed: review_id, user_id, business_id, stars, useful,
#     funny, cool, text, + review_timestamp (or date string)
#   • Enriched (TE.2): adds ~30 temporal columns, text analytics
#     (text_length, text_word_count, text_sentence_count), and total_votes
#
# Each section runs review-domain checks FIRST (always applicable),
# then temporal-enrichment checks (conditional on TE.2 activation).
# Uses SINGLE aggregation passes for efficiency.

from pyspark.sql.functions import (
    year, month, dayofmonth, dayofyear, dayofweek, hour, minute, second,
    quarter, weekofyear, date_format, last_day, trunc, col, when, lit,
    floor, ceil, concat, sum as _sum, current_date, substring,
    length as spark_length, size, split, trim, regexp_replace, to_timestamp
)
import pandas as pd

print("=" * 80)
print("6.6) CROSS-COLUMN CONSISTENCY CHECKS  —  Review Table")
print("    Core review domain + temporal enrichment validation")
print("=" * 80)

total_rows = df.count()
all_cols = set(df.columns)
cc_findings = []  # accumulated across all 6.6.x cells

# ── Detect available column groups ──
has_ts         = "review_timestamp" in all_cols
has_date_str   = "date" in all_cols
has_temporal   = has_ts and all(c in all_cols for c in [
    "review_date", "review_year", "review_month", "review_day",
    "review_dow_number", "review_quarter", "review_week_of_year",
    "review_hour", "review_minute", "review_second",
])
has_text_anlyt = "text_length" in all_cols
has_total_votes = "total_votes" in all_cols
has_flags      = all(c in all_cols for c in ["is_weekend", "is_weekday"])

print(f"\n   📊 Total rows: {total_rows:,}  |  Columns: {len(all_cols)}")
print(f"   {'✅' if has_ts else '⚠️ '} Timestamp column: {'review_timestamp (TE.1 applied)' if has_ts else 'date (raw string — TE.1 not applied)'}")
print(f"   {'✅' if has_temporal else 'ℹ️ '} Temporal enrichment: {'active (TE.2)' if has_temporal else 'not active — temporal checks skipped'}")
print(f"   {'✅' if has_text_anlyt else 'ℹ️ '} Text analytics: {'active' if has_text_anlyt else 'not active'}")
print(f"   {'✅' if has_total_votes else 'ℹ️ '} Engagement aggregate: {'active' if has_total_votes else 'not active'}")

# ══════════════════════════════════════════════════════════════════════
# 6.6.1) Logical Relationship Validation
#   Derived column == deterministic function(source column).
#   Review domain: rating integrality, vote sums, text metrics.
#   Temporal: date-part extractions from review_timestamp.
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.6.1) Logical Relationship Validation")
print("    Derived columns must equal their deterministic source function")

# ── Part A: Review Core Derivation Checks ──
print(f"\n   ── A) Review Core Derivations ──")

core_checks = [
    # Stars integrality: Yelp ratings are always whole numbers
    ("stars == floor(stars) [integrality]",
     col("stars").isNotNull() & (col("stars").cast("double") != floor(col("stars").cast("double")))),
]

# total_votes = useful + funny + cool (if TE.2 added this column)
if has_total_votes:
    core_checks.append((
        "total_votes == useful + funny + cool",
        col("total_votes") != (col("useful") + col("funny") + col("cool"))))

# text_length = length(text) (if TE.2 added this column)
if has_text_anlyt:
    core_checks.append((
        "text_length == length(text)",
        col("text_length") != spark_length(col("text"))))
    if "text_word_count" in all_cols:
        core_checks.append((
            "text_word_count == size(split(text, whitespace))",
            col("text_word_count") != size(split(col("text"), "\\s+"))))

# date string ↔ timestamp consistency (if both somehow coexist)
if has_ts and has_date_str:
    core_checks.append((
        "review_timestamp == to_timestamp(date)",
        col("review_timestamp") != to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")))

agg_exprs = [_sum(when(c, 1).otherwise(0)).alias(f"c_{i}") for i, (_, c) in enumerate(core_checks)]
row = df.select(*agg_exprs).collect()[0]

for i, (desc, _) in enumerate(core_checks):
    n = int(row[f"c_{i}"])
    ok = n == 0
    print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} mismatches — {'PASS' if ok else 'FINDING'}")
    cc_findings.append({"Section": "6.6.1", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})

# ── Part B: Temporal Derivation Checks (if TE.2 remodel active) ──
if has_temporal:
    print(f"\n   ── B) Temporal Derivation Validation (18 checks) ──")
    ts = "review_timestamp"
    temp_checks = [
        ("review_date == date(ts)",              col("review_date") != col(ts).cast("date")),
        ("review_year == year(ts)",              col("review_year") != year(col(ts))),
        ("review_month == month(ts)",            col("review_month") != month(col(ts))),
        ("review_day == dayofmonth(ts)",          col("review_day") != dayofmonth(col(ts))),
        ("review_day_of_year == dayofyear(ts)",  col("review_day_of_year") != dayofyear(col(ts))),
        ("review_dow_number == dayofweek(ts)",   col("review_dow_number") != dayofweek(col(ts))),
        ("review_quarter == quarter(ts)",        col("review_quarter") != quarter(col(ts))),
        ("review_week_of_year == weekofyear(ts)",col("review_week_of_year") != weekofyear(col(ts))),
        ("review_hour == hour(ts)",              col("review_hour") != hour(col(ts))),
        ("review_minute == minute(ts)",          col("review_minute") != minute(col(ts))),
        ("review_second == second(ts)",          col("review_second") != second(col(ts))),
        ("review_month_name == MMMM format",     col("review_month_name") != date_format(col(ts), "MMMM")),
        ("review_month_abbr == MMM format",      col("review_month_abbr") != date_format(col(ts), "MMM")),
        ("review_dow_name == EEEE format",       col("review_dow_name") != date_format(col(ts), "EEEE")),
        ("review_dow_abbr == EEE format",        col("review_dow_abbr") != date_format(col(ts), "EEE")),
        ("review_year_month == yyyy-MM format",  col("review_year_month") != date_format(col(ts), "yyyy-MM")),
        ("review_year_quarter derivation",       col("review_year_quarter") != concat(year(col(ts)).cast("string"), lit("-Q"), quarter(col(ts)).cast("string"))),
        ("review_year_start_date == trunc(year)",col("review_year_start_date") != trunc(col(ts), "year")),
    ]
    agg_t = [_sum(when(c, 1).otherwise(0)).alias(f"t_{i}") for i, (_, c) in enumerate(temp_checks)]
    row_t = df.select(*agg_t).collect()[0]
    for i, (desc, _) in enumerate(temp_checks):
        n = int(row_t[f"t_{i}"])
        ok = n == 0
        print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} mismatches — {'PASS' if ok else 'FINDING'}")
        cc_findings.append({"Section": "6.6.1", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})
else:
    print(f"\n   ── B) Temporal Derivation Validation — SKIPPED ──")
    print(f"       ℹ️  Enriched columns not found. Uncomment & run TE.2 to enable.")

p = sum(1 for f in cc_findings if f["Section"] == "6.6.1" and f["Status"] == "PASS")
t = sum(1 for f in cc_findings if f["Section"] == "6.6.1")
print(f"\n   ✅ {p}/{t} checks passed (single aggregation pass per group)")
print("=" * 80)

6.6) CROSS-COLUMN CONSISTENCY CHECKS  —  Review Table
    Core review domain + temporal enrichment validation

   📊 Total rows: 6,990,280  |  Columns: 9
   ⚠️  Timestamp column: date (raw string — TE.1 not applied)
   ℹ️  Temporal enrichment: not active — temporal checks skipped
   ℹ️  Text analytics: not active
   ℹ️  Engagement aggregate: not active

────────────────────────────────────────────────────────────
📌 6.6.1) Logical Relationship Validation
    Derived columns must equal their deterministic source function

   ── A) Review Core Derivations ──
   ✅ stars == floor(stars) [integrality]: 0 mismatches — PASS

   ── B) Temporal Derivation Validation — SKIPPED ──
       ℹ️  Enriched columns not found. Uncomment & run TE.2 to enable.

   ✅ 1/1 checks passed (single aggregation pass per group)


In [0]:
# 6.6.2) Derived-Column Logic Checks
# ─────────────────────────────────────────────────────────────────────
# Business-logic transformations that go beyond direct function extraction.
# Review domain: ID format validation, text integrity, rating format.
# Temporal: half_year, AM/PM, time_of_day, boundary flags.

print(f"\n{'─'*60}")
print("📌 6.6.2) Derived-Column Logic Checks")
print("    Business-logic derivations: ID format, text integrity, temporal flags")

# ── Part A: Review Domain Logic ──
print(f"\n   ── A) Review Domain Logic ──")

review_logic = [
    # ID format: Yelp uses 22-char base62 hashes for all identifiers
    ("review_id is 22-char string",
     spark_length(col("review_id")) != 22),
    ("user_id is 22-char string",
     spark_length(col("user_id")) != 22),
    ("business_id is 22-char string",
     spark_length(col("business_id")) != 22),
    # ID character set: only alphanumeric + underscore + hyphen
    ("review_id chars valid [A-Za-z0-9_-]",
     spark_length(regexp_replace(col("review_id"), "[A-Za-z0-9_-]", "")) > 0),
    ("user_id chars valid [A-Za-z0-9_-]",
     spark_length(regexp_replace(col("user_id"), "[A-Za-z0-9_-]", "")) > 0),
    ("business_id chars valid [A-Za-z0-9_-]",
     spark_length(regexp_replace(col("business_id"), "[A-Za-z0-9_-]", "")) > 0),
    # Text not purely whitespace (a real review has substantive content)
    ("text is non-empty after trim",
     col("text").isNull() | (spark_length(trim(col("text"))) == 0)),
    # Stars is not null (every review must have a rating)
    ("stars is not null",
     col("stars").isNull()),
]

agg_r = [_sum(when(c, 1).otherwise(0)).alias(f"r_{i}") for i, (_, c) in enumerate(review_logic)]
row = df.select(*agg_r).collect()[0]
for i, (desc, _) in enumerate(review_logic):
    n = int(row[f"r_{i}"])
    ok = n == 0
    print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} mismatches — {'PASS' if ok else 'FINDING'}")
    cc_findings.append({"Section": "6.6.2", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})

# ── Part B: Temporal Business Logic (if TE.2 active) ──
if has_temporal and has_flags:
    print(f"\n   ── B) Temporal Business Logic (15 checks) ──")
    temp_logic = [
        ("review_half_year vs month",
         ((col("review_month") <= 6) & (col("review_half_year") != 1)) |
         ((col("review_month") > 6) & (col("review_half_year") != 2))),
        ("review_half_year_name vs month",
         ((col("review_month") <= 6) & (col("review_half_year_name") != "H1")) |
         ((col("review_month") > 6) & (col("review_half_year_name") != "H2"))),
        ("review_quarter_name vs quarter",
         col("review_quarter_name") != concat(lit("Q"), col("review_quarter").cast("string"))),
        ("review_am_pm vs hour",
         ((col("review_hour") < 12) & (col("review_am_pm") != "AM")) |
         ((col("review_hour") >= 12) & (col("review_am_pm") != "PM"))),
        ("review_time_of_day vs hour",
         ~(
             (col("review_hour").between(5, 11) & (col("review_time_of_day") == "Morning")) |
             (col("review_hour").between(12, 16) & (col("review_time_of_day") == "Afternoon")) |
             (col("review_hour").between(17, 20) & (col("review_time_of_day") == "Evening")) |
             ((~col("review_hour").between(5, 20)) & (col("review_time_of_day") == "Night"))
         )),
        ("review_hour_12 vs hour",
         ~(
             ((col("review_hour") == 0) & (col("review_hour_12") == 12)) |
             ((col("review_hour").between(1, 12)) & (col("review_hour_12") == col("review_hour"))) |
             ((col("review_hour") > 12) & (col("review_hour_12") == (col("review_hour") - 12)))
         )),
        ("review_hour_bin_4h vs hour",
         col("review_hour_bin_4h") != (floor(col("review_hour") / 4) * 4).cast("int")),
        ("is_month_start vs day",
         col("is_month_start") != (col("review_day") == 1)),
        ("is_month_end vs date",
         col("is_month_end") != (col("review_date") == last_day(col("review_date")))),
        ("is_year_start vs month/day",
         col("is_year_start") != ((col("review_month") == 1) & (col("review_day") == 1))),
        ("is_year_end vs month/day",
         col("is_year_end") != ((col("review_month") == 12) & (col("review_day") == 31))),
        ("is_weekend vs dow_number",
         col("is_weekend") != col("review_dow_number").isin(1, 7)),
        ("is_weekday vs is_weekend",
         col("is_weekday") == col("is_weekend")),
        ("review_month_start_date vs date",
         col("review_month_start_date") != trunc(col("review_date"), "month")),
        ("review_month_end_date vs date",
         col("review_month_end_date") != last_day(col("review_date"))),
    ]
    agg_t = [_sum(when(c, 1).otherwise(0)).alias(f"tl_{i}") for i, (_, c) in enumerate(temp_logic)]
    row_t = df.select(*agg_t).collect()[0]
    for i, (desc, _) in enumerate(temp_logic):
        n = int(row_t[f"tl_{i}"])
        ok = n == 0
        print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} mismatches — {'PASS' if ok else 'FINDING'}")
        cc_findings.append({"Section": "6.6.2", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})
else:
    print(f"\n   ── B) Temporal Business Logic — SKIPPED (TE.2 not active) ──")

p = sum(1 for f in cc_findings if f["Section"] == "6.6.2" and f["Status"] == "PASS")
t = sum(1 for f in cc_findings if f["Section"] == "6.6.2")
print(f"\n   ✅ {p}/{t} logic checks passed (single aggregation pass per group)")
print("=" * 80)


────────────────────────────────────────────────────────────
📌 6.6.2) Derived-Column Logic Checks
    Business-logic derivations: ID format, text integrity, temporal flags

   ── A) Review Domain Logic ──
   ✅ review_id is 22-char string: 0 mismatches — PASS
   ✅ user_id is 22-char string: 0 mismatches — PASS
   ✅ business_id is 22-char string: 0 mismatches — PASS
   ✅ review_id chars valid [A-Za-z0-9_-]: 0 mismatches — PASS
   ✅ user_id chars valid [A-Za-z0-9_-]: 0 mismatches — PASS
   ✅ business_id chars valid [A-Za-z0-9_-]: 0 mismatches — PASS
   ✅ text is non-empty after trim: 0 mismatches — PASS
   ✅ stars is not null: 0 mismatches — PASS

   ── B) Temporal Business Logic — SKIPPED (TE.2 not active) ──

   ✅ 8/8 logic checks passed (single aggregation pass per group)


In [0]:
# 6.6.3) Logical Contradiction Detection
# ─────────────────────────────────────────────────────────────────────
# Mutually exclusive states or impossible value combinations.
# Review domain: impossible ratings, negative votes, ID collisions.
# Temporal: flag contradictions, AM/PM vs time_of_day.

print(f"\n{'─'*60}")
print("📌 6.6.3) Logical Contradiction Detection")
print("    Mutually exclusive flags & impossible value combinations")

# ── Part A: Review Domain Contradictions ──
print(f"\n   ── A) Review Domain Contradictions ──")

review_contradictions = [
    # Impossible star ratings (Yelp scale is 1–5, always integer)
    ("stars < 1 [below Yelp minimum]",
     col("stars").isNotNull() & (col("stars") < 1)),
    ("stars > 5 [above Yelp maximum]",
     col("stars").isNotNull() & (col("stars") > 5)),
    ("stars is fractional [non-integer rating]",
     col("stars").isNotNull() & (col("stars").cast("double") != floor(col("stars").cast("double")))),
    # Negative vote counts (community votes cannot be negative)
    ("useful < 0 [negative vote count]",
     col("useful") < 0),
    ("funny < 0 [negative vote count]",
     col("funny") < 0),
    ("cool < 0 [negative vote count]",
     col("cool") < 0),
    # ID namespace collisions (different ID types should never match)
    ("review_id == user_id [ID namespace collision]",
     col("review_id") == col("user_id")),
    ("review_id == business_id [ID namespace collision]",
     col("review_id") == col("business_id")),
    ("user_id == business_id [ID namespace collision]",
     col("user_id") == col("business_id")),
    # Null core fields (a review must have all core identifiers)
    ("review_id is null [missing PK]",
     col("review_id").isNull()),
    ("user_id is null [missing FK]",
     col("user_id").isNull()),
    ("business_id is null [missing FK]",
     col("business_id").isNull()),
]

agg_r = [_sum(when(c, 1).otherwise(0)).alias(f"rc_{i}") for i, (_, c) in enumerate(review_contradictions)]
row = df.select(*agg_r).collect()[0]
for i, (desc, _) in enumerate(review_contradictions):
    n = int(row[f"rc_{i}"])
    ok = n == 0
    print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
    cc_findings.append({"Section": "6.6.3", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})

# ── Part B: Temporal Contradictions (if TE.2 active) ──
if has_temporal and has_flags:
    print(f"\n   ── B) Temporal Contradictions ──")
    temp_contradictions = [
        ("is_weekend AND is_weekday both true",
         (col("is_weekend") == True) & (col("is_weekday") == True)),
        ("is_weekend AND is_weekday both false",
         (col("is_weekend") == False) & (col("is_weekday") == False)),
        ("is_month_start AND is_month_end both true (impossible)",
         (col("is_month_start") == True) & (col("is_month_end") == True)),
        ("is_year_start AND is_year_end both true (impossible)",
         (col("is_year_start") == True) & (col("is_year_end") == True)),
        ("AM yet Afternoon or Evening",
         (col("review_am_pm") == "AM") & col("review_time_of_day").isin("Afternoon", "Evening")),
        ("PM yet Morning",
         (col("review_am_pm") == "PM") & (col("review_time_of_day") == "Morning")),
        ("half_year=1 but quarter > 2",
         (col("review_half_year") == 1) & (col("review_quarter") > 2)),
        ("half_year=2 but quarter < 3",
         (col("review_half_year") == 2) & (col("review_quarter") < 3)),
        ("is_weekend=true but dow_name not Sat/Sun",
         (col("is_weekend") == True) & ~col("review_dow_name").isin("Saturday", "Sunday")),
        ("is_weekday=true but dow_name is Sat/Sun",
         (col("is_weekday") == True) & col("review_dow_name").isin("Saturday", "Sunday")),
        ("year_quarter != year + quarter",
         col("review_year_quarter") != concat(
             col("review_year").cast("string"), lit("-Q"), col("review_quarter").cast("string"))),
        ("year_month != year + month",
         col("review_year_month") != concat(
             col("review_year").cast("string"), lit("-"),
             when(col("review_month") < 10, concat(lit("0"), col("review_month").cast("string")))
             .otherwise(col("review_month").cast("string")))),
    ]
    agg_t = [_sum(when(c, 1).otherwise(0)).alias(f"tc_{i}") for i, (_, c) in enumerate(temp_contradictions)]
    row_t = df.select(*agg_t).collect()[0]
    for i, (desc, _) in enumerate(temp_contradictions):
        n = int(row_t[f"tc_{i}"])
        ok = n == 0
        print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
        cc_findings.append({"Section": "6.6.3", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})
else:
    print(f"\n   ── B) Temporal Contradictions — SKIPPED (TE.2 not active) ──")

p = sum(1 for f in cc_findings if f["Section"] == "6.6.3" and f["Status"] == "PASS")
t = sum(1 for f in cc_findings if f["Section"] == "6.6.3")
print(f"\n   ✅ {p}/{t} contradiction checks passed")
print("=" * 80)


────────────────────────────────────────────────────────────
📌 6.6.3) Logical Contradiction Detection
    Mutually exclusive flags & impossible value combinations

   ── A) Review Domain Contradictions ──
   ✅ stars < 1 [below Yelp minimum]: 0 — PASS
   ✅ stars > 5 [above Yelp maximum]: 0 — PASS
   ✅ stars is fractional [non-integer rating]: 0 — PASS
   ⚠️  useful < 0 [negative vote count]: 4 — FINDING
   ⚠️  funny < 0 [negative vote count]: 2 — FINDING
   ⚠️  cool < 0 [negative vote count]: 1 — FINDING
   ✅ review_id == user_id [ID namespace collision]: 0 — PASS
   ✅ review_id == business_id [ID namespace collision]: 0 — PASS
   ⚠️  user_id == business_id [ID namespace collision]: 1 — FINDING
   ✅ review_id is null [missing PK]: 0 — PASS
   ✅ user_id is null [missing FK]: 0 — PASS
   ✅ business_id is null [missing FK]: 0 — PASS

   ── B) Temporal Contradictions — SKIPPED (TE.2 not active) ──

   ✅ 8/12 contradiction checks passed



---

#### 🟠 DQ-REVIEW-017 — User-ID / Business-ID Namespace Collision

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-017` |
| **Description** | **1 review** has an identical value for both `user_id` and `business_id` (`lJzfEleeQ1uYnYQp37LD9Q`). In Yelp's data model, user identifiers and business identifiers occupy distinct namespaces — a user’s 22-char hash should never equal a business’s 22-char hash under normal circumstances. This collision means either (a) a hash-generation defect mapped two different entities to the same key, (b) the record is a synthetic/test row that was not scrubbed before the academic dataset release, or (c) a data export artifact merged two ID columns. |
| **Detection Phase** | Phase 6.6 → **6.6.3) Logical Contradiction Detection** (Part A: Review Domain Contradictions) |
| **Detected In** | Cell 76: `6.6.3) Logical Contradiction Detection` |
| **Severity** | 🟠 **Medium** — only 1 row (0.00001% of 6,990,280), but the violation is semantically severe: any join, grouping, or aggregation that pivots on `user_id` or `business_id` independently will conflate this user with this business, producing incorrect user-level and business-level metrics |
| **Affected Record** | `review_id = YwEHOUJF51lMefLLD_DO6w`, `user_id = business_id = lJzfEleeQ1uYnYQp37LD9Q`, `stars = 1`, `useful = 0`, `funny = 0`, `cool = 0`, `date = 2021-07-19 17:44:12` |
| **Pattern** | The review is a **1-star review with zero engagement** (no useful, funny, or cool votes) posted in July 2021. The absence of any community interaction and the ID collision together suggest this may be a test or system-generated record rather than a genuine user review. |
| **Impact** | **(1)** User-level aggregations (e.g., average stars per user, reviews per user) will include a business entity as if it were a user. **(2)** Business-level aggregations will include a user entity as if it were a business. **(3)** FK joins from review → yelp_user and review → yelp_business may both succeed for this row, creating a spurious link. **(4)** Any deduplication logic keyed on `(user_id, business_id)` will treat this as a self-referential record. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag and quarantine the row** — add an `is_id_collision` boolean column and exclude the row from analytical queries | Non-destructive: preserves the raw record for investigation while preventing it from distorting downstream aggregations. Simple WHERE clause (`WHERE NOT is_id_collision`) in Silver views. |
| 2 | **Delete the row** at the Silver layer during cleaning | With only 1 affected row out of \~7M, removal has zero statistical impact. Eliminates the contamination source entirely. However, hard deletes reduce auditability. |
| 3 | **NULL out the `business_id`** (or `user_id`) to break the collision while preserving the record | Retains the review for text/NLP analysis while severing the problematic ID link. The row would appear in user-level analyses (with a NULL business) or vice versa, depending on which ID is nullified. Requires a policy decision on which ID to trust. |
| 4 | **Cross-reference against `yelp_user` and `yelp_business`** to determine which ID is valid | Look up `lJzfEleeQ1uYnYQp37LD9Q` in both reference tables. If it exists in one but not the other, the valid table tells us which column is correct and the other can be NULLed. If it exists in both, this confirms a genuine hash collision at the source. |

##### ✅ Best Logical Solution

**Solution 4 (Cross-reference) followed by Solution 1 (Flag and quarantine).**

##### 🔧 Chosen Cleaning Approach: **Quarantine self-referential rows** (`user_id == business_id`) to `yelp_poc_uc.quarantine.yelp_review_id_collision`. **RCA confirmed:** ID `lJzfEleeQ1uYnYQp37LD9Q` is a genuine hash collision — exists in both `yelp_user` ("Karen", since 2011) and `yelp_business` ("Dollar Car Rental", Nashville TN). Additionally, 3,204 distinct IDs appear in both the `user_id` and `business_id` columns across the dataset (affecting 13,506 + 149,818 rows). These broader collisions don’t break standard joins and require no flag column — they are documented as a known pattern. The single self-referential row is simply quarantined and removed from Silver.

---

In [0]:
# 6.6.4) Cross-Column Range Logic
# ─────────────────────────────────────────────────────────────────────
# Range relationships between related columns.
# Review domain: star bounds, vote plausibility, text/ID length.
# Temporal: date hierarchy, Yelp-era plausibility.

print(f"\n{'─'*60}")
print("📌 6.6.4) Cross-Column Range Logic")
print("    Cross-column plausibility & range bounds")

# ── Part A: Review Domain Range Checks ──
print(f"\n   ── A) Review Domain Ranges ──")

review_ranges = [
    # Star rating bounded range — Yelp only allows {1, 2, 3, 4, 5}
    ("stars ∈ {1, 2, 3, 4, 5}",
     col("stars").isNotNull() & ~col("stars").isin(1, 2, 3, 4, 5)
     & ~col("stars").cast("double").isin(1.0, 2.0, 3.0, 4.0, 5.0)),
    # Vote counts: non-negative and plausible upper bound
    ("useful ∈ [0, 10000] [plausible]",
     (col("useful") < 0) | (col("useful") > 10000)),
    ("funny ∈ [0, 10000] [plausible]",
     (col("funny") < 0) | (col("funny") > 10000)),
    ("cool ∈ [0, 10000] [plausible]",
     (col("cool") < 0) | (col("cool") > 10000)),
    # Text length plausibility (non-empty, < 100K chars)
    ("text length ∈ [1, 100000]",
     col("text").isNull() | (spark_length(col("text")) < 1) | (spark_length(col("text")) > 100000)),
    # ID length exactness (Yelp hashes are always 22 chars)
    ("review_id length == 22",
     spark_length(col("review_id")) != 22),
    ("user_id length == 22",
     spark_length(col("user_id")) != 22),
    ("business_id length == 22",
     spark_length(col("business_id")) != 22),
    # Cross-vote plausibility: cool > useful+funny with cool>10 is highly implausible
    # (Yelp allows independent voting, so this is a soft heuristic)
    ("cool ≤ useful + funny [plausibility, cool>10]",
     (col("cool") > (col("useful") + col("funny"))) & (col("cool") > 10)),
]

agg_r = [_sum(when(c, 1).otherwise(0)).alias(f"rr_{i}") for i, (_, c) in enumerate(review_ranges)]
row = df.select(*agg_r).collect()[0]
for i, (desc, _) in enumerate(review_ranges):
    n = int(row[f"rr_{i}"])
    ok = n == 0
    print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
    cc_findings.append({"Section": "6.6.4", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})

# ── Part B: Temporal Range Checks (if TE.2 active) ──
if has_temporal:
    print(f"\n   ── B) Temporal Range Checks ──")
    temp_ranges = [
        ("month_start_date ≤ review_date",
         col("review_month_start_date") > col("review_date")),
        ("review_date ≤ month_end_date",
         col("review_date") > col("review_month_end_date")),
        ("year_start_date ≤ review_date",
         col("review_year_start_date") > col("review_date")),
        ("month_start_date ≤ month_end_date",
         col("review_month_start_date") > col("review_month_end_date")),
        ("year_start_date ≤ month_start_date",
         col("review_year_start_date") > col("review_month_start_date")),
        # Yelp domain plausibility (founded 2004, no future dates)
        ("review year ≥ 2004 [Yelp founding]",
         col("review_year") < 2004),
        ("review_date ≤ today [no future reviews]",
         col("review_date") > current_date()),
        # Value-range bounds for temporal fields
        ("day_of_year ≤ 366",
         col("review_day_of_year") > 366),
        ("week_of_year ≤ 53",
         col("review_week_of_year") > 53),
        ("hour_12 ∈ [1, 12]",
         (col("review_hour_12") < 1) | (col("review_hour_12") > 12)),
        ("hour_bin_4h ∈ {0,4,8,12,16,20}",
         ~col("review_hour_bin_4h").isin(0, 4, 8, 12, 16, 20)),
        ("day ∈ [1, 31]",
         (col("review_day") > 31) | (col("review_day") < 1)),
        ("month ∈ [1, 12]",
         (col("review_month") > 12) | (col("review_month") < 1)),
        ("hour ∈ [0, 23]",
         (col("review_hour") > 23) | (col("review_hour") < 0)),
    ]
    agg_t = [_sum(when(c, 1).otherwise(0)).alias(f"tr_{i}") for i, (_, c) in enumerate(temp_ranges)]
    row_t = df.select(*agg_t).collect()[0]
    for i, (desc, _) in enumerate(temp_ranges):
        n = int(row_t[f"tr_{i}"])
        ok = n == 0
        print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
        cc_findings.append({"Section": "6.6.4", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})
else:
    print(f"\n   ── B) Temporal Range Checks — SKIPPED (TE.2 not active) ──")

p = sum(1 for f in cc_findings if f["Section"] == "6.6.4" and f["Status"] == "PASS")
t = sum(1 for f in cc_findings if f["Section"] == "6.6.4")
print(f"\n   ✅ {p}/{t} range checks passed")
print("=" * 80)


────────────────────────────────────────────────────────────
📌 6.6.4) Cross-Column Range Logic
    Cross-column plausibility & range bounds

   ── A) Review Domain Ranges ──
   ✅ stars ∈ {1, 2, 3, 4, 5}: 0 — PASS
   ⚠️  useful ∈ [0, 10000] [plausible]: 4 — FINDING
   ⚠️  funny ∈ [0, 10000] [plausible]: 2 — FINDING
   ⚠️  cool ∈ [0, 10000] [plausible]: 1 — FINDING
   ✅ text length ∈ [1, 100000]: 0 — PASS
   ✅ review_id length == 22: 0 — PASS
   ✅ user_id length == 22: 0 — PASS
   ✅ business_id length == 22: 0 — PASS
   ⚠️  cool ≤ useful + funny [plausibility, cool>10]: 915 — FINDING

   ── B) Temporal Range Checks — SKIPPED (TE.2 not active) ──

   ✅ 5/9 range checks passed


---

#### 🟡 DQ-REVIEW-018 — Disproportionate Cool Vote Counts (cool > useful + funny)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-018` |
| **Description** | **915 reviews** (0.013% of 6,990,280) have a `cool` vote count that exceeds the combined `useful + funny` count, with `cool > 10`. On Yelp, `useful`, `funny`, and `cool` are independently cast vote types, so this is not inherently impossible — but the pattern is statistically unusual because `cool` votes are typically correlated with `useful` votes. A review that many people found "cool" but few found "useful" or "funny" deviates from the expected engagement pattern. |
| **Detection Phase** | Phase 6.6 → **6.6.4) Cross-Column Range Logic** (Part A: Review Domain Ranges) |
| **Detected In** | Cell 78: `6.6.4) Cross-Column Range Logic` |
| **Severity** | 🟡 **Low / Informational** — 915 rows (0.013%), not a hard data defect but a plausibility anomaly worth documenting for downstream awareness |
| **Affected Scope** | 915 rows — `cool` range [11, 360] (avg 15.6), `useful` range [0, 359] (avg 9.6), `funny` range [0, 21] (avg 1.7), excess (`cool - useful - funny`) range [1, 27] (avg 4.3) |
| **Pattern** | **96.4% of affected reviews are 4–5 star ratings** (626 five-star, 256 four-star, 24 three-star, 3 two-star, 6 one-star). This strongly suggests the pattern reflects **high-quality positive reviews** that are stylishly written — users click “cool” to acknowledge the writing quality rather than its informational value (“useful”) or humor (“funny”). The top excess cases show reviews with 23–30 cool votes and 0–5 useful/funny votes, consistent with aspirational restaurant or travel reviews. |
| **Impact** | **(1)** Any composite engagement metric that assumes `cool ≤ useful + funny` will incorrectly flag these reviews as anomalous. **(2)** Engagement-weighted models (e.g., review quality scoring) may overweight cool votes if the three vote types are assumed to be balanced. **(3)** If using vote counts as features for NLP or fraud detection, the disproportionate cool pattern could confuse classifiers that expect correlated vote distributions. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **No action required — document as known pattern** | The 915 rows are genuine reviews with legitimate (if disproportionate) community voting behavior. The excess is modest (avg 4.3 extra cool votes) and the affected reviews are overwhelmingly high-quality positive reviews. Removing or modifying them would distort the real engagement signal. |
| 2 | **Add an `is_cool_disproportionate` flag** for downstream awareness | Non-destructive annotation that lets analysts and model builders decide how to handle these rows. Particularly useful if building vote-based feature engineering where balanced vote distributions are assumed. |
| 3 | **Cap cool votes at `useful + funny + N` (e.g., N=10)** during feature engineering (not raw data) | Only relevant in modelling contexts where vote balance matters. Preserves raw data integrity while providing a transformed feature. Should be applied at the Gold/feature-engineering layer, never at Bronze or Silver. |

##### ✅ Best Logical Solution

**Solution 1 (No action — document as known pattern).** The 915 affected reviews are genuine high-quality content with legitimate voting behavior. The "cool" vote type on Yelp captures an aesthetic dimension (well-written, inspiring, aspirational) that is distinct from "useful" (informational) and "funny" (humorous), making an imbalance entirely plausible for certain review styles. The excess is small (avg 4.3) and concentrated in 4–5 star reviews. Capping or removing these values would destroy real engagement signal. Instead, document the pattern so downstream consumers can account for it if building models that assume vote-type correlation.

---

In [0]:
# 6.6.5) Dependency Rule Validation
# ─────────────────────────────────────────────────────────────────────
# If column A has value X, then column B must (or must not) have value Y.
# Review domain: vote-text dependencies, engagement patterns.
# Temporal: boundary cascading rules.

print(f"\n{'─'*60}")
print("📌 6.6.5) Dependency Rule Validation")
print("    Multi-column conditional requirements")

# ── Part A: Review Domain Dependencies ──
print(f"\n   ── A) Review Domain Dependencies ──")

review_deps = [
    # Empty/null text should have zero engagement (votes require content to evaluate)
    ("text is empty → useful should be 0",
     (col("text").isNull() | (spark_length(trim(col("text"))) == 0)) & (col("useful") > 0)),
    ("text is empty → funny should be 0",
     (col("text").isNull() | (spark_length(trim(col("text"))) == 0)) & (col("funny") > 0)),
    ("text is empty → cool should be 0",
     (col("text").isNull() | (spark_length(trim(col("text"))) == 0)) & (col("cool") > 0)),
    # High engagement on trivially short text is suspicious
    ("total_engagement > 50 → text length ≥ 10",
     ((col("useful") + col("funny") + col("cool")) > 50) &
     col("text").isNotNull() & (spark_length(col("text")) < 10)),
    # Null stars should not have any votes (incomplete review record)
    ("stars is null → total votes == 0",
     col("stars").isNull() & ((col("useful") + col("funny") + col("cool")) > 0)),
    # Every review must have complete attribution
    ("review_id not null → user_id not null",
     col("review_id").isNotNull() & col("user_id").isNull()),
    ("review_id not null → business_id not null",
     col("review_id").isNotNull() & col("business_id").isNull()),
]

agg_r = [_sum(when(c, 1).otherwise(0)).alias(f"rd_{i}") for i, (_, c) in enumerate(review_deps)]
row = df.select(*agg_r).collect()[0]
for i, (desc, _) in enumerate(review_deps):
    n = int(row[f"rd_{i}"])
    ok = n == 0
    print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
    cc_findings.append({"Section": "6.6.5", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})

# ── Part B: Temporal Dependencies (if TE.2 active) ──
if has_temporal and has_flags:
    print(f"\n   ── B) Temporal Dependency Rules ──")
    temp_deps = [
        ("is_year_start → half_year=1, quarter=1, month=1, day=1, is_month_start",
         (col("is_year_start") == True) & ~(
             (col("review_half_year") == 1) & (col("review_quarter") == 1) &
             (col("review_month") == 1) & (col("review_day") == 1) &
             (col("is_month_start") == True)
         )),
        ("is_year_end → half_year=2, quarter=4, month=12, day=31, is_month_end",
         (col("is_year_end") == True) & ~(
             (col("review_half_year") == 2) & (col("review_quarter") == 4) &
             (col("review_month") == 12) & (col("review_day") == 31) &
             (col("is_month_end") == True)
         )),
        ("is_month_start → date == month_start_date",
         (col("is_month_start") == True) & (col("review_date") != col("review_month_start_date"))),
        ("is_month_end → date == month_end_date",
         (col("is_month_end") == True) & (col("review_date") != col("review_month_end_date"))),
        ("Morning → AM",
         (col("review_time_of_day") == "Morning") & (col("review_am_pm") != "AM")),
        ("Afternoon → PM",
         (col("review_time_of_day") == "Afternoon") & (col("review_am_pm") != "PM")),
        ("Evening → PM",
         (col("review_time_of_day") == "Evening") & (col("review_am_pm") != "PM")),
        ("H1 → quarter ∈ {1,2} AND month ∈ 1-6",
         (col("review_half_year_name") == "H1") & ~(
             col("review_quarter").isin(1, 2) & (col("review_month") <= 6)
         )),
        ("H2 → quarter ∈ {3,4} AND month ∈ 7-12",
         (col("review_half_year_name") == "H2") & ~(
             col("review_quarter").isin(3, 4) & (col("review_month") >= 7)
         )),
        ("is_year_start → is_month_start [cascading]",
         (col("is_year_start") == True) & (col("is_month_start") != True)),
        ("is_year_end → is_month_end [cascading]",
         (col("is_year_end") == True) & (col("is_month_end") != True)),
    ]
    agg_t = [_sum(when(c, 1).otherwise(0)).alias(f"td_{i}") for i, (_, c) in enumerate(temp_deps)]
    row_t = df.select(*agg_t).collect()[0]
    for i, (desc, _) in enumerate(temp_deps):
        n = int(row_t[f"td_{i}"])
        ok = n == 0
        print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
        cc_findings.append({"Section": "6.6.5", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})
else:
    print(f"\n   ── B) Temporal Dependency Rules — SKIPPED (TE.2 not active) ──")

p = sum(1 for f in cc_findings if f["Section"] == "6.6.5" and f["Status"] == "PASS")
t = sum(1 for f in cc_findings if f["Section"] == "6.6.5")
print(f"\n   ✅ {p}/{t} dependency rule checks passed")
print("=" * 80)


────────────────────────────────────────────────────────────
📌 6.6.5) Dependency Rule Validation
    Multi-column conditional requirements

   ── A) Review Domain Dependencies ──
   ✅ text is empty → useful should be 0: 0 — PASS
   ✅ text is empty → funny should be 0: 0 — PASS
   ✅ text is empty → cool should be 0: 0 — PASS
   ✅ total_engagement > 50 → text length ≥ 10: 0 — PASS
   ✅ stars is null → total votes == 0: 0 — PASS
   ✅ review_id not null → user_id not null: 0 — PASS
   ✅ review_id not null → business_id not null: 0 — PASS

   ── B) Temporal Dependency Rules — SKIPPED (TE.2 not active) ──

   ✅ 7/7 dependency rule checks passed


In [0]:
# 6.6.6) Category Mapping Consistency
# ─────────────────────────────────────────────────────────────────────
# Code ↔ description mappings between related column PAIRS.
# Review domain: no native code→label columns in the core schema.
# Temporal: month/dow number→name, quarter→name, abbreviations.

print(f"\n{'─'*60}")
print("📌 6.6.6) Category Mapping Consistency")
print("    Code ↔ label alignment between related column pairs")

if has_temporal:
    # ── Build expected month name from month number ──
    expected_month = (
        when(col("review_month") == 1, "January")
        .when(col("review_month") == 2, "February")
        .when(col("review_month") == 3, "March")
        .when(col("review_month") == 4, "April")
        .when(col("review_month") == 5, "May")
        .when(col("review_month") == 6, "June")
        .when(col("review_month") == 7, "July")
        .when(col("review_month") == 8, "August")
        .when(col("review_month") == 9, "September")
        .when(col("review_month") == 10, "October")
        .when(col("review_month") == 11, "November")
        .when(col("review_month") == 12, "December")
    )

    # ── Build expected DOW name from DOW number (1=Sun…7=Sat) ──
    expected_dow = (
        when(col("review_dow_number") == 1, "Sunday")
        .when(col("review_dow_number") == 2, "Monday")
        .when(col("review_dow_number") == 3, "Tuesday")
        .when(col("review_dow_number") == 4, "Wednesday")
        .when(col("review_dow_number") == 5, "Thursday")
        .when(col("review_dow_number") == 6, "Friday")
        .when(col("review_dow_number") == 7, "Saturday")
    )

    mapping_checks = [
        ("month number → month_name",
         col("review_month_name") != expected_month),
        ("dow_number → dow_name",
         col("review_dow_name") != expected_dow),
        ("month_name → month_abbr (first 3 chars)",
         col("review_month_abbr") != substring(col("review_month_name"), 1, 3)),
        ("dow_name → dow_abbr (first 3 chars)",
         col("review_dow_abbr") != substring(col("review_dow_name"), 1, 3)),
        ("quarter → quarter_name (Q + quarter)",
         col("review_quarter_name") != concat(lit("Q"), col("review_quarter").cast("string"))),
        ("half_year → half_year_name (H + half_year)",
         col("review_half_year_name") != concat(lit("H"), col("review_half_year").cast("string"))),
        ("Q1 → month ∈ {1,2,3}",
         (col("review_quarter") == 1) & ~col("review_month").isin(1, 2, 3)),
        ("Q2 → month ∈ {4,5,6}",
         (col("review_quarter") == 2) & ~col("review_month").isin(4, 5, 6)),
        ("Q3 → month ∈ {7,8,9}",
         (col("review_quarter") == 3) & ~col("review_month").isin(7, 8, 9)),
        ("Q4 → month ∈ {10,11,12}",
         (col("review_quarter") == 4) & ~col("review_month").isin(10, 11, 12)),
    ]

    agg_m = [_sum(when(c, 1).otherwise(0)).alias(f"mc_{i}") for i, (_, c) in enumerate(mapping_checks)]
    row = df.select(*agg_m).collect()[0]
    for i, (desc, _) in enumerate(mapping_checks):
        n = int(row[f"mc_{i}"])
        ok = n == 0
        print(f"   {'✅' if ok else '⚠️ '} {desc}: {n:,} — {'PASS' if ok else 'FINDING'}")
        cc_findings.append({"Section": "6.6.6", "Check": desc, "Mismatches": n, "Status": "PASS" if ok else "FINDING"})

    p = sum(1 for f in cc_findings if f["Section"] == "6.6.6" and f["Status"] == "PASS")
    t = sum(1 for f in cc_findings if f["Section"] == "6.6.6")
    print(f"\n   ✅ {p}/{t} mapping checks passed")
else:
    print(f"\n   ℹ️  The core review schema has no code→label column pairs.")
    print(f"       Category mapping checks apply to temporal enrichment columns")
    print(f"       (month→name, dow→name, quarter→label) which require TE.2.")
    print(f"       SKIPPED — uncomment & run TE.2 to enable.")

print("=" * 80)


────────────────────────────────────────────────────────────
📌 6.6.6) Category Mapping Consistency
    Code ↔ label alignment between related column pairs

   ℹ️  The core review schema has no code→label column pairs.
       Category mapping checks apply to temporal enrichment columns
       (month→name, dow→name, quarter→label) which require TE.2.
       SKIPPED — uncomment & run TE.2 to enable.


In [0]:
# 6.6.7) Cross-Column Consistency — Consolidated Summary
# ─────────────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("6.6) CROSS-COLUMN CONSISTENCY — CONSOLIDATED SUMMARY")
print("=" * 80)

findings_df = pd.DataFrame(cc_findings)
display(findings_df)

total_checks = len(cc_findings)
violations = sum(1 for f in cc_findings if f["Status"] == "FINDING")
passed = total_checks - violations

# Per-section breakdown
print(f"\n{'─'*60}")
print("Per-Section Breakdown:")
for section in ["6.6.1", "6.6.2", "6.6.3", "6.6.4", "6.6.5", "6.6.6"]:
    section_items = [f for f in cc_findings if f["Section"] == section]
    if not section_items:
        continue
    s_pass = sum(1 for f in section_items if f["Status"] == "PASS")
    s_total = len(section_items)
    icon = "✅" if s_pass == s_total else "⚠️ "
    section_names = {
        "6.6.1": "Logical Relationship Validation",
        "6.6.2": "Derived-Column Logic Checks",
        "6.6.3": "Logical Contradiction Detection",
        "6.6.4": "Cross-Column Range Logic",
        "6.6.5": "Dependency Rule Validation",
        "6.6.6": "Category Mapping Consistency",
    }
    print(f"   {icon} {section} {section_names[section]}: {s_pass}/{s_total} passed")

print(f"\n{'─'*60}")
print(f"TOTAL: {total_checks} cross-column checks performed")
print(f"   ✅ Passed:   {passed}")
print(f"   ⚠️  Findings: {violations}")

if violations == 0:
    print("\n✅ All cross-column consistency checks passed — review data is internally consistent.")
else:
    print(f"\n⚠️  {violations} cross-column inconsistency(ies) — review sections above.")

print("\nCROSS-COLUMN CONSISTENCY CHECKS COMPLETE")
print("=" * 80)


6.6) CROSS-COLUMN CONSISTENCY — CONSOLIDATED SUMMARY


Section,Check,Mismatches,Status
6.6.1,stars == floor(stars) [integrality],0,PASS
6.6.2,review_id is 22-char string,0,PASS
6.6.2,user_id is 22-char string,0,PASS
6.6.2,business_id is 22-char string,0,PASS
6.6.2,review_id chars valid [A-Za-z0-9_-],0,PASS
6.6.2,user_id chars valid [A-Za-z0-9_-],0,PASS
6.6.2,business_id chars valid [A-Za-z0-9_-],0,PASS
6.6.2,text is non-empty after trim,0,PASS
6.6.2,stars is not null,0,PASS
6.6.3,stars < 1 [below Yelp minimum],0,PASS



────────────────────────────────────────────────────────────
Per-Section Breakdown:
   ✅ 6.6.1 Logical Relationship Validation: 1/1 passed
   ✅ 6.6.2 Derived-Column Logic Checks: 8/8 passed
   ⚠️  6.6.3 Logical Contradiction Detection: 8/12 passed
   ⚠️  6.6.4 Cross-Column Range Logic: 5/9 passed
   ✅ 6.6.5 Dependency Rule Validation: 7/7 passed

────────────────────────────────────────────────────────────
TOTAL: 37 cross-column checks performed
   ✅ Passed:   29
   ⚠️  Findings: 8

⚠️  8 cross-column inconsistency(ies) — review sections above.

CROSS-COLUMN CONSISTENCY CHECKS COMPLETE



### 6.7) REFERENTIAL INTEGRITY CHECKS
*Ensure relationships between the review table and parent/sibling tables are valid.*

The review table has **two foreign keys** — `business_id → yelp_business` and `user_id → yelp_user` — plus **stored aggregates** in both parent tables (`review_count`, `stars`, `average_stars`, vote totals) that must reconcile against computed values.

- **Foreign Key Validity** — Every FK value exists in the parent table's PK.
  - *View 1 — Orphan records:* `review.business_id` has no matching `yelp_business.business_id`.
  - *View 2 — Missing parents:* Parent entity exists but has zero reviews.
  - *View 3 — Invalid FK format:* FK value deviates from Yelp's 22-char alphanumeric hash.

- **Lookup Table Conformance** — Review FKs are well-represented across sibling tables.
  - *Business dimension:* What % of reviewed businesses also appear in `yelp_checkin` and `yelp_tip`?
  - *User dimension:* What % of reviewing users also appear in `yelp_tip`?

- **Cross-Table Aggregate Reconciliation** — Computed aggregates match stored values.
  - *Example:* `COUNT(reviews per business)` should equal `yelp_business.review_count`.
  - *Example:* `AVG(review.stars per user)` should approximate `yelp_user.average_stars`.
  - *Example:* `SUM(review.useful per user)` should align with `yelp_user.useful`.

In [0]:
# 6.7.1) FK Validity — business_id → yelp_business & user_id → yelp_user
# ─────────────────────────────────────────────────────────────────────
# The review table has TWO foreign keys:
#   FK-1: business_id → yelp_business.business_id
#   FK-2: user_id → yelp_user.user_id
# Each FK is validated with three views:
#   View 1 — Orphan records: child FK values with no matching parent PK
#   View 2 — Missing parents: parent rows with zero child references
#   View 3 — Invalid FK format: values that deviate from Yelp's 22-char hash

from pyspark.sql.functions import (
    col, count as _count, broadcast, sum as _sum, when, lit
)
import pandas as pd

print("=" * 80)
print("6.7.1) FK VALIDITY")
print("    review.business_id → yelp_business  &  review.user_id → yelp_user")
print("    Three views per FK: orphan records, missing parents, invalid format")
print("=" * 80)

findings_671 = []
id_pattern = r"^[a-zA-Z0-9_-]{22}$"

def validate_fk(child_df, parent_df, fk_col, parent_label, findings_list):
    """Validate a single FK relationship with 3 views."""
    child_ids = child_df.select(fk_col).distinct()
    parent_ids = broadcast(parent_df.select(fk_col).distinct())
    total_child = child_ids.count()
    total_parent = parent_ids.count()
    print(f"\n   Distinct {fk_col} in review:       {total_child:,}")
    print(f"   Distinct {fk_col} in {parent_label}: {total_parent:,}")

    # ── View 1: Orphan Records ──
    print(f"\n{'─'*60}")
    print(f"📌 View 1 — Orphan Records")
    print(f"   review.{fk_col} values with NO matching parent in {parent_label}")
    orphans = child_ids.join(parent_ids, fk_col, "left_anti")
    orphan_count = orphans.count()
    status = "PASS" if orphan_count == 0 else "FINDING"
    print(f"   {'✅' if orphan_count == 0 else '⚠️ '} Orphan {fk_col}s: {orphan_count:,} — {status}")
    if orphan_count > 0:
        samples = [r[0] for r in orphans.limit(10).collect()]
        print(f"      Examples: {samples}")
        affected = child_df.join(broadcast(orphans), fk_col, "inner").count()
        print(f"      Affected review rows: {affected:,}")
    findings_list.append({"FK": fk_col, "Check": "View 1: Orphan Records",
        "Description": f"review.{fk_col} NOT IN {parent_label}", "Count": orphan_count, "Status": status})

    # ── View 2: Missing Parent Records ──
    print(f"\n{'─'*60}")
    print(f"📌 View 2 — Missing Parent Records")
    print(f"   {parent_label} rows with ZERO reviews")
    missing = parent_ids.join(child_ids, fk_col, "left_anti")
    missing_count = missing.count()
    missing_pct = missing_count / total_parent * 100 if total_parent > 0 else 0
    print(f"   ℹ️  {parent_label} with zero reviews: {missing_count:,} ({missing_pct:.1f}%)")
    if missing_pct > 50:
        status_m = "FINDING"
        print(f"   ⚠️  Over half of {parent_label} rows have no reviews — data gap")
    elif missing_pct > 30:
        status_m = "WARNING"
        print(f"   ⚠️  Significant portion of {parent_label} rows have no reviews")
    else:
        status_m = "INFO"
        print(f"   ✅ Acceptable — not all entities necessarily have review activity")
    findings_list.append({"FK": fk_col, "Check": "View 2: Missing Parents",
        "Description": f"{parent_label} with zero reviews ({missing_pct:.1f}%)", "Count": missing_count, "Status": status_m})

    # ── View 3: Invalid FK Format ──
    print(f"\n{'─'*60}")
    print(f"📌 View 3 — Invalid FK Format")
    print(f"   {fk_col} pattern (22-char [A-Za-z0-9_-]) — both sides")
    child_bad = child_df.filter(~col(fk_col).rlike(id_pattern)).count()
    parent_bad = parent_df.filter(~col(fk_col).rlike(id_pattern)).count()
    print(f"   {'✅' if child_bad == 0 else '⚠️ '} Malformed {fk_col} in review:       {child_bad:,}")
    print(f"   {'✅' if parent_bad == 0 else '⚠️ '} Malformed {fk_col} in {parent_label}: {parent_bad:,}")
    total_bad = child_bad + parent_bad
    status_f = "PASS" if total_bad == 0 else "FINDING"
    findings_list.append({"FK": fk_col, "Check": "View 3: Invalid FK Format",
        "Description": f"Malformed {fk_col} (child + parent)", "Count": total_bad, "Status": status_f})

# ═══ FK-1: business_id → yelp_business ═══
print(f"\n{'━'*60}")
print("FK-1: review.business_id → yelp_business.business_id")
print(f"{'━'*60}")
try:
    df_business = spark.table("yelp_poc_uc.bronze_final.yelp_business")
    validate_fk(df, df_business, "business_id", "yelp_business", findings_671)
except Exception as e:
    print(f"   ⚠️  Could not load yelp_business: {e}")
    findings_671.append({"FK": "business_id", "Check": "FK Validity", "Description": str(e), "Count": "N/A", "Status": "SKIPPED"})

# ═══ FK-2: user_id → yelp_user ═══
print(f"\n\n{'━'*60}")
print("FK-2: review.user_id → yelp_user.user_id")
print(f"{'━'*60}")
try:
    df_user = spark.table("yelp_poc_uc.bronze_final.yelp_user")
    validate_fk(df, df_user, "user_id", "yelp_user", findings_671)
except Exception as e:
    print(f"   ⚠️  Could not load yelp_user: {e}")
    findings_671.append({"FK": "user_id", "Check": "FK Validity", "Description": str(e), "Count": "N/A", "Status": "SKIPPED"})

# ══ Summary ══
print(f"\n{'='*80}")
print("6.7.1 SUMMARY — FK Validity (2 FKs × 3 views)")
print("=" * 80)
findings_671_df = pd.DataFrame(findings_671)
display(findings_671_df)
violations = sum(1 for f in findings_671 if f.get("Status") == "FINDING")
print(f"{'✅' if violations == 0 else '⚠️ '} {violations} FK validity finding(s)")
print("=" * 80)

6.7.1) FK VALIDITY
    review.business_id → yelp_business  &  review.user_id → yelp_user
    Three views per FK: orphan records, missing parents, invalid format

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FK-1: review.business_id → yelp_business.business_id
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   Distinct business_id in review:       150,346
   Distinct business_id in yelp_business: 150,346

────────────────────────────────────────────────────────────
📌 View 1 — Orphan Records
   review.business_id values with NO matching parent in yelp_business
   ✅ Orphan business_ids: 0 — PASS

────────────────────────────────────────────────────────────
📌 View 2 — Missing Parent Records
   yelp_business rows with ZERO reviews
   ℹ️  yelp_business with zero reviews: 0 (0.0%)
   ✅ Acceptable — not all entities necessarily have review activity

────────────────────────────────────────────────────────────
📌 View 3 — Invalid FK Format
   business_id pattern (22

FK,Check,Description,Count,Status
business_id,View 1: Orphan Records,review.business_id NOT IN yelp_business,0,PASS
business_id,View 2: Missing Parents,yelp_business with zero reviews (0.0%),0,INFO
business_id,View 3: Invalid FK Format,Malformed business_id (child + parent),0,PASS
user_id,View 1: Orphan Records,review.user_id NOT IN yelp_user,32,FINDING
user_id,View 2: Missing Parents,yelp_user with zero reviews (0.0%),0,INFO
user_id,View 3: Invalid FK Format,Malformed user_id (child + parent),0,PASS


⚠️  1 FK validity finding(s)


In [0]:
# 6.7.2) Lookup Table Conformance
# ─────────────────────────────────────────────────────────────────────
# Cross-reference review FKs against sibling Yelp tables to verify
# data ecosystem consistency across the five-table Yelp schema.
#   A) business_id overlap with yelp_checkin and yelp_tip
#   B) user_id overlap with yelp_tip

from pyspark.sql.functions import col, broadcast
import pandas as pd

print("=" * 80)
print("6.7.2) LOOKUP TABLE CONFORMANCE")
print("    Cross-referencing review FKs against sibling Yelp tables")
print("=" * 80)

findings_672 = []

def check_overlap(source_ids, sibling_table, sibling_label, fk_col, total_source, findings_list):
    """Compare FK overlap between review and a sibling table."""
    print(f"\n{'─'*60}")
    print(f"📌 review.{fk_col} ↔ {sibling_label}.{fk_col}")
    try:
        df_sib = spark.table(sibling_table)
        sib_ids = df_sib.select(fk_col).distinct()
        total_sib = sib_ids.count()

        only_source = source_ids.join(broadcast(sib_ids), fk_col, "left_anti").count()
        only_sib = sib_ids.join(broadcast(source_ids), fk_col, "left_anti").count()
        overlap = source_ids.join(broadcast(sib_ids), fk_col, "inner").count()
        overlap_pct = overlap / total_source * 100 if total_source > 0 else 0

        print(f"   Distinct {fk_col} in review:        {total_source:,}")
        print(f"   Distinct {fk_col} in {sibling_label}:  {total_sib:,}")
        print(f"   Overlap (in both):                  {overlap:,} ({overlap_pct:.1f}% of review)")
        print(f"   Only in review (no {sibling_label}): {only_source:,}")
        print(f"   Only in {sibling_label} (no review): {only_sib:,}")

        status = "PASS" if overlap_pct > 80 else ("WARNING" if overlap_pct > 50 else "FINDING")
        print(f"   {'✅' if status == 'PASS' else '⚠️ '} Overlap coverage: {overlap_pct:.1f}% — {status}")

        findings_list.append({
            "Dimension": fk_col, "Check": f"review ↔ {sibling_label}",
            "Review IDs": total_source, f"{sibling_label} IDs": total_sib,
            "Overlap": overlap, "Overlap %": f"{overlap_pct:.1f}%",
            "Only Review": only_source, f"Only {sibling_label}": only_sib,
            "Status": status
        })
    except Exception as e:
        print(f"   ⚠️  Could not load {sibling_table}: {e}")
        findings_list.append({"Dimension": fk_col, "Check": f"review ↔ {sibling_label}", "Status": "SKIPPED"})

# ── Part A: business_id cross-referencing ──
print(f"\n   ── A) business_id Cross-Referencing ──")
review_bids = df.select("business_id").distinct()
total_review_bids = review_bids.count()

check_overlap(review_bids, "yelp_poc_uc.bronze_final.yelp_checkin", "yelp_checkin", "business_id", total_review_bids, findings_672)
check_overlap(review_bids, "yelp_poc_uc.bronze_final.yelp_tip", "yelp_tip", "business_id", total_review_bids, findings_672)

# ── Part B: user_id cross-referencing ──
print(f"\n\n   ── B) user_id Cross-Referencing ──")
review_uids = df.select("user_id").distinct()
total_review_uids = review_uids.count()

check_overlap(review_uids, "yelp_poc_uc.bronze_final.yelp_tip", "yelp_tip", "user_id", total_review_uids, findings_672)

# ══ Summary ══
print(f"\n{'='*80}")
print("6.7.2 SUMMARY — Lookup Table Conformance")
print("=" * 80)
findings_672_df = pd.DataFrame(findings_672)
display(findings_672_df)
v = sum(1 for f in findings_672 if f.get("Status") == "FINDING")
print(f"{'✅' if v == 0 else '⚠️ '} {v} conformance finding(s)")
print("=" * 80)

6.7.2) LOOKUP TABLE CONFORMANCE
    Cross-referencing review FKs against sibling Yelp tables

   ── A) business_id Cross-Referencing ──

────────────────────────────────────────────────────────────
📌 review.business_id ↔ yelp_checkin.business_id
   Distinct business_id in review:        150,346
   Distinct business_id in yelp_checkin:  131,930
   Overlap (in both):                  131,930 (87.8% of review)
   Only in review (no yelp_checkin): 18,416
   Only in yelp_checkin (no review): 0
   ✅ Overlap coverage: 87.8% — PASS

────────────────────────────────────────────────────────────
📌 review.business_id ↔ yelp_tip.business_id
   Distinct business_id in review:        150,346
   Distinct business_id in yelp_tip:  106,193
   Overlap (in both):                  106,193 (70.6% of review)
   Only in review (no yelp_tip): 44,153
   Only in yelp_tip (no review): 0
   ⚠️  Overlap coverage: 70.6% — WARNING


   ── B) user_id Cross-Referencing ──

──────────────────────────────────────────────

Dimension,Check,Review IDs,yelp_checkin IDs,Overlap,Overlap %,Only Review,Only yelp_checkin,Status,yelp_tip IDs,Only yelp_tip
business_id,review ↔ yelp_checkin,150346,131930.0,131930,87.8%,18416,0.0,PASS,null,null
business_id,review ↔ yelp_tip,150346,null,106193,70.6%,44153,null,WARNING,106193.0,0.0
user_id,review ↔ yelp_tip,1987929,null,301758,15.2%,1686171,null,FINDING,301758.0,0.0


⚠️  1 conformance finding(s)


---
#### 🚠 DQ-REVIEW-019 — Orphan User IDs (FK Violation)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-019` |
| **Description** | **32 distinct `user_id` values** in the review table have no matching parent record in `yelp_user`. These 32 orphan user_ids affect **33 review rows** (one user has 2 reviews). All orphan user_ids conform to the valid 22-char hash format, so the values are structurally correct but simply absent from the user table. |
| **Detection Phase** | Phase 6.7 → **6.7.1) FK Validity** (FK-2: review.user_id → yelp_user.user_id, View 1: Orphan Records) |
| **Detected In** | Cell 84: `6.7.1) FK Validity` |
| **Severity** | 🟠 **Medium** — only 33 rows (0.00047% of 6,990,280), but any FK violation means these reviews cannot be joined to user metadata (yelping_since, elite status, friend network, compliments). User-level aggregations will silently exclude these 32 users if using INNER JOIN, or produce NULLs if using LEFT JOIN. |
| **Affected Records** | 32 distinct `user_id`s including: `G0PWeUgNeGDobntevJlJ1g`, `0oMk8hhMqiSt4G1BJjjG6g`, `3N6-acEgosQSbipmBZKoSg`, `qH_QwXBgA4Z7WyCjtwRi2A`, `lzpM_Vf2rKA4ivGtAIOH4w`, `tquAg8GqbhN5k6Hkd23M0A`, `u8cq-5zzD7dPSa3LR8rIMw`, `5iBVQ3OeK8lV4Z_4PXc1Xw`, `5-L9Oo6PyUiBjeqj1GsJyQ`, `OxvEeexqWWeWkPwkKtTy2Q` (10 of 32 shown). |
| **Root Cause** | Likely users who were deleted, suspended, or merged after their reviews were posted. The Yelp academic dataset is a static export; user deletions that occurred between the user-table snapshot and the review-table snapshot would create orphans. |
| **Impact** | **(1)** INNER JOINs between review and yelp_user will silently drop 33 rows. **(2)** Any user-level feature engineering (e.g., user tenure, elite status, friend count) will be NULL for these reviews. **(3)** Deduplication or user-cohort analyses will miss these users entirely. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag with `is_orphan_user = true`** and use LEFT JOINs | Non-destructive. Preserves reviews for text/NLP analysis while making the FK gap explicit. |
| 2 | **Exclude the 33 rows** at the Silver layer | With 0.00047% impact, removal has zero statistical effect. Simplifies downstream joins. |
| 3 | **Create stub user records** with NULL metadata | Ensures INNER JOINs retain these rows. Artificial but complete. |

##### ✅ Best Logical Solution: Solution 1 (flag and LEFT JOIN).

##### 🔧 Chosen Cleaning Approach: **Quarantine the 33 orphan-user rows** to `yelp_poc_uc.quarantine.yelp_review_orphan_user`. **RCA confirmed:** All 33 reviews posted on 2022-01-19 within a ~2.3-hour window at the dataset boundary. 100% zero engagement, complete absence from all other tables. Pipeline tracing confirmed 0 errors in staging — source-level Yelp export timing gap, not an ingestion issue.

---

#### 🟡 DQ-REVIEW-020 — Low Review↔Tip User Overlap

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-020` |
| **Description** | Only **15.2%** of reviewing users (301,758 out of 1,987,929) also appear in `yelp_tip`. The remaining **1,686,171 users (84.8%)** have reviews but zero tips. Conversely, **all** tip users also have reviews (0 tip-only users). The business_id dimension shows better overlap: 87.8% with `yelp_checkin` and 70.6% with `yelp_tip`. |
| **Detection Phase** | Phase 6.7 → **6.7.2) Lookup Table Conformance** (Part B: user_id Cross-Referencing) |
| **Detected In** | Cell 85: `6.7.2) Lookup Table Conformance` |
| **Severity** | 🟡 **Low** — This is a **behavioral data gap**, not a data defect. Tips are a voluntary, lower-frequency user action; most Yelp users write reviews but never leave tips. The 15.2% overlap is consistent with expected engagement asymmetry. However, it becomes medium-severity if downstream models assume tip coverage mirrors review coverage (e.g., using tip sentiment as a feature for all users). |
| **Impact** | **(1)** Joining review and tip tables on `user_id` via INNER JOIN drops 84.8% of review users. **(2)** User-level features derived from tips (tip count, tip text sentiment) will be NULL for the majority. **(3)** Any analysis of "users who both review and tip" is limited to 15.2% of the user base. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Use LEFT JOIN** when combining review and tip tables on `user_id` | Preserves all review users; tip columns are NULL for the 84.8% non-tippers. Prevents silent data loss. |
| 2 | **Add `has_tips` boolean** to the Silver-layer user dimension | Lets downstream consumers explicitly segment by tip availability. |
| 3 | **Document the 15.2% coverage** in the data contract | Prevents future analysts from assuming 1:1 correspondence. |

##### ✅ Best Logical Solution: Solution 1 + Solution 3.

---

#### 🟠 DQ-REVIEW-021 — Cross-Table Aggregate Drift (Business & User)

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-021` |
| **Description** | Significant drift between **computed review aggregates** and **stored values** in `yelp_business` and `yelp_user`. The Yelp academic dataset is a geographic subset of Yelp's full platform; user and business tables carry lifetime counters that reflect activity beyond this dataset's scope. Key metrics: |
| **Detection Phase** | Phase 6.7 → **6.7.3) Cross-Table Aggregate Reconciliation** (Checks 1–5) |
| **Detected In** | Cell 87: `6.7.3) Cross-Table Aggregate Reconciliation` |

##### Drift Summary

| Check | Exact Match | Direction | Anomalous Cases |
| --- | --- | --- | --- |
| `review_count` per business | 55.2% (67,362 mismatches) | 67,353 computed > stored | Most businesses have MORE reviews in the dataset than `business.review_count` claims |
| `avg stars` per business | 99.7% within 0.5 | Balanced | Only 408 outliers beyond 0.5 — PASS |
| `review_count` per user | 24.0% (1,511,189 mismatches) | 1,493,524 stored > computed | Expected: users review outside this geographic scope |
| `avg stars` per user | 62.7% within 0.5 | N/A | 741,724 beyond 0.5 tolerance |
| `useful` votes per user | 44.3% exact | 1,995 computed > stored | Anomalous: dataset has more useful votes than user table claims |
| `funny` votes per user | 64.4% exact | 485 computed > stored | Minor anomaly |
| `cool` votes per user | 61.7% exact | 388 computed > stored | Minor anomaly |

| Attribute | Detail |
| --- | --- |
| **Severity** | 🟠 **Medium** — The drift is **structurally expected** for a geographic subset: `yelp_user.review_count` and `average_stars` reflect a user's global Yelp history, while this dataset only contains reviews for businesses in selected metros. However, two anomalies warrant investigation: **(a)** 67,353 businesses where computed review count **exceeds** stored `review_count` (the business table may have been snapshotted before the final review export), and **(b)** 17,665 users and 1,995/485/388 users where computed review counts or vote totals **exceed** stored values (these should never happen if stored values reflect lifetime totals). |
| **Root Cause** | The five Yelp academic dataset tables are exported as **independent snapshots** at potentially different points in time. The `review_count`, `stars`, `average_stars`, and vote columns in `yelp_business` and `yelp_user` are **cached aggregates** that may lag behind the actual review table. Additionally, reviews removed for policy violations after the review-table export but before the business/user-table export would reduce the stored counter without affecting the review-table rows. |
| **Impact** | **(1)** Using `yelp_business.review_count` directly will undercount for 67,353 businesses. **(2)** Using `yelp_user.review_count` will overcount for 1,493,524 users and undercount for 17,665. **(3)** Vote totals from `yelp_user` are unreliable for 36–56% of users. **(4)** `yelp_user.average_stars` differs by >0.5 stars for 37.3% of users. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Always compute aggregates from the review table** rather than using stored counters | Guarantees consistency within the dataset. Use `COUNT(*)` per business/user, `AVG(stars)`, and `SUM(useful/funny/cool)` from the review table directly. |
| 2 | **Add computed columns** to Silver-layer business/user dimensions | Replace `yelp_business.review_count` with `computed_review_count` and flag the delta: `review_count_drift = stored - computed`. |
| 3 | **Flag the 17,665 anomalous users** (computed > stored) for investigation | These violate the invariant that stored lifetime counters should always be ≥ dataset-subset counters. May indicate data export defects. |

##### ✅ Best Logical Solution: Solution 1 + Solution 2.
Never trust pre-computed counters from parent tables in the Yelp academic dataset. Always derive aggregates from the review table at the Silver layer, and carry the drift as a diagnostic column for data quality monitoring.

---

In [0]:
# 6.7.3) Cross-Table Aggregate Reconciliation
# ─────────────────────────────────────────────────────────────────────
# The review table's aggregates must reconcile against stored values
# in yelp_business and yelp_user. Both parent tables carry pre-computed
# counters (review_count, stars, average_stars, useful, funny, cool).
# Any drift reveals stale snapshots or dataset-boundary effects.
# Checks:
#   1) review_count per business vs yelp_business.review_count
#   2) avg stars per business vs yelp_business.stars
#   3) review_count per user vs yelp_user.review_count
#   4) avg stars per user vs yelp_user.average_stars
#   5) vote totals per user vs yelp_user vote columns
#   6) temporal span consistency across tables
#   7) isolated review businesses (no checkin or tip presence)

from pyspark.sql.functions import (
    col, count as _count, sum as _sum,
    min as spark_min, max as spark_max, broadcast,
    when, abs as spark_abs, percentile_approx,
    avg as spark_avg, round as spark_round
)
import pandas as pd

print("=" * 80)
print("6.7.3) CROSS-TABLE AGGREGATE RECONCILIATION")
print("    Comparing computed review aggregates vs stored values in parent tables")
print("=" * 80)

findings_673 = []

# ═══ Check 1: review_count per business ═══
print(f"\n{'─'*60}")
print("📌 Check 1: COUNT(reviews) per business vs yelp_business.review_count")
try:
    df_business = spark.table("yelp_poc_uc.bronze_final.yelp_business")
    computed_biz = df.groupBy("business_id").agg(_count("*").alias("computed"))
    stored_biz = df_business.select("business_id", col("review_count").alias("stored"))
    comp = computed_biz.join(broadcast(stored_biz), "business_id", "inner")
    total = comp.count()
    mismatches = comp.filter(col("computed") != col("stored"))
    mm_count = mismatches.count()
    match_pct = (total - mm_count) / total * 100 if total > 0 else 0
    status = "PASS" if mm_count == 0 else ("WARNING" if match_pct > 95 else "FINDING")
    print(f"   Businesses compared: {total:,}")
    print(f"   {'✅' if mm_count == 0 else '⚠️ '} Exact matches: {total - mm_count:,} ({match_pct:.1f}%) | Mismatches: {mm_count:,} — {status}")
    if mm_count > 0:
        higher = mismatches.filter(col("stored") > col("computed")).count()
        print(f"   Direction: stored > computed in {higher:,} cases (expected for dataset-window drift)")
        top5 = mismatches.withColumn("diff", col("stored") - col("computed")).orderBy(spark_abs(col("diff")).desc()).limit(5).collect()
        for r in top5:
            print(f"      {r['business_id']}: computed={r['computed']}, stored={r['stored']}, diff={r['diff']}")
    findings_673.append({"Check": "review_count per business", "Description": f"{match_pct:.1f}% exact ({mm_count:,} mismatches)", "Status": status})
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "review_count per business", "Description": str(e), "Status": "SKIPPED"})

# ═══ Check 2: avg stars per business ═══
print(f"\n{'─'*60}")
print("📌 Check 2: AVG(review.stars) per business vs yelp_business.stars")
try:
    computed_stars = df.groupBy("business_id").agg(spark_round(spark_avg("stars"), 1).alias("computed"))
    stored_stars = df_business.select("business_id", col("stars").alias("stored"))
    comp_s = computed_stars.join(broadcast(stored_stars), "business_id", "inner")
    total_s = comp_s.count()
    exact_s = comp_s.filter(spark_abs(col("computed") - col("stored")) <= 0.05).count()
    within_half = comp_s.filter(spark_abs(col("computed") - col("stored")) <= 0.5).count()
    outliers = total_s - within_half
    exact_pct = exact_s / total_s * 100 if total_s > 0 else 0
    half_pct = within_half / total_s * 100 if total_s > 0 else 0
    status = "PASS" if half_pct > 95 else ("WARNING" if half_pct > 80 else "FINDING")
    print(f"   Businesses compared: {total_s:,}")
    print(f"   Within 0.05 (near-exact): {exact_s:,} ({exact_pct:.1f}%)")
    print(f"   Within 0.5 tolerance:     {within_half:,} ({half_pct:.1f}%)")
    print(f"   {'✅' if outliers == 0 else '⚠️ '} Beyond 0.5: {outliers:,} — {status}")
    findings_673.append({"Check": "avg stars per business", "Description": f"{exact_pct:.1f}% near-exact, {half_pct:.1f}% within 0.5", "Status": status})
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "avg stars per business", "Description": str(e), "Status": "SKIPPED"})

# ═══ Check 3: review_count per user ═══
print(f"\n{'─'*60}")
print("📌 Check 3: COUNT(reviews) per user vs yelp_user.review_count")
try:
    df_user = spark.table("yelp_poc_uc.bronze_final.yelp_user")
    computed_u = df.groupBy("user_id").agg(_count("*").alias("computed"))
    stored_u = df_user.select("user_id", col("review_count").alias("stored"))
    comp_u = computed_u.join(broadcast(stored_u), "user_id", "inner")
    total_u = comp_u.count()
    mm_u = comp_u.filter(col("computed") != col("stored"))
    mm_u_count = mm_u.count()
    match_u_pct = (total_u - mm_u_count) / total_u * 100 if total_u > 0 else 0
    status = "PASS" if mm_u_count == 0 else ("WARNING" if match_u_pct > 95 else "FINDING")
    print(f"   Users compared: {total_u:,}")
    print(f"   {'✅' if mm_u_count == 0 else '⚠️ '} Exact matches: {total_u - mm_u_count:,} ({match_u_pct:.1f}%) | Mismatches: {mm_u_count:,} — {status}")
    if mm_u_count > 0:
        higher_u = mm_u.filter(col("stored") > col("computed")).count()
        lower_u = mm_u_count - higher_u
        print(f"   Direction: stored > computed: {higher_u:,} | computed > stored: {lower_u:,}")
        print(f"   ℹ️  stored > computed is expected: user may have reviews outside this dataset's scope")
        if lower_u > 0:
            print(f"   ⚠️  {lower_u:,} users have MORE computed reviews than stored — anomalous")
        diff_u = mm_u.withColumn("diff", col("stored") - col("computed")).agg(
            spark_avg("diff").alias("avg"), percentile_approx("diff", 0.5).alias("median"),
            spark_min("diff").alias("min"), spark_max("diff").alias("max")
        ).collect()[0]
        print(f"   Diff (stored-computed): avg={diff_u['avg']:.1f}, median={diff_u['median']}, range=[{diff_u['min']}, {diff_u['max']}]")
    findings_673.append({"Check": "review_count per user", "Description": f"{match_u_pct:.1f}% exact ({mm_u_count:,} mismatches)", "Status": status})
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "review_count per user", "Description": str(e), "Status": "SKIPPED"})

# ═══ Check 4: avg stars per user ═══
print(f"\n{'─'*60}")
print("📌 Check 4: AVG(review.stars) per user vs yelp_user.average_stars")
try:
    computed_us = df.groupBy("user_id").agg(spark_round(spark_avg("stars"), 2).alias("computed"))
    stored_us = df_user.select("user_id", col("average_stars").alias("stored"))
    comp_us = computed_us.join(broadcast(stored_us), "user_id", "inner")
    total_us = comp_us.count()
    exact_us = comp_us.filter(spark_abs(col("computed") - col("stored")) <= 0.05).count()
    within_half_us = comp_us.filter(spark_abs(col("computed") - col("stored")) <= 0.5).count()
    exact_us_pct = exact_us / total_us * 100 if total_us > 0 else 0
    half_us_pct = within_half_us / total_us * 100 if total_us > 0 else 0
    status = "PASS" if half_us_pct > 90 else ("WARNING" if half_us_pct > 70 else "FINDING")
    print(f"   Users compared: {total_us:,}")
    print(f"   Within 0.05: {exact_us:,} ({exact_us_pct:.1f}%) | Within 0.5: {within_half_us:,} ({half_us_pct:.1f}%)")
    print(f"   {'✅' if half_us_pct > 90 else '⚠️ '} Beyond 0.5: {total_us - within_half_us:,} — {status}")
    if half_us_pct <= 90:
        print(f"   ℹ️  Expected: yelp_user.average_stars covers full history; this dataset may be a subset")
    findings_673.append({"Check": "avg stars per user", "Description": f"{exact_us_pct:.1f}% near-exact, {half_us_pct:.1f}% within 0.5", "Status": status})
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "avg stars per user", "Description": str(e), "Status": "SKIPPED"})

# ═══ Check 5: vote totals per user ═══
print(f"\n{'─'*60}")
print("📌 Check 5: SUM(review.useful/funny/cool) per user vs yelp_user columns")
try:
    computed_votes = df.groupBy("user_id").agg(
        _sum("useful").alias("c_useful"), _sum("funny").alias("c_funny"), _sum("cool").alias("c_cool")
    )
    stored_votes = df_user.select(
        "user_id", col("useful").alias("s_useful"), col("funny").alias("s_funny"), col("cool").alias("s_cool")
    )
    comp_v = computed_votes.join(broadcast(stored_votes), "user_id", "inner")
    total_v = comp_v.count()
    for vt in ["useful", "funny", "cool"]:
        c, s = f"c_{vt}", f"s_{vt}"
        exact = comp_v.filter(col(c) == col(s)).count()
        exact_pct = exact / total_v * 100 if total_v > 0 else 0
        comp_higher = comp_v.filter(col(c) > col(s)).count()
        status = "PASS" if exact_pct > 80 else ("WARNING" if exact_pct > 50 else "FINDING")
        print(f"   {'✅' if status == 'PASS' else '⚠️ '} {vt}: {exact:,}/{total_v:,} exact ({exact_pct:.1f}%) — {status}")
        if comp_higher > 0:
            print(f"      ⚠️  {comp_higher:,} users have computed > stored (anomalous)")
        findings_673.append({"Check": f"{vt} votes per user", "Description": f"{exact_pct:.1f}% exact, computed>stored: {comp_higher:,}", "Status": status})
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "Vote totals per user", "Description": str(e), "Status": "SKIPPED"})

# ═══ Check 6: temporal span consistency ═══
print(f"\n{'─'*60}")
print("📌 Check 6: Temporal span consistency across Yelp tables")
try:
    review_range = df.agg(spark_min("date").alias("earliest"), spark_max("date").alias("latest")).collect()[0]
    print(f"   Review: {str(review_range['earliest'])[:19]} → {str(review_range['latest'])[:19]}")
    for t_name, t_label, t_col in [
        ("yelp_poc_uc.bronze_final.yelp_tip", "yelp_tip", "date"),
        ("yelp_poc_uc.bronze_final.yelp_checkin", "yelp_checkin", "date"),
    ]:
        try:
            t_range = spark.table(t_name).agg(spark_min(t_col).alias("e"), spark_max(t_col).alias("l")).collect()[0]
            print(f"   {t_label}: {str(t_range['e'])[:19]} → {str(t_range['l'])[:19]}")
            r_e, r_l = str(review_range['earliest'])[:10], str(review_range['latest'])[:10]
            t_e, t_l = str(t_range['e'])[:10], str(t_range['l'])[:10]
            overlaps = (r_e <= t_l and t_e <= r_l)
            status = "PASS" if overlaps else "FINDING"
            print(f"   {'✅' if overlaps else '⚠️ '} Overlap with {t_label}: {overlaps} — {status}")
            findings_673.append({"Check": f"Temporal overlap: review ↔ {t_label}", "Description": f"{'Overlapping' if overlaps else 'Disjoint'}", "Status": status})
        except Exception as e:
            print(f"   ⚠️  {t_label}: {e}")
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "Temporal span", "Description": str(e), "Status": "SKIPPED"})

# ═══ Check 7: isolated review businesses ═══
print(f"\n{'─'*60}")
print("📌 Check 7: Businesses in review but absent from ALL sibling tables")
try:
    r_bids = df.select("business_id").distinct()
    total_r = r_bids.count()
    ck_bids = spark.table("yelp_poc_uc.bronze_final.yelp_checkin").select("business_id").distinct()
    tp_bids = spark.table("yelp_poc_uc.bronze_final.yelp_tip").select("business_id").distinct()
    isolated = r_bids.join(broadcast(ck_bids), "business_id", "left_anti").join(broadcast(tp_bids), "business_id", "left_anti")
    iso_count = isolated.count()
    iso_pct = iso_count / total_r * 100 if total_r > 0 else 0
    status = "PASS" if iso_pct < 5 else ("WARNING" if iso_pct < 15 else "FINDING")
    print(f"   Review-only businesses (no checkins AND no tips): {iso_count:,} ({iso_pct:.1f}%)")
    print(f"   {'✅' if status == 'PASS' else '⚠️ '} Status: {status}")
    if iso_count > 0 and iso_pct >= 5:
        iso_reviews = df.join(broadcast(isolated), "business_id", "inner")
        iso_stats = iso_reviews.agg(_count("*").alias("n"), spark_avg("stars").alias("avg_s")).collect()[0]
        print(f"   These businesses account for {iso_stats['n']:,} reviews (avg stars: {iso_stats['avg_s']:.2f})")
    findings_673.append({"Check": "Isolated review businesses", "Description": f"{iso_count:,} ({iso_pct:.1f}%) — no checkins/tips", "Status": status})
except Exception as e:
    print(f"   ⚠️  Error: {e}")
    findings_673.append({"Check": "Isolated businesses", "Description": str(e), "Status": "SKIPPED"})

# ══ Summary ══
print(f"\n{'='*80}")
print("6.7.3 SUMMARY — Cross-Table Aggregate Reconciliation")
print("=" * 80)
findings_673_df = pd.DataFrame(findings_673)
display(findings_673_df)
v = sum(1 for f in findings_673 if f.get("Status") == "FINDING")
w = sum(1 for f in findings_673 if f.get("Status") == "WARNING")
if v == 0 and w == 0:
    print("✅ Cross-table aggregates are consistent and plausible.")
elif v == 0:
    print(f"⚠️  No critical findings, but {w} warning(s) — likely dataset-window drift.")
else:
    print(f"🚨 {v} finding(s) and {w} warning(s) — review above.")
print("=" * 80)

6.7.3) CROSS-TABLE AGGREGATE RECONCILIATION
    Comparing computed review aggregates vs stored values in parent tables

────────────────────────────────────────────────────────────
📌 Check 1: COUNT(reviews) per business vs yelp_business.review_count
   Businesses compared: 150,346
   ⚠️  Exact matches: 82,984 (55.2%) | Mismatches: 67,362 — FINDING
   Direction: stored > computed in 9 cases (expected for dataset-window drift)
      wz8ZPfySQczcPgSyd33-HQ: computed=3634, stored=3345, diff=-289
      PY9GRfzr4nTZeINf346QOw: computed=2715, stored=2486, diff=-229
      QHWYlmVbLC3K6eglWoHVvA: computed=3388, stored=3260, diff=-128
      j-qtdD55OLfSqfsWuQTDJg: computed=2884, stored=2761, diff=-123
      ac1AeYqs8Z4_e2X5M3if2A: computed=7516, stored=7400, diff=-116

────────────────────────────────────────────────────────────
📌 Check 2: AVG(review.stars) per business vs yelp_business.stars
   Businesses compared: 150,346
   Within 0.05 (near-exact): 35,988 (23.9%)
   Within 0.5 tolerance:    

Check,Description,Status
review_count per business,"55.2% exact (67,362 mismatches)",FINDING
avg stars per business,"23.9% near-exact, 99.7% within 0.5",PASS
review_count per user,"24.0% exact (1,511,189 mismatches)",FINDING
avg stars per user,"37.1% near-exact, 62.7% within 0.5",FINDING
useful votes per user,"44.3% exact, computed>stored: 1,995",FINDING
funny votes per user,"64.4% exact, computed>stored: 485",WARNING
cool votes per user,"61.7% exact, computed>stored: 388",WARNING
Temporal overlap: review ↔ yelp_tip,Overlapping,PASS
Temporal overlap: review ↔ yelp_checkin,Overlapping,PASS
Isolated review businesses,"14,107 (9.4%) — no checkins/tips",WARNING


🚨 4 finding(s) and 3 warning(s) — review above.


In [0]:
# 6.7.4) Referential Integrity Summary
# ─────────────────────────────────────────────────────────────────────
# Consolidated summary of all 6.7.x referential integrity checks.

import pandas as pd

print("=" * 80)
print("6.7) REFERENTIAL INTEGRITY — CONSOLIDATED SUMMARY")
print("=" * 80)

all_ri_findings = []

for f in findings_671:
    all_ri_findings.append({
        "Section": "6.7.1 FK Validity",
        "FK": f.get("FK", ""),
        "Check": f["Check"],
        "Detail": f.get("Description", ""),
        "Status": f["Status"]
    })

for f in findings_672:
    all_ri_findings.append({
        "Section": "6.7.2 Lookup Conformance",
        "FK": f.get("Dimension", ""),
        "Check": f["Check"],
        "Detail": f.get("Overlap %", f.get("Description", "")),
        "Status": f["Status"]
    })

for f in findings_673:
    all_ri_findings.append({
        "Section": "6.7.3 Aggregate Reconciliation",
        "FK": "",
        "Check": f["Check"],
        "Detail": f.get("Description", ""),
        "Status": f["Status"]
    })

all_ri_df = pd.DataFrame(all_ri_findings)
display(all_ri_df)

status_counts = all_ri_df["Status"].value_counts().to_dict()
findings_count = status_counts.get("FINDING", 0)
warnings_count = status_counts.get("WARNING", 0)
passed_count = status_counts.get("PASS", 0)
info_count = status_counts.get("INFO", 0)
skipped_count = status_counts.get("SKIPPED", 0)

print(f"\n   Total checks: {len(all_ri_findings)}")
print(f"   ✅ PASS:    {passed_count}")
print(f"   ℹ️  INFO:    {info_count}")
print(f"   ⚠️  WARNING: {warnings_count}")
print(f"   🚨 FINDING: {findings_count}")
print(f"   ⏭️  SKIPPED: {skipped_count}")

if findings_count == 0 and warnings_count == 0:
    print(f"\n✅ Referential integrity is solid — both FK relationships (business_id, user_id)")
    print(f"   valid, lookup tables consistent, and cross-table aggregates reconciled.")
elif findings_count == 0:
    print(f"\n⚠️  No critical findings, but {warnings_count} warning(s) to review.")
    print(f"   Expected: dataset-window drift causes stored counters to exceed computed values.")
else:
    print(f"\n🚨 {findings_count} critical finding(s) and {warnings_count} warning(s) — action needed.")

print("\nREFERENTIAL INTEGRITY CHECKS COMPLETE")
print("=" * 80)

6.7) REFERENTIAL INTEGRITY — CONSOLIDATED SUMMARY


Section,FK,Check,Detail,Status
6.7.1 FK Validity,business_id,View 1: Orphan Records,review.business_id NOT IN yelp_business,PASS
6.7.1 FK Validity,business_id,View 2: Missing Parents,yelp_business with zero reviews (0.0%),INFO
6.7.1 FK Validity,business_id,View 3: Invalid FK Format,Malformed business_id (child + parent),PASS
6.7.1 FK Validity,user_id,View 1: Orphan Records,review.user_id NOT IN yelp_user,FINDING
6.7.1 FK Validity,user_id,View 2: Missing Parents,yelp_user with zero reviews (0.0%),INFO
6.7.1 FK Validity,user_id,View 3: Invalid FK Format,Malformed user_id (child + parent),PASS
6.7.2 Lookup Conformance,business_id,review ↔ yelp_checkin,87.8%,PASS
6.7.2 Lookup Conformance,business_id,review ↔ yelp_tip,70.6%,WARNING
6.7.2 Lookup Conformance,user_id,review ↔ yelp_tip,15.2%,FINDING
6.7.3 Aggregate Reconciliation,,review_count per business,"55.2% exact (67,362 mismatches)",FINDING



   Total checks: 19
   ✅ PASS:    7
   ℹ️  INFO:    2
   ⚠️  WARNING: 4
   🚨 FINDING: 6
   ⏭️  SKIPPED: 0

🚨 6 critical finding(s) and 4 warning(s) — action needed.

REFERENTIAL INTEGRITY CHECKS COMPLETE



### 6.8) BUSINESS RULE & ACCURACY CHECKS
*Review-domain plausibility, behavioral anomalies, and accuracy signals that are not merely schema, null, or simple range issues.*

- **Review Business Rules** — Suspicious or implausible review behavior in Yelp context.
  - *Example:* one user posts an unusually high number of reviews in a single day.
  - *Example:* the same user reuses near-identical review text across multiple businesses.
  - *Example:* an ultra-short review attracts unusually high vote totals.

- **Statistical Outlier Detection** — Extreme values in review text length, engagement, or entity-level review volume.

- **Distribution Anomaly Detection** — Unusual shifts in yearly review volume, rating mix, weekday/weekend posting behavior, or seasonality.

- **Concentration & Pareto Analysis** — Whether review activity is dominated by a small subset of businesses.

- **Business Activity Span Analysis** — Whether business review lifecycles are unnaturally compressed, dormant long before the dataset cutoff, or unusually intense.


In [0]:
# 6.8.1) Business Rule Violations
# ─────────────────────────────────────────────────────────────────────
# Review-specific plausibility rules that go beyond simple schema/range checks.
# Earlier deterministic findings are cross-referenced; new review-domain rules
# focus on suspicious reviewer behavior, templated text reuse, and engagement.

from pyspark.sql.functions import (
    col, count as _count, countDistinct, sum as _sum, length, lower, trim,
    regexp_replace, to_timestamp, to_date
)
import pandas as pd

print("=" * 80)
print("6.8.1) BUSINESS RULE VIOLATIONS")
print("    Review-domain plausibility rules for reviewer behavior and engagement")
print("=" * 80)

findings_68 = []

df68 = (
    df
    .withColumn("review_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("review_date", to_date(col("review_ts")))
    .withColumn("text_length", length(trim(col("text"))))
    .withColumn("text_norm", regexp_replace(lower(trim(col("text"))), r"\s+", " "))
    .withColumn("total_votes", col("useful") + col("funny") + col("cool"))
)

total_rows = df68.count()

print(f"\n{'─'*60}")
print("📌 Previously Verified Checks")
already_done = [
    ("Stars restricted to {1,2,3,4,5}",      "6.2.1 / 6.2.2",            "PASS"),
    ("Negative vote counts",                  "6.2.1 / 6.6.3 / 6.6.4",   "FINDING (DQ-REVIEW-003 / DQ-REVIEW-016)"),
    ("Empty review text",                     "6.2.2",                    "PASS"),
    ("Pre-Yelp / implausible years",          "6.2.2 / 6.2.4",            "PASS"),
    ("Exact duplicate rows",                  "6.5.1",                    "PASS"),
    ("Duplicate review_id",                   "4.7 / 6.5.2",              "PASS"),
]
for check, ref, result in already_done:
    icon = "✅" if result == "PASS" else "🔗"
    print(f"   {icon} {check:<38} → {ref} [{result}]")

# ── Rule 1: unusually high daily posting volume per user ─────────────────
print(f"\n{'─'*60}")
print("📌 Rule 1: Daily reviews per user > 20")
print("   (A very high single-day posting volume may indicate automation or bulk importing)")

daily_user = (
    df68.groupBy("user_id", "review_date")
        .agg(_count("*").alias("daily_review_count"))
)
high_daily_users = daily_user.filter(col("daily_review_count") > 20)
high_daily_days = high_daily_users.count()
high_daily_distinct_users = high_daily_users.select("user_id").distinct().count()
status = "INFO" if high_daily_days > 0 else "PASS"
print(f"   {'ℹ️' if high_daily_days > 0 else '✅'} User-days above threshold: {high_daily_days:,} — {status}")
print(f"   Distinct users involved: {high_daily_distinct_users:,}")
if high_daily_days > 0:
    display(high_daily_users.orderBy(col("daily_review_count").desc()).limit(5))
findings_68.append({
    "Check": "High daily reviewer volume",
    "Detail": f"{high_daily_days:,} user-days across {high_daily_distinct_users:,} users with >20 reviews/day",
    "Status": status
})

# ── Rule 2: same user reuses near-identical text across businesses ───────
print(f"\n{'─'*60}")
print("📌 Rule 2: Same user reuses identical normalized text across ≥3 businesses")
print("   (Potential templated/spam reviewing behavior; minimum text length avoids trivial phrases)")

reused_text = (
    df68.filter(col("text_length") >= 20)
        .groupBy("user_id", "text_norm")
        .agg(
            countDistinct("business_id").alias("distinct_businesses"),
            _count("*").alias("review_rows")
        )
        .filter(col("distinct_businesses") >= 3)
)
reused_groups = reused_text.count()
reused_rows = reused_text.agg(_sum("review_rows").alias("rows")).collect()[0]["rows"] or 0
status = "INFO" if reused_groups > 0 else "PASS"
print(f"   {'ℹ️' if reused_groups > 0 else '✅'} Reused-text groups: {reused_groups:,} — {status}")
print(f"   Affected review rows: {reused_rows:,}")
if reused_groups > 0:
    display(reused_text.orderBy(col("distinct_businesses").desc(), col("review_rows").desc()).limit(5))
findings_68.append({
    "Check": "Repeated review text across businesses",
    "Detail": f"{reused_groups:,} user-text groups affecting {reused_rows:,} rows",
    "Status": status
})

# ── Rule 3: ultra-short reviews with strong community engagement ─────────
print(f"\n{'─'*60}")
print("📌 Rule 3: Reviews under 15 characters with total_votes ≥ 10")
print("   (A near-empty review receiving high engagement is unusual and worth reviewing)")

short_high_vote = df68.filter((col("text_length") < 15) & (col("total_votes") >= 10))
short_high_vote_count = short_high_vote.count()
status = "INFO" if short_high_vote_count > 0 else "PASS"
print(f"   {'ℹ️' if short_high_vote_count > 0 else '✅'} Suspicious short-but-high-vote reviews: {short_high_vote_count:,} — {status}")
if short_high_vote_count > 0:
    display(
        short_high_vote
        .select("review_id", "user_id", "business_id", "stars", "total_votes", "text")
        .orderBy(col("total_votes").desc(), col("stars").desc())
        .limit(5)
    )
findings_68.append({
    "Check": "Short reviews with high votes",
    "Detail": f"{short_high_vote_count:,} reviews with <15 chars and at least 10 total votes",
    "Status": status
})

# ── Rule 4: exceptionally high total vote counts per review ──────────────
print(f"\n{'─'*60}")
print("📌 Rule 4: Reviews with total_votes > 100")
print("   (Not invalid, but these highly visible reviews can strongly influence downstream modeling)")

viral_reviews = df68.filter(col("total_votes") > 100)
viral_review_count = viral_reviews.count()
status = "INFO" if viral_review_count > 0 else "PASS"
print(f"   {'ℹ️' if viral_review_count > 0 else '✅'} Highly engaged reviews: {viral_review_count:,} — {status}")
if viral_review_count > 0:
    display(
        viral_reviews
        .select("review_id", "business_id", "user_id", "stars", "useful", "funny", "cool", "total_votes")
        .orderBy(col("total_votes").desc())
        .limit(5)
    )
findings_68.append({
    "Check": "Highly engaged reviews",
    "Detail": f"{viral_review_count:,} reviews with more than 100 total votes",
    "Status": status
})

print(f"\n{'='*80}")
print(f"6.8.1 complete — {len(findings_68)} review-domain rules evaluated")
print("=" * 80)

6.8.1) BUSINESS RULE VIOLATIONS
    Review-domain plausibility rules for reviewer behavior and engagement

────────────────────────────────────────────────────────────
📌 Previously Verified Checks
   ✅ Stars restricted to {1,2,3,4,5}        → 6.2.1 / 6.2.2 [PASS]
   🔗 Negative vote counts                   → 6.2.1 / 6.6.3 / 6.6.4 [FINDING (DQ-REVIEW-003 / DQ-REVIEW-016)]
   ✅ Empty review text                      → 6.2.2 [PASS]
   ✅ Pre-Yelp / implausible years           → 6.2.2 / 6.2.4 [PASS]
   ✅ Exact duplicate rows                   → 6.5.1 [PASS]
   ✅ Duplicate review_id                    → 4.7 / 6.5.2 [PASS]

────────────────────────────────────────────────────────────
📌 Rule 1: Daily reviews per user > 20
   (A very high single-day posting volume may indicate automation or bulk importing)
   ℹ️ User-days above threshold: 732 — INFO
   Distinct users involved: 641


user_id,review_date,daily_review_count
n-lBS02-3yvlY5Q91mmwDA,2005-07-10,121
6EHUYIyEb-biAOAGRBqEpg,2019-09-01,109
sLqm9fRo_u81kJCIRdzrMg,2008-05-24,88
LwZJFLGxQwjjeOgpqTJnfw,2020-06-15,82
Cf8Jw9yX8RrG-l4PQoMkuw,2012-02-26,76



────────────────────────────────────────────────────────────
📌 Rule 2: Same user reuses identical normalized text across ≥3 businesses
   (Potential templated/spam reviewing behavior; minimum text length avoids trivial phrases)
   ℹ️ Reused-text groups: 145 — INFO
   Affected review rows: 557


user_id,text_norm,distinct_businesses,review_rows
H47NeEoS2T1ZYMEhugcQig,"do not park here! they are too quick to boot you! $144 to remove a boot. if you lose track of time (a normal affair in new orleans, having fun) and even if (like i did) you add time, they get you in between!!!! avoid all premium parking lots!! there are so many other options near by! trust me!",18,18
oAC2NWyUCB5FqnVSOscSkA,"i had a terrible experience with the customer service side of club fitness. i was in st. louis for 2 months this summer for my job, and i joined for those dates. the gym stipulates when you join that should you like to cancel, you need to do so online and 30 days before you are charged again. i knew when i would be leaving, so i put this in my calendar, and made sure to cancel on time. i followed through with this and double-checked on my last day by going up to the counter, asking them to make sure my account was cancelled, and they said everything looked good. i got back to my home in illinois, and have been charged every month since leaving. i called customer service, completely confused as to why i have been charged. they claim that they have no recollection of my cancellation online, and therefore will not reimburse me. additionally, i asked right then and there to cancel my account immediately, they said, ""you must cancel online only, we can't do it over the phone."" which probably means i will be charged again, since i need to give 30 days notice. unbelievable. the confirmation email from my original cancellation no longer exists (why would it, because the gym confirmed it with me on my last day at the gym) so i have no proof that i cancelled. it is their word against mine. i had a nice time working out at the gym, but i have never had to deal with such bad customer service in my life. i am out a lot of money, and it was never my fault. i say to you: stay away!!!!",11,11
qHZWwTeGdl66NhFHXntZ2Q,"at the height of the omicron surge, wendy's does not require their food handler employees to wear masks! this is the height of corporate irresponsibility and lack of caring for their customers as well as their employees. not going there any time soon and suggest you consider this wreckless no-health policy before you take a bite. if they can't be bothered, then neither can i.",10,10
EtNv9my1szqzoBAYixfyag,"we had mistakenly been short paying our car loan by a whopping $1.83 per month... totally our fault. but, instead of calling all of our contact numbers or sending us a letter, ssfcu reports this to the credit bureaus and dings my wife's and my credit by 100 points. when called about this, they said ""not our fault"". i will be ending my relationship as soon as i can, and i will go with a credit union that care enough to call/mail about things like this before killing my credit. a little human touch vs. automated calling would have solved this. all for $1.83/month...nice customer service!",10,10
NgVx1kHZf8jP25Q5RIU7Ew,great place to be. golds gym is hands down the best gym in the area. love that they're updating all of their clubs!,9,9



────────────────────────────────────────────────────────────
📌 Rule 3: Reviews under 15 characters with total_votes ≥ 10
   (A near-empty review receiving high engagement is unusual and worth reviewing)
   ℹ️ Suspicious short-but-high-vote reviews: 7 — INFO


review_id,user_id,business_id,stars,total_votes,text
5L1UABBQf0HXQQtq4bV5Mw,Ase_kJIYuT6yOsqqVPuWUA,NacQKO2s6W3GXTK3b1RoVA,3.0,30,..
CyK_p3d1woWEp0R0RB8Jjg,Ase_kJIYuT6yOsqqVPuWUA,2EOinQpyU4ChpyJ4KusO2A,5.0,26,See below
0i57NiyGEgi11r5hQ3DXXQ,lMY8NBPyzlPbbu-KBYfD9A,Ct6pupk8Cy6sI2mrIzF7UA,4.0,18,Great
nv_9SrpAqUK24JgmwhZOLQ,CMSiKCRbQallCbseIKisjw,yYMY4TUCGys5fFju6zODbg,1.0,15,O
h1YOtIbEcd5Wa3l30guLJg,fv6vl7th2L2GIwGAbtpHow,p8nHNmp_L_GsI3hRtRXrDw,5.0,12,Solid.



────────────────────────────────────────────────────────────
📌 Rule 4: Reviews with total_votes > 100
   (Not invalid, but these highly visible reviews can strongly influence downstream modeling)
   ℹ️ Highly engaged reviews: 3,070 — INFO


review_id,business_id,user_id,stars,useful,funny,cool,total_votes
b0HBLNn02tkpC9rIyPRc_Q,3RcQ7yTvJn9LLuXthTkWiQ,aMR0N5sy95Qz0sUjg1bmYA,1.0,1182,0,0,1182
pL_jmXOSPOMq71ZwrlRynQ,iRIHK8-EwpeffwvoO4nzIA,sMfFISqyKkxlLsRzYaL9Lw,1.0,112,792,107,1011
kTSpD7akIapty9AtLwJ11g,3RcQ7yTvJn9LLuXthTkWiQ,aPpOZxh2p1SoWxRL3ulYGQ,1.0,997,0,0,997
_KhSKjKKXoLgINKv7lTr0Q,3RcQ7yTvJn9LLuXthTkWiQ,CHN_EDxhElh9DS1Vud6-hA,1.0,969,0,0,969
TFlWosaJn3lVJwgm3nrVYA,3RcQ7yTvJn9LLuXthTkWiQ,9YOVI-bxiiOx0bTZ-J_Irw,1.0,933,0,0,933



6.8.1 complete — 4 review-domain rules evaluated


---
#### 🟡 DQ-REVIEW-022 — Suspicious Reviewer Behavior Patterns

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-022` |
| **Description** | The revised review-focused rule set surfaces several **behavioral anomaly patterns** rather than hard data-integrity failures: **732 user-day combinations** across **641 users** exceed **20 reviews in a single day** (top case: user `n-lBS02-3yvlY5Q91mmwDA` posted **121 reviews on 2005-07-10**), **145 user-text groups** reuse the same normalized review text across **3+ businesses** affecting **557 reviews**, **7 reviews** are ultra-short (`<15` chars) yet received **10+ votes**, and **3,070 reviews** accumulated **more than 100 total votes**. These patterns are plausible on an open review platform, but they are important for spam detection, reviewer-behavior analysis, and downstream model robustness. |
| **Detection Phase** | Phase 6.8 → **6.8.1) Business Rule Violations** |
| **Detected In** | Cell 90: `6.8.1) Business Rule Violations` |
| **Severity** | 🟡 **Moderate** — These are not schema violations, but they can distort trust, reviewer-level features, engagement models, and anomaly detection if treated as ordinary behavior. Severity increases for fraud/spam use cases. |
| **Affected Records** | 732 high-volume user-days, 557 reused-text review rows, 7 short-but-high-vote reviews, 3,070 highly engaged reviews. |
| **Root Cause** | Likely a mix of genuinely prolific reviewers, cross-posted complaint/praise text, viral reviews with unusual visibility, and a smaller subset of potentially templated or coordinated behavior. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag suspicious reviewer behavior** with booleans such as `is_high_daily_reviewer`, `is_reused_text_pattern`, and `is_highly_engaged_review` | Preserves all records while making risky behavioral patterns explicit. |
| 2 | **Review repeated-text patterns manually** for top users before filtering | Reused wording may represent legitimate copy/paste complaints across related businesses, not just spam. |
| 3 | **Down-weight high-engagement outliers** in reviewer-quality or reputation modeling | Prevents a small set of viral reviews from dominating downstream features. |

##### ✅ Best Logical Solution: Flag first, investigate top offenders second.


In [0]:
# 6.8.2) Statistical Outlier Detection
# ─────────────────────────────────────────────────────────────────────
# Review-level outliers: text length and total engagement votes.
# These are not automatic errors, but they are important for moderation,
# NLP robustness, and downstream model sensitivity.

from pyspark.sql.functions import (
    col, length, trim, mean, stddev, skewness, kurtosis,
    min as spark_min, max as spark_max, percentile_approx,
    substring
)

print("=" * 80)
print("6.8.2) STATISTICAL OUTLIER DETECTION")
print("    Review text-length and engagement outlier profiling")
print("=" * 80)

df68 = (
    df
    .withColumn("text_length", length(trim(col("text"))))
    .withColumn("total_votes", col("useful") + col("funny") + col("cool"))
)

total_reviews = df68.count()
print(f"\n   Total reviews profiled: {total_reviews:,}")

metric_specs = [
    ("text_length", "Review text length (characters)"),
    ("total_votes", "Total reaction votes per review")
]

for metric, label in metric_specs:
    print(f"\n{'─'*60}")
    print(f"📌 {label}")

    stats = df68.agg(
        mean(metric).alias("avg"),
        stddev(metric).alias("std"),
        spark_min(metric).alias("min_val"),
        spark_max(metric).alias("max_val"),
        percentile_approx(metric, [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).alias("pcts"),
        skewness(metric).alias("skew"),
        kurtosis(metric).alias("kurt")
    ).collect()[0]

    avg_val = stats["avg"] or 0
    std_val = stats["std"] or 0
    p01, p05, p25, p50, p75, p95, p99 = stats["pcts"]

    print(f"   Min / P01 / P05 / Median / P95 / P99 / Max")
    print(f"   {stats['min_val']:,} / {p01:,} / {p05:,} / {p50:,} / {p95:,} / {p99:,} / {stats['max_val']:,}")
    print(f"   Mean: {avg_val:,.2f}  |  Std: {std_val:,.2f}  |  Skew: {stats['skew']:.2f}  |  Kurtosis: {stats['kurt']:.2f}")

    # Z-score upper-tail outliers
    z_upper = avg_val + 3 * std_val
    z_outliers = df68.filter(col(metric) > z_upper).count() if std_val > 0 else 0
    print(f"   μ + 3σ threshold: {z_upper:,.2f}  |  Upper-tail Z outliers: {z_outliers:,}")

    # IQR upper-tail outliers
    iqr = p75 - p25
    mild_upper = p75 + 1.5 * iqr
    extreme_upper = p75 + 3.0 * iqr
    mild_cnt = df68.filter((col(metric) > mild_upper) & (col(metric) <= extreme_upper)).count()
    extreme_cnt = df68.filter(col(metric) > extreme_upper).count()
    print(f"   Tukey fences: mild>{mild_upper:,.2f}  |  extreme>{extreme_upper:,.2f}")
    print(f"   Mild upper outliers: {mild_cnt:,}  |  Extreme upper outliers: {extreme_cnt:,}")

    sample_cols = ["review_id", "business_id", "user_id", "stars", metric]
    if metric == "text_length":
        sample_cols.append(substring(col("text"), 1, 120).alias("text_preview"))
    else:
        sample_cols.extend(["useful", "funny", "cool"])

    display(df68.select(*sample_cols).orderBy(col(metric).desc()).limit(5))

    findings_68.append({
        "Check": f"{metric} Z-score outliers",
        "Detail": f"{z_outliers:,} reviews above μ+3σ ({z_upper:,.2f})",
        "Status": "INFO" if z_outliers > 0 else "PASS"
    })
    findings_68.append({
        "Check": f"{metric} extreme IQR outliers",
        "Detail": f"{extreme_cnt:,} reviews above outer fence ({extreme_upper:,.2f})",
        "Status": "INFO" if extreme_cnt > 0 else "PASS"
    })

print(f"\n{'='*80}")
print("6.8.2 complete — review-level outlier profiling finished")
print("=" * 80)

6.8.2) STATISTICAL OUTLIER DETECTION
    Review text-length and engagement outlier profiling

   Total reviews profiled: 6,990,280

────────────────────────────────────────────────────────────
📌 Review text length (characters)
   Min / P01 / P05 / Median / P95 / P99 / Max
   1 / 86 / 113 / 406 / 1,565 / 2,641 / 5,000
   Mean: 567.76  |  Std: 527.26  |  Skew: 2.76  |  Kurtosis: 11.90
   μ + 3σ threshold: 2,149.54  |  Upper-tail Z outliers: 138,640
   Tukey fences: mild>1,456.50  |  extreme>2,193.00
   Mild upper outliers: 290,670  |  Extreme upper outliers: 129,953


review_id,business_id,user_id,stars,text_length,text_preview
H6YAb7g-__1zFxUsRIJSKw,PuPRPHuGlOVhCdTcKUvS0A,0I5s0O5lwTFrcr2Zxt-0Pw,1.0,5000,"Hey, dumb physician assistant or whatever you are, maybe don't lead people into a room and tell them ""sorry for your los"
VjfdbDFRFOzisBkae4FxNQ,bJfcuUMgJUkHWpCUeAI4Iw,xc1opNkFCnAjsPXs7jqtfg,5.0,5000,"WEEKEND IN PHILLY, PART 4 I recently returned to my former stomping grounds in Philly for long foodie weekend: amazing"
L4u5yrNTdjy2RGAT2m147g,KVnF0iaOg1AKD0hNh8mV7w,CfuZTe3tk1b12vraHmhLpg,1.0,5000,"I really wanted to like this place, but there was just too much bad for me to rate the place higher than one star and wo"
7rpWsQ33sZOS-EQXzIXH6w,pRqLUBx3LPiKZ5PKU_fwdg,5tZ4HD6uH2ylYqXDvOO8eA,1.0,5000,"This company will screw you over, the gym is a dump, and employees are rude, including general manager, Everett. I join"
3WGqkDPqKIO2DaWlc-JrMQ,avKQ--Xf0s7C7yVSPC72Fg,3fCb2So6bBV_EK8RUQ6cug,4.0,5000,"Soaring above the trees, as you drive through downtown Tampa, you might be a bit confused as to your whereabouts. You m"



────────────────────────────────────────────────────────────
📌 Total reaction votes per review
   Min / P01 / P05 / Median / P95 / P99 / Max
   -3 / 0 / 0 / 1 / 8 / 21 / 1,182
   Mean: 2.01  |  Std: 6.20  |  Skew: 31.04  |  Kurtosis: 2541.90
   μ + 3σ threshold: 20.60  |  Upper-tail Z outliers: 72,559
   Tukey fences: mild>5.00  |  extreme>8.00
   Mild upper outliers: 283,303  |  Extreme upper outliers: 311,177


review_id,business_id,user_id,stars,total_votes,useful,funny,cool
b0HBLNn02tkpC9rIyPRc_Q,3RcQ7yTvJn9LLuXthTkWiQ,aMR0N5sy95Qz0sUjg1bmYA,1.0,1182,1182,0,0
pL_jmXOSPOMq71ZwrlRynQ,iRIHK8-EwpeffwvoO4nzIA,sMfFISqyKkxlLsRzYaL9Lw,1.0,1011,112,792,107
kTSpD7akIapty9AtLwJ11g,3RcQ7yTvJn9LLuXthTkWiQ,aPpOZxh2p1SoWxRL3ulYGQ,1.0,997,997,0,0
_KhSKjKKXoLgINKv7lTr0Q,3RcQ7yTvJn9LLuXthTkWiQ,CHN_EDxhElh9DS1Vud6-hA,1.0,969,969,0,0
TFlWosaJn3lVJwgm3nrVYA,3RcQ7yTvJn9LLuXthTkWiQ,9YOVI-bxiiOx0bTZ-J_Irw,1.0,933,933,0,0



6.8.2 complete — review-level outlier profiling finished


In [0]:
# 6.8.3) Distribution Anomaly Detection
# ─────────────────────────────────────────────────────────────────────
# Review-table distribution diagnostics: rating mix, year-over-year shifts,
# weekday/weekend behavior, posting hour patterns, and month seasonality.
# The goal is interpretability, not forcing every pattern to be uniform.

from pyspark.sql.functions import (
    col, count as _count, to_timestamp, year, month, hour,
    date_format, dayofweek, when
)
import numpy as np
import pandas as pd

print("=" * 80)
print("6.8.3) DISTRIBUTION ANOMALY DETECTION")
print("    Rating mix, temporal cadence, and seasonality diagnostics")
print("=" * 80)

df68 = (
    df
    .withColumn("review_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("review_year", year(col("review_ts")))
    .withColumn("review_month", month(col("review_ts")))
    .withColumn("review_month_name", date_format(col("review_ts"), "MMMM"))
    .withColumn("review_hour", hour(col("review_ts")))
    .withColumn("review_dow_number", when(dayofweek(col("review_ts")) == 1, 7).otherwise(dayofweek(col("review_ts")) - 1))
    .withColumn("review_dow_name", date_format(col("review_ts"), "EEEE"))
)

def shannon_entropy(observed):
    total = sum(observed)
    probs = [o / total for o in observed if o > 0]
    h = -sum(p * np.log2(p) for p in probs)
    max_h = np.log2(len(observed)) if len(observed) > 0 else 0
    return h, (h / max_h) if max_h > 0 else 0

# ── Rating distribution ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Rating Distribution")
star_pd = df68.groupBy("stars").agg(_count("*").alias("cnt")).orderBy("stars").toPandas()
star_pd["pct"] = star_pd["cnt"] / star_pd["cnt"].sum() * 100
display(star_pd)

positive_share = star_pd.loc[star_pd["stars"] >= 4, "pct"].sum()
negative_share = star_pd.loc[star_pd["stars"] <= 2, "pct"].sum()
polarized_share = star_pd.loc[star_pd["stars"].isin([1, 5]), "pct"].sum()
print(f"   Positive share (4–5★): {positive_share:.1f}%")
print(f"   Negative share (1–2★): {negative_share:.1f}%")
print(f"   Polarized share (1★ + 5★): {polarized_share:.1f}%")
findings_68.append({
    "Check": "Polarized rating share",
    "Detail": f"{polarized_share:.1f}% of reviews are 1★ or 5★",
    "Status": "INFO"
})

# ── Year distribution and non-boundary YoY shifts ────────────────────────
print(f"\n{'─'*60}")
print("📌 Year Distribution & YoY Change")
year_pd = df68.groupBy("review_year").agg(_count("*").alias("cnt")).orderBy("review_year").toPandas()
year_pd["yoy_pct"] = year_pd["cnt"].pct_change() * 100
display(year_pd)

min_year = int(year_pd["review_year"].min())
max_year = int(year_pd["review_year"].max())
non_boundary_drops = year_pd[
    (year_pd["review_year"] > min_year) &
    (year_pd["review_year"] < max_year) &
    (year_pd["yoy_pct"] < -50)
]
print(f"   Boundary years: {min_year} and {max_year} are treated cautiously because partial coverage is common.")
if len(non_boundary_drops) > 0:
    print(f"   ⚠️ Non-boundary years with >50% YoY decline: {list(non_boundary_drops['review_year'].astype(int))}")
else:
    print("   ✅ No non-boundary year shows a >50% YoY collapse.")
findings_68.append({
    "Check": "Non-boundary YoY declines",
    "Detail": f"{len(non_boundary_drops)} year(s) with >50% decline excluding first/last year",
    "Status": "INFO" if len(non_boundary_drops) > 0 else "PASS"
})

# ── Day-of-week distribution ──────────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Day-of-Week Distribution")
dow_pd = df68.groupBy("review_dow_number", "review_dow_name").agg(_count("*").alias("cnt")).orderBy("review_dow_number").toPandas()
dow_pd["pct"] = dow_pd["cnt"] / dow_pd["cnt"].sum() * 100
display(dow_pd)

weekend_share = dow_pd.loc[dow_pd["review_dow_name"].isin(["Saturday", "Sunday"]), "pct"].sum()
peak_dow = dow_pd.loc[dow_pd["cnt"].idxmax()]
trough_dow = dow_pd.loc[dow_pd["cnt"].idxmin()]
_, dow_entropy_norm = shannon_entropy(dow_pd["cnt"].tolist())
print(f"   Weekend share: {weekend_share:.1f}%")
print(f"   Peak day: {peak_dow['review_dow_name']} ({peak_dow['pct']:.1f}%)")
print(f"   Lowest day: {trough_dow['review_dow_name']} ({trough_dow['pct']:.1f}%)")
print(f"   Normalized entropy: {dow_entropy_norm:.4f}")
findings_68.append({
    "Check": "Weekend review share",
    "Detail": f"{weekend_share:.1f}% of reviews arrive on Saturday/Sunday",
    "Status": "INFO"
})

# ── Hour-of-day distribution ──────────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Hour-of-Day Distribution")
hour_pd = df68.groupBy("review_hour").agg(_count("*").alias("cnt")).orderBy("review_hour").toPandas()
hour_pd["pct"] = hour_pd["cnt"] / hour_pd["cnt"].sum() * 100
display(hour_pd)

peak_hour = hour_pd.loc[hour_pd["cnt"].idxmax()]
trough_hour = hour_pd.loc[hour_pd["cnt"].idxmin()]
overnight_share = hour_pd.loc[hour_pd["review_hour"].between(0, 5), "pct"].sum()
print(f"   Peak hour: {int(peak_hour['review_hour']):02d}:00 ({peak_hour['pct']:.2f}%)")
print(f"   Lowest hour: {int(trough_hour['review_hour']):02d}:00 ({trough_hour['pct']:.2f}%)")
print(f"   Overnight share (00:00–05:59): {overnight_share:.2f}%")
findings_68.append({
    "Check": "Overnight posting share",
    "Detail": f"{overnight_share:.2f}% of reviews are posted between midnight and 6am",
    "Status": "INFO"
})

# ── Month distribution / seasonality ─────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Month Distribution & Seasonality")
month_pd = df68.groupBy("review_month", "review_month_name").agg(_count("*").alias("cnt")).orderBy("review_month").toPandas()
month_pd["pct"] = month_pd["cnt"] / month_pd["cnt"].sum() * 100
month_pd["seasonality_idx"] = month_pd["cnt"] / month_pd["cnt"].mean()
display(month_pd)

peak_month = month_pd.loc[month_pd["cnt"].idxmax()]
trough_month = month_pd.loc[month_pd["cnt"].idxmin()]
print(f"   Peak month: {peak_month['review_month_name']} ({peak_month['pct']:.2f}%, index={peak_month['seasonality_idx']:.3f})")
print(f"   Lowest month: {trough_month['review_month_name']} ({trough_month['pct']:.2f}%, index={trough_month['seasonality_idx']:.3f})")
findings_68.append({
    "Check": "Seasonality spread",
    "Detail": f"Peak month index {peak_month['seasonality_idx']:.3f} vs trough month index {trough_month['seasonality_idx']:.3f}",
    "Status": "INFO"
})

print(f"\n{'='*80}")
print("6.8.3 complete — review distribution diagnostics finished")
print("=" * 80)

6.8.3) DISTRIBUTION ANOMALY DETECTION
    Rating mix, temporal cadence, and seasonality diagnostics

────────────────────────────────────────────────────────────
📌 Rating Distribution


stars,cnt,pct
1.0,1069561,15.300688956665542
2.0,544240,7.785668099131938
3.0,691934,9.898516225387253
4.0,1452918,20.784832653341496
5.0,3231627,46.23029406547377


   Positive share (4–5★): 67.0%
   Negative share (1–2★): 23.1%
   Polarized share (1★ + 5★): 61.5%

────────────────────────────────────────────────────────────
📌 Year Distribution & YoY Change


review_year,cnt,yoy_pct
2005,854,null
2006,3853,351.17096018735367
2007,15363,298.7282636906307
2008,48226,213.91004361127384
2009,74387,54.24667191971135
2010,138587,86.30540282576256
2011,230813,66.54736735768867
2012,286570,24.15678492979165
2013,383950,33.98122622744879
2014,522275,36.02682640968877


   Boundary years: 2005 and 2022 are treated cautiously because partial coverage is common.
   ✅ No non-boundary year shows a >50% YoY collapse.

────────────────────────────────────────────────────────────
📌 Day-of-Week Distribution


...

   Weekend share: 31.6%
   Peak day: Sunday (16.4%)
   Lowest day: Thursday (13.1%)
   Normalized entropy: 0.9985

────────────────────────────────────────────────────────────
📌 Hour-of-Day Distribution


review_hour,cnt,pct
0,460765,6.591509925210436
1,457189,6.540353176124561
2,416472,5.957872932128613
3,339935,4.862966862557723
4,248771,3.5588130947544303
5,162636,2.326602081747798
6,100730,1.441000932723725
7,60031,0.858778189142638
8,38078,0.5447278220614911
9,30389,0.4347322281797009


   Peak hour: 18:00 (6.64%)
   Lowest hour: 09:00 (0.43%)
   Overnight share (00:00–05:59): 29.84%

────────────────────────────────────────────────────────────
📌 Month Distribution & Seasonality


review_month,review_month_name,cnt,pct,seasonality_idx
1,January,604532,8.648180044290072,1.0377816053148086
2,February,544125,7.784022957592543,0.934082754911105
3,March,598555,8.562675600977357,1.0275210721172827
4,April,551471,7.889111738013356,0.9466934085616027
5,May,586575,8.39129476930824,1.0069553723169886
6,June,601737,8.608195952093478,1.0329835142512174
7,July,654627,9.364818004429006,1.1237781605314807
8,August,636384,9.103841333966594,1.0924609600759911
9,September,565374,8.088002197336873,0.9705602636804248
10,October,571809,8.18005859564996,0.9816070314779951


   Peak month: July (9.36%, index=1.124)
   Lowest month: November (7.60%, index=0.912)

6.8.3 complete — review distribution diagnostics finished


---
#### 🟡 DQ-REVIEW-023 — Strong Rating Polarization & Non-Uniform Posting Cadence

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-023` |
| **Description** | The revised distribution analysis found **no non-boundary year with a >50% YoY decline**, which is a good sign after explicitly excluding the first and last dataset years from anomaly logic. However, the review table remains strongly **behaviorally non-uniform**: **61.5%** of reviews are either **1★ or 5★**, **67.0%** are positive (**4–5★**), **31.6%** arrive on weekends, and **29.84%** are posted between **00:00 and 05:59**. Monthly seasonality is present but moderate (seasonality index ranges from **0.912** to **1.124**). |
| **Detection Phase** | Phase 6.8 → **6.8.3) Distribution Anomaly Detection** |
| **Detected In** | Cell 93: `6.8.3) Distribution Anomaly Detection` |
| **Severity** | 🟡 **Informational / Moderate** — These are not data defects; they describe the behavioral shape of Yelp reviews. They matter because models or KPIs that assume balanced classes, uniform posting cadence, or smooth seasonality will be mis-specified. |
| **Affected Records** | Entire review table (**6,990,280** rows). |
| **Root Cause** | Typical review-platform behavior: users tend to post at emotional extremes, weekends generate more leisure/travel reviews, and overnight posting likely reflects asynchronous app usage plus timezone-normalized timestamps. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Keep explicit temporal features** such as hour, weekday/weekend, and month | These patterns carry real behavioral signal. |
| 2 | **Avoid assuming class balance** in rating models | The star distribution is clearly skewed toward 4★ and 5★ reviews. |
| 3 | **Treat first and last dataset years carefully** in trend analysis | Boundary years are structurally different and should be isolated from anomaly logic. |

##### ✅ Best Logical Solution: Retain these signals as modeled features rather than treating them as defects.


In [0]:
# 6.8.4) Concentration & Pareto Analysis
# ─────────────────────────────────────────────────────────────────────
# Assess how concentrated review activity is across businesses.
# This matters for representativeness: a small set of popular businesses
# can dominate descriptive statistics, model training, and evaluation.

from pyspark.sql.functions import col, count as _count
import numpy as np
import pandas as pd

print("=" * 80)
print("6.8.4) CONCENTRATION & PARETO ANALYSIS")
print("    How evenly are reviews distributed across businesses?")
print("=" * 80)

reviews_per_biz = df.groupBy("business_id").agg(_count("*").alias("review_count"))
biz_pd = reviews_per_biz.orderBy("review_count").toPandas()

total_reviews = int(biz_pd["review_count"].sum())
total_businesses = len(biz_pd)

print(f"\n   Total businesses: {total_businesses:,}")
print(f"   Total reviews:    {total_reviews:,}")

# ── Gini coefficient ──────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Gini Coefficient")
sorted_counts = np.sort(biz_pd["review_count"].values).astype(float)
n = len(sorted_counts)
index = np.arange(1, n + 1)
gini = (2.0 * np.sum(index * sorted_counts) / (n * np.sum(sorted_counts))) - (n + 1.0) / n
print(f"   Gini coefficient: {gini:.4f}")
if gini < 0.30:
    interpretation = "Low concentration"
elif gini < 0.50:
    interpretation = "Moderate concentration"
elif gini < 0.70:
    interpretation = "High concentration"
else:
    interpretation = "Very high concentration"
print(f"   Interpretation: {interpretation}")
findings_68.append({
    "Check": "Business concentration (Gini)",
    "Detail": f"Gini = {gini:.4f} ({interpretation.lower()})",
    "Status": "INFO"
})

# ── Pareto / top-N concentration ──────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Pareto (Top-N%) Analysis")
biz_desc = biz_pd.sort_values("review_count", ascending=False).reset_index(drop=True)
biz_desc["cum_reviews"] = biz_desc["review_count"].cumsum()
biz_desc["cum_pct"] = biz_desc["cum_reviews"] / total_reviews * 100

for pct in [1, 5, 10, 20, 50]:
    top_n = max(1, int(total_businesses * pct / 100))
    cum_pct = biz_desc.iloc[top_n - 1]["cum_pct"]
    print(f"   Top {pct:>2}% businesses ({top_n:>6,}) → {cum_pct:>5.1f}% of all reviews")

idx_80 = biz_desc[biz_desc["cum_pct"] >= 80].index[0] + 1
pct_biz_80 = idx_80 / total_businesses * 100
print(f"\n   80% of all reviews come from {idx_80:,} businesses ({pct_biz_80:.1f}%)")
findings_68.append({
    "Check": "Pareto concentration",
    "Detail": f"{pct_biz_80:.1f}% of businesses account for 80% of reviews",
    "Status": "INFO"
})

# ── Sparse-tail businesses ────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Sparse-Tail Businesses")
single_cnt = int((biz_pd["review_count"] == 1).sum())
le5_cnt = int((biz_pd["review_count"] <= 5).sum())
print(f"   Businesses with exactly 1 review: {single_cnt:,} ({single_cnt / total_businesses * 100:.1f}%)")
print(f"   Businesses with ≤ 5 reviews:      {le5_cnt:,} ({le5_cnt / total_businesses * 100:.1f}%)")
findings_68.append({
    "Check": "Single-review businesses",
    "Detail": f"{single_cnt:,} businesses ({single_cnt / total_businesses * 100:.1f}%) have exactly one review",
    "Status": "INFO"
})
findings_68.append({
    "Check": "Low-volume business tail",
    "Detail": f"{le5_cnt:,} businesses ({le5_cnt / total_businesses * 100:.1f}%) have five or fewer reviews",
    "Status": "INFO"
})

# ── Largest businesses by review count ────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Highest-Volume Businesses")
display(reviews_per_biz.orderBy(col("review_count").desc()).limit(5))

print(f"\n{'='*80}")
print("6.8.4 complete — concentration analysis finished")
print("=" * 80)

6.8.4) CONCENTRATION & PARETO ANALYSIS
    How evenly are reviews distributed across businesses?

   Total businesses: 150,346
   Total reviews:    6,990,280

────────────────────────────────────────────────────────────
📌 Gini Coefficient
   Gini coefficient: 0.6784
   Interpretation: High concentration

────────────────────────────────────────────────────────────
📌 Pareto (Top-N%) Analysis
   Top  1% businesses ( 1,503) →  19.5% of all reviews
   Top  5% businesses ( 7,517) →  43.5% of all reviews
   Top 10% businesses (15,034) →  57.9% of all reviews
   Top 20% businesses (30,069) →  73.1% of all reviews
   Top 50% businesses (75,173) →  90.8% of all reviews

   80% of all reviews come from 41,906 businesses (27.9%)

────────────────────────────────────────────────────────────
📌 Sparse-Tail Businesses
   Businesses with exactly 1 review: 0 (0.0%)
   Businesses with ≤ 5 reviews:      12,901 (8.6%)

────────────────────────────────────────────────────────────
📌 Highest-Volume Businesse

business_id,review_count
_ab50qdWOk0DdB6XOrBitw,7673
ac1AeYqs8Z4_e2X5M3if2A,7516
GXFMD0Z4jEVZBCsbPf4CTQ,6160
ytynqOUb3hjKeJfRj5Tshw,5778
oBNrLz4EDhiscSlbOl8uAw,5264



6.8.4 complete — concentration analysis finished


---
#### 🟡 DQ-REVIEW-024 — High Review Concentration Across Businesses

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-024` |
| **Description** | Review activity is materially concentrated across businesses. The revised business-level concentration analysis shows a **Gini coefficient of 0.6784** (high concentration). The top **1% of businesses (1,503)** account for **19.5%** of all reviews, and **27.9% of businesses (41,906)** account for **80%** of total review volume. At the sparse end, **12,901 businesses (8.6%)** have **5 or fewer reviews**, while the largest business (`_ab50qdWOk0DdB6XOrBitw`) has **7,673 reviews**. |
| **Detection Phase** | Phase 6.8 → **6.8.4) Concentration & Pareto Analysis** |
| **Detected In** | Cell 95: `6.8.4) Concentration & Pareto Analysis` |
| **Severity** | 🟡 **Moderate** — This is a platform-distribution characteristic, not a defect, but it can bias aggregates, evaluation sets, and models toward popular businesses. |
| **Affected Records** | Entire review table (**6,990,280** rows) and all **150,346** reviewed businesses. |
| **Root Cause** | Expected long-tail marketplace behavior: a relatively small share of highly visible businesses attracts a disproportionately large share of reviews. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Create activity tiers** for businesses | Makes reporting and segmentation more interpretable than raw counts alone. |
| 2 | **Use log-transformed review-count features** in modeling | Reduces the dominance of the right tail. |
| 3 | **Sample or stratify by activity band** when building evaluation/training datasets | Prevents popular businesses from overwhelming less-reviewed businesses. |

##### ✅ Best Logical Solution: Segment by activity tier and log-transform review-count features for modeling.


In [0]:
# 6.8.5) Business Activity Span Analysis
# ─────────────────────────────────────────────────────────────────────
# Analyze each business's review lifecycle within the review dataset:
# first/last review, active duration, dormancy before dataset cutoff,
# and normalized review intensity.

from pyspark.sql.functions import (
    col, count as _count, min as spark_min, max as spark_max,
    datediff, months_between, mean, percentile_approx,
    when, lit, to_timestamp, to_date
)

print("=" * 80)
print("6.8.5) BUSINESS ACTIVITY SPAN ANALYSIS")
print("    Temporal span, dormancy, and review intensity per business")
print("=" * 80)

df68 = (
    df
    .withColumn("review_ts", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("review_date", to_date(col("review_ts")))
)

dataset_bounds = df68.agg(
    spark_min("review_date").alias("dataset_start"),
    spark_max("review_date").alias("dataset_end")
).collect()[0]
dataset_end = dataset_bounds["dataset_end"]

print(f"\n   Dataset span: {dataset_bounds['dataset_start']} → {dataset_end}")

biz_span = (
    df68.groupBy("business_id")
        .agg(
            _count("*").alias("review_count"),
            spark_min("review_date").alias("first_review"),
            spark_max("review_date").alias("last_review")
        )
        .withColumn("active_days", datediff(col("last_review"), col("first_review")))
        .withColumn("days_before_dataset_end", datediff(lit(dataset_end), col("last_review")))
)

total_biz = biz_span.count()
span_stats = biz_span.agg(
    mean("active_days").alias("avg_span"),
    percentile_approx("active_days", [0.25, 0.50, 0.75]).alias("span_pcts"),
    spark_min("active_days").alias("min_span"),
    spark_max("active_days").alias("max_span"),
    mean("days_before_dataset_end").alias("avg_gap"),
    percentile_approx("days_before_dataset_end", 0.50).alias("median_gap")
).collect()[0]

print(f"\n   Total businesses: {total_biz:,}")
print(f"   Active span (days between first and last review):")
print(f"      Min:    {span_stats['min_span']:,}")
print(f"      P25:    {span_stats['span_pcts'][0]:,}")
print(f"      Median: {span_stats['span_pcts'][1]:,}")
print(f"      P75:    {span_stats['span_pcts'][2]:,}")
print(f"      Max:    {span_stats['max_span']:,}")
print(f"      Mean:   {span_stats['avg_span']:,.0f}")

# ── Zero-span businesses ──────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Zero-Span Businesses")
zero_span = biz_span.filter(col("active_days") == 0).count()
print(f"   Businesses whose reviews all occur on one calendar day: {zero_span:,} ({zero_span / total_biz * 100:.1f}%)")
findings_68.append({
    "Check": "Zero-span businesses",
    "Detail": f"{zero_span:,} businesses ({zero_span / total_biz * 100:.1f}%) have activity compressed to one day",
    "Status": "INFO"
})

# ── Dormancy before dataset cutoff ────────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Dormant Before Dataset End (>365 days before final review date in dataset)")
dormant_1yr = biz_span.filter(col("days_before_dataset_end") > 365).count()
print(f"   Dormant > 1 year before dataset cutoff: {dormant_1yr:,} ({dormant_1yr / total_biz * 100:.1f}%)")
print(f"   Mean gap to dataset end:   {span_stats['avg_gap']:,.0f} days")
print(f"   Median gap to dataset end: {span_stats['median_gap']:,} days")
findings_68.append({
    "Check": "Dormant businesses before cutoff",
    "Detail": f"{dormant_1yr:,} businesses ({dormant_1yr / total_biz * 100:.1f}%) have no review in the last dataset year",
    "Status": "INFO"
})

# ── Review intensity per active month ─────────────────────────────────────
print(f"\n{'─'*60}")
print("📌 Review Intensity (reviews per active month)")
biz_intensity = (
    biz_span
    .withColumn(
        "active_months",
        when(col("active_days") > 0, months_between(col("last_review"), col("first_review")) + 1.0)
        .otherwise(lit(1.0))
    )
    .withColumn("reviews_per_month", col("review_count") / col("active_months"))
)

intensity_stats = biz_intensity.agg(
    mean("reviews_per_month").alias("avg"),
    percentile_approx("reviews_per_month", [0.50, 0.95, 0.99]).alias("pcts"),
    spark_max("reviews_per_month").alias("max_val")
).collect()[0]

high_intensity = biz_intensity.filter(col("reviews_per_month") > 50).count()
print(f"   Mean reviews/month:   {intensity_stats['avg']:.2f}")
print(f"   Median reviews/month: {intensity_stats['pcts'][0]:.2f}")
print(f"   P95 reviews/month:    {intensity_stats['pcts'][1]:.2f}")
print(f"   P99 reviews/month:    {intensity_stats['pcts'][2]:.2f}")
print(f"   Max reviews/month:    {intensity_stats['max_val']:.2f}")
print(f"   Businesses above 50 reviews/month: {high_intensity:,}")
if high_intensity > 0:
    display(
        biz_intensity
        .select("business_id", "review_count", "first_review", "last_review", "active_days", "reviews_per_month")
        .orderBy(col("reviews_per_month").desc())
        .limit(5)
    )
findings_68.append({
    "Check": "High-intensity businesses",
    "Detail": f"{high_intensity:,} businesses exceed 50 reviews per active month",
    "Status": "INFO" if high_intensity > 0 else "PASS"
})

print(f"\n{'='*80}")
print("6.8.5 complete — business lifecycle analysis finished")
print("=" * 80)

6.8.5) BUSINESS ACTIVITY SPAN ANALYSIS
    Temporal span, dormancy, and review intensity per business

   Dataset span: 2005-02-16 → 2022-01-19

   Total businesses: 150,346
   Active span (days between first and last review):
      Min:    0
      P25:    1,379
      Median: 2,366
      P75:    3,429
      Max:    6,170
      Mean:   2,452

────────────────────────────────────────────────────────────
📌 Zero-Span Businesses
   Businesses whose reviews all occur on one calendar day: 3 (0.0%)

────────────────────────────────────────────────────────────
📌 Dormant Before Dataset End (>365 days before final review date in dataset)
   Dormant > 1 year before dataset cutoff: 62,613 (41.6%)
   Mean gap to dataset end:   615 days
   Median gap to dataset end: 235 days

────────────────────────────────────────────────────────────
📌 Review Intensity (reviews per active month)
   Mean reviews/month:   0.65
   Median reviews/month: 0.24
   P95 reviews/month:    2.64
   P99 reviews/month:    6.20
 

business_id,review_count,first_review,last_review,active_days,reviews_per_month
GXFMD0Z4jEVZBCsbPf4CTQ,6160,2012-08-10,2022-01-18,3448,53.91304347643434
9pwzHmhCpgJwfaIP9jWoCg,94,2021-12-07,2022-01-07,31,47.0
_ab50qdWOk0DdB6XOrBitw,7673,2006-09-18,2022-01-17,5600,41.48290896306111
ac1AeYqs8Z4_e2X5M3if2A,7516,2006-09-30,2022-01-19,5590,40.70510132781395
3IUg1tfMbxq6a10WSiK9wQ,43,2022-01-16,2022-01-19,3,39.20588247978373



6.8.5 complete — business lifecycle analysis finished


---
#### 🟡 DQ-REVIEW-025 — Dormant and High-Intensity Business Lifecycle Patterns

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-025` |
| **Description** | The revised lifecycle analysis uses the **dataset end date** rather than the current calendar date, which makes the signal much more meaningful. Only **3 businesses** have a true **zero-span** lifecycle (all reviews on one day), but **62,613 businesses (41.6%)** were already **dormant for more than 1 year before the dataset cutoff** on **2022-01-19**. On the opposite end, only **1 business** exceeded **50 reviews per active month**: `GXFMD0Z4jEVZBCsbPf4CTQ` with **6,160 reviews** over **3,448 active days** (**53.9 reviews/month**). |
| **Detection Phase** | Phase 6.8 → **6.8.5) Business Activity Span Analysis** |
| **Detected In** | Cell 97: `6.8.5) Business Activity Span Analysis` |
| **Severity** | 🟡 **Moderate** — Not erroneous, but highly relevant for lifecycle modeling, churn-style analysis, and any attempt to compare businesses without accounting for review recency and active span. |
| **Affected Records** | 3 zero-span businesses, 62,613 dormant-before-cutoff businesses, 1 very high-intensity business. |
| **Root Cause** | Natural business lifecycle heterogeneity: some businesses peaked early and went quiet long before snapshot end, while a tiny number sustained extremely dense review activity. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Flag lifecycle states** such as `is_zero_span`, `is_dormant_pre_cutoff`, and `is_high_intensity` | Makes lifecycle segmentation explicit and reusable downstream. |
| 2 | **Use dataset-relative recency** instead of today-based recency in historical EDA | Prevents stale snapshots from making every business look inactive. |
| 3 | **Create lifecycle segments** for downstream modeling/reporting | Separates flash, dormant, steady, and hyper-active businesses. |

##### ✅ Best Logical Solution: Use dataset-relative lifecycle flags and segment businesses before downstream modeling.


In [0]:
# 6.8.6) Business Rule & Accuracy — Consolidated Summary
# ─────────────────────────────────────────────────────────────────────

import pandas as pd

print("=" * 80)
print("6.8) BUSINESS RULE & ACCURACY — CONSOLIDATED SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame(findings_68)
display(summary_df)

status_counts = summary_df["Status"].value_counts().to_dict() if not summary_df.empty else {}
pass_count = status_counts.get("PASS", 0)
info_count = status_counts.get("INFO", 0)
warning_count = status_counts.get("WARNING", 0)
finding_count = status_counts.get("FINDING", 0)

print(f"\n   Total checks: {len(findings_68)}")
print(f"   ✅ PASS:     {pass_count}")
print(f"   ℹ️  INFO:     {info_count}")
print(f"   ⚠️  WARNING:  {warning_count}")
print(f"   🚨 FINDING:  {finding_count}")

if finding_count == 0 and warning_count == 0:
    print("\n✅ No hard business-rule failures were introduced in 6.8.")
    print("   The section mainly surfaces interpretable review-behavior anomalies and distributional edge cases for downstream review.")
elif finding_count == 0:
    print(f"\n⚠️  No critical failures, but {warning_count} warning(s) require attention.")
else:
    print(f"\n🚨 {finding_count} critical finding(s) and {warning_count} warning(s) detected — review the section above.")

print("\n" + "=" * 80)
print("BUSINESS RULE & ACCURACY CHECKS COMPLETE")
print("=" * 80)

6.8) BUSINESS RULE & ACCURACY — CONSOLIDATED SUMMARY


Check,Detail,Status
High daily reviewer volume,732 user-days across 641 users with >20 reviews/day,INFO
Repeated review text across businesses,145 user-text groups affecting 557 rows,INFO
Short reviews with high votes,7 reviews with <15 chars and at least 10 total votes,INFO
Highly engaged reviews,"3,070 reviews with more than 100 total votes",INFO
text_length Z-score outliers,"138,640 reviews above μ+3σ (2,149.54)",INFO
text_length extreme IQR outliers,"129,953 reviews above outer fence (2,193.00)",INFO
total_votes Z-score outliers,"72,559 reviews above μ+3σ (20.60)",INFO
total_votes extreme IQR outliers,"311,177 reviews above outer fence (8.00)",INFO
Polarized rating share,61.5% of reviews are 1★ or 5★,INFO
Non-boundary YoY declines,0 year(s) with >50% decline excluding first/last year,PASS



   Total checks: 20
   ✅ PASS:     1
   ℹ️  INFO:     19
   ⚠️  WARNING:  0
   🚨 FINDING:  0

✅ No hard business-rule failures were introduced in 6.8.
   The section mainly surfaces interpretable review-behavior anomalies and distributional edge cases for downstream review.

BUSINESS RULE & ACCURACY CHECKS COMPLETE


### 6.9) TIMELINESS CHECKS
*Review-table-adapted assessment of dataset currency, ingestion observability, timestamp integrity, and historical-snapshot boundary effects.*

For `yelp_poc_uc.bronze_final.yelp_review`, timeliness should be interpreted as a **historical snapshot quality** question, not as a real-time event-stream SLA question. This section therefore focuses on:

- **Snapshot Freshness & Coverage**
  - Earliest / latest review timestamps
  - Calendar span, recent-year concentration, and whether the dataset is stale relative to today

- **Ingestion Observability**
  - Whether the table contains row-level load / ingestion metadata
  - Whether Delta table history can provide only a batch-level proxy

- **Snapshot Lag & Boundary Period Risk**
  - Whether the latest review date is far older than the latest table commit date
  - Whether the first / last calendar years are incomplete and can bias trend analysis

- **Temporal Consistency of Raw Review Timestamps**
  - Can the raw `date` string be parsed cleanly?
  - Do parsed components agree with the original timestamp text?
  - Are any records future-dated or malformed?

- **Temporal Gaps**
  - Kept in **6.4.6** as a completeness issue, with a cross-reference here so timeliness interpretation stays consistent.

This makes 6.9 specific to the review table’s actual structure: a raw `date` string plus Delta table metadata, rather than assumed pre-built load timestamp columns or downstream derived date attributes.

In [0]:
# 6.9.1) Snapshot Freshness & Coverage
# ─────────────────────────────────────────────────────────────────────
# Review-table-specific timeliness profile.
# Builds a self-contained temporal base from the raw `date` string so this
# section does not depend on any temporary remodeled columns from earlier cells.

from pyspark.sql.functions import (
    col, to_timestamp, to_date, year, month, date_format,
    min as spark_min, max as spark_max, count as _count,
    countDistinct, sum as _sum, when, datediff, current_date
)
import pandas as pd

print("=" * 80)
print("6.9.1) SNAPSHOT FRESHNESS & COVERAGE")
print("    Currency, temporal span, boundary-year completeness, and recency mix")
print("=" * 80)

TABLE_NAME = globals().get("TABLE_NAME", "yelp_poc_uc.bronze_final.yelp_review")
expected_cols = {"review_id", "user_id", "business_id", "stars", "useful", "funny", "cool", "text", "date"}
if "df" not in globals() or not expected_cols.issubset(set(df.columns)):
    df = spark.table(TABLE_NAME)
    print(f"   Reloaded raw review table from: {TABLE_NAME}")
else:
    print(f"   Using in-memory DataFrame for: {TABLE_NAME}")

# Initialize cumulative findings list for all 6.9 subcells
findings_69 = []

review_timeliness_df = (
    df.select("review_id", "user_id", "business_id", "date")
      .withColumn("review_timestamp", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
      .withColumn("review_date", to_date(col("review_timestamp")))
      .withColumn("review_year", year(col("review_timestamp")))
      .withColumn("review_month", month(col("review_timestamp")))
      .withColumn("review_year_month", date_format(col("review_timestamp"), "yyyy-MM"))
)

profile_69 = review_timeliness_df.agg(
    _count("*").alias("total_rows"),
    _sum(when(col("review_timestamp").isNull() & col("date").isNotNull(), 1).otherwise(0)).alias("parse_failures"),
    spark_min("review_timestamp").alias("earliest_ts"),
    spark_max("review_timestamp").alias("latest_ts"),
    spark_min("review_date").alias("earliest_date"),
    spark_max("review_date").alias("latest_date"),
    countDistinct("review_year").alias("distinct_years"),
    datediff(spark_max(col("review_date")), spark_min(col("review_date"))).alias("span_days"),
    datediff(current_date(), spark_max(col("review_date"))).alias("days_stale")
).collect()[0]

total_rows_69   = int(profile_69["total_rows"])
parse_failures  = int(profile_69["parse_failures"])
earliest_ts_69  = profile_69["earliest_ts"]
latest_ts_69    = profile_69["latest_ts"]
earliest_dt_69  = profile_69["earliest_date"]
latest_dt_69    = profile_69["latest_date"]
distinct_years_69 = int(profile_69["distinct_years"])
span_days_69    = int(profile_69["span_days"])
days_stale_69   = int(profile_69["days_stale"])
span_years_69   = round(span_days_69 / 365.25, 1)

print(f"\n{'─'*60}")
print("📌 Raw timestamp parseability")
print(f"   Rows:                {total_rows_69:,}")
print(f"   Failed timestamp parses: {parse_failures:,}")
findings_69.append({
    "Check": "Review timestamp parseability",
    "Detail": f"{parse_failures:,} failed parses from raw `date` strings",
    "Status": "PASS" if parse_failures == 0 else "FINDING"
})

print(f"\n{'─'*60}")
print("📌 Temporal range")
print(f"   Earliest timestamp:  {earliest_ts_69}")
print(f"   Latest timestamp:    {latest_ts_69}")
print(f"   Earliest date:       {earliest_dt_69}")
print(f"   Latest date:         {latest_dt_69}")
print(f"   Distinct years:      {distinct_years_69}")
print(f"   Span:                {span_days_69:,} days (~{span_years_69} years)")
findings_69.append({
    "Check": "Snapshot coverage span",
    "Detail": f"{earliest_dt_69} to {latest_dt_69} ({span_days_69:,} days across {distinct_years_69} years)",
    "Status": "PASS" if span_days_69 >= 365 else "INFO"
})

print(f"\n{'─'*60}")
print("📌 Snapshot freshness relative to today")
print(f"   Days since latest review: {days_stale_69:,} days (~{days_stale_69 / 365.25:.1f} years)")
if days_stale_69 <= 30:
    freshness_status_69 = "PASS"
    print("   ✅ Fresh enough for near-current analysis")
elif days_stale_69 <= 365:
    freshness_status_69 = "INFO"
    print("   ⚠️  Moderately stale — current-state interpretation requires caution")
else:
    freshness_status_69 = "INFO"
    print("   ⚠️  Historical snapshot — not suitable for present-day operational conclusions")
findings_69.append({
    "Check": "Snapshot freshness",
    "Detail": f"Latest review date {latest_dt_69}; stale by {days_stale_69:,} days",
    "Status": freshness_status_69
})

review_yearly_69 = (
    review_timeliness_df
    .groupBy("review_year")
    .agg(_count("*").alias("review_count"))
    .orderBy("review_year")
)
yearly_pd_69 = review_yearly_69.toPandas()
yearly_pd_69["pct_of_total"] = (yearly_pd_69["review_count"] / yearly_pd_69["review_count"].sum() * 100).round(2)

first_year_69 = int(yearly_pd_69["review_year"].min())
latest_year_69 = int(yearly_pd_69["review_year"].max())
recent_3y_rows_69 = int(yearly_pd_69[yearly_pd_69["review_year"] >= latest_year_69 - 2]["review_count"].sum())
recent_3y_pct_69 = recent_3y_rows_69 / total_rows_69 * 100

boundary_years_69 = (
    review_timeliness_df
    .filter(col("review_year").isin(first_year_69, latest_year_69))
    .groupBy("review_year")
    .agg(
        countDistinct("review_month").alias("months_present"),
        _count("*").alias("review_count"),
        spark_min("review_date").alias("first_review_in_year"),
        spark_max("review_date").alias("last_review_in_year")
    )
    .orderBy("review_year")
)
boundary_rows_69 = {int(r["review_year"]): r.asDict() for r in boundary_years_69.collect()}
first_year_months_69 = int(boundary_rows_69[first_year_69]["months_present"])
latest_year_months_69 = int(boundary_rows_69[latest_year_69]["months_present"])

print(f"\n{'─'*60}")
print("📌 Recency mix and boundary-year completeness")
print(f"   Last 3 calendar years ({latest_year_69 - 2}–{latest_year_69}): {recent_3y_rows_69:,} rows ({recent_3y_pct_69:.1f}% of total)")
print(f"   First year {first_year_69}:  {first_year_months_69}/12 months present")
print(f"   Latest year {latest_year_69}: {latest_year_months_69}/12 months present")

findings_69.append({
    "Check": "Recent-year concentration",
    "Detail": f"{recent_3y_rows_69:,} rows ({recent_3y_pct_69:.1f}%) fall in {latest_year_69 - 2}–{latest_year_69}",
    "Status": "INFO"
})
findings_69.append({
    "Check": "Boundary-year completeness",
    "Detail": f"{first_year_69} has {first_year_months_69}/12 months; {latest_year_69} has {latest_year_months_69}/12 months",
    "Status": "INFO" if (first_year_months_69 < 12 or latest_year_months_69 < 12) else "PASS"
})

review_monthly_tail_69 = (
    review_timeliness_df
    .groupBy("review_year_month")
    .agg(_count("*").alias("review_count"))
    .orderBy(col("review_year_month").desc())
    .limit(12)
)

print(f"\n{'─'*60}")
print("📋 Recent yearly distribution")
display(review_yearly_69.orderBy(col("review_year").desc()).limit(10))

print(f"\n📋 Latest 12 year-month buckets in the snapshot")
display(review_monthly_tail_69)

timeliness_profile_69 = {
    "table_name": TABLE_NAME,
    "total_rows": total_rows_69,
    "parse_failures": parse_failures,
    "earliest_ts": earliest_ts_69,
    "latest_ts": latest_ts_69,
    "earliest_date": earliest_dt_69,
    "latest_date": latest_dt_69,
    "distinct_years": distinct_years_69,
    "span_days": span_days_69,
    "span_years": span_years_69,
    "days_stale": days_stale_69,
    "first_year": first_year_69,
    "latest_year": latest_year_69,
    "first_year_months_present": first_year_months_69,
    "latest_year_months_present": latest_year_months_69,
    "recent_3y_rows": recent_3y_rows_69,
    "recent_3y_pct": recent_3y_pct_69,
}

print("=" * 80)

6.9.1) SNAPSHOT FRESHNESS & COVERAGE
    Currency, temporal span, boundary-year completeness, and recency mix
   Using in-memory DataFrame for: yelp_poc_uc.bronze_final.yelp_review

────────────────────────────────────────────────────────────
📌 Raw timestamp parseability
   Rows:                6,990,280
   Failed timestamp parses: 0

────────────────────────────────────────────────────────────
📌 Temporal range
   Earliest timestamp:  2005-02-16 03:23:22
   Latest timestamp:    2022-01-19 19:48:45
   Earliest date:       2005-02-16
   Latest date:         2022-01-19
   Distinct years:      18
   Span:                6,181 days (~16.9 years)

────────────────────────────────────────────────────────────
📌 Snapshot freshness relative to today
   Days since latest review: 1,732 days (~4.7 years)
   ⚠️  Historical snapshot — not suitable for present-day operational conclusions

────────────────────────────────────────────────────────────
📌 Recency mix and boundary-year completeness
   Last 

review_year,review_count
2022,31665
2021,618189
2020,554557
2019,907284
2018,906362
2017,820048
2016,758882
2015,688415
2014,522275
2013,383950



📋 Latest 12 year-month buckets in the snapshot


review_year_month,review_count
2022-01,31665
2021-12,48479
2021-11,45224
2021-10,49304
2021-09,46397
2021-08,56439
2021-07,60455
2021-06,57846
2021-05,57762
2021-04,53781


---
#### 🟡 DQ-REVIEW-026 — Historical Snapshot Staleness and Partial Terminal Year

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-026` |
| **Description** | The review dataset is a **historical snapshot**, not a current feed. The latest review date is **2022-01-19**, which is **1,732 days (~4.7 years)** older than today (2026-10-17). The snapshot spans **2005-02-16 → 2022-01-19** across **6,990,280 rows**, **18 calendar years**, and **6,181 days (~16.9 years)**, but the boundary years are incomplete: **2005** begins in **February** and **2022** contains **January only**. The latest three calendar years (**2020–2022**) contribute **1,204,411 rows (17.2%)**, so trend analyses near the right edge of the series are shaped by both COVID-era behavior and the January-2022 cutoff. |
| **Detection Phase** | Phase 6.9 — Timeliness Checks → **6.9.1) Snapshot Freshness & Coverage** |
| **Detected In** | Cell 101: `6.9.1) Snapshot Freshness & Coverage` |
| **Severity** | 🟡 **INFO** — critical temporal context, not a row-level defect |
| **Affected Scope** | Entire review table (**6,990,280** rows) |
| **Impact** | Any dashboard, model, or narrative framed as “current Yelp behavior” would be temporally misleading. Trend lines near 2020–2022 mix real-world pandemic effects with an extraction cutoff, while year-over-year analysis that includes 2022 will understate annual activity because that year is only one month long in the data. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Carry an explicit `data_as_of_date` / snapshot date in downstream layers** | Makes the dataset’s currency machine-readable and prevents consumers from mistaking a historical snapshot for a live feed. |
| 2 | **Filter temporal trend analysis to complete years (`2006–2021`)** | Removes the partial boundary years while preserving the clean multi-year core of the series for seasonality and YoY comparisons. |
| 3 | **Add a temporal caveat to every downstream artifact** | A clear note such as “Review data available through 2022-01-19 only” is the fastest safeguard against misinterpretation. |
| 4 | **Refresh the source snapshot if the goal is current-state analysis** | The only true fix for present-day relevance is newer source data. |

##### ✅ Best Logical Solution

Use **Solutions 1 + 2 + 3 immediately**: store an explicit snapshot date, restrict time-series comparisons to complete years, and label downstream outputs with the data cutoff. If the business question requires 2023–2026 behavior, then **Solution 4** becomes mandatory.

---

In [0]:
# 6.9.2) Ingestion Observability & Table History
# ─────────────────────────────────────────────────────────────────────
# For this review table, timeliness observability has two levels:
#   1) row-level metadata columns inside the dataset
#   2) table-level Delta commit history outside the row schema
# We inspect both, because this is a historical snapshot rather than a live stream.

from pyspark.sql.functions import col, min as spark_min, max as spark_max, count as _count

print("=" * 80)
print("6.9.2) INGESTION OBSERVABILITY & TABLE HISTORY")
print("    Row-level ingestion metadata vs. Delta table-level history")
print("=" * 80)

load_ts_patterns = [
    "load_timestamp", "load_date", "load_ts", "etl_timestamp",
    "ingestion_timestamp", "ingested_at", "created_at",
    "_loaded_at", "_ingestion_time", "updated_at", "modified_at",
    "last_updated", "_metadata_timestamp",
]

cols_lower_69 = {c.lower(): c for c in df.columns}
row_level_ts_cols_69 = [cols_lower_69[p] for p in load_ts_patterns if p in cols_lower_69]

print(f"\n{'─'*60}")
print("📌 Row-level ingestion metadata search")
print(f"   Candidate patterns checked: {len(load_ts_patterns)}")
if row_level_ts_cols_69:
    print(f"   Found row-level metadata columns: {row_level_ts_cols_69}")
    for c in row_level_ts_cols_69:
        stats = df.agg(
            spark_min(col(c)).alias("min_val"),
            spark_max(col(c)).alias("max_val")
        ).collect()[0]
        print(f"   {c}: {stats['min_val']} → {stats['max_val']}")
    findings_69.append({
        "Check": "Row-level ingestion metadata",
        "Detail": f"Present columns: {row_level_ts_cols_69}",
        "Status": "PASS"
    })
else:
    print("   ℹ️  No row-level load / ingestion timestamp columns exist in the review table.")
    print("   This means true per-record source-to-ingestion latency cannot be measured from the row schema.")
    findings_69.append({
        "Check": "Row-level ingestion metadata",
        "Detail": "No load / ingestion timestamp columns present in row schema",
        "Status": "INFO"
    })

table_history_69 = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME}")
history_profile_69 = table_history_69.agg(
    _count("*").alias("history_events"),
    spark_min("timestamp").alias("earliest_commit_ts"),
    spark_max("timestamp").alias("latest_commit_ts")
).collect()[0]

history_events_69 = int(history_profile_69["history_events"])
earliest_commit_ts_69 = history_profile_69["earliest_commit_ts"]
latest_commit_ts_69 = history_profile_69["latest_commit_ts"]
operation_counts_69 = table_history_69.groupBy("operation").count().orderBy(col("count").desc(), col("operation"))

print(f"\n{'─'*60}")
print("📌 Delta table history")
print(f"   History events:       {history_events_69}")
print(f"   Earliest commit:      {earliest_commit_ts_69}")
print(f"   Latest commit:        {latest_commit_ts_69}")
print("   Recent history events:")
display(
    table_history_69
    .select("version", "timestamp", "operation", "userName", "clusterId")
    .orderBy(col("timestamp").desc())
    .limit(10)
)

print("📋 Operation counts")
display(operation_counts_69)

findings_69.append({
    "Check": "Delta table history availability",
    "Detail": f"{history_events_69} commit-history events available ({earliest_commit_ts_69} → {latest_commit_ts_69})",
    "Status": "PASS" if history_events_69 > 0 else "INFO"
})

timeliness_observability_69 = {
    "row_level_ts_cols": row_level_ts_cols_69,
    "history_events": history_events_69,
    "earliest_commit_ts": earliest_commit_ts_69,
    "latest_commit_ts": latest_commit_ts_69,
}

print("=" * 80)

6.9.2) INGESTION OBSERVABILITY & TABLE HISTORY
    Row-level ingestion metadata vs. Delta table-level history

────────────────────────────────────────────────────────────
📌 Row-level ingestion metadata search
   Candidate patterns checked: 13
   ℹ️  No row-level load / ingestion timestamp columns exist in the review table.
   This means true per-record source-to-ingestion latency cannot be measured from the row schema.

────────────────────────────────────────────────────────────
📌 Delta table history
   History events:       3
   Earliest commit:      2026-09-22 11:46:00
   Latest commit:        2026-10-16 02:01:37
   Recent history events:


version,timestamp,operation,userName,clusterId
2,2026-10-16T02:01:37.000Z,COMPUTE STATS,890f026c-4c18-48be-bebf-5dfc905b2d40,1016-015801-s7m14xg-v2n
1,2026-10-16T02:01:10.000Z,SET TBLPROPERTIES,890f026c-4c18-48be-bebf-5dfc905b2d40,1016-015801-s7m14xg-v2n
0,2026-09-22T11:46:00.000Z,CREATE OR REPLACE TABLE AS SELECT,ibrahim.mohamed@gbrands.com,0922-083211-lqvke12a-v2n


📋 Operation counts


operation,count
COMPUTE STATS,1
CREATE OR REPLACE TABLE AS SELECT,1
SET TBLPROPERTIES,1


In [0]:
# 6.9.3) Snapshot Lag & Boundary Period Risk
# ─────────────────────────────────────────────────────────────────────
# Real source-to-target latency is not measurable without row-level ingestion
# timestamps. For a historical review snapshot, the more meaningful question is:
# how far apart are (a) the latest event in the data and (b) the latest table
# commit, and are the boundary years partial in ways that distort trends?

from pyspark.sql.functions import col, count as _count, countDistinct, min as spark_min, max as spark_max

print("=" * 80)
print("6.9.3) SNAPSHOT LAG & BOUNDARY PERIOD RISK")
print("    Historical snapshot interpretation instead of real-time SLA latency")
print("=" * 80)

latest_review_date_69 = timeliness_profile_69["latest_date"]
latest_commit_ts_69 = timeliness_observability_69["latest_commit_ts"]
latest_commit_date_69 = latest_commit_ts_69.date() if latest_commit_ts_69 is not None else None
publish_lag_days_69 = (
    (latest_commit_date_69 - latest_review_date_69).days
    if latest_review_date_69 is not None and latest_commit_date_69 is not None
    else None
)

print(f"\n{'─'*60}")
print("📌 Snapshot lag")
print(f"   Latest review date:   {latest_review_date_69}")
print(f"   Latest table commit:  {latest_commit_ts_69}")
if publish_lag_days_69 is not None:
    print(f"   Gap between latest review and latest commit: {publish_lag_days_69:,} days")
    if publish_lag_days_69 > 30:
        print("   ℹ️  This behaves like a historical backfill / published snapshot, not a near-real-time pipeline.")
    findings_69.append({
        "Check": "Snapshot publication lag",
        "Detail": f"{publish_lag_days_69:,} days between latest review date and latest Delta commit",
        "Status": "INFO" if publish_lag_days_69 > 30 else "PASS"
    })
else:
    findings_69.append({
        "Check": "Snapshot publication lag",
        "Detail": "Could not compare latest review date to latest table commit",
        "Status": "INFO"
    })

first_year_69 = timeliness_profile_69["first_year"]
latest_year_69 = timeliness_profile_69["latest_year"]

boundary_periods_69 = (
    review_timeliness_df
    .filter(col("review_year").isin(first_year_69, latest_year_69))
    .groupBy("review_year")
    .agg(
        countDistinct("review_month").alias("months_present"),
        _count("*").alias("review_count"),
        spark_min("review_date").alias("first_review"),
        spark_max("review_date").alias("last_review")
    )
    .orderBy("review_year")
)

print(f"\n{'─'*60}")
print("📌 Boundary-year completeness")
display(boundary_periods_69)

boundary_period_rows_69 = {int(r["review_year"]): r.asDict() for r in boundary_periods_69.collect()}
first_year_months_69 = int(boundary_period_rows_69[first_year_69]["months_present"])
latest_year_months_69 = int(boundary_period_rows_69[latest_year_69]["months_present"])

print(f"   {first_year_69}: {first_year_months_69}/12 months present")
print(f"   {latest_year_69}: {latest_year_months_69}/12 months present")
findings_69.append({
    "Check": "Boundary-year trend risk",
    "Detail": f"{first_year_69} has {first_year_months_69}/12 months; {latest_year_69} has {latest_year_months_69}/12 months",
    "Status": "INFO" if (first_year_months_69 < 12 or latest_year_months_69 < 12) else "PASS"
})

print(f"\n{'─'*60}")
print("📋 Latest 12 months in the available snapshot")
display(review_monthly_tail_69)

print("=" * 80)

6.9.3) SNAPSHOT LAG & BOUNDARY PERIOD RISK
    Historical snapshot interpretation instead of real-time SLA latency

────────────────────────────────────────────────────────────
📌 Snapshot lag
   Latest review date:   2022-01-19
   Latest table commit:  2026-10-16 02:01:37
   Gap between latest review and latest commit: 1,731 days
   ℹ️  This behaves like a historical backfill / published snapshot, not a near-real-time pipeline.

────────────────────────────────────────────────────────────
📌 Boundary-year completeness


review_year,months_present,review_count,first_review,last_review
2005,11,854,2005-02-16,2005-12-31
2022,1,31665,2022-01-01,2022-01-19


   2005: 11/12 months present
   2022: 1/12 months present

────────────────────────────────────────────────────────────
📋 Latest 12 months in the available snapshot


review_year_month,review_count
2022-01,31665
2021-12,48479
2021-11,45224
2021-10,49304
2021-09,46397
2021-08,56439
2021-07,60455
2021-06,57846
2021-05,57762
2021-04,53781


---
#### 🟡 DQ-REVIEW-027 — Table-Level History Exists, but Row-Level Ingestion Metadata Is Missing

| Attribute | Detail |
| --- | --- |
| **Error ID** | `DQ-REVIEW-027` |
| **Description** | The review table contains **no row-level load / ingestion timestamp columns** across 13 checked naming patterns (`load_timestamp`, `load_date`, `load_ts`, `etl_timestamp`, `ingestion_timestamp`, `ingested_at`, `created_at`, `_loaded_at`, `_ingestion_time`, `updated_at`, `modified_at`, `last_updated`, `_metadata_timestamp`). However, the Delta table itself *does* expose **table-level history**: **3 commit events** are present in `DESCRIBE HISTORY`, with table creation on **2026-09-22** and the latest commit on **2026-10-16**. This means batch-level observability exists, but **true per-row ingestion timing and source-to-target latency do not**. The gap between the **latest review date (2022-01-19)** and the **latest Delta commit (2026-10-16)** is **1,731 days**, confirming that this table behaves as a recently materialized historical snapshot rather than a near-real-time feed. |
| **Detection Phase** | Phase 6.9 — Timeliness Checks → **6.9.2) Ingestion Observability & Table History** and **6.9.3) Snapshot Lag & Boundary Period Risk** |
| **Detected In** | Cell 103: `6.9.2) Ingestion Observability & Table History` and Cell 104: `6.9.3) Snapshot Lag & Boundary Period Risk` |
| **Severity** | 🟡 **INFO** — observability / governance gap, not a row-level data defect |
| **Affected Scope** | Entire review table (**6,990,280** rows) |
| **Impact** | **(1) No per-record latency measurement:** event-time-to-ingestion-time cannot be computed. **(2) No row-level audit trail:** individual records cannot be tied to a precise ingestion moment. **(3) Snapshot ambiguity:** without an explicit dataset-as-of field, downstream users may confuse a historical snapshot with a live pipeline. **(4) Limited SLA governance:** table history helps at the batch level, but it cannot replace row-level ingestion observability. |

##### Suggested Solutions

| # | Solution | Justification |
| --- | --- | --- |
| 1 | **Add `_ingestion_timestamp = current_timestamp()` during the next downstream transformation** | Standard row-level observability pattern for measuring event-to-ingestion delay and supporting auditability. |
| 2 | **Add an explicit `data_as_of_date` / snapshot release field** | Distinguishes historical-snapshot freshness from pipeline-latency freshness and prevents semantic confusion downstream. |
| 3 | **Retain and monitor `DESCRIBE HISTORY` as a batch-level proxy** | Useful immediately with no schema change; it provides commit timing and operation history even though it is not row-granular. |
| 4 | **Capture pipeline run identifiers and batch timestamps in downstream layers** | Complements row-level ingestion metadata with operational lineage and easier root-cause analysis. |

##### ✅ Best Logical Solution

Use **Solutions 1 + 2** as the primary fix: add a row-level ingestion timestamp for observability and a dataset-as-of field for semantics. Keep **Solution 3** as an immediate interim control, and add **Solution 4** if this PoC evolves into a production pipeline.

---

In [0]:
# 6.9.4) Temporal Consistency of Raw Review Timestamps
# ─────────────────────────────────────────────────────────────────────
# The review table stores a single raw `date` string rather than a full set of
# derived time attributes. So the correct consistency check is to validate the
# raw timestamp text itself and verify that parsed components round-trip cleanly.

from pyspark.sql.functions import (
    col, current_timestamp, when, sum as _sum, length, substring,
    date_format, year, month, dayofmonth, hour, minute, second
)

print("=" * 80)
print("6.9.4) TEMPORAL CONSISTENCY OF RAW REVIEW TIMESTAMPS")
print("    Validates raw `date` strings, parsed timestamps, and component agreement")
print("=" * 80)

total_rows_69 = timeliness_profile_69["total_rows"]

consistency_69 = review_timeliness_df.agg(
    _sum(when(length(col("date")) != 19, 1).otherwise(0)).alias("bad_length"),
    _sum(when(col("review_timestamp").isNull() & col("date").isNotNull(), 1).otherwise(0)).alias("parse_failures"),
    _sum(when(col("review_timestamp") > current_timestamp(), 1).otherwise(0)).alias("future_ts"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (date_format(col("review_timestamp"), "yyyy-MM-dd HH:mm:ss") != col("date")),
        1
    ).otherwise(0)).alias("roundtrip_mismatch"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (substring(col("date"), 1, 4).cast("int") != year(col("review_timestamp"))),
        1
    ).otherwise(0)).alias("year_mismatch"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (substring(col("date"), 6, 2).cast("int") != month(col("review_timestamp"))),
        1
    ).otherwise(0)).alias("month_mismatch"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (substring(col("date"), 9, 2).cast("int") != dayofmonth(col("review_timestamp"))),
        1
    ).otherwise(0)).alias("day_mismatch"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (substring(col("date"), 12, 2).cast("int") != hour(col("review_timestamp"))),
        1
    ).otherwise(0)).alias("hour_mismatch"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (substring(col("date"), 15, 2).cast("int") != minute(col("review_timestamp"))),
        1
    ).otherwise(0)).alias("minute_mismatch"),
    _sum(when(
        col("review_timestamp").isNotNull() &
        (substring(col("date"), 18, 2).cast("int") != second(col("review_timestamp"))),
        1
    ).otherwise(0)).alias("second_mismatch")
).collect()[0]

checks_69 = [
    ("Raw timestamp string length != 19", consistency_69["bad_length"]),
    ("Timestamp parse failures", consistency_69["parse_failures"]),
    ("Future-dated reviews", consistency_69["future_ts"]),
    ("Parsed timestamp does not round-trip to original string", consistency_69["roundtrip_mismatch"]),
    ("Year substring != parsed year", consistency_69["year_mismatch"]),
    ("Month substring != parsed month", consistency_69["month_mismatch"]),
    ("Day substring != parsed day", consistency_69["day_mismatch"]),
    ("Hour substring != parsed hour", consistency_69["hour_mismatch"]),
    ("Minute substring != parsed minute", consistency_69["minute_mismatch"]),
    ("Second substring != parsed second", consistency_69["second_mismatch"]),
]

for check_name, violations in checks_69:
    violations = int(violations)
    pct = violations / total_rows_69 * 100 if total_rows_69 > 0 else 0
    status = "PASS" if violations == 0 else "FINDING"
    icon = "✅" if violations == 0 else "⚠️ "
    print(f"   {icon} {check_name}: {violations:,} ({pct:.4f}%) — {status}")
    findings_69.append({
        "Check": check_name,
        "Detail": f"{violations:,} violations",
        "Status": status
    })

total_violations_69 = sum(int(v) for _, v in checks_69)
print(f"\n   Total violations across {len(checks_69)} checks: {total_violations_69:,}")
print("=" * 80)

6.9.4) TEMPORAL CONSISTENCY OF RAW REVIEW TIMESTAMPS
    Validates raw `date` strings, parsed timestamps, and component agreement
   ✅ Raw timestamp string length != 19: 0 (0.0000%) — PASS
   ✅ Timestamp parse failures: 0 (0.0000%) — PASS
   ✅ Future-dated reviews: 0 (0.0000%) — PASS
   ✅ Parsed timestamp does not round-trip to original string: 0 (0.0000%) — PASS
   ✅ Year substring != parsed year: 0 (0.0000%) — PASS
   ✅ Month substring != parsed month: 0 (0.0000%) — PASS
   ✅ Day substring != parsed day: 0 (0.0000%) — PASS
   ✅ Hour substring != parsed hour: 0 (0.0000%) — PASS
   ✅ Minute substring != parsed minute: 0 (0.0000%) — PASS
   ✅ Second substring != parsed second: 0 (0.0000%) — PASS

   Total violations across 10 checks: 0


In [0]:
# 6.9.5) Temporal Gaps (cross-reference)
# ─────────────────────────────────────────────────────────────────────
# Temporal gaps are intentionally owned by section 6.4 (completeness).
# Here we only restate the conclusion so the timeliness narrative remains
# aligned with the earlier gap analysis.

print("=" * 80)
print("6.9.5) TEMPORAL GAPS (cross-reference)")
print("=" * 80)

first_year_69 = timeliness_profile_69["first_year"]
latest_year_69 = timeliness_profile_69["latest_year"]
first_year_missing_69 = 12 - timeliness_profile_69["first_year_months_present"]
latest_year_missing_69 = 12 - timeliness_profile_69["latest_year_months_present"]
missing_total_69 = first_year_missing_69 + latest_year_missing_69
interior_start_69 = first_year_69 + 1
interior_end_69 = latest_year_69 - 1

print(f"\n   ℹ️  Temporal gap analysis was already completed in:")
print(f"       → 6.4.6) Temporal Gap Analysis")
print(f"       → DQ-REVIEW-012 — Temporal Gaps: Incomplete Boundary Years")
print(f"\n   Key findings carried forward into 6.9:")
print(f"       • {missing_total_69} missing year-months across the {first_year_69}–{latest_year_69} range")
print(f"       • {first_year_69}: missing {first_year_missing_69} month(s) at the left boundary")
print(f"       • {latest_year_69}: missing {latest_year_missing_69} month(s) at the right boundary")
print(f"       • All interior years ({interior_start_69}–{interior_end_69}) are fully represented")
print(f"       • Interpreted as partial boundary years, not random temporal data loss")
print(f"\n   ✅ No duplicate gap analysis needed here — use 6.4.6 as the primary completeness reference.")

findings_69.append({
    "Check": "Temporal Gaps",
    "Detail": f"Already analyzed in 6.4.6 (DQ-REVIEW-012) — {missing_total_69} missing year-months at boundary years",
    "Status": "CROSS-REF"
})

print("=" * 80)

6.9.5) TEMPORAL GAPS (cross-reference)

   ℹ️  Temporal gap analysis was already completed in:
       → 6.4.6) Temporal Gap Analysis
       → DQ-REVIEW-012 — Temporal Gaps: Incomplete Boundary Years

   Key findings carried forward into 6.9:
       • 12 missing year-months across the 2005–2022 range
       • 2005: missing 1 month(s) at the left boundary
       • 2022: missing 11 month(s) at the right boundary
       • All interior years (2006–2021) are fully represented
       • Interpreted as partial boundary years, not random temporal data loss

   ✅ No duplicate gap analysis needed here — use 6.4.6 as the primary completeness reference.


In [0]:
# 6.9.6) Timeliness Checks — Consolidated Summary
# ─────────────────────────────────────────────────────────────────────

import pandas as pd

print("=" * 80)
print("6.9) TIMELINESS CHECKS — CONSOLIDATED SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame(findings_69)
display(summary_df)

finding_count = sum(1 for f in findings_69 if f["Status"] == "FINDING")
info_count    = sum(1 for f in findings_69 if f["Status"] == "INFO")
pass_count    = sum(1 for f in findings_69 if f["Status"] == "PASS")
xref_count    = sum(1 for f in findings_69 if f["Status"] == "CROSS-REF")

print(f"\n   {len(findings_69)} timeliness checks performed:")
print(f"      ✅ PASS:      {pass_count}")
print(f"      ⚠️  FINDING:   {finding_count}")
print(f"      ℹ️  INFO:      {info_count}")
print(f"      🔗 CROSS-REF: {xref_count}")

if finding_count == 0:
    print("\n   ✅ No timestamp-integrity violations were detected in the raw review data.")
    print("   The main timeliness constraints are snapshot age, missing row-level ingestion metadata, and partial boundary years.")
else:
    print(f"\n   ⚠️  {finding_count} timestamp consistency issue(s) were detected — review the findings above.")

print("\nTIMELINESS CHECKS COMPLETE")
print("=" * 80)

6.9) TIMELINESS CHECKS — CONSOLIDATED SUMMARY


Check,Detail,Status
Review timestamp parseability,0 failed parses from raw `date` strings,PASS
Snapshot coverage span,"2005-02-16 to 2022-01-19 (6,181 days across 18 years)",PASS
Snapshot freshness,"Latest review date 2022-01-19; stale by 1,732 days",INFO
Recent-year concentration,"1,204,411 rows (17.2%) fall in 2020–2022",INFO
Boundary-year completeness,2005 has 11/12 months; 2022 has 1/12 months,INFO
Row-level ingestion metadata,No load / ingestion timestamp columns present in row schema,INFO
Delta table history availability,3 commit-history events available (2026-09-22 11:46:00 → 2026-10-16 02:01:37),PASS
Snapshot publication lag,"1,731 days between latest review date and latest Delta commit",INFO
Boundary-year trend risk,2005 has 11/12 months; 2022 has 1/12 months,INFO
Raw timestamp string length != 19,0 violations,PASS



   20 timeliness checks performed:
      ✅ PASS:      13
      ⚠️  FINDING:   0
      ℹ️  INFO:      6
      🔗 CROSS-REF: 1

   ✅ No timestamp-integrity violations were detected in the raw review data.
   The main timeliness constraints are snapshot age, missing row-level ingestion metadata, and partial boundary years.

TIMELINESS CHECKS COMPLETE


---
# 7) Root Cause Analysis (RCA)
## DQ-REVIEW-014 · DQ-REVIEW-017 · DQ-REVIEW-019

*Deep-dive investigation into three data quality findings to determine definitive root causes and inform remediation decisions at the Silver layer.*

| Finding | Severity | Summary | Root Cause Hypothesis |
| --- | --- | --- | --- |
| **DQ-REVIEW-014** | 🟠 Medium | 212,788 user-business pairs have >1 review (244,520 excess rows, 3.5%) | Yelp Academic Dataset retains historical review versions |
| **DQ-REVIEW-017** | 🟠 Medium | 1 review has identical `user_id` and `business_id` | Hash collision or test record |
| **DQ-REVIEW-019** | 🟠 Medium | 32 orphan `user_id`s (33 rows) with no parent in `yelp_user` | Snapshot timing gap between table exports |

In [0]:
# RCA-1) DQ-REVIEW-014 — Multi-Review User-Business Pairs
# Hypothesis: Dataset retains historical review edits as separate rows.

from pyspark.sql.functions import (
    col, count as _count, sum as _sum, mean, length, when,
    row_number, lag, datediff, to_timestamp, percentile_approx, lit,
    round as spark_round, min as spark_min, max as spark_max
)
from pyspark.sql.window import Window
import pandas as pd

df = spark.table("yelp_poc_uc.bronze_final.yelp_review")
total_rows = df.count()

# ════ EVIDENCE A: How many reviews do multi-review pairs have? ════
print("\n═" * 80)
print("EVIDENCE A — Multi-Review Pair Frequency")
print("═" * 80)

ub_freq = (
    df.groupBy("user_id", "business_id")
    .agg(_count("*").alias("review_count"))
    .filter(col("review_count") > 1)
)
total_pairs = ub_freq.count()
total_affected_rows = ub_freq.agg(_sum("review_count")).collect()[0][0]

freq_dist = (
    ub_freq.groupBy("review_count")
    .agg(
        _count("*").alias("num_pairs"),
        _sum("review_count").alias("total_rows_involved")
    )
    .withColumn("pct_of_pairs", spark_round(col("num_pairs") / lit(total_pairs) * 100, 2))
    .withColumn("pct_of_dataset", spark_round(col("total_rows_involved") / lit(total_rows) * 100, 2))
    .orderBy("review_count")
)
display(freq_dist)
print(f"\n   Affected: {total_pairs:,} pairs / {total_affected_rows:,} rows out of {total_rows:,} total")

# ════ EVIDENCE B: Time gap between consecutive reviews for same user-business ════
print("\n═" * 80)
print("EVIDENCE B — Days Between Consecutive Reviews (Same User-Business)")
print("═" * 80)

w = Window.partitionBy("user_id", "business_id").orderBy("date")
df_gaps = (
    df.withColumn("prev_date", lag("date").over(w))
    .withColumn("prev_stars", lag("stars").over(w))
    .filter(col("prev_date").isNotNull())
    .withColumn("days_gap", datediff(to_timestamp(col("date")), to_timestamp(col("prev_date"))))
    .withColumn("star_change", (col("stars") - col("prev_stars")).cast("int"))
)

gap_buckets = (
    df_gaps.withColumn("gap_bucket",
        when(col("days_gap") == 0, "0: Same day")
        .when(col("days_gap") <= 1, "1: Within 1 day")
        .when(col("days_gap") <= 7, "2: 2–7 days")
        .when(col("days_gap") <= 30, "3: 8–30 days")
        .when(col("days_gap") <= 90, "4: 1–3 months")
        .when(col("days_gap") <= 365, "5: 3–12 months")
        .when(col("days_gap") <= 730, "6: 1–2 years")
        .otherwise("7: 2+ years")
    )
    .groupBy("gap_bucket")
    .agg(
        _count("*").alias("count"),
        spark_round(mean("days_gap"), 1).alias("avg_days_in_bucket")
    )
    .withColumn("pct", spark_round(col("count") / lit(df_gaps.count()) * 100, 1))
    .orderBy("gap_bucket")
)
display(gap_buckets)

# ════ EVIDENCE C: Did users change their star rating? ════
print("\n═" * 80)
print("EVIDENCE C — Star Rating Change Between Consecutive Versions")
print("═" * 80)

star_change_summary = (
    df_gaps
    .withColumn("direction",
        when(col("star_change") > 0, "↑ Upgraded")
        .when(col("star_change") < 0, "↓ Downgraded")
        .otherwise("→ Same rating")
    )
    .groupBy("star_change", "direction")
    .agg(_count("*").alias("count"))
    .withColumn("pct", spark_round(col("count") / lit(df_gaps.count()) * 100, 1))
    .orderBy("star_change")
)
display(star_change_summary)

# ════ EVIDENCE D: First vs Last version — what changed? ════
print("\n═" * 80)
print("EVIDENCE D — First Version vs Last Version Comparison")
print("═" * 80)

w_desc = Window.partitionBy("user_id", "business_id").orderBy(col("date").desc())
df_v = (
    df.withColumn("rn_asc", row_number().over(w))
    .withColumn("rn_desc", row_number().over(w_desc))
    .withColumn("ver_count", _count("*").over(Window.partitionBy("user_id", "business_id")))
    .filter(col("ver_count") > 1)
)

first_r = df_v.filter(col("rn_asc") == 1).select(
    "user_id", "business_id",
    col("stars").alias("first_stars"), col("useful").alias("first_useful"),
    length(col("text")).alias("first_text_len"), col("date").alias("first_date")
)
last_r = df_v.filter(col("rn_desc") == 1).select(
    "user_id", "business_id",
    col("stars").alias("last_stars"), col("useful").alias("last_useful"),
    length(col("text")).alias("last_text_len"), col("date").alias("last_date")
)
comp = first_r.join(last_r, ["user_id", "business_id"])

comp_agg = comp.select(
    lit("Stars").alias("metric"),
    spark_round(mean("first_stars"), 2).alias("avg_first_version"),
    spark_round(mean("last_stars"), 2).alias("avg_last_version")
).unionByName(
    comp.select(
        lit("Useful votes").alias("metric"),
        spark_round(mean("first_useful"), 2).alias("avg_first_version"),
        spark_round(mean("last_useful"), 2).alias("avg_last_version")
    )
).unionByName(
    comp.select(
        lit("Text length (chars)").alias("metric"),
        spark_round(mean("first_text_len"), 0).alias("avg_first_version"),
        spark_round(mean("last_text_len"), 0).alias("avg_last_version")
    )
)
display(comp_agg)

# Rating shift breakdown
rating_shift = (
    comp.withColumn("shift",
        when(col("last_stars") > col("first_stars"), "↑ Upgraded")
        .when(col("last_stars") < col("first_stars"), "↓ Downgraded")
        .otherwise("→ Same")
    )
    .groupBy("shift").agg(_count("*").alias("pairs"))
    .withColumn("pct", spark_round(col("pairs") / lit(comp.count()) * 100, 1))
    .orderBy("shift")
)
print("\nRating shift (first → last):")
display(rating_shift)

# ════ EVIDENCE E: Sample multi-review pairs — see the actual data ════
print("\n═" * 80)
print("EVIDENCE E — Top 10 Most-Revised User-Business Pairs (actual rows)")
print("═" * 80)

top_pair = ub_freq.orderBy(col("review_count").desc()).limit(1).collect()[0]
top_uid, top_bid = top_pair["user_id"], top_pair["business_id"]

print(f"\nMost-revised pair: user={top_uid}, business={top_bid} ({top_pair['review_count']} reviews)")
display(
    df.filter((col("user_id") == top_uid) & (col("business_id") == top_bid))
    .select("review_id", "stars", "useful", "funny", "cool",
            length(col("text")).alias("text_length"), "date")
    .orderBy("date")
)


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE A — Multi-Review Pair Frequency
════════════════════════════════════════════════════════════════════════════════


review_count,num_pairs,total_rows_involved,pct_of_pairs,pct_of_dataset
2,190088,380176,89.33,5.44
3,17601,52803,8.27,0.76
4,3278,13112,1.54,0.19
5,987,4935,0.46,0.07
6,412,2472,0.19,0.04
7,168,1176,0.08,0.02
8,101,808,0.05,0.01
9,47,423,0.02,0.01
10,32,320,0.02,0.0
11,23,253,0.01,0.0



   Affected: 212,788 pairs / 457,308 rows out of 6,990,280 total

═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE B — Days Between Consecutive Reviews (Same User-Business)
════════════════════════════════════════════════════════════════════════════════


gap_bucket,count,avg_days_in_bucket,pct
0: Same day,1294,0.0,0.5
1: Within 1 day,4680,1.0,1.9
2: 2–7 days,22736,4.0,9.3
3: 8–30 days,31034,17.5,12.7
4: 1–3 months,37287,56.7,15.2
5: 3–12 months,70115,206.6,28.7
6: 1–2 years,38220,522.6,15.6
7: 2+ years,39154,1322.2,16.0



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE C — Star Rating Change Between Consecutive Versions
════════════════════════════════════════════════════════════════════════════════


...

metric,avg_first_version,avg_last_version
Stars,3.63,3.56
Useful votes,1.23,1.62
Text length (chars),693.0,624.0



Rating shift (first → last):


shift,pairs,pct
↑ Upgraded,45652,21.5
→ Same,121282,57.0
↓ Downgraded,45854,21.5



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE E — Top 10 Most-Revised User-Business Pairs (actual rows)
════════════════════════════════════════════════════════════════════════════════

Most-revised pair: user=a0DnfD31lNdiBTY2-YBBFA, business=C6YaSrMAzy3jJqinlFVudw (40 reviews)


review_id,stars,useful,funny,cool,text_length,date
DsyfvuzQE7QqXkQTSAHGTQ,4.0,1,1,1,483,2016-05-10 21:40:50
uwgclIe5TiBsD0Gm6F0jSw,4.0,0,0,0,837,2016-05-25 18:43:02
KoO09b389kGXvAkl1BASVg,4.0,1,0,0,431,2016-06-21 19:18:52
K3hcA_8vnIAQ8kBOMPyU4A,4.0,0,0,0,335,2016-06-30 18:03:29
qxj6MRYO6awJtWWUjBtdWQ,4.0,0,0,0,418,2016-08-15 19:04:26
x7SEbFrRM6swcKBVbtJhcg,5.0,1,1,1,324,2016-09-01 00:47:36
4cglLsOoPIHyDMRI2qVZdA,5.0,1,1,1,518,2016-09-07 17:10:16
FdZxYXaB1l7tqu14rQzRGg,5.0,0,0,0,577,2016-09-09 17:48:28
K1vb7j4KugO2lY2ECTCNtQ,5.0,2,1,1,305,2016-09-16 17:19:38
J2QjCAh617dn-rQO86k9Ow,5.0,0,0,0,363,2016-11-04 16:51:35


In [0]:
# RCA-2) DQ-REVIEW-017 — User-ID / Business-ID Namespace Collision
# Hypothesis: (a) genuine hash collision, (b) test record, or (c) export artifact.

from pyspark.sql.functions import col, count as _count, length, lit, when
import pandas as pd

df = spark.table("yelp_poc_uc.bronze_final.yelp_review")
user_tbl = spark.table("yelp_poc_uc.bronze_final.yelp_user")
biz_tbl  = spark.table("yelp_poc_uc.bronze_final.yelp_business")
tip_tbl  = spark.table("yelp_poc_uc.bronze_final.yelp_tip")
checkin_tbl = spark.table("yelp_poc_uc.bronze_final.yelp_checkin")

COLLISION_ID = "lJzfEleeQ1uYnYQp37LD9Q"

# ════ EVIDENCE A: Does this ID exist in yelp_user? ════
print("═" * 80)
print("EVIDENCE A — yelp_user record for collision ID")
print("═" * 80)
display(
    user_tbl.filter(col("user_id") == COLLISION_ID)
    .select("user_id", "name", "review_count", "yelping_since",
            "useful", "funny", "cool", "fans", "elite", "average_stars")
)

# ════ EVIDENCE B: Does this ID exist in yelp_business? ════
print("\n═" * 80)
print("EVIDENCE B — yelp_business record for collision ID")
print("═" * 80)
display(
    biz_tbl.filter(col("business_id") == COLLISION_ID)
    .select("business_id", "name", "city", "state", "stars",
            "review_count", "is_open", "categories")
)

# ════ EVIDENCE C: The actual review where user_id == business_id ════
print("\n═" * 80)
print("EVIDENCE C — The colliding review record (full row)")
print("═" * 80)
display(
    df.filter((col("user_id") == COLLISION_ID) & (col("business_id") == COLLISION_ID))
    .select("review_id", "user_id", "business_id", "stars",
            "useful", "funny", "cool", "date",
            col("text").substr(1, 500).alias("text_preview"))
)

# ════ EVIDENCE D: Cross-table footprint — where else does this ID appear? ════
print("\n═" * 80)
print("EVIDENCE D — Cross-table footprint of collision ID")
print("═" * 80)

footprint = pd.DataFrame([
    {"table": "yelp_review",   "role": "user_id",     "rows": df.filter(col("user_id") == COLLISION_ID).count()},
    {"table": "yelp_review",   "role": "business_id",  "rows": df.filter(col("business_id") == COLLISION_ID).count()},
    {"table": "yelp_user",     "role": "user_id",      "rows": user_tbl.filter(col("user_id") == COLLISION_ID).count()},
    {"table": "yelp_business", "role": "business_id",   "rows": biz_tbl.filter(col("business_id") == COLLISION_ID).count()},
    {"table": "yelp_tip",      "role": "user_id",      "rows": tip_tbl.filter(col("user_id") == COLLISION_ID).count()},
    {"table": "yelp_tip",      "role": "business_id",   "rows": tip_tbl.filter(col("business_id") == COLLISION_ID).count()},
    {"table": "yelp_checkin",  "role": "business_id",   "rows": checkin_tbl.filter(col("business_id") == COLLISION_ID).count()},
])
display(footprint)

# ════ EVIDENCE E: Other reviews by this user vs for this business ════
print("\n═" * 80)
print("EVIDENCE E — Context: other reviews by this user & for this business")
print("═" * 80)

print("\nOther reviews by user_id (is this a real active user?):")
display(
    df.filter(col("user_id") == COLLISION_ID)
    .select("review_id", "business_id", "stars", "useful", "date",
            length(col("text")).alias("text_len"))
    .orderBy("date")
)

print("\nSample reviews for business_id (is this a real business?):")
display(
    df.filter(col("business_id") == COLLISION_ID)
    .select("review_id", "user_id", "stars", "useful", "date",
            length(col("text")).alias("text_len"))
    .orderBy(col("date").desc())
    .limit(10)
)

════════════════════════════════════════════════════════════════════════════════
EVIDENCE A — yelp_user record for collision ID
════════════════════════════════════════════════════════════════════════════════


user_id,name,review_count,yelping_since,useful,funny,cool,fans,elite,average_stars
lJzfEleeQ1uYnYQp37LD9Q,Karen,52,2011-07-04 01:58:45,36,6,10,1,,3.09



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE B — yelp_business record for collision ID
════════════════════════════════════════════════════════════════════════════════


business_id,name,city,state,stars,review_count,is_open,categories
lJzfEleeQ1uYnYQp37LD9Q,Dollar Car Rental,Nashville,TN,2.5,72,1,"Car Rental, Hotels & Travel"



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE C — The colliding review record (full row)
════════════════════════════════════════════════════════════════════════════════


...

table,role,rows
yelp_review,user_id,1
yelp_review,business_id,74
yelp_user,user_id,1
yelp_business,business_id,1
yelp_tip,user_id,0
yelp_tip,business_id,4
yelp_checkin,business_id,1



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE E — Context: other reviews by this user & for this business
════════════════════════════════════════════════════════════════════════════════

Other reviews by user_id (is this a real active user?):


review_id,business_id,stars,useful,date,text_len
YwEHOUJF51lMefLLD_DO6w,lJzfEleeQ1uYnYQp37LD9Q,1.0,0,2021-07-19 17:44:12,1079



Sample reviews for business_id (is this a real business?):


review_id,user_id,stars,useful,date,text_len
has03Me5-aMDbUn4FnfEwQ,Qa1ZwnfAhWjplMJRBmjt9w,1.0,0,2021-11-19 17:45:17,720
ZOyLlQxpbTgkxjlavNcrFg,QtSJ0c4AiWV9AvAmh4fQPA,1.0,0,2021-10-14 21:46:25,1329
cC1kRh7L7SxEr4n-6kZGPA,EHPfKwUPwLq8iq1ccA2fHg,1.0,0,2021-10-14 00:23:01,1059
30UKAp5C1f5gGj0bCsg8GQ,nRNqsnELU6_PLoZGB8RkBQ,1.0,0,2021-10-10 00:46:58,753
cBdkg6xQpn3lcGPzYT5fqg,2jJoqEbm5ICChrdOZ-msAQ,1.0,0,2021-09-19 16:35:24,621
YwEHOUJF51lMefLLD_DO6w,lJzfEleeQ1uYnYQp37LD9Q,1.0,0,2021-07-19 17:44:12,1079
UyPiGSeSYFljE_P-EiWjMw,hzEjOGOElvopRlqraAdCeA,5.0,0,2021-06-29 08:13:23,434
JZ3mL34Ne4n5IyAjBtVoHw,H-NP7YFyXkQzZC091jM2Uw,1.0,1,2021-06-09 17:32:04,184
FXxWQkyY_wc2w0luIPPeew,SapAkHRKYPmzzyRSCakLug,1.0,2,2021-03-30 22:41:09,255
cuMUM0Kgry4B3nDVieJYTg,o--zHBo8ihFUeqBhbx7Qag,3.0,5,2021-03-20 18:27:52,773


In [0]:
# RCA-3) DQ-REVIEW-019 — Orphan User IDs (FK Violation)
# Hypothesis: Snapshot timing gap — yelp_user exported before these accounts existed.

from pyspark.sql.functions import (
    col, count as _count, min as spark_min, max as spark_max, mean,
    sum as _sum, length, when, countDistinct, lit, round as spark_round
)
import pandas as pd

df = spark.table("yelp_poc_uc.bronze_final.yelp_review")
user_tbl = spark.table("yelp_poc_uc.bronze_final.yelp_user")
biz_tbl  = spark.table("yelp_poc_uc.bronze_final.yelp_business")
tip_tbl  = spark.table("yelp_poc_uc.bronze_final.yelp_tip")

# Identify orphan user_ids
orphan_ids_df = (
    df.select("user_id").distinct()
    .join(user_tbl.select("user_id").distinct(), "user_id", "left_anti")
)
orphan_list = [r[0] for r in orphan_ids_df.collect()]
orphan_reviews = df.filter(col("user_id").isin(orphan_list))

# ════ EVIDENCE A: All orphan reviews — inspect every single row ════
print("═" * 80)
print(f"EVIDENCE A — All {len(orphan_list)} Orphan Users / {orphan_reviews.count()} Reviews")
print("═" * 80)
display(
    orphan_reviews
    .select(
        "review_id", "user_id", "business_id", "stars",
        "useful", "funny", "cool",
        length(col("text")).alias("text_length"),
        col("date").substr(1, 19).alias("review_timestamp")
    )
    .orderBy("date")
)

# ════ EVIDENCE B: Temporal pattern — are these clustered? ════
print("\n═" * 80)
print("EVIDENCE B — Temporal Pattern vs Dataset Boundary")
print("═" * 80)

ds_boundary = df.agg(spark_max("date").alias("dataset_max_date")).collect()[0][0]

temporal = orphan_reviews.agg(
    spark_min("date").alias("orphan_earliest"),
    spark_max("date").alias("orphan_latest"),
    countDistinct(col("date").substr(1, 10)).alias("distinct_calendar_dates"),
    _count("*").alias("total_reviews")
).withColumn("dataset_max_date", lit(ds_boundary))
display(temporal)

# ════ EVIDENCE C: Star rating distribution of orphans vs full dataset ════
print("\n═" * 80)
print("EVIDENCE C — Star Distribution: Orphan Reviews vs Full Dataset")
print("═" * 80)

orphan_count = orphan_reviews.count()

orphan_stars = (
    orphan_reviews.groupBy("stars").agg(_count("*").alias("orphan_count"))
    .withColumn("orphan_pct", spark_round(col("orphan_count") / lit(orphan_count) * 100, 1))
)
dataset_stars = (
    df.groupBy("stars").agg(_count("*").alias("dataset_count"))
    .withColumn("dataset_pct", spark_round(col("dataset_count") / lit(total_rows) * 100, 1))
)
star_comparison = orphan_stars.join(dataset_stars, "stars", "full").orderBy("stars")
display(star_comparison)

# ════ EVIDENCE D: Engagement — do orphan reviews have any votes? ════
print("\n═" * 80)
print("EVIDENCE D — Engagement Profile of Orphan Reviews")
print("═" * 80)

eng = orphan_reviews.agg(
    _sum("useful").alias("total_useful"),
    _sum("funny").alias("total_funny"),
    _sum("cool").alias("total_cool"),
    _sum(
        when((col("useful") == 0) & (col("funny") == 0) & (col("cool") == 0), 1).otherwise(0)
    ).alias("rows_with_zero_engagement"),
    _count("*").alias("total_rows")
)
display(eng)

# ════ EVIDENCE E: Do these orphan-review businesses actually exist? ════
print("\n═" * 80)
print("EVIDENCE E — Business Validity & Cross-Table Presence")
print("═" * 80)

orphan_biz_ids = orphan_reviews.select("business_id").distinct()
biz_valid_count = orphan_biz_ids.join(biz_tbl.select("business_id").distinct(), "business_id", "inner").count()
tip_presence = tip_tbl.filter(col("user_id").isin(orphan_list)).count()

validity = pd.DataFrame([
    {"check": "Orphan business_ids found in yelp_business", "result": f"{biz_valid_count}/{orphan_biz_ids.count()}", "status": "ALL VALID" if biz_valid_count == orphan_biz_ids.count() else "GAPS"},
    {"check": "Orphan user_ids found in yelp_tip",          "result": str(tip_presence), "status": "ABSENT" if tip_presence == 0 else "PRESENT"},
    {"check": "Orphan user_ids found in yelp_user",         "result": "0", "status": "ABSENT (by definition)"},
])
display(validity)

# ════ EVIDENCE F: Sample review text — are these real reviews? ════
print("\n═" * 80)
print("EVIDENCE F — Sample Review Texts (first 300 chars)")
print("═" * 80)
display(
    orphan_reviews
    .select(
        "stars", col("date").substr(1, 10).alias("date"),
        length(col("text")).alias("text_length"),
        col("text").substr(1, 300).alias("text_preview")
    )
    .orderBy("date")
    .limit(8)
)

════════════════════════════════════════════════════════════════════════════════
EVIDENCE A — All 32 Orphan Users / 33 Reviews
════════════════════════════════════════════════════════════════════════════════


review_id,user_id,business_id,stars,useful,funny,cool,text_length,review_timestamp
XuqkOMPkmKsKFzFQg7F82A,dWZlWFtsEXFVq_vulT00lA,P5qMWIfibfgYOwTTm5qkyg,1.0,0,0,0,296,2022-01-19 17:22:03
G8n21nKcVNR-noZJqz3sEw,vq2H7lJ73VwXMDqC8DiImw,sgFb5gCtljLBb3Pjt9smoQ,5.0,0,0,0,1696,2022-01-19 17:23:55
an39xQvirUGSlrzmOnEHuw,4bqSo-Xfd8Yu5BMDsd947Q,Tk0S72jSlnleZA1uxfNn_A,5.0,0,0,0,629,2022-01-19 17:27:45
1pcF0fYcXZ2V9QUgtFF8Lw,G0PWeUgNeGDobntevJlJ1g,mQZpnPY7o2u3I90rbzZNQg,1.0,0,0,0,843,2022-01-19 17:30:20
EZ0mbYE2xvG7g9rpn1_zmg,I200IyE9DCxJvvof2wnO6A,4H6KdEMRlSrpFktX2k5gxg,5.0,0,0,0,118,2022-01-19 17:31:13
JVuwa9WSsFe52HDmK4wD2g,sxxnBQb15fOyg30JInIKqw,P5qMWIfibfgYOwTTm5qkyg,5.0,0,0,0,1556,2022-01-19 17:32:01
d34uBmQ8ubYcL_yFpNatOw,qH_QwXBgA4Z7WyCjtwRi2A,LAL-W7lTr2krLkGpTcPIXg,5.0,0,0,0,125,2022-01-19 17:43:28
aiqDd7oFKug-tS4lv9-B8g,5-L9Oo6PyUiBjeqj1GsJyQ,4z_3A9Bx6DMjuTJnqF9oTA,1.0,0,0,0,738,2022-01-19 17:44:56
GZmjLeVfDktqx7xZvAHBSg,MaengE6zJ6k_d5e6nwnVaA,kZTwub3IkBCDHYW5J5pRrA,1.0,0,0,0,202,2022-01-19 17:45:25
ojv1NeMghN0rYviQ7pgbFg,77n3enAMdlka0pZ82GT9VQ,nB6wv6EY7ExXdehFs6sB4A,5.0,0,0,0,592,2022-01-19 17:51:12



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE B — Temporal Pattern vs Dataset Boundary
════════════════════════════════════════════════════════════════════════════════


orphan_earliest,orphan_latest,distinct_calendar_dates,total_reviews,dataset_max_date
2022-01-19 17:22:03,2022-01-19 19:43:00,1,33,2022-01-19 19:48:45



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE C — Star Distribution: Orphan Reviews vs Full Dataset
════════════════════════════════════════════════════════════════════════════════


...

total_useful,total_funny,total_cool,rows_with_zero_engagement,total_rows
0,0,0,33,33



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE E — Business Validity & Cross-Table Presence
════════════════════════════════════════════════════════════════════════════════


check,result,status
Orphan business_ids found in yelp_business,32/32,ALL VALID
Orphan user_ids found in yelp_tip,0,ABSENT
Orphan user_ids found in yelp_user,0,ABSENT (by definition)



═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
EVIDENCE F — Sample Review Texts (first 300 chars)
════════════════════════════════════════════════════════════════════════════════


stars,date,text_length,text_preview
1.0,2022-01-19,1044,"Terrible customer service. we had granite countertops installed throughout our home a few years ago. The counter in the kitchen is pulling away from the wall so I called Three Rivers Granite on November 29,2021 to explain what was happening and get an estimate to fix it. the office staff would ne"
5.0,2022-01-19,153,Beautiful venue. Incredible live music. Staple for me when visiting New Orleans. Romantic setting. Prices are a bit fancy but the experience is worth it.
5.0,2022-01-19,160,Dr Rhett Bruner and his entire staff are beyond great. From his front staff always always so professional and welcoming to Dr Bruner himself; all are WONDERFUL!
1.0,2022-01-19,2097,I recently had a 30 year old gas furnance go out during a massive snow storm that hit Grass Valley CA. The storm was so bad it knocked out power for 72 hours and the roads were shut down for almost a week. The governor of CA had to issue a state of emergency because of it. I am currently a active du
5.0,2022-01-19,125,We love the pretzels here. Always hot and fresh. The staff is friendly and fast. Always a treat when we get mart pretzels!
4.0,2022-01-19,1159,First time trying this new spot and for the most part it did not disappoint! Good portion of food with plenty of flavor. We had the fish spread as an appetizer and was probably the least flavorful of anything we ordered. Not to say it was bad but just a bit bland. We also ordered two of the seafood
5.0,2022-01-19,170,"I love this place. The food is to die for"" delicious! The service is slow, though. They need more people, but in these challenging times, that is easier said than done."
1.0,2022-01-19,296,"Exorbitant pricing vs. the little bit of material and labor that is actually performed They will remind you that they're legal team will keep you bound in the agreement sign during your time of uncertainty, lack of education of the processes and the pricing as well as pulling fear over your head"


---
## RCA Summary — Root Causes & Decision Logic

### What the Evidence Shows

| Finding | Root Cause (from evidence above) | How We Know |
| --- | --- | --- |
| **DQ-REVIEW-014** | Dataset retains historical review edits as separate rows | Evidence A: 89% of pairs have exactly 2 versions. Evidence B: median 160-day gap (organic re-visit). Evidence C: 41.7% changed their rating. Evidence D: older versions accumulate more votes. Evidence E: top pair has 40 revisions spanning years. |
| **DQ-REVIEW-017** | Genuine hash collision — user "Karen" and business "Dollar Car Rental" share the same 22-char ID | Evidence A+B: ID resolves to real entities in both parent tables. Evidence C: review text ("DO NOT RENT…") matches the car rental business. Evidence D: ID has full cross-table footprint as both a user and a business. |
| **DQ-REVIEW-019** | Snapshot timing gap — `yelp_user` table was exported before these accounts were captured | Evidence A: all 33 reviews on the exact same date (2022-01-19) within \~2.3 hours. Evidence B: this date IS the dataset boundary. Evidence D: 100% zero engagement (brand new). Evidence E: complete absence from every other table. |

---

### Remediation Decision Guide

#### DQ-REVIEW-014 — Multi-Review Pairs (244,520 excess rows, 3.5%)

| Option | Approach | When to Choose | Trade-off |
| --- | --- | --- | --- |
| **A** | Add `is_latest_review` flag (recommended) | Default for most pipelines | Non-destructive; supports both current-state and historical analysis |
| **B** | Keep only latest review per (user, business) | Space-constrained or simple aggregation-only use cases | Loses historical review evolution data |
| **C** | Retain all versions as-is | NLP sentiment drift study, review-update behavior research | Downstream must always dedup for aggregations |

#### DQ-REVIEW-017 — Namespace Collision (1 row, 0.00001%)

| Option | Approach | When to Choose | Trade-off |
| --- | --- | --- | --- |
| **A** | Flag with `is_id_collision = true` (recommended) | Default — preserves record, prevents aggregation contamination | Requires WHERE clause discipline |
| **B** | Delete the row | Zero-tolerance FK integrity policy | Loses a genuine review with valid text |
| **C** | NULL out `business_id` | Preserve for user-level NLP analysis only | Breaks business-level joins for this row |

#### DQ-REVIEW-019 — Orphan Users (33 rows, 0.00047%)

| Option | Approach | When to Choose | Trade-off |
| --- | --- | --- | --- |
| **A** | Flag with `is_orphan_user = true` + use LEFT JOINs (recommended) | Default — preserves reviews, makes gap explicit | User metadata columns will be NULL |
| **B** | Exclude the 33 rows at Silver | Strict referential integrity required | Negligible data loss (0.00047%) |
| **C** | Create stub user records with NULL metadata | Need INNER JOIN compatibility without data loss | Introduces synthetic records |

---

### ✅ Chosen Silver-Layer Implementation (Post Human Review, 2026-10-20)

| Finding | Chosen Approach | Implementation |
| --- | --- | --- |
| **DQ-014** | Option A — `is_latest_review` flag (SCD pattern) | `ROW_NUMBER() OVER (PARTITION BY user_id, business_id ORDER BY review_timestamp DESC, review_id DESC) = 1` |
| **DQ-017** | **Quarantine** self-referential row (no flag column) | 1 row → `quarantine.yelp_review_id_collision` |
| **DQ-019** | **Quarantine** orphan-user rows | 33 rows → `quarantine.yelp_review_orphan_user` |

```
Silver review table (yelp_poc_uc.silver_cleaned.yelp_review):
  • is_latest_review   BOOLEAN  — ROW_NUMBER() OVER (PARTITION BY user_id, business_id ORDER BY review_timestamp DESC, review_id DESC) = 1
  • is_short_review    BOOLEAN  — LENGTH(TRIM(text)) < 10 AND text contains alphanumeric chars

Quarantine tables:
  • quarantine.yelp_review_null_text       — 57 rows  (NULL/empty text)
  • quarantine.yelp_review_id_collision    — 1 row   (user_id == business_id)
  • quarantine.yelp_review_orphan_user     — 33 rows (user_id not in yelp_user)

Reconciliation: Silver + ALL quarantine = Bronze (6,990,280)
```

Standard analytics queries should filter: `WHERE is_latest_review`